# casmi26 e3fu2 (ours: two-pipeline fusion)

Ours: the two builds below run one after the other in subprocesses (each sees only its own datasets) and their top-25 lists are fused per molecule by weighted reciprocal rank (k=10, wa=1, wb=1); if B fails or runs out of time the file is A's. Built by enveda-casmi26/kernels/e3/build_fu.py.

## A: build.py base=f465 ownco=1 lfrag=1 nn=0.25

# casmi26 fu2_x (yasunorim)

Base: the pipeline of [Candidate-aware PubChem promotion 0.465](https://www.kaggle.com/code/sanpier/candidate-aware-pubchem-promotion-0-465) by Saner Turhaner (its embedded frozen_0465.ipynb, version 7) = [CASMI 26: The Data Behind Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures) by dmitriigluzdov, v16 (public LB 0.464), + sanpier's candidate-specific rank-1 PubChem promotion (public LB 0.465); everything they credit (cells below, unchanged). Ours: the base's fragment scorer (frag_rescore) scores with our learned peak credits (bucket = cleavage level x hydrogen shift of the cheapest explanation, linear and sqrt intensity; listwise fit on 4,432 train isomer panels; held out MRR 0.723 -> 0.748 on enveda-180, 0.567 -> 0.584 on np-examples). Ours (ownco=1): engine 2's candidate pool also takes the structures of our own COCONUT table (yasunorim/casmi26-own-coconut-candidates, built from the official COCONUT dump; 38k structures more than the public COCONUT table) that the pool does not have, fingerprinted like the training structures. Ported (not ours): sanpier's neural rerank of the final 25 ([PubChem + Neural025 - LB 0.466](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466), model dataset sanpier/casmi-spectrum-model-v1), reciprocal-rank weight 0.25, offset 3, as the last step. Built by enveda-casmi26/kernels/e3/build.py in github.com/yasumorishima/kaggle-competitions.

## B: build.py base=v22 ownco=1

# casmi26 fu2_x (yasunorim)

Base: [casmi26-v22-n1-challenger](https://www.kaggle.com/code/ahmedberatozer/casmi26-v22-n1-challenger) by Ahmed Berat Ozer (= his v15-n1a + the N1 challenger of flexonafft, casmi26-closing-the-evidence-gap v47, also dmitriigluzdov v16 public LB 0.464: GLACIER proxies kept where ICEBERG scores exist, joint ICEBERG + proxy-GLACIER re-ordering), Apache-2.0 lineage, and everything it credits (cells below, unchanged). Ours (ownco=1): engine 2's candidate pool also takes the structures of our own COCONUT table (yasunorim/casmi26-own-coconut-candidates, built from the official COCONUT dump; 38k structures more than the public COCONUT table) that the pool does not have, fingerprinted like the training structures. This build adds nothing (reference run). Built by enveda-casmi26/kernels/e3/build.py in github.com/yasumorishima/kaggle-competitions.

In [ ]:
# ours: two-pipeline fusion orchestrator (built by enveda-casmi26/kernels/e3/build_fu.py)
import json, os, shutil, signal, subprocess, sys, time, csv
T0 = time.time()
W = '/kaggle/working'
SCRIPTS = {"A": "# --- ours (fusion orchestrator): this script's own inputs only (all names kept inside a function) ---\ndef _ours_fu_install_filter(allowed, other_slugs):\n    import glob as gm\n    og, oig = gm.glob, gm.iglob\n    def keep(p):\n        p = str(p)\n        if not p.startswith('/kaggle/input/'):\n            return True\n        parts = p[len('/kaggle/input/'):].split('/')\n        if parts[0] == 'datasets' and len(parts) >= 3:\n            return parts[1] + '/' + parts[2] in allowed\n        return parts[0] not in other_slugs\n    def g(*a, **k):\n        return [p for p in og(*a, **k) if keep(p)]\n    def ig(*a, **k):\n        return (p for p in oig(*a, **k) if keep(p))\n    gm.glob, gm.iglob = g, ig\n    print('ours: input filter', sorted(allowed), flush=True)\n_ours_fu_install_filter(set(['dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets', 'dmitriigluzdov/casmi26-fold-safe-fpnet', 'dmitriigluzdov/casmi26-natural-product-knowledge-table', 'dmitriigluzdov/casmi26-natural-product-panel-benchmark', 'dmitriigluzdov/casmi26-pubchem-popularity-prior', 'sanpier/casmi-spectrum-model-v1', 'yasunorim/casmi26-own-coconut-candidates']), set(['casmi-chebi-lmsd-structures', 'casmi26-fp-models-v2', 'casmi26-fpnet-full1', 'casmi26-glacier', 'casmi26-iceberg', 'casmi26-pubchem-tier', 'casmi26-pubchemlite-cc0-candidate-fingerprints', 'casmi26-ranker-features', 'casmi26-simulated-ranker-rows', 'casmi26-v2-pool', 'casmi26-v3-models', 'casmi26-v4b-models', 'coconut-casmi26-candidates', 'rdkit-2026-3-3-wheel']))\ndel _ours_fu_install_filter\ntry:\n    from IPython.display import display\nexcept Exception:\n    display = print\n\n# ===== cell 0 =====\nimport os, sys, subprocess, importlib.metadata, time\nos.environ['PYTHONHASHSEED'] = '0'\nNOTEBOOK_STARTED = time.monotonic()   # record total notebook time, including the retained analysis\nfrom pathlib import Path\nif not Path('/kaggle/input').exists() and Path('.runtime/rdkit2026033').exists():\n    sys.path.insert(0, str(Path('.runtime/rdkit2026033').resolve()))\ntry:\n    installed = importlib.metadata.version('rdkit')\nexcept importlib.metadata.PackageNotFoundError:\n    installed = ''\nif installed.replace('.03.', '.3.') != '2026.3.3':\n    tag=f'cp{sys.version_info.major}{sys.version_info.minor}'\n    wheels=sorted(Path('/kaggle/input').rglob(f'rdkit-2026.3.3-{tag}-*.whl'))\n    assert wheels, f'Attach the RDKit 2026.3.3 offline wheel for {tag}.'\n    subprocess.check_call([sys.executable,'-m','pip','install','--no-index','--no-deps','-q',str(wheels[0])], env={**os.environ, 'PYTHONHASHSEED': '0'})\nfrom rdkit import rdBase\nassert rdBase.rdkitVersion == '2026.03.3', rdBase.rdkitVersion\nprint('Official-metric RDKit:', rdBase.rdkitVersion)\n\n\n# ===== cell 1 =====\nfrom pathlib import Path\nimport os\nimport hashlib\nimport platform\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom IPython.display import display, Markdown\n\nfrom rdkit import Chem, rdBase\nfrom rdkit.Chem import Descriptors, Draw, rdMolDescriptors, rdFingerprintGenerator\nfrom rdkit.Chem.Scaffolds import MurckoScaffold\nfrom rdkit import DataStructs\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.preprocessing import StandardScaler\n\nSEED = 20260915\nrng = np.random.default_rng(SEED)\npd.set_option(\"display.max_colwidth\", 100)\npd.set_option(\"display.max_rows\", 100)\nsns.set_theme(style=\"whitegrid\", context=\"notebook\")\nplt.rcParams.update({\"figure.figsize\": (11, 5.5), \"axes.titleweight\": \"bold\", \"figure.dpi\": 120})\n\n# Sharp inline figures, with PNG retained for readers that do not support SVG.\nfrom matplotlib_inline.backend_inline import set_matplotlib_formats\nset_matplotlib_formats('svg', 'png')\nplt.rcParams.update({\n    'figure.dpi': 160, 'savefig.dpi': 200, 'savefig.bbox': 'tight',\n    'font.size': 12, 'axes.labelsize': 12, 'axes.titlesize': 13,\n    'xtick.labelsize': 10, 'ytick.labelsize': 10,\n    'axes.spines.top': False, 'axes.spines.right': False,\n    'svg.fonttype': 'none', 'figure.facecolor': 'white',\n})\n\n\n# ===== cell 2 =====\ndef locate_data_dir():\n    candidates = [\n        Path(\"/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra\"),\n        Path(\"/kaggle/input/enveda-casmi26-molecule-id-mass-spectra\"),\n        Path(\"/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra\"),\n        Path(\"data/raw\"),\n        Path(\"../data/raw\"),\n    ]\n    for path in candidates:\n        if all((path / name).exists() for name in (\"train.parquet\", \"test.parquet\", \"sample_submission.csv\")):\n            return path.resolve()\n    raise FileNotFoundError(\"Could not find train.parquet, test.parquet, and sample_submission.csv\")\n\n\nDATA = locate_data_dir()\nTRAIN_PATH = DATA / \"train.parquet\"\nTEST_PATH = Path(os.environ.get(\"CASMI_TEST_PATH\", DATA / \"test.parquet\"))\nSAMPLE_PATH = Path(os.environ.get(\"CASMI_SAMPLE_PATH\", DATA / \"sample_submission.csv\"))\n\n# ===== cell 3 =====\n# Locate each mounted dataset by a file it contains (mount paths differ between older and newer notebooks).\nimport glob, json as _json\nfrom pathlib import Path\n\n\ndef dataset_dir(slug, marker):\n    hits = sorted(glob.glob(f'/kaggle/input/**/{slug}/{marker}', recursive=True), key=len)\n    return Path(hits[0]).parent if hits else None\n\n\nV4G = dataset_dir('casmi26-attributed-v4g-reproduction-assets', 'ATTRIBUTION.md')\nPOP = dataset_dir('casmi26-pubchem-popularity-prior', 'pc_lsid.npy')\nPANEL = dataset_dir('casmi26-natural-product-panel-benchmark', 'panel.parquet')\nTABLE = dataset_dir('casmi26-natural-product-knowledge-table', 'natural_products.parquet')\nFOLD = dataset_dir('casmi26-fold-safe-fpnet', 'MODEL_INPUTS.json')\nassert FOLD is not None, 'Attach the updated casmi26-fold-safe-fpnet dataset'\n\n# ===== cell 4 =====\ntry:\n    rows = []\n    for name, path, unit in (('train.parquet', TRAIN_PATH, 'a labelled spectrum'),\n                             ('test.parquet', TEST_PATH, 'an unlabelled spectrum')):\n        meta = pq.ParquetFile(path).metadata\n        rows.append((name, unit, meta.num_rows, meta.num_columns))\n    sub = pd.read_csv(SAMPLE_PATH, usecols=['molecule_id'])\n    rows.append(('sample_submission.csv', 'a molecule to answer', len(sub), 2))\n    display(pd.DataFrame(rows, columns=['file', 'one row is', 'rows', 'columns']).style.hide(axis='index')\n            .format({'rows': '{:,}'}))\n    n_train = pq.read_table(TRAIN_PATH, columns=['inchikey14']).column('inchikey14').to_pandas().nunique()\n    n_test = pq.read_table(TEST_PATH, columns=['molecule_id']).column('molecule_id').to_pandas().nunique()\n    print(f'training structures (distinct inchikey14): {n_train:,}; test molecules: {n_test:,}')\n    del sub\nexcept Exception as exc:\n    print('competition file table unavailable:', type(exc).__name__, exc)\n\n# ===== cell 5 =====\nROLE = {\n    'casmi26-v4b-models': 'main engine: code, four LightGBM boosters, generator',\n    'casmi26-fpnet-full1': 'main engine: full1 fingerprint model',\n    'casmi26-v2-pool': 'main engine: 710,701-structure candidate pool with fragments and fingerprints',\n    'casmi26-v3-models': 'PubChem channel: FPNet A+B models and code; ensemble adds fold-safe FPNet from the fifth dataset',\n    'casmi26-fp-models-v2': 'second engine: single and merged fingerprint models',\n    'casmi26-ranker-features': 'second engine: ranker training features',\n    'casmi26-simulated-ranker-rows': 'second engine: simulated ranker training rows',\n    'coconut-casmi26-candidates': 'second engine: COCONUT candidates with fingerprints',\n    'chebi-lipidmaps-casmi26': 'earlier ChEBI and LIPID MAPS candidate bank (kept for reproducibility)',\n    'casmi-chebi-lmsd-structures': 'second engine: standardised ChEBI and LIPID MAPS candidates (sergiomalv)',\n    'casmi26-pubchemlite-cc0-candidate-fingerprints': 'second engine: PubChemLite CC0 candidates (franciscoangulo)',\n    'casmi26-iceberg': 'forward model ICEBERG and its inference wrapper',\n    'casmi26-glacier': 'forward model GLACIER and its inference wrapper',\n    'hengck23-dreams-enveda-casmi26': 'candidates for unfilled ranks (mass-compatible DreaMS structures)',\n    'rdkit-2026-3-3-wheel': 'RDKit 2026.03.3 (metric identity)',\n    'rdkit-wheels': 'RDKit 2025.03.6 (simulator environment)',\n}\ntry:\n    rows = []\n    for d in sorted(p for p in V4G.iterdir() if p.is_dir()):\n        files = [f for f in d.rglob('*') if f.is_file()]\n        rows.append((d.name, ROLE.get(d.name, ''), len(files), round(sum(f.stat().st_size for f in files) / 1e9, 3)))\n    flat = [f for f in V4G.iterdir() if f.is_file()]\n    rows.append(('(top level)', 'original v4g inference inputs and attribution files', len(flat),\n                 round(sum(f.stat().st_size for f in flat) / 1e9, 3)))\n    display(pd.DataFrame(rows, columns=['folder', 'role', 'files', 'size (GB)']).style.hide(axis='index')\n            .format({'files': '{:,}', 'size (GB)': '{:.3f}'}))\nexcept Exception as exc:\n    print('folder table unavailable:', type(exc).__name__)\n\n# ===== cell 6 =====\ntry:\n    cols = ['inchikey14', 'compound_name', 'pubchem_cid', 'pop', 'n_depositors', 'n_vendors',\n            'src_spectral_library', 'src_metabolite_pathway', 'src_chebi', 'log1p_mesh']\n    pool = pd.read_csv(POP / 'pool_popularity.csv', usecols=cols)\n    keys = ['RYYVLZVUVIJVGH', 'LPHGQDQBBGAPDZ']        # caffeine, isocaffeine (both C8H10N4O2)\n    t = pool[pool.inchikey14.isin(keys)].set_index('inchikey14').loc[keys].reset_index()\n    t = t[['compound_name', 'inchikey14', 'pubchem_cid', 'pop', 'n_depositors', 'n_vendors', 'log1p_mesh',\n           'src_spectral_library', 'src_metabolite_pathway', 'src_chebi']]\n    display(t.rename(columns={'compound_name': 'compound', 'n_depositors': 'depositors', 'n_vendors': 'vendors',\n                              'log1p_mesh': 'ln(1 + MeSH terms)', 'src_spectral_library': 'spectral library',\n                              'src_metabolite_pathway': 'metabolite/pathway', 'src_chebi': 'ChEBI'})\n            .style.hide(axis='index').format({'pop': '{:.2f}', 'ln(1 + MeSH terms)': '{:.2f}'}))\n    del pool\nexcept Exception as exc:\n    print('popularity table unavailable:', type(exc).__name__, exc)\n\n# ===== cell 7 =====\ntry:\n    b = pd.read_csv(POP / 'BENCHMARK_SOURCES.csv')\n    b = b[b.channel.eq('PubChem exact formula')]\n    names = {'src_spectral_library': 'spectral libraries', 'src_metabolite_pathway': 'metabolite / pathway resources',\n             'src_chebi': 'ChEBI', 'src_toxicology': 'EPA toxicology', 'src_lipidmaps': 'LIPID MAPS',\n             'log1p_n_depositors': 'ln(1 + distinct depositors)', 'log1p_mesh': 'ln(1 + MeSH terms)'}\n    b = b[b.signal.isin(names)].assign(signal=lambda d: d.signal.map(names))\n    out = pd.DataFrame({'signal': b.signal, 'answers with signal > 0 (%)': 100 * b.answer_nonzero_fraction,\n                        'competing candidates with signal > 0 (%)': 100 * b.decoy_nonzero_fraction_query_mean})\n    for w in ('0.10', '0.25', '0.50'):\n        if f'delta_{w}_mrr25' in b.columns:\n            out[f'\u0394 MRR@25, w = {w}'] = b[f'delta_{w}_mrr25'].values\n    print(f'documentation prior alone: MRR@25 = {float(b.baseline_pop_mrr25.iloc[0]):.4f} (250 molecules)')\n    fmt = {c: '{:+.4f}' for c in out.columns if c.startswith('\u0394')}\n    fmt.update({c: '{:.1f}' for c in out.columns if c.endswith('(%)')})\n    display(out.style.hide(axis='index').format(fmt))\nexcept Exception as exc:\n    print('benchmark table unavailable:', type(exc).__name__, exc)\n\n# ===== cell 8 =====\ntry:\n    panel = pd.read_parquet(PANEL / 'panel.parquet')\n    summary = {'molecules': len(panel)}\n    for c in ('n_spectra', 'n_views', 'spectra'):\n        if c in panel.columns and pd.api.types.is_numeric_dtype(panel[c]):\n            summary['spectra'] = int(panel[c].sum()); break\n    for c in ('true_mass', 'neutral_mass', 'mass'):\n        if c in panel.columns:\n            summary['median neutral mass (Da)'] = float(panel[c].median()); break\n    display(pd.DataFrame([summary]).style.hide(axis='index')\n            .format({'spectra': '{:,}', 'median neutral mass (Da)': '{:.1f}'}))\n    rows = []\n    for f in sorted(PANEL.glob('*.parquet')):\n        names = pq.read_schema(f).names\n        rows.append((f.name, pq.ParquetFile(f).metadata.num_rows, len(names),\n                     ', '.join(names[:7]) + (', \u2026' if len(names) > 7 else '')))\n    for d in sorted(p for p in PANEL.iterdir() if p.is_dir()):\n        rows.append((d.name + '/', None, None, ', '.join(sorted(p.name for p in d.iterdir())[:5])))\n    display(pd.DataFrame(rows, columns=['file', 'rows', 'columns', 'contents']).style.hide(axis='index')\n            .format({'rows': lambda v: '' if pd.isna(v) else f'{int(v):,}',\n                     'columns': lambda v: '' if pd.isna(v) else f'{int(v)}'}))\nexcept Exception as exc:\n    print('panel summary unavailable:', type(exc).__name__, exc)\n\ntry:\n    prior_summary = (pd.read_csv(PANEL / 'prior_rule_summary.csv')\n                     .replace({'rule': {'v10 composite prior': 'composite prior (Section 2)'}})\n                     .rename(columns={'queries': 'molecules', 'truth_present': 'answer among candidates',\n                                      'mrr25': 'MRR@25', 'delta_documentation': '\u0394 vs documentation'}))\n    print('Prior-only ranking of the PubChem candidates with the answer formula; 250 molecules, ties averaged')\n    display(prior_summary.style.hide(axis='index').format(\n        {'MRR@25': '{:.4f}', '\u0394 vs documentation': '{:+.4f}'}))\nexcept Exception as exc:\n    print('cached public prior replay unavailable:', type(exc).__name__, exc)\n\ntry:\n    audit = _json.loads((FOLD / 'SPLIT_AUDIT.json').read_text())\n    contract = _json.loads((FOLD / 'MODEL_INPUTS.json').read_text())\n    display(pd.DataFrame([{\n        'fold-safe output bits': contract['output_bits'],\n        'excluded structure rows': audit['excluded_structure_rows'],\n        'panel identities excluded': audit['panel_truth_keys_excluded'],\n        'panel identities in its training set': audit['panel_training_overlap'],\n    }]).style.hide(axis='index').format('{:,}'))\n    replay = pd.DataFrame(_json.loads((FOLD / 'PANEL_REPLAY_PUBLIC.json').read_text())['notebook_summary'])\n    replay = replay[replay.protocol.eq('v10_prior') &\n                    (replay['fold0 weight'].eq(0) | np.isclose(replay['fold0 weight'], 1 / 3))].copy()\n    replay.insert(0, 'fingerprint logits', np.where(replay['fold0 weight'].eq(0), 'A+B', 'A+B + 0.5 fold0'))\n    print('PubChem-channel shortlist with the documentation prior; 250 molecules')\n    display(replay.drop(columns=['protocol', 'fold0 weight']).style.hide(axis='index').format(\n        {c: '{:.4f}' for c in ['MRR@25', 'top-1', 'delta MRR', 'delta MRR 95% low', 'delta MRR 95% high']}))\nexcept Exception as exc:\n    print('fold-safe public audit unavailable:', type(exc).__name__, exc)\n\n\n# ===== cell 9 =====\ntry:\n    nt = pd.read_parquet(TABLE / 'natural_products.parquet',\n                         columns=['np_pathway', 'in_casmi_train', 'in_casmi_pool', 'pubmed_articles'])\n    candidate_joins = pd.read_parquet(TABLE / 'candidate_joins.parquet',\n                                     columns=['ik14', 'tier_row', 'pubchemlite_row'])\n    keys = pd.read_parquet(TABLE / 'natural_products.parquet', columns=['ik14']).ik14\n    assert candidate_joins.ik14.equals(keys)\n    nt = nt.assign(in_pubchem_tier=candidate_joins.tier_row.ge(0).to_numpy(),\n                   in_pubchemlite=candidate_joins.pubchemlite_row.ge(0).to_numpy())\n    nt = nt.assign(np_pathway=nt.np_pathway.fillna('unclassified'),\n                   with_pubmed=nt.pubmed_articles.fillna(0).gt(0),\n                   in_casmi_train=nt.in_casmi_train.astype(bool), in_casmi_pool=nt.in_casmi_pool.astype(bool))\n    def composition(g):\n        return pd.Series({'natural products': len(g),\n                          'with training spectrum (%)': 100 * g.in_casmi_train.mean(),\n                          'in candidate pool (%)': 100 * g.in_casmi_pool.mean(),\n                          'with PubMed article (%)': 100 * g.with_pubmed.mean(),\n                          'in PubChem tier (%)': 100 * g.in_pubchem_tier.mean(),\n                          'in PubChemLite add-on (%)': 100 * g.in_pubchemlite.mean()})\n    t = pd.DataFrame({k: composition(g) for k, g in nt.groupby('np_pathway')}).T\n    t = t.sort_values('natural products', ascending=False)\n    t.loc['all'] = composition(nt)\n    t.index.name = 'pathway'\n    fmt = {c: '{:.1f}' for c in t.columns if c.endswith('(%)')}\n    fmt['natural products'] = '{:,.0f}'\n    display(t.style.format(fmt))\n    del nt\n    pcl_docs = pd.read_parquet(POP / 'pubchemlite_annotations.parquet',\n                               columns=['tier_row', 'lsid', 'lpmid', 'lotus_refs'])\n    display(pd.DataFrame([{\n        'PubChemLite add-on candidates': len(pcl_docs),\n        'matched to pinned PubChem tier (%)': 100 * pcl_docs.tier_row.ge(0).mean(),\n        'with documentation prior (%)': 100 * (pcl_docs.lsid + pcl_docs.lpmid).gt(0).mean(),\n        'with LOTUS references (%)': 100 * pcl_docs.lotus_refs.gt(0).mean(),\n    }]).style.hide(axis='index').format({'PubChemLite add-on candidates': '{:,}', **{\n        c: '{:.1f}' for c in ['matched to pinned PubChem tier (%)',\n                              'with documentation prior (%)', 'with LOTUS references (%)']}}))\n    del pcl_docs\nexcept Exception as exc:\n    print('knowledge table unavailable:', type(exc).__name__, exc)\n\n# ===== cell 10 =====\n# EDA has finished: release scalar tables and RDKit objects before constructing the search index.\nimport gc\nfor _name in (\"train\", \"unique_structures\", \"descriptor_sample\", \"desc\", \"fp_sample\", \"records\",\n              \"preview_pool\", \"representatives\", \"mols\", \"X\", \"projection\", \"library_pairs\"):\n    globals().pop(_name, None)\nplt.close(\"all\")\ngc.collect()\n\n\n# ===== cell 11 =====\n'''CASMI26: Metric-Identity Fusion \u2014 Aggressive Experiment\nOne vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. Preserve the existing PubChem gate and same-formula GLACIER logic. This tests duplicate-vote bias, not a coefficient search. It can change top-1 and reduce the score.\nRun the complete repaired @obstacledeveloper adaptation first, then this independent overlay. submission_reference.csv is the completed baseline from the same run; experiment_audit.csv/json record changes and file hashes. This is not stacked with the other experiment. Same 17 source inputs and GPU environment; no additional dataset required. No new public score has been measured. A visible smoke run is not validation.'''\n\n\n# ===== cell 12 =====\n'''Credits & Attribution\n@obstacledeveloper \u2014 direct source: CASMI26 V1.1 Metric Audit and Safe Tail, version 1, scriptVersionId 356393001. Source: https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail?scriptVersionId=356393001\nThe source page displays public score 0.433; this adaptation has not reproduced that result.\n@bobthebot369 \u2014 inherited Sovereign Zenith ensemble and confidence-promotion ideas: https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith/notebook\n@ahmedberatozer \u2014 engine, ranker, pool, PubChem, full FPNet and simulator packages: https://www.kaggle.com/ahmedberatozer\n@prvsiyan \u2014 analog propagation, spectral matching and fingerprint models: https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline\n@dmitriigluzdov \u2014 PubChem popularity prior: https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior\n@hengck23 \u2014 inherited hengck23-dreams-enveda-casmi26 structural candidate resource: https://www.kaggle.com/hengck23\nOfficial metric implementation, version 14: https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank\nICEBERG/GLACIER research tooling: https://github.com/coleygroup/ms-pred\nCompetition: https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra\nRDKit official metric wheel: https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel\nChEBI/LIPID MAPS and COCONUT resources remain credited in inherited code and input metadata.\nThe public notebook declares Apache-2.0; upstream datasets/components retain their own licences. Substantial code, including embedded modules, is reused verbatim. Our adaptation changes presentation and adds provenance, not authorship of the engines. Upstream comments are retained as provenance, not independently established component-performance claims.'''\n\n\n# ===== cell 13 =====\n'''Experiment provenance and reasoning\nDirect starting point: @obstacledeveloper, linked in Credits & Attribution below. The upstream engine/fusion/simulator code is substantially reused, not claimed as original. Our contribution here is the separately audited overlay described above. Previous analog expansion, top-1 shielding, adduct-specific merging and single-pass variants did not improve our confirmed 0.420 result. These experiments test structural ranking changes instead of repeating those parameter changes. Metric identity uses the inherited official canonicalization and heavy-atom composition helpers. GLACIER tooling: https://github.com/coleygroup/ms-pred ; research: https://arxiv.org/abs/2606.29161 . Global cross-formula ranking is our unvalidated hypothesis, not a claim made by that paper. Public leaderboard feedback does not establish private-set generalization.'''\n\n\n# ===== cell 14 =====\n'''# CASMI26 V1.1 \u2014 metric audit and safe tail\n\nStarting point: this account's V1 submission, public MRR **0.425** on 2026-10-08.\nThis is a conservative candidate-coverage experiment, not a claimed leaderboard improvement.\n\nChanges: pin official RDKit 2026.03.3; preserve the V1 inference/ranking pipeline;\ndeduplicate only after ranking using the official metric identity; append ranked overflow\nand mass-compatible DreaMS candidates into vacant slots; assert every original matchable\nidentity has equal or better rank. The old DreaMS branch was bypassed by `NameError: Path`.\n\nNo hidden-test labels, manual labels, or network access are used. The visible test overlaps\ntraining data, so its score is an operational check and is **not** a validation estimate.\n\nSource attribution: copied V1 ensemble from BobtheBot369 / Ahmed Berat \u00d6zer / prvsiyan\nand the other authors credited in its code. Scoring reference: Kaggle Competition Metrics,\n`https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank`, version 14, Apache-2.0.\n'''\n\n\n# ===== cell 15 =====\n'''## 1. Setup\n\nLocates every dataset, installs the competition's RDKit wheel, sets the popularity-prior parameters for the PubChem channel (`POP_LAM = 0.25`, `POP_UNION = 200`) and checks the v4b model files against their manifest.\n'''\n\n\n# ===== cell 16 =====\n'''CASMI26 v4n = v4i + ICE_LAM = GL_LAM = 1.0 (v4l, LB 0.380) + engine FP bank fpnet_full1 (v4m, LB 0.378)'''\n'''CASMI26 v4i = v4h + PubChem-only channel pass-1 depth N1 1000 -> 5000 (recall; gate/slots unchanged)'''\n'''CASMI26 v4h = v4g + GLACIER third re-scoring term: z(ranker) + 0.5 z(ICEBERG) + 0.5 z(GLACIER) (casmi26-glacier)'''\n'''casmi26 v28 = v17 + pool popularity re-rank of the base list (mu 0.15, mode A); v17 = Ahmed v4g + popularity prior in the PubChem list (see make_v17.py)'''\n'''CASMI26 v4f = v4b (FPNet A engine bank, LB 0.354) + ICEBERG post-ranker isomer re-scoring = the v3.6 model (v1 engine + FPNet A + moderate generation + gated PubChem channel, LB 0.358)'''\n'''+ fe_v4 feature families computed on the SAME v1 engine run (derivation evidence / class-3 priors, fragmentation 2.0,'''\n'''analog-structure relations, DreamsFP views) + a ranker retrained on the v1-engine simulation sim1x with them.'''\n'''Built by work/analysis/fe_v4/build_v4v1.py -- edit there.'''\nimport os, sys, glob, re, subprocess, time, json, hashlib, pickle\nfrom pathlib import Path\nT0 = time.time()\nLIB_TAU, REL_TH = 0.9, 600.0\nTOPN, ICE_LAM, ICE_BUDGET, ICE_PC = 60, 1.0, 5400, True\nSLOTS_AGG, SLOTS_GENTLE = [2, 4, 6, 8, 10], [4, 8, 12, 16, 20]\nos.makedirs('/kaggle/working/numba_cache', exist_ok=True)\nos.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'\ndef find(pattern):\n    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)\n    if not hits:\n        raise FileNotFoundError(pattern)\n    return hits[0]\ndef sha256(path, n=1 << 22):\n    h = hashlib.sha256()\n    with open(path, 'rb') as f:\n        for b in iter(lambda: f.read(n), b''):\n            h.update(b)\n    return h.hexdigest()\nFOLD0_PATH = str(FOLD / 'fpnet_fold0.pt')\nassert sha256(FOLD0_PATH) == 'ddf6eaaa8e541a16e3f0357ed30f0346e529054f75d5619aedb6024dcb1bd7f0', 'Untested fold-safe checkpoint'\nos.environ['CASMI_FPNET_FOLD0'] = FOLD0_PATH\nprint('PubChem fold-safe model:', FOLD0_PATH, flush=True)\nPOP_LAM, POP_UNION = 0.15, 200            # v17: popularity prior for the PubChem-only list\nos.environ['CASMI_POP_DIR'] = str(POP); os.environ['CASMI_POP_LAM'] = str(POP_LAM); os.environ['CASMI_POP_UNION'] = str(POP_UNION)\nprint('v17 popularity prior', POP_LAM, POP_UNION, os.environ['CASMI_POP_DIR'])\nSRC_BONUS, SRC_BITS = 0.5, 1   # documentation prior + bonus for depositor-source groups (pc_src_mask.npy, dataset v5)\nos.environ['CASMI_SRC_BONUS'] = str(SRC_BONUS); os.environ['CASMI_SRC_BITS'] = str(SRC_BITS)\nassert os.path.exists(os.path.join(os.environ['CASMI_POP_DIR'], 'pc_src_mask.npy')), 'pc_src_mask.npy missing'\nprint('depositor-source bonus', SRC_BONUS, 'bits', SRC_BITS, flush=True)\nNP_BONUS = 1.0   # natural-product bonus: COCONUT, LOTUS or NPAtlas membership (pc_np.npy, popularity prior)\nos.environ['CASMI_NP_BONUS'] = str(NP_BONUS)\nassert os.path.exists(os.path.join(os.environ['CASMI_POP_DIR'], 'pc_np.npy')), 'pc_np.npy missing'\nprint('natural-product bonus', NP_BONUS, flush=True)\nLOTUS_W = 0.5   # literature prior: ln(1 + LOTUS references) from the natural-product knowledge table\nos.environ['CASMI_LOTUS_W'] = str(LOTUS_W)\nassert (POP / 'aligned_public_priors.py').is_file(), 'aligned_public_priors.py missing'\nassert sha256(POP / 'aligned_public_priors.py') == '89873c7312c16be2180d44bb9eb3c4e9ce19c821e74ab92bb41666ea0c0f9248', 'Untested aligned prior helper'\nassert (POP / 'pc_lotus_refs.npy').is_file(), 'pc_lotus_refs.npy missing'\nprint('LOTUS literature prior', LOTUS_W, flush=True)\nassert (POP / 'pc_lsynonyms.npy').is_file(), 'pc_lsynonyms.npy missing (popularity dataset of 19.10 or later)'\nprint('PubChem synonym prior 0.5 (pc_lsynonyms.npy)', flush=True)\nfor _n in ('delta_pc_mass.npy', 'growth_pc_mass.npy', 'delta_pc_lsynonyms.npy', 'growth_pc_lsynonyms.npy'):\n    assert (POP / _n).is_file(), _n + ' missing (popularity dataset of 20.10 or later)'\nimport numpy as _np\nprint('PubChem supplements: legacy delta', len(_np.load(POP / 'delta_pc_mass.npy', mmap_mode='r')), 'rows + growth',\n      len(_np.load(POP / 'growth_pc_mass.npy', mmap_mode='r')), 'rows', flush=True)\ntag = f'cp{sys.version_info.major}{sys.version_info.minor}'\nwhl = [w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w and Path(w).name.startswith('rdkit-2026.3.3-')]\nassert whl, 'Official RDKit 2026.3.3 wheel is missing'\nprint('rdkit wheel:', whl)\nr = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], capture_output=True, text=True)\nprint(r.stdout[-500:], r.stderr[-500:])\nr.check_returncode()\nimport rdkit; print('rdkit', rdkit.__version__)\nassert rdkit.__version__ == '2026.03.3'\nV4CODE = os.path.dirname(os.path.dirname(find('casmi26-v4b-models/**/casmi/engine.py')))\nV4MOD = os.path.dirname(find('casmi26-v4b-models/**/MANIFEST.json'))\nV3CODE = os.path.dirname(os.path.dirname(find('casmi26-v3-models/**/casmi/engine.py')))\nV3MOD = os.path.dirname(find('casmi26-v3-models/**/ranker_0.pkl'))\nPOOL_DIR = os.path.dirname(find('pool_meta.parquet'))\nPC_DIR = os.path.dirname(find('pc_smiles.npy'))\nCOMP = os.path.dirname(find('test.parquet'))\nMAN = json.load(open(os.path.join(V4MOD, 'MANIFEST.json')))\nbad = []\nfor rel, h in sorted(MAN['files'].items()):\n    p = os.path.join(V4CODE, rel[len('code/'):]) if rel.startswith('code/') else os.path.join(V4MOD, rel)\n    if not os.path.exists(p) or sha256(p) != h:\n        bad.append(rel)\nassert not bad, f'dataset files differ from MANIFEST.json: {bad}'\nprint('V4CODE', V4CODE, '\\nV4MOD', V4MOD, '\\nV3CODE', V3CODE, '\\nPOOL', POOL_DIR, '\\nPC', PC_DIR, '\\nCOMP', COMP)\nprint(f\"{len(MAN['files'])} files match MANIFEST.json; families {MAN['families']}\", f'{time.time()-T0:.0f}s')\n\n'''Compatibility repair: select the simulator RDKit 2025.3.6 wheel by interpreter and platform tags, including bundled .whl.ice files. Fail before inference when no compatible build exists; preserve the main RDKit 2026.03.3 environment.'''\n\ndef simulator_rdkit_wheel(pkg):\n    from pip._vendor.packaging.tags import sys_tags\n    from pip._vendor.packaging.utils import parse_wheel_filename\n    compatible = set(sys_tags())\n    bundled = sorted(glob.glob(os.path.join(pkg, 'wheels', 'rdkit-2025.3.6-*.whl*')))\n    external = sorted(glob.glob('/kaggle/input/**/rdkit-2025.3.6-*.whl', recursive=True))\n    for path in bundled + external:\n        name = os.path.basename(path).removesuffix('.ice')\n        _, version, _, tags = parse_wheel_filename(name)\n        if str(version) == '2025.3.6' and compatible.intersection(tags):\n            return path\n    raise FileNotFoundError(\n        f'RDKit 2025.3.6 wheel compatible with Python {sys.version_info.major}.{sys.version_info.minor} '\n        f'and this platform is required for ICEBERG/GLACIER; candidates: {bundled + external}')\n\n_simulator_pkg = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))\nprint('Simulator RDKit wheel:', simulator_rdkit_wheel(_simulator_pkg))\n\n\n# ===== cell 17 =====\n# v42 determinism policy, adapted from its restored tree to E006's original mounts.\n# E006 cell6 has already verified the original v4b manifest and all its hashes.\ndef _v43_patch_engine(source_root, work_root):\n    import hashlib\n    from pathlib import Path\n    source_root = Path(source_root).resolve(strict=True)\n    work_root = Path(work_root)\n    original = source_root / 'casmi' / 'engine.py'\n    raw = original.read_bytes()\n    old = 'ad = max(set(adducts), key=adducts.count)'\n    new = 'ad = max(adducts, key=adducts.count)'\n    text = raw.decode('utf-8')\n    assert text.count(old) == 1, 'Expected exactly one source main-engine adduct tie'\n    patched = text.replace(old, new, 1).encode('utf-8')\n    assert source_root != work_root.resolve() and source_root not in work_root.resolve().parents\n    work_root.mkdir(parents=True, exist_ok=True)\n    # Preserve the original relative source tree. Only engine.py is copied/changed.\n    for source in sorted(source_root.rglob('*')):\n        if '__pycache__' in source.parts or source.suffix == '.pyc':\n            continue\n        dest = work_root / source.relative_to(source_root)\n        if source.is_dir():\n            dest.mkdir(parents=True, exist_ok=True)\n        elif source == original:\n            assert not dest.is_symlink(), 'Patched engine must be a work-tree file'\n            if dest.exists():\n                assert dest.read_bytes() == patched, 'Existing work-tree patch differs'\n            else:\n                dest.write_bytes(patched)\n        else:\n            if dest.is_symlink():\n                assert dest.resolve(strict=True) == source.resolve(strict=True)\n            else:\n                assert not dest.exists(), 'Existing work-tree source differs'\n                dest.symlink_to(source)\n    assert original.read_bytes() == raw, 'Mounted source changed'\n    assert (work_root / 'casmi/engine.py').read_bytes() == patched\n    return dict(applied=True, original_source_root=str(source_root), work_source_root=str(work_root),\n        original_sha256=hashlib.sha256(raw).hexdigest(), patched_sha256=hashlib.sha256(patched).hexdigest(),\n        original_line=old, patched_line=new, subprocess_pythonhashseed='0',\n        torch_seed=0, cuda_seed=0, cudnn_deterministic=True, cudnn_benchmark=False)\n\nV43_DETERMINISM = _v43_patch_engine(V4CODE, '/kaggle/working/v43_v4code')\nV4CODE = V43_DETERMINISM['work_source_root']\nos.environ['PYTHONHASHSEED'] = '0'\n\n\n# ===== cell 18 =====\n\"\"\"Read-only E004 observer, before the source smoke switch.\"\"\"\n\n\ndef _e004_preflight(scope):\n    import json\n    import os\n    import platform\n    import subprocess\n    import pandas as pd\n    import rdkit\n\n    slug = 'enveda-CASMI26-molecule-id-mass-spectra'\n    comp = os.path.realpath(scope['COMP'])\n    assert comp in {f'/kaggle/input/{slug}', f'/kaggle/input/competitions/{slug}'}\n    assert platform.python_version_tuple()[:2] == ('3', '12')\n    assert rdkit.__version__ == '2026.03.3'\n    # Do not initialize a parent CUDA context before the source's Engine2 subprocess.\n    gpu = subprocess.check_output(\n        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()\n    assert 'T4' in gpu, gpu\n    ids = sorted(set(pd.read_parquet(os.path.join(comp, 'test.parquet'),\n                                    columns=['molecule_id']).molecule_id.astype(str)))\n    sample = pd.read_csv(os.path.join(comp, 'sample_submission.csv'), dtype=str)\n    assert ids and set(sample.molecule_id) == set(ids)\n    result = dict(runtime_input_path=comp, input_ids=ids, python=platform.python_version(),\n                  rdkit=rdkit.__version__, gpu=gpu)\n    print('E004_PREFLIGHT', json.dumps({k: v for k, v in result.items() if k != 'input_ids'}))\n    return result\n\n\nE004_PREFLIGHT = _e004_preflight(globals())\ndel _e004_preflight\n\n\n# ===== cell 19 =====\n'''## 2. Commit switch\n\nWhen the visible placeholder test is detected, the notebook runs only 12 molecules with a short ICEBERG budget, so saving a version is quick. The scoring rerun on the hidden test runs everything. Nothing else depends on this switch.\n'''\n\n\n# ===== cell 20 =====\n'''SMOKE switch: commits (dummy test) run 12 molecules; the scoring rerun (hidden test) runs everything.'''\nimport pandas as pd, hashlib\n_te = pd.read_parquet(os.path.join(COMP, 'test.parquet'), columns=['molecule_id', 'spectrum_id'])\n_sig = hashlib.md5(','.join(sorted(_te.spectrum_id.astype(str))).encode()).hexdigest()\nIS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or _sig != '654e030e4173780e945252b4a5adf70b'\nSMOKE_N = 12\nif not IS_RERUN:\n    SM = '/kaggle/working/smoke_comp'; os.makedirs(SM, exist_ok=True)\n    _full = pd.read_parquet(os.path.join(COMP, 'test.parquet'))\n    _keep = sorted(_full.molecule_id.unique())[:SMOKE_N]\n    _full[_full.molecule_id.isin(_keep)].to_parquet(os.path.join(SM, 'test.parquet'))\n    for _f in ['train.parquet', 'sample_submission.csv']:\n        if not os.path.exists(os.path.join(SM, _f)):\n            os.symlink(os.path.join(COMP, _f), os.path.join(SM, _f))\n    COMP = SM\n    ICE_BUDGET = 300\nprint('RERUN', IS_RERUN, '| sig', _sig, '| COMP', COMP, '| ICE_BUDGET', ICE_BUDGET)\n\n\n# ===== cell 21 =====\n'''## 3. Engine 2: two-ranker engine (BIO + AFIX)\n\nThis engine runs first in its own process, while GPU memory is still empty. It writes a top-40 list per molecule with tautomer-canonical InChIKey-14 keys.\n\n- **Candidates** come from COCONUT, ChEBI/LIPID MAPS (BIO) and the training structures.\n- **Evidence** comes from library matching (including adduct-shifted matches), analog propagation, prvsiyan's fingerprint models and MetFrag-lite fragment matching.\n- **Ranking** blends two gradient-boosted rankers.\n'''\n\n\n# ===== cell 22 =====\n'''Our engine (two-ranker + BIO + AFIX) in a subprocess, then weighted reciprocal-rank fusion with the v4g lists.'''\nENG_DIR = '/kaggle/working/eng'; os.makedirs(ENG_DIR, exist_ok=True)\nENG_FILES = {'pv.py': '\"\"\"prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied\\nverbatim where possible so local simulations and the Kaggle notebook share one source of truth.\\nAdditions are marked `# ADDED`.\\n\"\"\"\\nimport math\\nimport numpy as np\\nfrom numba import njit, prange\\n\\n\\nclass CFG:\\n    PPM_WIN = 10.0\\n    PPM_FALLBACK = 30.0\\n    INT_FLOOR = 0.002\\n    MAX_PEAKS = 256\\n    MZ_TOL = 0.01\\n    INT_POWER = 1.0\\n    ENT_WEIGHT = True\\n    ANALOG_WIN = 200.0\\n    N_ANALOG = 80\\n    SIM_POWER = 3.0\\n    W1_PRIORS = (0.30, 0.60)\\n    SEEDS = (0, 1, 2, 3, 4, 5, 6, 7)  # 8-seed bag (ghazarosbarseghyan91 v14 seeds)\\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\\n    TOPN = 25\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# similarity kernels\\n# ----------------------------------------------------------------------------------------------\\n@njit(cache=True, fastmath=True)\\ndef _clean(mz, it, floor, topk, power, ent_weight):\\n    n = len(mz)\\n    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)\\n    mx = 0.0\\n    for i in range(n):\\n        if it[i] > mx: mx = it[i]\\n    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)\\n    thr = floor * mx; c = 0\\n    for i in range(n):\\n        if it[i] >= thr: c += 1\\n    idx = np.empty(c, np.int64); j = 0\\n    for i in range(n):\\n        if it[i] >= thr: idx[j] = i; j += 1\\n    if c > topk:\\n        v = np.empty(c, np.float32)\\n        for i in range(c): v[i] = it[idx[i]]\\n        o = np.argsort(v)[c - topk:]\\n        k2 = np.empty(topk, np.int64)\\n        for i in range(topk): k2[i] = idx[o[i]]\\n        k2.sort(); idx = k2; c = topk\\n    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)\\n    s = 0.0\\n    for i in range(c):\\n        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v\\n    if s > 0:\\n        for i in range(c): oi[i] /= s\\n    if ent_weight:\\n        S = 0.0\\n        for i in range(c):\\n            if oi[i] > 0: S -= oi[i] * np.log(oi[i])\\n        if S < 3.0:\\n            w = 0.25 + 0.25 * S; s2 = 0.0\\n            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]\\n            if s2 > 0:\\n                for i in range(c): oi[i] /= s2\\n    return om, oi\\n\\n\\n@njit(cache=True, fastmath=True)\\ndef entropy_sim(qmz, qp, cmz, cp, tol):\\n    i = 0; j = 0; n = len(qmz); m = len(cmz)\\n    SA = 0.0\\n    for x in range(n):\\n        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])\\n    SB = 0.0\\n    for x in range(m):\\n        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])\\n    SAB = 0.0; tot = 0.0\\n    buf = np.empty(n + m, np.float64); b = 0\\n    while i < n and j < m:\\n        d = qmz[i] - cmz[j]\\n        if d < -tol: buf[b] = qp[i]; i += 1; b += 1\\n        elif d > tol: buf[b] = cp[j]; j += 1; b += 1\\n        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1\\n    while i < n: buf[b] = qp[i]; i += 1; b += 1\\n    while j < m: buf[b] = cp[j]; j += 1; b += 1\\n    for x in range(b): tot += buf[x]\\n    if tot <= 0: return 0.0\\n    for x in range(b):\\n        v = buf[x] / tot\\n        if v > 0: SAB -= v * np.log(v)\\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\\n\\n\\n@njit(cache=True, fastmath=True, parallel=True)\\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\\n    out = np.zeros(len(cand), np.float32)\\n    for k in prange(len(cand)):\\n        c = cand[k]; a = off[c]; b = off[c + 1]\\n        if b <= a: continue\\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\\n        if len(cm) == 0: continue\\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\\n    return out\\n\\n\\n@njit(cache=True, fastmath=True)\\ndef entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):\\n    a = entropy_sim(qmz, qp, cmz, cp, tol)\\n    if shift > -0.001 and shift < 0.001: return a\\n    sm = np.empty(len(cmz), np.float32)\\n    for i in range(len(cmz)): sm[i] = cmz[i] + shift\\n    b = entropy_sim(qmz, qp, sm, cp, tol)\\n    return a if a > b else b\\n\\n\\n# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)\\n@njit(cache=True, fastmath=True, parallel=True)\\ndef search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):\\n    out = np.zeros(hi - lo, np.float32)\\n    for k in prange(hi - lo):\\n        a = roff[lo + k]; b = roff[lo + k + 1]\\n        if b <= a: continue\\n        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])\\n    return out\\n\\n\\n# ADDED: library search where each reference is shifted by (query precursor - reference precursor):\\n# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)\\n@njit(cache=True, fastmath=True, parallel=True)\\ndef search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):\\n    out = np.zeros(len(cand), np.float32)\\n    for k in prange(len(cand)):\\n        c = cand[k]; a = off[c]; b = off[c + 1]\\n        if b <= a: continue\\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\\n        if len(cm) == 0: continue\\n        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])\\n    return out\\n\\n\\n# ADDED: clean a list of spectra into a flat store\\n@njit(cache=True)\\ndef clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):\\n    n = len(rows)\\n    lens = np.zeros(n + 1, np.int64)\\n    tmp_m = []\\n    tmp_i = []\\n    for k in range(n):\\n        r = rows[k]\\n        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)\\n        tmp_m.append(cm); tmp_i.append(cp)\\n        lens[k + 1] = lens[k] + len(cm)\\n    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)\\n    for k in range(n):\\n        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]\\n    return lens, om, oi\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# adducts\\n# ----------------------------------------------------------------------------------------------\\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\\nE = 0.00054857990; PROTON = MASS[\\'H\\'] - E; H2O = 2 * MASS[\\'H\\'] + MASS[\\'O\\']\\nNH4 = MASS[\\'N\\'] + 4 * MASS[\\'H\\']; FORMATE = MASS[\\'C\\'] + 2 * MASS[\\'H\\'] + 2 * MASS[\\'O\\']\\nACETATE = 2 * MASS[\\'C\\'] + 4 * MASS[\\'H\\'] + 2 * MASS[\\'O\\']\\nADDUCTS = {\\n    \"[M+H]+\": (1, 1, PROTON), \"[M+NH4]+\": (1, 1, NH4 - E), \"[M+Na]+\": (1, 1, MASS[\\'Na\\'] - E),\\n    \"[M+K]+\": (1, 1, MASS[\\'K\\'] - E), \"[M-H2O+H]+\": (1, 1, PROTON - H2O), \"[M-2H2O+H]+\": (1, 1, PROTON - 2 * H2O),\\n    \"[M+2H]2+\": (1, 2, 2 * PROTON), \"[M]+\": (1, 1, -E), \"[M-H2O]+\": (1, 1, -E - H2O),\\n    \"[M+CH3OH+H]+\": (1, 1, PROTON + MASS[\\'C\\'] + 4 * MASS[\\'H\\'] + MASS[\\'O\\']),\\n    \"[M+CH3CN+H]+\": (1, 1, PROTON + 2 * MASS[\\'C\\'] + 3 * MASS[\\'H\\'] + MASS[\\'N\\']),\\n    \"[M-H]-\": (1, 1, -PROTON), \"[M-H2O-H]-\": (1, 1, -PROTON - H2O), \"[M+CH2O2-H]-\": (1, 1, FORMATE - PROTON),\\n    \"[M+C2H4O2-H]-\": (1, 1, ACETATE - PROTON), \"[M+Cl]-\": (1, 1, MASS[\\'Cl\\'] + E), \"[M]-\": (1, 1, E),\\n    \"[M-2H]-\": (1, 2, -2 * PROTON), \"[M+Na-2H]-\": (1, 1, MASS[\\'Na\\'] - 2 * PROTON),\\n    \"[2M+H]+\": (2, 1, PROTON), \"[2M+Na]+\": (2, 1, MASS[\\'Na\\'] - E), \"[2M+NH4]+\": (2, 1, NH4 - E),\\n    \"[2M+K]+\": (2, 1, MASS[\\'K\\'] - E), \"[2M-H]-\": (2, 1, -PROTON), \"[2M+CH2O2-H]-\": (2, 1, FORMATE - PROTON),\\n    \"[2M+C2H4O2-H]-\": (2, 1, ACETATE - PROTON), \"[2M+Na-2H]-\": (2, 1, MASS[\\'Na\\'] - 2 * PROTON),\\n    \"[3M+H]+\": (3, 1, PROTON), \"[3M-H]-\": (3, 1, -PROTON),\\n}\\n\\n\\ndef neutral_mass(mz, adduct):\\n    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)\\n    for a, (n, z, d) in ADDUCTS.items():\\n        m = (ad == a)\\n        if m.any(): out[m] = (mz[m] * z - d) / n\\n    return out\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# ranking features (31)\\n# ----------------------------------------------------------------------------------------------\\nN_ANALOG = 80\\nP_SIM = 3.0\\nN_FEAT = 31\\n\\n\\ndef _rank_norm(x):\\n    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)\\n\\n\\ndef _z(x):\\n    s = x.std()\\n    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)\\n\\n\\ndef rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):\\n    nc = cand_fp.shape[0]\\n    cf = cand_fp.astype(np.float32); cs = cf.sum(1)\\n    lv = np.asarray(cand_lib, np.float32)\\n    lvmax = float(lv.max()) if nc else 0.0\\n    if analog_fp is not None and len(analog_sim):\\n        af = analog_fp.astype(np.float32); asum = af.sum(1)\\n        inter = cf @ af.T\\n        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)\\n        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)\\n        ap = (tan * (w ** P_SIM)[None, :]).max(1)\\n        a1 = (tan * w[None, :]).max(1)\\n        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])\\n        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)\\n    else:\\n        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0\\n    apmax = float(ap.max()) if nc else 0.0\\n    if model_logits is not None:\\n        raw = cf @ np.asarray(model_logits, np.float32)\\n        nrm = raw / np.sqrt(np.maximum(cs, 1.0))\\n        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),\\n                 (raw == raw.max()).astype(np.float32)]\\n    else:\\n        mfeat = [np.zeros(nc, np.float32)] * 6\\n    if model_logits is not None and nc:\\n        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))\\n        lbest = int(np.argmax(lv)) if lvmax > 0 else -1\\n        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0\\n        abest = int(np.argmax(ap)) if apmax > 0 else -1\\n        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0\\n        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),\\n                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]\\n    else:\\n        xfeat = [np.zeros(nc, np.float32)] * 6\\n    if frag is not None:\\n        fr = np.asarray(frag, np.float32)\\n        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]\\n    else:\\n        ffeat = [np.zeros(nc, np.float32)] * 4\\n    return np.column_stack([\\n        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),\\n        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,\\n        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),\\n        np.full(nc, np.log(max(nc, 1))),\\n        *mfeat, *ffeat, *xfeat,\\n    ]).astype(np.float32)\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# MetFrag-lite\\n# ----------------------------------------------------------------------------------------------\\nAMU = {\\'C\\': 12.0, \\'H\\': 1.00782503207, \\'N\\': 14.0030740048, \\'O\\': 15.9949146196, \\'P\\': 30.97376163,\\n       \\'S\\': 31.97207100, \\'F\\': 18.99840322, \\'Cl\\': 34.96885268, \\'Br\\': 78.9183371, \\'I\\': 126.904473,\\n       \\'Na\\': 22.9897692809, \\'K\\': 38.96370668, \\'Si\\': 27.9769265325, \\'B\\': 11.0093054, \\'Se\\': 79.9165213}\\nH = AMU[\\'H\\']\\n\\n\\ndef mol_graph(smi):\\n    from rdkit import Chem, RDLogger\\n    RDLogger.DisableLog(\\'rdApp.*\\')\\n    m = Chem.MolFromSmiles(smi)\\n    if m is None: return None\\n    n = m.GetNumAtoms()\\n    w = np.zeros(n)\\n    for a in m.GetAtoms():\\n        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H\\n    if (w == 0).any(): return None\\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\\n    return w, bonds, n\\n\\n\\ndef _components(n, bonds, drop):\\n    adj = [[] for _ in range(n)]\\n    for i, (a, b) in enumerate(bonds):\\n        if i in drop: continue\\n        adj[a].append(b); adj[b].append(a)\\n    seen = np.zeros(n, bool); comps = []\\n    for s in range(n):\\n        if seen[s]: continue\\n        stack = [s]; seen[s] = True; cur = [s]\\n        while stack:\\n            u = stack.pop()\\n            for v in adj[u]:\\n                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)\\n        comps.append(cur)\\n    return comps\\n\\n\\ndef fragment_masses(smi, max_breaks=2, max_bonds=34):\\n    g = mol_graph(smi)\\n    if g is None: return np.zeros(0)\\n    w, bonds, n = g\\n    nb = len(bonds)\\n    if nb == 0 or nb > max_bonds: return np.array([w.sum()])\\n    out = {w.sum()}\\n    for i in range(nb):\\n        for c in _components(n, bonds, {i}):\\n            out.add(float(w[c].sum()))\\n    if max_breaks >= 2:\\n        for i in range(nb):\\n            for j in range(i + 1, nb):\\n                for c in _components(n, bonds, {i, j}):\\n                    out.add(float(w[c].sum()))\\n    return np.array(sorted(out))\\n\\n\\ndef frag_masses_safe(smi):\\n    try: return fragment_masses(smi)\\n    except Exception: return np.zeros(0)\\n\\n\\n# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)\\nCARRIERS = {\\n    \"[M+H]+\": (PROTON,), \"[M-H2O+H]+\": (PROTON,), \"[M-2H2O+H]+\": (PROTON,),\\n    \"[M+NH4]+\": (PROTON, NH4 - E), \"[M+Na]+\": (PROTON, MASS[\\'Na\\'] - E), \"[M+K]+\": (PROTON, MASS[\\'K\\'] - E),\\n    \"[M-H]-\": (-PROTON,), \"[M-H2O-H]-\": (-PROTON,), \"[M+CH2O2-H]-\": (-PROTON, FORMATE - PROTON),\\n    \"[M+Cl]-\": (-PROTON, MASS[\\'Cl\\'] + E),\\n}\\n\\n\\ndef explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\\n    car = CARRIERS.get(adduct, (PROTON,) if \"+\" in adduct[-2:] else (-PROTON,))\\n    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))\\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\\n    if tot <= 0: return 0.0\\n    idx = np.searchsorted(ion, peak_mz)\\n    ok = np.zeros(len(peak_mz), bool)\\n    for off in (-1, 0):\\n        k = np.clip(idx + off, 0, len(ion) - 1)\\n        ok |= np.abs(ion[k] - peak_mz) <= tol\\n    return float(w[ok].sum() / tot)\\n\\n\\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\\n    ion = []\\n    for dh in h_shifts:\\n        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))\\n    ion = np.sort(np.concatenate(ion))\\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\\n    if tot <= 0: return 0.0\\n    idx = np.searchsorted(ion, peak_mz)\\n    ok = np.zeros(len(peak_mz), bool)\\n    for off in (-1, 0):\\n        k = np.clip(idx + off, 0, len(ion) - 1)\\n        ok |= np.abs(ion[k] - peak_mz) <= tol\\n    return float(w[ok].sum() / tot)', 'pv_fp.py': '\"\"\"Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch.\"\"\"\\nimport math\\nimport numpy as np\\n# ----------------------------------------------------------------------------------------------\\n# spectrum -> fingerprint model\\n# ----------------------------------------------------------------------------------------------\\nimport torch, torch.nn as nn, torch.nn.functional as F\\n\\nFP_MAX_PEAKS = 128\\nADDUCT_LIST = [\"[M+H]+\", \"[M+NH4]+\", \"[M+Na]+\", \"[M+K]+\", \"[M-H2O+H]+\", \"[M-2H2O+H]+\", \"[M]+\",\\n               \"[M-H]-\", \"[M-H2O-H]-\", \"[M+CH2O2-H]-\", \"[M+C2H4O2-H]-\", \"[M+Cl]-\", \"[M]-\",\\n               \"[M+2H]2+\", \"[M-2H]-\", \"[2M+H]+\", \"[2M+Na]+\", \"[2M+NH4]+\", \"[2M-H]-\", \"[2M+K]+\",\\n               \"[2M+CH2O2-H]-\", \"[2M+C2H4O2-H]-\", \"[2M+Na-2H]-\", \"[M+Na-2H]-\", \"[M-H2O]+\", \"<unk>\"]\\nADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}\\nINSTR_LIST = [\"timsTOF\", \"Orbitrap\", \"QTOF\", \"IT\", \"other\"]\\n\\n\\ndef instr_family(s):\\n    if s is None: return 4\\n    t = str(s).lower()\\n    if \\'timstof\\' in t: return 0\\n    if \\'orbitrap\\' in t or \\'qft\\' in t or \\'ftms\\' in t or \\'hybrid ft\\' in t or \\'itft\\' in t or \\'exactive\\' in t: return 1\\n    if \\'tof\\' in t: return 2\\n    if \\'trap\\' in t or \\'qq\\' in t: return 3\\n    return 4\\n\\n\\ndef prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\\n    keep = (mz <= prec_mz + 1.5)\\n    mz, it = mz[keep], it[keep]\\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\\n    mx = it.max()\\n    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\\n    keep = it >= floor * mx\\n    mz, it = mz[keep], it[keep]\\n    if len(mz) > max_peaks:\\n        order = np.argsort(-it)\\n        bucket = (mz // win).astype(np.int64)\\n        cnt = {}; sel = []\\n        for i in order:\\n            b = bucket[i]; c = cnt.get(b, 0)\\n            if c < per_win: cnt[b] = c + 1; sel.append(i)\\n        sel = np.array(sel)\\n        if len(sel) > max_peaks:\\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\\n        elif len(sel) < max_peaks:\\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\\n            need = max_peaks - len(sel)\\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\\n        mz, it = mz[sel], it[sel]\\n    o = np.argsort(mz)\\n    mz, it = mz[o], it[o]\\n    v = np.sqrt(it / it.max())\\n    return mz.astype(np.float32), v.astype(np.float32)\\n\\n\\nclass SinEmb(nn.Module):\\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\\n        super().__init__()\\n        n = dim // 2\\n        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\\n        self.register_buffer(\\'inv\\', (2 * math.pi) / wav)\\n\\n    def forward(self, x):\\n        a = x.unsqueeze(-1) * self.inv\\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\\n\\n\\nclass Block(nn.Module):\\n    def __init__(self, d, h, drop):\\n        super().__init__(); self.h = h\\n        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)\\n        self.n2 = nn.LayerNorm(d)\\n        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\\n        self.drop = nn.Dropout(drop)\\n\\n    def forward(self, x, pad):\\n        B, N, D = x.shape; y = self.n1(x)\\n        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\\n        m = (~pad)[:, None, None, :]\\n        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)\\n        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\\n        return x + self.drop(self.ff(self.n2(x)))\\n\\n\\nclass FPNet(nn.Module):\\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\\n        super().__init__()\\n        self.d = d\\n        self.mz_emb = SinEmb(d)\\n        self.nl_emb = SinEmb(d)\\n        self.pk = nn.Linear(2 * d + 1, d)\\n        self.prec_emb = SinEmb(d)\\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\\n        self.gl = nn.Linear(d + 3, d)\\n        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\\n        self.norm = nn.LayerNorm(d)\\n        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\\n\\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\\n        B, N = mz.shape\\n        nl = (prec[:, None] - mz).clamp(min=0)\\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\\n        g = self.gl(torch.cat([self.prec_emb(prec),\\n                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\\n                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\\n        x = torch.cat([g.unsqueeze(1), p], 1)\\n        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\\n        for b in self.blocks: x = b(x, pad)\\n        x = self.norm(x)\\n        cls = x[:, 0]\\n        msk = (~pad[:, 1:]).float().unsqueeze(-1)\\n        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\\n        return self.head(torch.cat([cls, mean], -1))\\n\\n\\ndef load_fp_models(paths, dev):\\n    single, merged = [], []\\n    nbits = None\\n    for pth in paths:\\n        ck = torch.load(pth, map_location=\\'cpu\\', weights_only=False)\\n        net = FPNet(ck[\\'nbits\\'], d=ck[\\'d\\'], layers=ck[\\'layers\\']).to(dev).eval()\\n        net.load_state_dict(ck[\\'model\\'])\\n        (merged if \\'merged\\' in str(pth).replace(\\'\\\\\\\\\\', \\'/\\').split(\\'/\\')[-1] else single).append(net)\\n        nbits = ck[\\'nbits\\']\\n    return single, merged, nbits\\n\\n\\ndef merge_peaks(specs):\\n    \"\"\"specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum.\"\"\"\\n    specs = [(a, b) for a, b in specs if len(a)]\\n    if not specs: return np.zeros(0), np.zeros(0)\\n    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])\\n    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])\\n    o = np.argsort(mz); mz, it = mz[o], it[o]\\n    keep = np.ones(len(mz), bool)\\n    for j in range(1, len(mz)):\\n        if mz[j] - mz[j - 1] < 0.005:\\n            if it[j] >= it[j - 1]: keep[j - 1] = False\\n            else: keep[j] = False\\n    return mz[keep], it[keep]\\n\\n\\n@torch.no_grad()\\ndef logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):\\n    \"\"\"P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode.\"\"\"\\n    keep = [i for i, (a, _) in enumerate(P) if len(a)]\\n    if not keep: return None\\n    P = [P[i] for i in keep]\\n    B = len(P); N = max(len(a) for a, _ in P)\\n    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)\\n    for i, (a, b) in enumerate(P):\\n        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False\\n    T = lambda x: torch.as_tensor(x, device=dev)\\n    args = (T(mz), T(it), T(pad),\\n            T(np.asarray(prec, np.float32)[keep]),\\n            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX[\\'<unk>\\']) for a in np.asarray(adduct, object)[keep]])),\\n            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),\\n            T(np.asarray(ce, np.float32)[keep]),\\n            T(np.asarray(mode, np.float32)[keep]))\\n    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)\\n\\n\\ndef molecule_logits(models, specs, prec, adduct, instr, ce, mode):\\n    \"\"\"prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).\\n    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum.\"\"\"\\n    single, merged, dev = models\\n    out = []\\n    if single:\\n        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]\\n        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)\\n        if za is not None: out.append(za)\\n    if merged:\\n        mz, it = merge_peaks(specs)\\n        pm = float(np.median(prec))\\n        P = [prep_peaks(mz, it, pm)]\\n        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])\\n        if zb is not None: out.append(zb)\\n    return np.mean(out, axis=0) if out else None', 'casmi_engine.py': '\"\"\"End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,\\nbut reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).\\n\\nStages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission\\n\"\"\"\\nimport os, sys, glob, time, pickle, math\\nos.environ.setdefault(\"OPENBLAS_NUM_THREADS\", \"1\")\\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\\nimport pv\\nfrom pv import CFG\\nimport re as _re\\n_MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100, P=30.97376163,\\n             Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473, Si=27.9769265325, Se=79.9165213,\\n             Na=22.9897692809, K=38.96370668, B=11.0093054, As=74.9215965, Fe=55.9349375, D=2.0141017778)\\n_TOK = _re.compile(r\"([A-Z][a-z]?)(\\\\d*)\")\\n\\n\\ndef formula_mass(f):\\n    \"\"\"ADDED: neutral monoisotopic mass by arithmetic on molecular_formula (prvsiyan fix; immune to\\n    riken\\'s 0.005 Da precursor precision and gnps adduct mislabels). NaN if an element is untabulated.\"\"\"\\n    if not isinstance(f, str) or not f: return np.nan\\n    m = 0.0\\n    for el, n in _TOK.findall(f):\\n        if el not in _MONO: return np.nan\\n        m += _MONO[el] * (int(n) if n else 1)\\n    return m\\n\\nPOOL = None\\n\\n\\nclass RANK:\\n    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged\\n    SEEDS = (0, 1, 2, 3, 4, 5)  # 6-seed bag (ghazarosbarseghyan91 v14 seeds)\\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\\n\\nT0 = time.time()\\nLOCAL = os.environ.get(\"CASMI_LOCAL\") == \"1\"\\nROOTS = [r\"C:\\\\Users\\\\HW-LEE\\\\Desktop\\\\CASMI\"] if LOCAL else [\"/kaggle/input\"]\\n\\n\\ndef find(name):\\n    for root in ROOTS:\\n        hits = glob.glob(os.path.join(root, \"**\", name), recursive=True)\\n        if hits: return sorted(hits, key=len)[0]\\n    raise FileNotFoundError(name)\\n\\n\\ndef log(msg):\\n    print(f\"[{time.time()-T0:6.0f}s] {msg}\", flush=True)\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# library\\n# ---------------------------------------------------------------------------------------------\\ndef load_library(path):\\n    \"\"\"Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB.\"\"\"\\n    f = pq.ParquetFile(path)\\n    n_rows = f.metadata.num_rows\\n    npk = 0\\n    for i in range(f.num_row_groups):\\n        c = f.read_row_group(i, columns=[\"ms2_mzs\"]).column(0).combine_chunks()\\n        npk += len(c.values)\\n    off = np.zeros(n_rows + 1, np.int64)\\n    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)\\n    prec, add, pol, ik, smi = [], [], [], [], []\\n    fo, tims = [], []\\n    r0 = p0 = 0\\n    for i in range(f.num_row_groups):\\n        t = f.read_row_group(i, columns=[\"inchikey14\", \"normalized_smiles\", \"adduct\", \"precursor_mz\", \"ionization_mode\",\\n                                         \"ms2_mzs\", \"ms2_normalized_intensities\", \"molecular_formula\", \"instrument_type\"])\\n        mzc = t.column(\"ms2_mzs\").combine_chunks(); itc = t.column(\"ms2_normalized_intensities\").combine_chunks()\\n        o = mzc.offsets.to_numpy().astype(np.int64)\\n        n = len(o) - 1; k = int(o[-1] - o[0])\\n        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\\n        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\\n        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])\\n        prec.append(t.column(\"precursor_mz\").to_numpy(zero_copy_only=False).astype(np.float64))\\n        add += t.column(\"adduct\").cast(pa.string()).to_pylist()\\n        pol.append(np.array(t.column(\"ionization_mode\").cast(pa.string()).to_pylist(), dtype=object) == \"positive\")\\n        ik += t.column(\"inchikey14\").cast(pa.string()).to_pylist()\\n        smi += t.column(\"normalized_smiles\").cast(pa.string()).to_pylist()\\n        fo += t.column(\"molecular_formula\").cast(pa.string()).to_pylist()\\n        tims.append(np.array([isinstance(x, str) and x.lower().replace(\"-\", \"\").replace(\" \", \"\") == \"timstof\"\\n                              for x in t.column(\"instrument_type\").cast(pa.string()).to_pylist()]))\\n        r0 += n; p0 += k\\n        del t, mzc, itc\\n    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)\\n    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)\\n    nmp = pv.neutral_mass(prec, add)\\n    _fm = {x: formula_mass(x) for x in set(v for v in fo if isinstance(v, str))}\\n    nmf = np.array([_fm.get(x, np.nan) if isinstance(x, str) else np.nan for x in fo])\\n    nm = np.where(np.isfinite(nmf), nmf, nmp)\\n    tims = np.concatenate(tims)\\n    log(f\"library mass index: formula {np.isfinite(nmf).mean():.1%}, timsTOF rows {tims.mean():.1%}\")\\n    del fo\\n    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol, tims=tims,\\n                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)\\n# ---------------------------------------------------------------------------------------------\\n_g = {}\\n\\n\\ndef _fp_init():\\n    from rdkit import RDLogger\\n    from rdkit.Chem import rdFingerprintGenerator\\n    RDLogger.DisableLog(\"rdApp.*\")\\n    _g[\"bits\"] = np.load(find(\"fp_bits.npy\"))\\n    _g[\"m2\"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\\n    _g[\"m3\"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\\n    _g[\"rk\"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\\n\\n\\ndef fp_and_mass(smi):\\n    from rdkit import Chem\\n    from rdkit.Chem import MACCSkeys\\n    from rdkit.Chem.Descriptors import ExactMolWt\\n    if not _g: _fp_init()\\n    m = Chem.MolFromSmiles(smi)\\n    if m is None: return None\\n    try:\\n        fp = np.concatenate([_g[\"m2\"].GetFingerprintAsNumPy(m).astype(np.uint8),\\n                             _g[\"m3\"].GetFingerprintAsNumPy(m).astype(np.uint8),\\n                             _g[\"rk\"].GetFingerprintAsNumPy(m).astype(np.uint8),\\n                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g[\"bits\"]]\\n        return np.packbits(fp), float(ExactMolWt(m))\\n    except Exception:\\n        return None\\n\\n\\ndef canon_key(smi):\\n    from rdkit import Chem, RDLogger\\n    from rdkit.Chem.MolStandardize import rdMolStandardize\\n    RDLogger.DisableLog(\"rdApp.*\")\\n    if \"te\" not in _g: _g[\"te\"] = rdMolStandardize.TautomerEnumerator()\\n    try:\\n        m = Chem.MolFromSmiles(smi)\\n        return Chem.MolToInchiKey(_g[\"te\"].Canonicalize(m))[:14] if m is not None else None\\n    except Exception:\\n        return None\\n\\n\\ndef build_pool(L, workers):\\n    from multiprocessing import Pool as MPool\\n    d = os.path.dirname(find(\"coco_fp.npy\"))\\n    cm = pickle.load(open(os.path.join(d, \"coco_meta.pkl\"), \"rb\"))\\n    co_fp = np.load(os.path.join(d, \"coco_fp.npy\")); co_mass = np.load(os.path.join(d, \"coco_mass.npy\"))\\n    co_keys = np.asarray(cm[\"keys\"], dtype=object); co_smi = np.asarray(cm[\"smiles\"], dtype=object)\\n    # ADDED: ChEBI + LIPID MAPS (prvsiyan/chebi-lipidmaps-casmi26), same 6,930-bit packed fingerprints\\n    bd = os.path.dirname(find(os.path.join(\"casmi-chebi-lmsd-structures\", \"bio_fp.npy\"))); bm = pickle.load(open(os.path.join(bd, \"bio_meta.pkl\"), \"rb\"))\\n    bkeys = np.asarray(bm[\"keys\"], dtype=object); bnew = ~pd.Series(bkeys).isin(set(co_keys)).values\\n    co_fp = np.vstack([co_fp, np.load(os.path.join(bd, \"bio_fp.npy\"))[bnew]])\\n    co_mass = np.concatenate([co_mass, np.load(os.path.join(bd, \"bio_mass.npy\"))[bnew]])\\n    co_keys = np.concatenate([co_keys, bkeys[bnew]])\\n    co_smi = np.concatenate([co_smi, np.asarray(bm[\"smiles\"], dtype=object)[bnew]])\\n    log(f\"+ ChEBI/LIPID MAPS {int(bnew.sum()):,} new structures\")\\n    # PubChemLite CC0 candidates (idea: guhongbin/casmi26-v447-pubchemlite; block as in ahmedberatozer/casmi26-v14-pubchemlite;\\n    # data: franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints, copied into the attributed v4g assets).\\n    # New arrays are built first and swapped in only if every step succeeds; any failure keeps the previous pool.\\n    try:\\n        pcd = os.path.dirname(find(os.path.join(\"casmi26-pubchemlite-cc0-candidate-fingerprints\", \"pubchemlite_fp.npy\")))\\n        pcm = pickle.load(open(os.path.join(pcd, \"pubchemlite_meta.pkl\"), \"rb\"))\\n        pckeys = np.asarray(pcm[\"keys\"], dtype=object)\\n        pcnew = ~pd.Series(pckeys).isin(set(co_keys)).values\\n        _fp = np.vstack([co_fp, np.load(os.path.join(pcd, \"pubchemlite_fp.npy\"))[pcnew]])\\n        _ms = np.concatenate([co_mass, np.load(os.path.join(pcd, \"pubchemlite_mass.npy\"))[pcnew]])\\n        _ks = np.concatenate([co_keys, pckeys[pcnew]])\\n        _sm = np.concatenate([co_smi, np.asarray(pcm[\"smiles\"], dtype=object)[pcnew]])\\n        assert len(_fp) == len(_ms) == len(_ks) == len(_sm), (len(_fp), len(_ms), len(_ks), len(_sm))\\n        co_fp, co_mass, co_keys, co_smi = _fp, _ms, _ks, _sm\\n        log(f\"+ PubChemLite CC0 {int(pcnew.sum()):,} new structures\")\\n    except Exception as _pe:\\n        log(f\"PubChemLite CC0 skipped: {_pe!r}\")\\n    tr = pd.DataFrame({\"ik\": L[\"ik\"], \"smi\": L[\"smi\"]}).dropna().drop_duplicates(\"ik\")\\n    try:  # ours (yasunorim, 10-21): own COCONUT table, structures not yet in the pool, fingerprinted like the training structures\\n        _od = [p for p in glob.glob(\"/kaggle/input/**/coco_meta.pkl\", recursive=True) if \"own-coconut\" in p]\\n        if _od:\\n            _om = pickle.load(open(_od[0], \"rb\"))\\n            _ot = pd.DataFrame({\"ik\": np.asarray(_om[\"keys\"], dtype=object), \"smi\": np.asarray(_om[\"smiles\"], dtype=object)}).dropna()\\n            _ot = _ot[~_ot.ik.isin(set(co_keys)) & ~_ot.ik.isin(set(tr.ik))].drop_duplicates(\"ik\")\\n            tr = pd.concat([tr, _ot], ignore_index=True)\\n            log(f\"+ ours: own COCONUT {len(_ot):,} new structures\")\\n        else:\\n            log(\"own COCONUT not mounted\")\\n    except Exception as _oe:\\n        log(f\"own COCONUT skipped: {_oe!r}\")\\n    tr = tr[~tr.ik.isin(set(co_keys))]\\n    log(f\"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}\")\\n    with MPool(workers) as mp:\\n        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)\\n    ok = np.array([r is not None for r in res])\\n    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])\\n    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])\\n    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])\\n    good = np.isfinite(mass)\\n    o = np.argsort(np.where(good, mass, 1e18), kind=\"mergesort\")[: good.sum()]\\n    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])\\n    P[\"k2i\"] = pd.Series(np.arange(len(o)), index=P[\"keys\"]); P[\"k2i\"] = P[\"k2i\"][~P[\"k2i\"].index.duplicated()]\\n    log(f\"pool {len(o):,} structures\")\\n    return P\\n\\n\\ndef pool_fps(idx):\\n    return np.unpackbits(np.asarray(POOL[\"fp\"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]\\n\\n\\ndef pool_window(t, ppm):\\n    a = np.searchsorted(POOL[\"mass\"], t * (1 - ppm / 1e6), \"left\")\\n    b = np.searchsorted(POOL[\"mass\"], t * (1 + ppm / 1e6), \"right\")\\n    return np.arange(a, b)\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)\\n# ---------------------------------------------------------------------------------------------\\ndef _reps(L, rows):\\n    npk = np.diff(L[\"off\"])[rows]\\n    df = pd.DataFrame({\"row\": rows, \"p\": L[\"row_p\"][rows], \"npk\": npk, \"tims\": L[\"tims\"][rows]})\\n    # ADDED: prefer the test instrument (timsTOF) representative, then the richest spectrum\\n    df = df.sort_values([\"p\", \"tims\", \"npk\", \"row\"], ascending=[True, False, False, True]).drop_duplicates(\"p\")\\n    rep = df.row.values; rep_nm = L[\"nm\"][rep]\\n    o = np.argsort(rep_nm, kind=\"mergesort\"); rep = rep[o]\\n    roff, rmz, rit = pv.clean_store(rep, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\\n    return dict(rep=rep, rep_p=L[\"row_p\"][rep], rep_nm=L[\"nm\"][rep], roff=roff, rmz=rmz, rit=rit)\\n\\n\\ndef build_index(L):\\n    ok = np.isfinite(L[\"nm\"]) & (L[\"row_p\"] >= 0)\\n    rows = np.where(ok)[0]\\n    o = np.argsort(L[\"nm\"][rows], kind=\"mergesort\")\\n    I = dict(lib_rows=rows[o], lib_nm=L[\"nm\"][rows[o]])\\n    I[\"all\"] = _reps(L, rows)\\n    I[\"pos\"] = _reps(L, rows[L[\"pol\"][rows] == 1])\\n    I[\"neg\"] = _reps(L, rows[L[\"pol\"][rows] == -1])\\n    log(f\"index: {len(rows):,} library rows; representatives {len(I[\\'all\\'][\\'rep\\']):,}\")\\n    return I\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# per-molecule channels (identical to e02_channels.molecule_channels)\\n# ---------------------------------------------------------------------------------------------\\ndef clean(mz, it):\\n    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\\n                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\\n\\n\\ndef analog_search(R, Q, target):\\n    lo = np.searchsorted(R[\"rep_nm\"], target - CFG.ANALOG_WIN, \"left\")\\n    hi = np.searchsorted(R[\"rep_nm\"], target + CFG.ANALOG_WIN, \"right\")\\n    asim = np.zeros(hi - lo, np.float32)\\n    if hi > lo and Q:\\n        shift = (target - R[\"rep_nm\"][lo:hi]).astype(np.float32)\\n        for qm, qp in Q:\\n            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R[\"roff\"], R[\"rmz\"], R[\"rit\"], CFG.MZ_TOL, shift))\\n    return R[\"rep_p\"][lo:hi], asim\\n\\n\\ndef top_analogs(p, s):\\n    if len(p) == 0: return []\\n    df = pd.DataFrame({\"p\": p, \"s\": s}).groupby(\"p\", sort=False).s.max().sort_values(ascending=False, kind=\"mergesort\")\\n    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))\\n\\n\\ndef molecule_channels(L, I, specs, precs, pols, target):\\n    Q = [clean(a, b) for a, b in specs]\\n    keep = [i for i, (a, _) in enumerate(Q) if len(a)]\\n    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]\\n    tol = target * CFG.PPM_WIN / 1e6\\n    lo = np.searchsorted(I[\"lib_nm\"], target - tol, \"left\"); hi = np.searchsorted(I[\"lib_nm\"], target + tol, \"right\")\\n    crow = I[\"lib_rows\"][lo:hi]\\n    u, inv = np.unique(L[\"row_p\"][crow], return_inverse=True)\\n    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)\\n    for j, (qm, qp) in enumerate(Q):\\n        if len(crow) == 0: break\\n        s = pv.search(qm, qp, crow, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\\n                      CFG.INT_POWER, CFG.ENT_WEIGHT)\\n        np.maximum.at(M[j], inv, s)\\n        sh = (precs[j] - L[\"prec\"][crow]).astype(np.float32)\\n        s2 = pv.search_shift_rows(qm, qp, crow, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\\n                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)\\n        np.maximum.at(MS[j], inv, s2)\\n    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)\\n    z = np.zeros(0, np.float32)\\n    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)\\n    p, s = analog_search(I[\"all\"], Q, target)\\n    ana = top_analogs(p, s)\\n    ps, ss = [], []\\n    for sign, name in ((1, \"pos\"), (-1, \"neg\")):\\n        QQ = [q for q, pl in zip(Q, pols) if pl == sign]\\n        if QQ:\\n            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)\\n    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []\\n    return lib, ana, anapol\\n\\n\\ndef parse_ce(v):\\n    try:\\n        a = np.abs(np.atleast_1d(np.asarray(v, float)))\\n        return float(a.mean()) if len(a) else 25.0\\n    except Exception:\\n        return 25.0\\n\\n\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# ranking features: prvsiyan\\'s 31 + our blocks (identical to e05_rank.py BLOCKS)\\n# ---------------------------------------------------------------------------------------------\\ndef _rel(v):\\n    v = np.asarray(v, np.float32)\\n    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]\\n\\n\\ndef _analog_feats(cfp, ana):\\n    nc = len(cfp)\\n    if not ana: return np.zeros((nc, 5), np.float32)\\n    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)\\n    inter = cf @ afp.T\\n    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)\\n    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)\\n    ap = (tan * (w ** 3)[None, :]).max(1)\\n    ap6 = (tan * (w ** 6)[None, :]).max(1)\\n    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)\\n\\n\\ndef blk_base(r, cfp):\\n    ana = r[\"ana\"]\\n    afp = pool_fps([p for p, _ in ana]) if ana else None\\n    sims = np.array([s for _, s in ana], np.float32)\\n    return pv.rank_features(cfp, r[\"lib\"], afp, sims, r[\"zlog\"], r[\"frag\"])\\n\\n\\ndef blk_mass(r, cfp):\\n    ppm = (POOL[\"mass\"][r[\"cand\"]] - r[\"target\"]) / r[\"target\"] * 1e6\\n    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)\\n\\n\\ndef blk_libx(r, cfp):\\n    cnt = np.log1p(r[\"lib_cnt\"])\\n    return np.column_stack(_rel(r[\"lib_mean\"]) + _rel(r[\"libsh\"]) + [cnt, cnt - cnt.max()]).astype(np.float32)\\n\\n\\ndef blk_anapol(r, cfp):\\n    return _analog_feats(cfp, r[\"anapol\"])\\n\\n\\ndef blk_fragx(r, cfp):\\n    return np.column_stack(_rel(r[\"frag_ad\"]) + _rel(r[\"frag_mean\"]) + [r[\"frag_pol\"]]).astype(np.float32)\\n\\n\\nBLOCKS = {\"base\": blk_base, \"mass\": blk_mass, \"libx\": blk_libx, \"anapol\": blk_anapol, \"fragx\": blk_fragx}\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# run\\n# ---------------------------------------------------------------------------------------------\\ndef compute_channels(L, I, models, te):\\n    mols = list(te.groupby(\"molecule_id\", sort=True))\\n    recs = []\\n    for gi, (mid, sub) in enumerate(mols):\\n        specs = []\\n        for r in sub.itertuples():\\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\\n            k_ = a_ <= float(r.precursor_mz) + 2.0\\n            a_, b_ = a_[k_], b_[k_]\\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\\n        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)\\n        nms = nms[np.isfinite(nms)]\\n        rec = dict(mid=mid, cand=np.zeros(0, np.int64))\\n        if len(nms):\\n            target = float(np.median(nms))\\n            modes = np.where(sub.ionization_mode.astype(str).values == \"positive\", 1.0, -1.0)\\n            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)\\n            cand = pool_window(target, CFG.PPM_WIN)\\n            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)\\n            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)\\n            import pv_fp\\n            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),\\n                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),\\n                                         ces, modes)\\n            u, lmax, lmean, lsh, lcnt = lib\\n            pos = {int(x): i for i, x in enumerate(u)}\\n            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0\\n            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,\\n                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),\\n                       ana=ana, anapol=anapol)\\n            for nm_, arr in ((\"lib\", lmax), (\"lib_mean\", lmean), (\"libsh\", lsh), (\"lib_cnt\", lcnt)):\\n                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v\\n        recs.append(rec)\\n        if gi % 50 == 0: log(f\"  channels {gi}/{len(mols)}\")\\n    return mols, recs\\n\\n\\ndef compute_frag(recs, workers):\\n    from multiprocessing import Pool as MPool\\n    uc = np.unique(np.concatenate([r[\"cand\"] for r in recs]))\\n    log(f\"MetFrag-lite on {len(uc):,} candidates\")\\n    with MPool(workers) as mp:\\n        fr = mp.map(pv.frag_masses_safe, list(POOL[\"smiles\"][uc]), chunksize=16)\\n    fmap = dict(zip(uc.tolist(), fr))\\n    for r in recs:\\n        if not len(r[\"cand\"]): continue\\n        peaks = []\\n        for a_, b_ in r.pop(\"specs\"):\\n            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\\n        F1 = np.zeros((len(r[\"cand\"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\\n        for j, c in enumerate(r[\"cand\"]):\\n            fm = fmap[int(c)]\\n            for k, (x, y) in enumerate(peaks):\\n                F1[j, k] = pv.explain_score(fm, x, y, mode=r[\"mode\"], tol=CFG.MZ_TOL)\\n                F2[j, k] = pv.explain_score_adduct(fm, x, y, r[\"adducts\"][k], tol=CFG.MZ_TOL)\\n                F3[j, k] = pv.explain_score(fm, x, y, mode=r[\"modes\"][k], tol=CFG.MZ_TOL)\\n        r[\"frag\"] = F1.max(1); r[\"frag_ad\"] = F2.max(1); r[\"frag_mean\"] = F3.mean(1); r[\"frag_pol\"] = F3.max(1)\\n\\n\\ndef _hgb(X, Y, W, seed, gbm):\\n    from sklearn.ensemble import HistGradientBoostingClassifier\\n    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)\\n    return m\\n\\n\\ndef fit_rankers(rank_train_path):\\n    \"\"\"Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim).\"\"\"\\n    z = np.load(rank_train_path, allow_pickle=True)\\n    X, Y, S = z[\"X\"], z[\"Y\"], z[\"S\"]\\n    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]\\n    log(f\"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features\")\\n    return rankers, [str(b) for b in z[\"blocks\"]]\\n\\n\\ndef fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):\\n    \"\"\"prvsiyan\\'s ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim).\"\"\"\\n    z = np.load(pv_rank_path)\\n    rankers = [_hgb(z[\"X\"], z[\"Y\"], np.where(z[\"M\"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]\\n    log(f\"prvsiyan ranker: {len(rankers)} GBMs on {z[\\'X\\'].shape[0]:,} rows x {z[\\'X\\'].shape[1]} features\")\\n    return rankers, z[\"X\"].shape[1]\\n\\n\\ndef score_molecules(recs, rankers, blocks, use_fp):\\n    out = {}\\n    for r in recs:\\n        if not len(r[\"cand\"]): continue\\n        rr = r if use_fp else dict(r, zlog=None)\\n        cfp = pool_fps(r[\"cand\"])\\n        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\\n        out[r[\"mid\"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)\\n    return out\\n\\n\\ndef blend_scores(a, b, wa):\\n    \"\"\"Within-molecule rank blend (both inputs are per-candidate probabilities).\"\"\"\\n    out = {}\\n    for mid in a:\\n        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])\\n        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]\\n    return out\\n\\n\\ndef make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):\\n    \"\"\"score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all.\"\"\"\\n    from multiprocessing import Pool as MPool\\n    cand = {r[\"mid\"]: r[\"cand\"] for r in recs if len(r[\"cand\"])}\\n    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind=\"mergesort\")][:topn + 15] for mid, sc in S.items()}\\n               for n, S in score_sets.items()}\\n    allc = [v for o in ordered.values() for v in o.values()]\\n    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)\\n    with MPool(workers) as mp:\\n        ck = mp.map(canon_key, list(POOL[\"smiles\"][short]), chunksize=64)\\n    ckey = dict(zip(short.tolist(), ck))\\n    samp = pd.read_csv(sample_path)\\n    subs = {}\\n    for n, o in ordered.items():\\n        rows = []\\n        for mid, _ in mols:\\n            out, seen = [], set()\\n            for c in o.get(mid, []):\\n                k = ckey.get(int(c)) or POOL[\"keys\"][c]\\n                if k in seen: continue\\n                seen.add(k); out.append(POOL[\"smiles\"][c])\\n                if len(out) == topn: break\\n            while len(out) < topn:\\n                out.append(\"CCO\")\\n            rows.append((mid, \";\".join(out[:topn])))\\n        sub = samp[[\"molecule_id\"]].merge(pd.DataFrame(rows, columns=[\"molecule_id\", \"smiles\"]), on=\"molecule_id\", how=\"left\")\\n        sub[\"smiles\"] = sub[\"smiles\"].fillna(\"CCO\")\\n        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0\\n        assert (sub.smiles.str.split(\";\").map(len) <= topn).all()\\n        subs[n] = sub\\n    return subs\\n\\n\\ndef main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):\\n    global POOL\\n    import torch\\n    import pv_fp\\n    L = load_library(train_path); log(f\"library {len(L[\\'prec\\']):,} spectra\")\\n    POOL = build_pool(L, workers)\\n    L[\"row_p\"] = POOL[\"k2i\"].reindex(L[\"ik\"]).fillna(-1).values.astype(np.int64)\\n    del L[\"ik\"], L[\"smi\"]\\n    I = build_index(L)\\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\\n    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)\\n    log(f\"fp models: {len(single)} single + {len(merged)} merged on {dev}\")\\n    te = pq.read_table(test_path).to_pandas()\\n    if limit:\\n        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]\\n    log(f\"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules\")\\n    mols, recs = compute_channels(L, I, (single, merged, dev), te)\\n    del L, I\\n    compute_frag(recs, workers)\\n    ours, blocks = fit_rankers(our_rank_path)\\n    s_ours = score_molecules(recs, ours, blocks, use_fp=False)\\n    del ours\\n    pvr, nfeat = fit_pv_rankers(pv_rank_path)\\n    s_pv = score_molecules(recs, pvr, [\"base\"], use_fp=True)\\n    del pvr\\n    subs = make_submissions(mols, recs, {\"blend\": blend_scores(s_pv, s_ours, w_pv), \"pv\": s_pv, \"ours\": s_ours},\\n                            sample_path, workers)\\n    log(\"submissions ready\")\\n    return subs, recs, {\"pv\": s_pv, \"ours\": s_ours}\\n'}\nfor _n, _src in ENG_FILES.items():\n    open(os.path.join(ENG_DIR, _n), 'w').write(_src)\nopen(os.path.join(ENG_DIR, 'eng_runner.py'), 'w').write('\"\"\"Our engine (two-ranker, BIO + AFIX) -> {molecule_id: {smiles: [...], keys: [...]}} (blend order, top 40).\"\"\"\\nimport os, sys, json, time\\nimport numpy as np\\nif __name__ == \\'__main__\\':\\n    here = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\\n    test, train, sample, out = sys.argv[1:5]\\n    import casmi_engine as E\\n    _orig_find = E.find\\n    def _find(n):                                  # v4g ships its own fp_bits.npy (10,226 bits): take prvsiyan\\'s 6,930\\n        if n == \\'fp_bits.npy\\':\\n            return os.path.join(os.path.dirname(_orig_find(\\'coco_fp.npy\\')), \\'fp_bits.npy\\')\\n        return _orig_find(n)\\n    E.find = _find\\n    E.RANK.W_A = (0.35, 0.55); E.RANK.SEEDS = (0, 1); E.CFG.N_ANALOG = 200\\n    import glob\\n    fp_models = sorted(p for p in glob.glob(\\'/kaggle/input/**/fp_*.pt\\', recursive=True) if \\'casmi26-fp-models-v2\\' in p)\\n    print(\\'engine fp models\\', fp_models, flush=True)\\n    T0 = time.time()\\n    subs, recs, S = E.main(test, train, sample, E.find(\\'sim_rank_rows_nofp.npz\\'), E.find(\\'rank_train.npz\\'), fp_models,\\n                           workers=os.cpu_count(), w_pv=0.88)\\n    bl = E.blend_scores(S[\\'pv\\'], S[\\'ours\\'], 0.88)\\n    res = {}\\n    for r in recs:\\n        mid = r[\\'mid\\']\\n        if mid not in bl:\\n            continue\\n        order = np.argsort(-bl[mid], kind=\\'mergesort\\')[:80]\\n        smis, keys, seen = [], [], set()\\n        for c in r[\\'cand\\'][order]:\\n            s = E.POOL[\\'smiles\\'][c]; k = E.canon_key(s)\\n            if k is None or k in seen:\\n                continue\\n            seen.add(k); smis.append(s); keys.append(k)\\n            if len(smis) >= 40:\\n                break\\n        res[str(mid)] = dict(smiles=smis, keys=keys)\\n    json.dump(res, open(out, \\'w\\'))\\n    print(\\'engine lists\\', len(res), f\\'{time.time()-T0:.0f}s\\', flush=True)\\n')\nENG = {}\ntry:\n    _t0 = time.time()\n    rr = subprocess.run([sys.executable, os.path.join(ENG_DIR, 'eng_runner.py'), os.path.join(COMP, 'test.parquet'),\n                         os.path.join(COMP, 'train.parquet'), os.path.join(COMP, 'sample_submission.csv'),\n                         '/kaggle/working/eng_lists.json'], capture_output=True, text=True, timeout=4 * 3600, cwd=ENG_DIR, env={**os.environ, 'PYTHONHASHSEED': '0'})\n    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])\n    ENG = json.load(open('/kaggle/working/eng_lists.json'))\n    print('engine lists', len(ENG), f'{time.time()-_t0:.0f}s')\nexcept Exception as e:\n    print('ENGINE FAILED -> v4g lists kept:', repr(e))\n\n\n# ===== cell 23 =====\n'''## 4. PubChem-only channel with a popularity prior\n\nThis channel searches about 105 million PubChem structures that are not in the main candidate pool. Its fingerprint logits combine FPNet A+B with the panel-excluded fold-safe FPNet (normalized fold0 fraction 0.333333). The raw score retains the A+B coefficient for fixed promotion-margin thresholds. Its score combines the spectrum fit with how well documented a structure is.\n\n$$\\\\text{score} = z(f\\\\cdot z) + 0.15\\\\,\\\\big(\\\\log(1+\\\\#\\\\text{PubChem substances}) + \\\\log(1+\\\\#\\\\text{PubMed papers})\\\\big)$$\n\nThe 200 most documented structures in the mass window are always added to the shortlist. A gate later decides how many PubChem candidates enter the final list. This prior added +0.004 on the leaderboard.\n'''\n\n\n# ===== cell 24 =====\nV17_PATCH = '\\n\\n# ---- v17 patch: popularity prior (log1p SID + log1p PMID from PubChem, mapped onto this store by stereo-stripped SMILES) ----\\n_POP_DIR = os.environ.get(\\'CASMI_POP_DIR\\'); _POP_LAM = float(os.environ.get(\\'CASMI_POP_LAM\\', \\'0\\')); _POP_UNION = int(os.environ.get(\\'CASMI_POP_UNION\\', \\'0\\'))\\n_SRC_BONUS = float(os.environ.get(\\'CASMI_SRC_BONUS\\', \\'0\\')); _SRC_BITS = int(os.environ.get(\\'CASMI_SRC_BITS\\', \\'0\\'))   # v5 depositor-source bonus\\n_NP_BONUS = float(os.environ.get(\\'CASMI_NP_BONUS\\', \\'0\\'))   # natural-product membership bonus (data notebook v5)\\n_LOTUS_W = float(os.environ.get(\\'CASMI_LOTUS_W\\', \\'0\\'))   # pre-aligned literature prior\\n_SYN_W = 0.5\\n_PARENT_PENALTY = 0.0\\n_AMBIGUITY = False\\n_init_worker0 = init_worker\\n\\n\\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\\n    _init_worker0(pc_dir, bits_path, pool_meta_path, code_dir)\\n    if _POP_DIR:\\n        _n = len(_W[\\'mass\\'])\\n        if _SYN_W:\\n            _W[\\'syn\\'] = np.load(os.path.join(_POP_DIR, \\'pc_lsynonyms.npy\\'), mmap_mode=\\'r\\')\\n            assert _W[\\'syn\\'].shape == (_n,), \\'Synonym tier-row alignment differs\\'\\n        if _PARENT_PENALTY:\\n            _W[\\'parent_rows\\'] = np.load(os.path.join(_POP_DIR, \\'pc_different_parent_rows.npy\\'), mmap_mode=\\'r\\')\\n            assert _W[\\'parent_rows\\'].ndim == 1\\n            assert not len(_W[\\'parent_rows\\']) or int(_W[\\'parent_rows\\'][-1]) < _n\\n        if _AMBIGUITY:\\n            _W[\\'formula_count\\'] = np.load(os.path.join(_POP_DIR, \\'pc_formula_isomers.npy\\'), mmap_mode=\\'r\\')\\n            assert _W[\\'formula_count\\'].shape == (_n,), \\'Formula tier-row alignment differs\\'\\n        _W[\\'ls\\'] = np.load(os.path.join(_POP_DIR, \\'pc_lsid.npy\\'), mmap_mode=\\'r\\')\\n        _W[\\'lp\\'] = np.load(os.path.join(_POP_DIR, \\'pc_lpmid.npy\\'), mmap_mode=\\'r\\')\\n        if _SRC_BONUS > 0 and _SRC_BITS:\\n            _W[\\'sm\\'] = np.load(os.path.join(_POP_DIR, \\'pc_src_mask.npy\\'), mmap_mode=\\'r\\')\\n        if _NP_BONUS > 0:\\n            _W[\\'np\\'] = np.load(os.path.join(_POP_DIR, \\'pc_np.npy\\'), mmap_mode=\\'r\\')\\n        if _LOTUS_W > 0:\\n            import sys as _sys\\n            _sys.path.insert(0, _POP_DIR)\\n            from aligned_public_priors import load_lotus_refs, lotus_log_prior\\n            _W[\\'lotus\\'] = load_lotus_refs(_POP_DIR, tier_rows=len(_W[\\'mass\\']))\\n            _W[\\'lotus_log\\'] = lotus_log_prior\\n\\n\\ndef probe_one(task):\\n    mid, target, z = task\\n    t0 = time.time()\\n    chem = _W[\\'chem\\']; Chem = _W[\\'Chem\\']; gen = _W[\\'ecfp4\\']\\n    tol = float(target) * PPM * 1e-6\\n    smis, window_priors = _window_bundle(float(target))\\n    n = len(smis)\\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0, fz_top=None)\\n    if n == 0:\\n        diag[\\'secs\\'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    raw_e = _W[\\'raw_e\\']; z_e = z[_W[\\'sel_e\\']].astype(np.float32)\\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\\n    for i, s in enumerate(smis):\\n        m = Chem.MolFromSmiles(s)\\n        if m is None:\\n            continue\\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\\n    sc1 = E @ z_e\\n    sc1[~ok] = -np.inf\\n    n1 = min(PC_N1, int(ok.sum()))\\n    diag[\\'n_pass1\\'] = n1\\n    if n1 == 0:\\n        diag[\\'secs\\'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\\n    use_pop = (\\'ls\\' in _W) and _POP_LAM > 0\\n    pop = (np.asarray(window_priors[\\'ls\\'], np.float32) + np.asarray(window_priors[\\'lp\\'], np.float32)) if use_pop else np.zeros(n, np.float32)\\n    if use_pop and \\'sm\\' in _W:\\n        pop = pop + _SRC_BONUS * ((np.asarray(window_priors[\\'sm\\']) & _SRC_BITS) != 0).astype(np.float32)\\n    if use_pop and \\'np\\' in _W:\\n        pop = pop + _NP_BONUS * (np.asarray(window_priors[\\'np\\']) != 0).astype(np.float32)\\n    if use_pop and \\'lotus\\' in _W:\\n        _lr = _W[\\'lotus_log\\'](window_priors[\\'lotus\\'])\\n        pop = pop + _LOTUS_W * _lr\\n    if use_pop and \\'syn\\' in _W:\\n        pop = pop + _SYN_W * np.asarray(window_priors[\\'syn\\'], np.float32)\\n    if use_pop and \\'parent_rows\\' in _W:\\n        _flags = np.zeros(n, dtype=bool)\\n        pop = pop - _PARENT_PENALTY * _flags.astype(np.float32)\\n    if use_pop and _POP_UNION > 0:\\n        pp = np.where(ok, pop, -np.inf); ku = min(_POP_UNION, int(ok.sum()))\\n        top1 = np.union1d(top1, np.argpartition(-pp, ku - 1)[:ku])\\n    bits = _W[\\'bits\\']\\n    fz, idx = [], []\\n    for i in top1:\\n        fp = chem.raw_fingerprint(smis[i])\\n        if fp is None:\\n            continue\\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\\n    diag[\\'n_pass2\\'] = len(idx)\\n    if not idx:\\n        diag[\\'secs\\'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    fz = np.asarray(fz, np.float64); idx = np.asarray(idx)\\n    keyc = {}\\n    def key(j):\\n        if j not in keyc:\\n            keyc[j] = chem.score_key(smis[idx[j]]); diag[\\'n_keyed\\'] += 1\\n        return keyc[j]\\n    for j in np.argsort(-fz, kind=\\'stable\\'):        # gate quantity = v16\\'s pc_fz[0] (best non-pool structure by raw f.z)\\n        k = key(j)\\n        if k is not None and k not in _W[\\'pool_keys\\']:\\n            diag[\\'fz_top\\'] = float(fz[j]); break\\n    zf = (fz - fz.mean()) / (fz.std() + 1e-9)\\n    _lam = _POP_LAM\\n    if use_pop and \\'formula_count\\' in _W:\\n        _count = np.maximum(np.asarray(np.ones(n, dtype=np.uint16), np.float64)[idx], 1.0)\\n        _lam = _POP_LAM * np.clip(np.log1p(_count) / np.log1p(1000.0), 0.5, 1.5)\\n    sc = zf + _lam * pop[idx]\\n    order = np.argsort(-sc, kind=\\'stable\\')\\n    promotion_evidence = {}\\n    out, out_s, out_k, out_p, out_sc, seen = [], [], [], [], [], set()\\n    for j in order:\\n        k = key(j)\\n        if k is None or k in seen:\\n            continue\\n        seen.add(k)\\n        if k in _W[\\'pool_keys\\']:\\n            diag[\\'n_in_pool\\'] += 1\\n            continue\\n        out.append(smis[idx[j]]); out_s.append(float(sc[j])); out_k.append(k); out_p.append(float(pop[idx[j]])); out_sc.append(float(sc[j]))\\n        promotion_evidence[k] = dict(smiles=smis[idx[j]], raw_fz=float(fz[j]),\\n                                     S=float(sc[j]), pop=float(pop[idx[j]]))\\n        if len(out) >= K:\\n            break\\n    diag[\\'n_listed\\'] = len(out)\\n    diag[\\'promotion_evidence\\'] = promotion_evidence\\n    diag[\\'S\\'] = float(out_sc[0]) if out_sc else 0.0\\n    diag[\\'top_pop\\'] = float(out_p[0]) if out_p else 0.0\\n    diag[\\'secs\\'] = time.time() - t0\\n    return mid, out, out_s, out_k, diag\\n\\n\\n\"\"\"Appended to the current embedded PubChem module; no full-tier reordering.\"\"\"\\n_init_worker_single_store = init_worker\\n\\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\\n    _init_worker_single_store(pc_dir, bits_path, pool_meta_path, code_dir)\\n    assert _POP_DIR, \\'Aligned popularity dataset required for PubChem store union\\'\\n    assert not _AMBIGUITY and not _PARENT_PENALTY, \\'Supplementary formula/parent priors have not been measured\\'\\n    base = dict(name=\\'tier\\', mass=_W[\\'mass\\'], off=_W[\\'off\\'], buf=_W[\\'buf\\'])\\n    for key in (\\'ls\\', \\'lp\\', \\'sm\\', \\'np\\', \\'lotus\\', \\'syn\\'):\\n        if key in _W:\\n            assert _W[key].shape == (len(base[\\'mass\\']),), (key, \\'tier-row alignment\\')\\n            base[key] = _W[key]\\n    _W[\\'pc_stores\\'] = [base]\\n    for prefix in (\\'delta_pc\\', \\'growth_pc\\'):\\n        mass_path = os.path.join(_POP_DIR, prefix + \\'_mass.npy\\')\\n        if not os.path.isfile(mass_path):\\n            continue\\n        store = dict(name=prefix)\\n        for key, suffix in [(\\'mass\\',\\'mass\\'), (\\'off\\',\\'off\\'), (\\'buf\\',\\'smiles\\'), (\\'ls\\',\\'lsid\\'), (\\'lp\\',\\'lpmid\\'),\\n                            (\\'sm\\',\\'src_mask\\'), (\\'np\\',\\'np\\'), (\\'lotus\\',\\'lotus_refs\\'), (\\'syn\\',\\'lsynonyms\\'),\\n                            (\\'src_observed\\',\\'src_observed\\')]:\\n            path = os.path.join(_POP_DIR, prefix + \\'_\\' + suffix + \\'.npy\\')\\n            assert os.path.isfile(path), \\'Missing aligned supplementary prior: \\' + path\\n            store[key] = np.load(path, mmap_mode=\\'r\\')\\n        n = len(store[\\'mass\\'])\\n        assert store[\\'off\\'].shape == (n + 1,) and store[\\'buf\\'].ndim == 1\\n        assert int(store[\\'off\\'][0]) == 0 and int(store[\\'off\\'][-1]) == len(store[\\'buf\\'])\\n        assert np.all(store[\\'mass\\'][1:] >= store[\\'mass\\'][:-1]), prefix + \\' masses are not sorted\\'\\n        for key in (\\'ls\\',\\'lp\\',\\'sm\\',\\'np\\',\\'lotus\\',\\'syn\\',\\'src_observed\\'):\\n            assert store[key].shape == (n,), (prefix, key, \\'row alignment\\')\\n        _W[\\'pc_stores\\'].append(store)\\n\\ndef _window_bundle(target):\\n    \"\"\"Return SMILES and row-aligned priors gathered from independent sorted stores.\"\"\"\\n    target = float(target)\\n    tol = target * PPM * 1e-6\\n    smis, parts, counts = [], {key: [] for key in (\\'ls\\',\\'lp\\',\\'sm\\',\\'np\\',\\'lotus\\',\\'syn\\',\\'src_observed\\')}, {}\\n    for store in _W[\\'pc_stores\\']:\\n        a = int(np.searchsorted(store[\\'mass\\'], target - tol, \\'left\\'))\\n        b = int(np.searchsorted(store[\\'mass\\'], target + tol, \\'right\\'))\\n        counts[store[\\'name\\']] = b - a\\n        if b <= a:\\n            continue\\n        off = np.asarray(store[\\'off\\'][a:b + 1], np.int64)\\n        raw = bytes(store[\\'buf\\'][off[0]:off[-1]])\\n        first = int(off[0])\\n        smis.extend(raw[int(off[i])-first:int(off[i+1])-first].decode(\\'ascii\\') for i in range(b-a))\\n        for key in parts:\\n            if key in store:\\n                values = np.asarray(store[key][a:b])\\n            else:\\n                values = np.full(b-a, 1 if key == \\'src_observed\\' else 0, np.float32)\\n            assert values.shape == (b-a,), (store[\\'name\\'], key)\\n            parts[key].append(values)\\n    priors = {key: np.concatenate(values) if values else np.zeros(0, np.float32) for key, values in parts.items()}\\n    assert all(len(values) == len(smis) for values in priors.values())\\n    _W[\\'last_window_counts\\'] = counts\\n    return smis, priors\\n'\n'''PubChem-only channel in its own process (S1 code snapshot + FPNet A+B of casmi26-v3-models) -- identical to v3.6'''\nCORE = '\"\"\"P_pubchem probe core (lever 3). Embedded verbatim into the Kaggle notebook (written to /kaggle/working and imported,\\nso worker processes can import it) and imported directly by the local smoke test.\\n\\nPer molecule:\\n  1. f.z logits of the A+B FPNet ensemble, exactly like Engine.model_logits: z = 0.5 * (mean per-spectrum logits +\\n     mean per-polarity merged-spectrum logits); models averaged inside ModelBank.\\n  2. PubChem window: +-10 ppm around the target neutral mass (median over the molecule\\'s spectra), no subsampling.\\n  3. pass 1 (src/casmi/pubchem.py tier): partial f.z on the ECFP4 block of the selected bits, keep top PC_N1 = 1000.\\n  4. pass 2: full selected-bit fingerprints, full f.z, sort descending.\\n  5. walk down the f.z order, compute the metric score key (tautomer-canonical InChIKey14); keep a structure only if\\n     its key is NOT in the candidate pool (train U COCONUT) and not already listed; stop at 25.\\n  An empty list becomes the single \\'CCO\\' fallback (never padded).\\nReading: sum of reciprocal ranks of PubChem-only truths / ~130 public molecules.\\n\"\"\"\\nfrom __future__ import annotations\\n\\nimport os\\nimport time\\n\\nimport numpy as np\\n\\nPC_N1 = 5000\\nPPM = 10.0\\nK = 25\\n\\n_W = {}\\n\\n\\n# ------------------------------------------------------------------------------------------ model part (main process)\\ndef molecule_logits(bank, spectra):\\n    \"\"\"spectra: list of dict(mz, it, mode, adduct, prec, ce, ce_n). Returns z_avg (nbits,) float32 or None.\"\"\"\\n    from casmi import chem, fpnet\\n    from casmi.spectra import merge_spectra\\n    items_s = []\\n    for s in spectra:\\n        pm, pi = fpnet.prep_peaks(s[\\'mz\\'], s[\\'it\\'], s[\\'prec\\'])\\n        ce_ok = s[\\'ce\\'] is not None and not np.isnan(s[\\'ce\\'])\\n        items_s.append(dict(mz=pm, it=pi, prec=float(s[\\'prec\\']), adduct_ix=chem.adduct_index(s[\\'adduct\\']),\\n                            ce=float(s[\\'ce\\']) if ce_ok else 0.0, ce_known=1.0 if ce_ok else 0.0,\\n                            n_merged=min(int(s.get(\\'ce_n\\', 1)), 8), mode=float(s[\\'mode\\'])))\\n    items_m = []\\n    for md in (1, -1):\\n        grp = [s for s in spectra if s[\\'mode\\'] == md]\\n        if not grp:\\n            continue\\n        mm, ii = merge_spectra([(s[\\'mz\\'], s[\\'it\\']) for s in grp])\\n        prec = float(np.median([s[\\'prec\\'] for s in grp]))\\n        pm, pi = fpnet.prep_peaks(mm, ii, prec)\\n        ces = [s[\\'ce\\'] for s in grp if s[\\'ce\\'] is not None and not np.isnan(s[\\'ce\\'])]\\n        adducts = [s[\\'adduct\\'] for s in grp]\\n        ad = max(adducts, key=adducts.count)          # deterministic tie-break (first occurrence)\\n        items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad),\\n                            ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\\n                            n_merged=min(sum(max(1, int(s.get(\\'ce_n\\', 1))) for s in grp), 8), mode=float(md)))\\n    if not items_s:\\n        return None\\n    zs, _ = bank.logits_el(items_s)\\n    zm, _ = bank.logits_el(items_m)\\n    # 1.5 * mean(A, B, fold0) = mean(A, B) + 0.5 * fold0.\\n    # Positive scaling preserves shortlist/order/z-scores while retaining the A+B coefficient in raw-margin gates.\\n    return (1.5 * 0.5 * (zs.mean(0) + zm.mean(0))).astype(np.float32)\\n\\n\\n# ------------------------------------------------------------------------------------------ PubChem part (workers)\\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\\n    import sys\\n    if code_dir and code_dir not in sys.path:\\n        sys.path.insert(0, code_dir)\\n    import pandas as pd\\n    from casmi import chem\\n    _W[\\'chem\\'] = chem\\n    _W[\\'mass\\'] = np.load(os.path.join(pc_dir, \\'pc_mass.npy\\'), mmap_mode=\\'r\\')\\n    _W[\\'off\\'] = np.load(os.path.join(pc_dir, \\'pc_off.npy\\'), mmap_mode=\\'r\\')\\n    _W[\\'buf\\'] = np.load(os.path.join(pc_dir, \\'pc_smiles.npy\\'), mmap_mode=\\'r\\')\\n    bits = np.load(bits_path)\\n    _W[\\'bits\\'] = bits\\n    _W[\\'sel_e\\'] = np.where(bits < 4096)[0]\\n    _W[\\'raw_e\\'] = bits[_W[\\'sel_e\\']]\\n    _W[\\'pool_keys\\'] = set(pd.read_parquet(pool_meta_path, columns=[\\'key\\']).key.values.tolist())\\n    r = chem._rdkit()\\n    _W[\\'ecfp4\\'] = r[\\'gen\\'].GetMorganGenerator(radius=2, fpSize=4096)\\n    _W[\\'Chem\\'] = r[\\'Chem\\']\\n\\n\\ndef _window(target):\\n    tol = target * PPM * 1e-6\\n    a = int(np.searchsorted(_W[\\'mass\\'], target - tol, \\'left\\')); b = int(np.searchsorted(_W[\\'mass\\'], target + tol, \\'right\\'))\\n    off = np.asarray(_W[\\'off\\'][a:b + 1]).astype(np.int64)\\n    if b <= a:\\n        return []\\n    raw = bytes(_W[\\'buf\\'][off[0]:off[-1]])\\n    base = off[0]\\n    return [raw[off[i] - base:off[i + 1] - base].decode(\\'ascii\\') for i in range(b - a)]\\n\\n\\ndef probe_one(task):\\n    \"\"\"task = (molecule_id, target_mass, z (nbits,) float32). Returns (molecule_id, [smiles], [f.z], [score keys], diag dict).\"\"\"\\n    mid, target, z = task\\n    t0 = time.time()\\n    chem = _W[\\'chem\\']; Chem = _W[\\'Chem\\']; gen = _W[\\'ecfp4\\']\\n    smis = _window(float(target))\\n    n = len(smis)\\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0)\\n    if n == 0:\\n        diag[\\'secs\\'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    raw_e = _W[\\'raw_e\\']; z_e = z[_W[\\'sel_e\\']].astype(np.float32)\\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\\n    for i, s in enumerate(smis):\\n        m = Chem.MolFromSmiles(s)\\n        if m is None:\\n            continue\\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\\n    sc1 = E @ z_e\\n    sc1[~ok] = -np.inf\\n    n1 = min(PC_N1, int(ok.sum()))\\n    diag[\\'n_pass1\\'] = n1\\n    if n1 == 0:\\n        diag[\\'secs\\'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\\n    bits = _W[\\'bits\\']\\n    fz, idx = [], []\\n    for i in top1:\\n        fp = chem.raw_fingerprint(smis[i])\\n        if fp is None:\\n            continue\\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\\n    diag[\\'n_pass2\\'] = len(idx)\\n    order = np.argsort(-np.asarray(fz), kind=\\'stable\\')\\n    out, out_fz, out_k, seen = [], [], [], set()\\n    for o in order:\\n        s = smis[idx[o]]\\n        k = chem.score_key(s)\\n        diag[\\'n_keyed\\'] += 1\\n        if k is None or k in seen:\\n            continue\\n        seen.add(k)\\n        if k in _W[\\'pool_keys\\']:\\n            diag[\\'n_in_pool\\'] += 1\\n            continue\\n        out.append(s); out_fz.append(float(fz[o])); out_k.append(k)\\n        if len(out) >= K:\\n            break\\n    diag[\\'n_listed\\'] = len(out)\\n    diag[\\'secs\\'] = time.time() - t0\\n    return mid, out, out_fz, out_k, diag\\n'\nRUNNER = '\"\"\"PubChem-only channel for the final submission (runs in its own process so its casmi package\\n(S1 snapshot, FPNet A+B) never collides with the v1 engine\\'s casmi package).\\n\\nPer molecule: z = A+B logits (probe_core2.molecule_logits, identical to the probe that scored 0.037);\\nbest_pool_fz = max f.z over the pool window (+-10 ppm, fallback 30 ppm) with the same z (as P1 logits.py);\\nPubChem-only list = probe_core2.probe_one (top 25 by f.z, keys not in the pool) with f.z values and score keys.\\nWrites a json {molecule_id: {pc, pc_fz, pc_keys, best_pool_fz, target}}.\\nusage: python pc_runner.py <v3_code_dir> <v3_models_dir> <pool_dir> <pubchem_dir> <test_parquet> <out_json> <n_workers> [max_mols]\\n\"\"\"\\nimport os, sys, json, time\\nimport numpy as np, pandas as pd\\n\\nif __name__ == \\'__main__\\':\\n    V3CODE, V3MOD, POOL_DIR, PC_DIR, TEST, OUT, NW = sys.argv[1:8]\\n    MAXM = int(sys.argv[8]) if len(sys.argv) > 8 else 0\\n    NW = int(NW)\\n    sys.path.insert(0, V3CODE); sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\\n    import torch, glob\\n    import probe_core2 as pc\\n    from casmi import fpnet, chem\\n    T0 = time.time()\\n    models = sorted(glob.glob(os.path.join(V3MOD, \\'fpnet_*.pt\\')))\\n    assert len(models) == 2, models\\n    fold0 = os.environ[\\'CASMI_FPNET_FOLD0\\']\\n    assert os.path.isfile(fold0), fold0\\n    dev = \\'cuda\\' if torch.cuda.is_available() else \\'cpu\\'\\n    bank = fpnet.ModelBank(models + [fold0], device=dev)\\n    bits = np.load(os.path.join(POOL_DIR, \\'fp_bits.npy\\')); nbits = len(bits)\\n    assert bank.nbits == nbits, (bank.nbits, nbits)\\n    assert np.array_equal(bits, np.load(os.path.join(os.path.dirname(fold0), \\'fp_bits.npy\\'))), \\'Fingerprint bit order differs\\'\\n    pmass = pd.read_parquet(os.path.join(POOL_DIR, \\'pool_meta.parquet\\'), columns=[\\'mass\\']).mass.values.astype(np.float64)\\n    assert np.all(np.diff(pmass) >= 0), \\'pool not mass-sorted\\'\\n    pfp = np.load(os.path.join(POOL_DIR, \\'pool_fp.npy\\'), mmap_mode=\\'r\\')\\n    te = pd.read_parquet(TEST)\\n    te[\\'nm\\'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\\n    tasks, extra = [], {}\\n    groups = list(te.groupby(\\'molecule_id\\', sort=False))\\n    if MAXM:\\n        groups = groups[:MAXM]\\n    for mid, sub in groups:\\n        nms = sub.nm.values[np.isfinite(sub.nm.values)]\\n        if not len(nms):\\n            continue\\n        target = float(np.median(nms))\\n        spectra = []\\n        for r in sub.itertuples():\\n            ce = r.collision_energy_ev\\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\\n                                mode=1 if r.ionization_mode == \\'positive\\' else -1, adduct=r.adduct,\\n                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,\\n                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\\n        spectra = spectra[:16]\\n        try:\\n            z = pc.molecule_logits(bank, spectra)\\n        except Exception as e:\\n            print(\\'ERROR logits\\', mid, repr(e), flush=True); z = None\\n        if z is None:\\n            continue\\n        tol = target * 10e-6\\n        a = np.searchsorted(pmass, target - tol, \\'left\\'); b = np.searchsorted(pmass, target + tol, \\'right\\')\\n        if b <= a:\\n            tol = target * 30e-6\\n            a = np.searchsorted(pmass, target - tol, \\'left\\'); b = np.searchsorted(pmass, target + tol, \\'right\\')\\n        if b > a:\\n            F = np.unpackbits(np.asarray(pfp[a:b]), axis=1)[:, :nbits].astype(np.float64)\\n            best = float((F @ z.astype(np.float64)).max())\\n        else:\\n            best = float(\\'nan\\')\\n        extra[mid] = dict(best_pool_fz=best, target=target)\\n        tasks.append((mid, target, z))\\n    del bank; torch.cuda.empty_cache()\\n    print(f\\'logits for {len(tasks)} molecules {time.time()-T0:.0f}s\\', flush=True)\\n    res = {}\\n    import multiprocessing as mp\\n    ctx = mp.get_context(\\'fork\\' if sys.platform != \\'win32\\' else \\'spawn\\')\\n    with ctx.Pool(NW, initializer=pc.init_worker, initargs=(PC_DIR, os.path.join(POOL_DIR, \\'fp_bits.npy\\'),\\n                                                            os.path.join(POOL_DIR, \\'pool_meta.parquet\\'), V3CODE)) as P:\\n        for k, (mid, smis, fzs, keys, d) in enumerate(P.imap_unordered(pc.probe_one, tasks, chunksize=1)):\\n            res[mid] = dict(pc=smis, pc_fz=fzs, pc_keys=keys, fz_top=d.get(\\'fz_top\\'), S=d.get(\\'S\\'), top_pop=d.get(\\'top_pop\\', 0.0), promotion_evidence=d.get(\\'promotion_evidence\\', {}), **extra[mid])\\n            if (k + 1) % 25 == 0 or k + 1 == len(tasks):\\n                print(f\\'  pubchem {k+1}/{len(tasks)} {time.time()-T0:.0f}s\\', flush=True)\\n    json.dump(res, open(OUT, \\'w\\'))\\n    print(\\'pc_runner done\\', len(res), f\\'{time.time()-T0:.0f}s\\', flush=True)\\n'\nopen('/kaggle/working/probe_core2.py', 'w').write(CORE + V17_PATCH)\nopen('/kaggle/working/pc_runner.py', 'w').write(RUNNER)\nprint('v17 patch appended to probe_core2:', 'def probe_one' in V17_PATCH)\nPC = {}\nPath('/kaggle/working/pc_lists.json').unlink(missing_ok=True)\ntry:\n    rr = subprocess.run([sys.executable, '/kaggle/working/pc_runner.py', V3CODE, V3MOD, POOL_DIR, PC_DIR,\n                         os.path.join(COMP, 'test.parquet'), '/kaggle/working/pc_lists.json', '4'],\n                        capture_output=True, text=True, timeout=4 * 3600, env={**os.environ, 'PYTHONHASHSEED': '0'})\n    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])\n    rr.check_returncode()\n    PC = json.load(open('/kaggle/working/pc_lists.json'))\nexcept Exception as e:\n    print('PUBCHEM CHANNEL FAILED -> base lists only:', repr(e))\nprint('pubchem lists', len(PC), f'{time.time()-T0:.0f}s')\n\n\n# ===== cell 25 =====\n'''## 5. Engine 1: Ahmed Berat \u00d6zer's v4n\n\nThis step builds the spectral library from `train.parquet` and links the precomputed candidate pool: training structures plus COCONUT, about 711k in total. The pool comes with FPNet fingerprints and fragment tables.\n'''\n\n\n# ===== cell 26 =====\n'''v1 engine code (the 0.358 model's engine) + fe_v4; pool symlinks; spectrum cache from train.parquet'''\nsys.path.insert(0, V4CODE)\nos.makedirs('work', exist_ok=True)\nfor f in ['pool_meta.parquet', 'pool_fp.npy', 'pool_frag_off.npy', 'pool_frag_mass.npy', 'fp_bits.npy', 'train_structs.parquet', 'train_fp_sel.npy']:\n    src = os.path.join(POOL_DIR, f)\n    dst = f'work/{f}'\n    if not os.path.exists(dst):\n        os.symlink(src, dst)\nfrom casmi.build import build_spec_cache\nbuild_spec_cache(os.path.join(COMP, 'train.parquet'), 'work')\n\n\"\"\"Notebook-embedded, trained-SID-only extension. No runtime structure chemistry.\"\"\"\n\ndef _build_external_reference_cache(base, table, pins):\n    import hashlib, os, shutil, tempfile\n    from pathlib import Path\n    import numpy as np\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    from casmi import chem\n    from casmi.build import LIB_IX, instr_family\n    from casmi.spectra import clean_peaks\n\n    def check(ok, reason):\n        if not ok:\n            raise ValueError(reason)\n\n    def digest(path):\n        h = hashlib.sha256()\n        with open(path, 'rb') as f:\n            for block in iter(lambda: f.read(4 << 20), b''):\n                h.update(block)\n        return h.hexdigest()\n\n    target = None\n    try:\n        check(table is not None, 'knowledge table unavailable')\n        source = Path(table) / 'external_reference_spectra'\n        base = Path(base)\n        for name, expected in pins['files'].items():\n            path = source / name\n            check(path.is_file() and path.stat().st_size <= 32 << 20, 'external file unavailable or too large')\n            check(digest(path) == expected, 'external file checksum differs')\n        for name, expected in pins['base_files'].items():\n            check(digest(base / name) == expected, 'training structure/pool version differs')\n        links = pq.read_table(source / 'external_scorer_sid_links.parquet').to_pylist()\n        check(len(links) == pins['keys'], 'link count differs')\n        k2link = {r['scorer_key']: r for r in links}\n        check(len(k2link) == len(links), 'duplicate scorer links')\n        quality = pq.read_table(source / 'external_record_quality.parquet').to_pylist()\n        pf = pq.ParquetFile(source / 'external_delta.parquet')\n        check(pf.metadata.num_rows == len(quality) == pins['delta_rows'], 'delta/quality length differs')\n        off = np.load(base / 'spec_off.npy', mmap_mode='r')\n        mz = np.load(base / 'spec_mz.npy', mmap_mode='r')\n        it = np.load(base / 'spec_it.npy', mmap_mode='r')\n        meta = pq.ParquetFile(base / 'spec_meta.parquet')\n        n = meta.metadata.num_rows\n        check(off.dtype == np.int64 and mz.dtype == it.dtype == np.float32, 'CSR dtype differs')\n        check(len(off) == n + 1 and off[0] == 0 and off[-1] == len(mz) == len(it), 'CSR size differs')\n        for start in range(0, n, 1 << 18):\n            check(np.all(np.diff(off[start:min(n, start + (1 << 18)) + 1]) >= 0), 'CSR offsets decrease')\n        for start in range(0, len(mz), 1 << 20):\n            a, b = mz[start:start + (1 << 20)], it[start:start + (1 << 20)]\n            check(np.all(np.isfinite(a)) and np.all(np.isfinite(b)) and np.all(a > 0) and np.all(b >= 0), 'invalid base peaks')\n        required = {'sid','lib','instr','adduct_ix','mode','precursor_mz','n_clean','nm','adduct',\n                    'precursor_error_ppm','collision_energy_orig_units','base_peak_intensity','num_peaks',\n                    'ce_mean','ce_min','ce_max','ce_n'}\n        check(set(meta.schema_arrow.names) == required, 'cache metadata schema differs')\n        meta_row = 0\n        for batch in meta.iter_batches(batch_size=65536, columns=['sid','n_clean']):\n            sids = np.asarray(batch.column(0))\n            check(np.all((sids >= 0) & (sids < pins['structure_rows'])), 'base SID out of range')\n            check(np.array_equal(np.asarray(batch.column(1)), np.diff(off[meta_row:meta_row + len(sids) + 1])), 'base peak counts differ')\n            meta_row += len(sids)\n        extra_meta, extra_mz, extra_it, used_keys = [], [], [], set()\n        sources = {'MassBank':'massbank', 'MoNA':'mona', 'GNPS':'gnps', 'MSnLib':'pluskal_ms2'}\n        row = 0\n        for batch in pf.iter_batches(batch_size=128):\n            for r in batch.to_pylist():\n                q = quality[row]\n                check(q['delta_row'] == row and all(q[c] == r[c] for c in ['source','record_id','scorer_key','spectrum_sha256','licence']), 'quality join differs')\n                row += 1\n                link = k2link.get(r['scorer_key'])\n                check(link is not None, 'scorer link missing')\n                if link['sid'] is None or not r['in_train_structure'] or q['mass_check_status'] != 'within_diagnostic_threshold':\n                    continue\n                check(0 <= link['sid'] < pins['structure_rows'] and r['formal_charge'] == 0, 'invalid linked SID/charge')\n                check(abs(r['neutral_mono_mass'] - link['mass']) < 1e-5, 'linked structure mass differs')\n                check(r['adduct'] in chem.ADDUCTS and r['ionization_mode'] in ['positive','negative'], 'unsupported ion metadata')\n                check(bool(r['licence']), 'record licence missing')\n                mode = 1 if r['ionization_mode'] == 'positive' else -1\n                if chem.ADDUCTS[r['adduct']][3] != mode:\n                    continue\n                x, y = np.asarray(r['ms2_mzs'], np.float64), np.asarray(r['ms2_normalized_intensities'], np.float64)\n                check(len(x) == len(y) and np.all(np.isfinite(x)) and np.all(np.isfinite(y)) and np.all(x > 0) and np.all(y >= 0), 'invalid external peaks')\n                prec = float(r['precursor_mz'])\n                check(np.isfinite(prec) and prec > 0, 'invalid precursor')\n                x, y = clean_peaks(x, y, prec, 0.001, 512, 2.0)\n                if not len(x):\n                    continue\n                check(np.all(np.diff(x) >= 0) and np.all(np.isfinite(y)) and len(x) <= 512, 'cleaned peaks invalid')\n                ce = r['collision_energy_ev']\n                known_ce = ce is not None and np.isfinite(ce) and r['collision_energy_units'] == 'eV'\n                ce = float(ce) if known_ce else None\n                d = dict(adduct=r['adduct'], precursor_mz=prec, precursor_error_ppm=None,\n                         collision_energy_orig_units=r['collision_energy_units'], base_peak_intensity=None,\n                         num_peaks=len(r['ms2_mzs']), ce_mean=ce, ce_min=ce, ce_max=ce,\n                         ce_n=1 if known_ce else 0, sid=int(link['sid']), lib=LIB_IX[sources[r['source']]],\n                         instr=instr_family(r['instrument_type']), adduct_ix=chem.adduct_index(r['adduct']),\n                         mode=mode, nm=float(chem.neutral_mass([prec], [r['adduct']])[0]), n_clean=len(x))\n                extra_meta.append(d); extra_mz.append(x); extra_it.append(y); used_keys.add(r['scorer_key'])\n        check(len(extra_meta) == pins['added_rows'] and len(used_keys) == pins['added_keys'], 'eligible extension count differs')\n        check(len(extra_meta) > 0, 'no eligible external references')\n        target = Path(tempfile.mkdtemp(prefix='external_reference_', dir=base.parent))\n        os.symlink((base / 'train_structs.parquet').resolve(), target / 'train_structs.parquet')\n        counts = np.array([len(x) for x in extra_mz], np.int64)\n        eo = np.r_[0, np.cumsum(counts)]\n        for name, original, additions in [('spec_mz.npy', mz, extra_mz), ('spec_it.npy', it, extra_it)]:\n            out = np.lib.format.open_memmap(target / name, mode='w+', dtype=np.float32, shape=(len(original) + int(eo[-1]),))\n            for start in range(0, len(original), 1 << 20):\n                end = min(len(original), start + (1 << 20))\n                out[start:end] = original[start:end]\n            for j, values in enumerate(additions):\n                out[len(original) + eo[j]:len(original) + eo[j + 1]] = values\n            out.flush(); del out\n        out = np.lib.format.open_memmap(target / 'spec_off.npy', mode='w+', dtype=np.int64, shape=(n + len(extra_meta) + 1,))\n        for start in range(0, len(off), 1 << 20):\n            out[start:min(len(off), start + (1 << 20))] = off[start:start + (1 << 20)]\n        out[n + 1:] = eo[1:] + off[-1]\n        out.flush(); del out\n        with pq.ParquetWriter(target / 'spec_meta.parquet', meta.schema_arrow) as writer:\n            for batch in meta.iter_batches(batch_size=65536):\n                writer.write_batch(batch)\n            writer.write_table(pa.Table.from_pylist(extra_meta, schema=meta.schema_arrow))\n        # Verify every copied base byte by bounded array comparison; no second full CSR in RAM.\n        for name, original in [('spec_off.npy',off), ('spec_mz.npy',mz), ('spec_it.npy',it)]:\n            extended = np.load(target / name, mmap_mode='r')\n            for start in range(0, len(original), 1 << 20):\n                end = min(len(original), start + (1 << 20))\n                check(np.array_equal(original[start:end], extended[start:end]), 'base CSR changed')\n            if name == 'spec_off.npy':\n                check(extended[-1] == len(mz) + eo[-1] and np.all(np.diff(extended[n:]) > 0), 'appended offsets invalid')\n            del extended\n        check(pq.ParquetFile(target / 'spec_meta.parquet').metadata.num_rows == n + len(extra_meta), 'extended metadata size differs')\n        return dict(path=str(target), added=len(extra_meta), keys=len(used_keys), base_rows=n, error=None)\n    except Exception as exc:\n        if target is not None:\n            shutil.rmtree(target, ignore_errors=True)\n        return dict(path=None, added=0, keys=0, error=f'{type(exc).__name__}: ' + ' '.join(str(exc).split())[:200])\n\n\ndef _external_reference_library(state):\n    import gc\n    import numpy as np\n    from casmi.library import Library\n    if state['path'] is not None:\n        try:\n            library = Library(state['path'])\n            if len(library.sid) != state['base_rows'] + state['added'] or not np.all(np.diff(library.smass_sorted) >= 0):\n                raise ValueError('extended Library index differs')\n            if np.any(library.sid < 0) or np.any(library.sid >= len(library.struct_mass)):\n                raise ValueError('structure/SID alignment differs')\n            return library\n        except Exception as exc:\n            state['error'] = f'{type(exc).__name__}: ' + ' '.join(str(exc).split())[:200]\n            state['path'] = None\n            library = None\n            gc.collect()\n    return Library('work')\n\n\ndef _external_reference_engine(state, library, pool, train_fp, bank):\n    import gc\n    from casmi.library import Library\n    from casmi.engine import Engine, EngineCfg\n    if state['path'] is not None:\n        try:\n            if len(train_fp) != len(library.struct_mass):\n                raise ValueError('fingerprint/SID alignment differs')\n            engine = Engine(library, pool, train_fp, EngineCfg(generate=True), bank)\n            print(f\"external reference spectra: {state['added']} added for {state['keys']} structures\", flush=True)\n            return library, engine\n        except Exception as exc:\n            state['error'] = f'{type(exc).__name__}: ' + ' '.join(str(exc).split())[:200]\n            state['path'] = None\n            library = engine = None\n            gc.collect()\n        library = Library('work')\n    print('external reference spectra disabled: ' + str(state['error']), flush=True)\n    return library, Engine(library, pool, train_fp, EngineCfg(generate=True), bank)\n\n_EXTERNAL_REFERENCE_PINS = {'files': {'external_delta.parquet': 'e75558832935f28e85a5419022c4cfba4d0d4b49f1063544c317c8ae61c2ac41', 'external_record_quality.parquet': '4f5b227d1c874624182c76e8f8406ba3ee86521744831cb461d6198c898b6c9e', 'external_scorer_sid_links.parquet': '8cd61b8f110bfd6686684d330609a1d0f0d7cc96eab32b9cdd3352939ef443ef'}, 'base_files': {'train_structs.parquet': 'f20aa1cac23af42e430a6633707fda34e48dea97114bdc3e443897f692df3656', 'pool_meta.parquet': '82b8c40315611289b9a4004b3af18d6d3402b3cd5baa4ee5723da07b7e2f6bb0'}, 'keys': 782, 'delta_rows': 1384, 'structure_rows': 275810, 'added_rows': 751, 'added_keys': 469}\n_EXTERNAL_REFERENCE_STATE = _build_external_reference_cache('work', TABLE, _EXTERNAL_REFERENCE_PINS)\n\nimport casmi; print('casmi from', casmi.__file__)\nsys.path.insert(1, os.path.join(V4CODE, 'fe_v4'))\nprint(f'cache built {time.time()-T0:.0f}s')\n\n\n# ===== cell 27 =====\n'''v1 engine exactly as in v3.6 (FPNet A, EngineCfg(generate=True)) wrapped by V1FE (recording hooks + families)'''\nimport numpy as np, pandas as pd, torch, lightgbm as lgb\nfrom casmi.library import Library\nfrom casmi.pool import Pool\nfrom casmi.engine import Engine, EngineCfg, FEATURES\nfrom casmi import fpnet, chem\nimport v1engine\ndev = 'cuda' if torch.cuda.is_available() else 'cpu'\ntorch.manual_seed(0)\ntorch.cuda.manual_seed_all(0)\ntorch.backends.cudnn.deterministic = True\ntorch.backends.cudnn.benchmark = False\nL = _external_reference_library(_EXTERNAL_REFERENCE_STATE); P = Pool('work')\ntfp = np.load('work/train_fp_sel.npy')\n_FULL = glob.glob('/kaggle/input/**/casmi26-fpnet-full1/**/fpnet_full1.pt', recursive=True)   # v4m\nbank = fpnet.ModelBank([_FULL[0] if _FULL else os.path.join(V4MOD, 'fpnet_0.pt')], device=dev)\nprint('engine FP bank:', _FULL[0] if _FULL else 'fpnet_0 (A) FALLBACK', flush=True)\nL, E = _external_reference_engine(_EXTERNAL_REFERENCE_STATE, L, P, tfp, bank)\nV = v1engine.V1FE(E, [os.path.join(V4MOD, m['file']) for m in MAN['fe_models']], MAN['families'], device=dev)\nif 'derivation' in MAN['families']:\n    n_bad = V.mods['derivation'].self_test()\n    print('derivation prior self-test mismatches:', n_bad)\n    assert n_bad == 0, 'RDKit build differs from the one the derivation priors were built with'\nRK = pickle.load(open(os.path.join(V4MOD, MAN['ranker']['file']), 'rb'))\nRK_FEATS = list(RK['features']); RK_B = [lgb.Booster(model_str=s) for s in RK['boosters']]\ndef rank_score(X, names):\n    pos = {n: i for i, n in enumerate(names)}\n    miss = [f for f in RK_FEATS if f not in pos]\n    assert not miss, f'ranker features not produced (would be zero-filled): {miss}'\n    Xs = np.ascontiguousarray(X[:, [pos[f] for f in RK_FEATS]], dtype=np.float32)\n    return np.mean([b.predict(Xs, num_threads=4) for b in RK_B], axis=0)\nprint('engine ready; ranker features', len(RK_FEATS), 'boosters', len(RK_B), f'{time.time()-T0:.0f}s')\n\n\n# ===== cell 28 =====\n'''## 6. Main candidate lists + pool popularity re-rank\n\nFor each molecule, v4n scores candidates with FPNet fingerprints, its feature families and a 4-booster LightGBM ranker, and keeps the top 60. The list is then re-ranked by\n\n$$z(\\\\text{ranker score}) + 0.15\\\\,\\\\big(\\\\log(1+\\\\#\\\\text{substances}) + \\\\log(1+\\\\#\\\\text{papers})\\\\big)$$\n\nOnly pool structures are re-ranked. Generated (novel) candidates keep their positions and form their own simulator groups. This step added +0.010 on the leaderboard.\n'''\n\n\n# ===== cell 29 =====\n'''all molecules -> base lists (v3.6 spectra construction and ordering)'''\nte = pd.read_parquet(os.path.join(COMP, 'test.parquet'))\nte['nm'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\nmols = list(te.groupby('molecule_id', sort=False))\nBASE, n_err = {}, 0\nPOOLPOP_MU = 0.15\nPOOL_POP = np.load(find('pool_lsid.npy')).astype(np.float64) + np.load(find('pool_lpmid.npy')).astype(np.float64)\nassert len(POOL_POP) == len(P.key), (len(POOL_POP), len(P.key))\nn_pp = 0\nprint('v19 pool popularity re-rank, mu', POOLPOP_MU, 'pool', len(POOL_POP))\nfor gi, (mid, sub) in enumerate(mols):\n    nms = sub.nm.values[np.isfinite(sub.nm.values)]\n    smis, keys, lib_max, scs, forms = [], [], 0.0, [], []\n    pids_ = []\n    if len(nms):\n        target = float(np.median(nms))\n        spectra = []\n        for r in sub.itertuples():\n            ce = r.collision_energy_ev\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                                mode=1 if r.ionization_mode == 'positive' else -1, adduct=r.adduct,\n                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,\n                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\n        try:\n            C, X, F, names, info = V.run(spectra, target)\n            lib_max = float(info.get('lib_max', 0.0))\n            if len(C.pid):\n                score = rank_score(np.concatenate([X, F], 1), list(FEATURES) + list(names))\n                order = np.argsort(-score, kind='stable')\n                seen = set()\n                for i in order:\n                    k = C.key[i]\n                    if k in seen: continue\n                    seen.add(k); smis.append(C.smiles[i]); keys.append(k); scs.append(float(score[i])); forms.append(C.formula[i]); pids_.append(int(C.pid[i]))\n                    if len(smis) >= TOPN: break\n        except Exception as e:\n            print('ERROR', mid, repr(e)); n_err += 1\n    if POOLPOP_MU > 0 and len(scs) > 1:            # v19: popularity re-rank of the top-60 base list\n        try:\n            s_ = np.asarray(scs, np.float64); z_ = (s_ - s_.mean()) / (s_.std() + 1e-9)\n            pid_ = np.asarray(pids_); isg_ = pid_ < 0\n            f_ = z_ + POOLPOP_MU * np.where(isg_, 0.0, POOL_POP[np.maximum(pid_, 0)])\n            pool_i = np.where(~isg_)[0]; o_ = np.empty(len(pid_), np.int64)\n            o_[isg_] = np.where(isg_)[0]; o_[~isg_] = pool_i[np.argsort(-f_[pool_i], kind='stable')]   # generated keep their slots\n            smis = [smis[i] for i in o_]; keys = [keys[i] for i in o_]; scs = [float(f_[i]) for i in o_]\n            forms = [forms[i] + ('|gen' if isg_[i] else '') for i in o_]      # generated form their own ICEBERG groups\n            n_pp += 1\n        except Exception as e:\n            print('poolpop failed', mid, repr(e))\n    BASE[mid] = (smis, keys, lib_max, scs, forms)\n    if (gi + 1) % 25 == 0 or gi == len(mols) - 1:\n        print(f'  {gi+1}/{len(mols)} molecules {time.time()-T0:.0f}s', flush=True)\nprint('errors', n_err, '| popularity re-ranked lists', n_pp)\n'''Keep ranked overflow for append-only final coverage.'''\nBASE_TAIL_CANDIDATES = {str(k): list(v[0]) for k, v in BASE.items()}\n\n\n# ===== cell 30 =====\n'''## 7. ICEBERG + GLACIER spectrum simulators\n\nBoth models predict each candidate's spectrum and compare it with the measured one. Candidates with the same molecular formula are then re-ranked by\n\n$$z(\\\\text{score}) + 1.0\\\\,z(\\\\text{ICEBERG}) + 1.0\\\\,z(\\\\text{GLACIER})$$\n\nMost wrong answers share the right answer's formula, so this is where isomers get separated.\n\n- Second-engine candidates join the simulator input, so they are scored too.\n- **GLACIER uses [M+H]+ spectra only.** Restricting it this way added +0.006 on the leaderboard.\n'''\n\n\n# ===== cell 31 =====\n'''--- CLAW FIX (b418/b419): stage health is now RECORDED, never silently swallowed ---'''\n'''`glob('/kaggle/input/**/x')` WITHOUT recursive=True matches one directory level only, so it'''\n'''misses /kaggle/input/datasets/<owner>/<slug>/. That silently disabled ICEBERG+GLACIER:'''\n'''ICE FAILED -> ranker order kept: FileNotFoundError('cp313 rdkit wheel not found')  [b417 log]'''\n'''rdkit-2025.3.6-cp312-...whl is not a supported wheel on this platform               [b417 log]'''\n'''recursive=True finds the wheel under BOTH mount layouts (same 2025.3.6 cp313 file either way).'''\nCLAW_ICE_HEALTH = 'ICE NOT RUN'\nCLAW_GL_HEALTH = 'GL NOT RUN'\nBASE_ICE = False         # b421: base-list ICEBERG fusion OFF (b420 showed it costs 0.009); see header\n'''ICEBERG post-ranker isomer re-scoring (casmi26-iceberg). Optional: any failure keeps the ranker order.'''\nimport gc\nICE_SCORES, ice_stats = {}, dict(molecules=0, changed_top25=0, changed_top1=0)\n\n\n'''--- CLAW FIX (b415c): make ICEBERG/GLACIER runnable on a cp313 Kaggle image ---'''\n'''The ICEBERG package bundles rdkit-2025.3.6 built for **cp312**; this image is cp313,'''\n'''so pip rejects that wheel and install_site() raises before anything is scored'''\n'''(n_mols_scored == 0 for ICEBERG *and* GLACIER). The MANIFEST pins 2025.3.6 because'''\n'''\"FragmentEngine Kekulize differs in 2026.03\", so swapping in the runtime's 2026.03'''\n'''rdkit would change the scoring. Instead we install the SAME 2025.3.6 release built'''\n'''for cp313, so the pinned version is honoured exactly and RDKIT_REQ passes natively.'''\n''''''\n'''install_site() is idempotent via <site>/.ice_site_ok, which it compares against the'''\n'''name list of `pkg/wheels`. We pre-build the site and write that exact list, so both'''\n'''runners short-circuit and never invoke the failing pip.'''\ndef _claw_preseed_site(pkg, site):\n    import glob as _g, os as _o, shutil as _s, subprocess as _sp, sys as _sys\n    srcs = sorted(_g.glob(_o.path.join(pkg, 'wheels', '*.whl.ice')) +\n                  _g.glob(_o.path.join(pkg, 'wheels', '*.whl')))\n    # install everything EXCEPT the cp-tag-mismatched rdkit\n    keep = [w for w in srcs if not _o.path.basename(w).lower().startswith('rdkit')]\n    dropped = [_o.path.basename(w) for w in srcs if w not in keep]\n    new = [simulator_rdkit_wheel(pkg)]\n    keep = keep + new[:1]\n    _o.makedirs(site, exist_ok=True)\n    wdir = _o.path.join(site, '.wheels')\n    _o.makedirs(wdir, exist_ok=True)\n    whl = []\n    for w in keep:\n        dst = _o.path.join(wdir, _o.path.basename(w)[:-4] if w.endswith('.ice') else _o.path.basename(w))\n        _s.copyfile(w, dst)\n        whl.append(dst)\n    r = _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',\n                 '--no-compile', '--disable-pip-version-check', '--upgrade', '--target', site] + whl,\n                stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True)\n    print('[claw] site preseed rc', r.returncode, (r.stdout or '')[-400:], flush=True)\n    if r.returncode != 0:\n        raise RuntimeError('claw preseed pip failed: ' + (r.stdout or '')[-600:])\n    _s.rmtree(wdir, ignore_errors=True)\n    # the marker MUST equal exactly what install_site() derives from the PACKAGE list\n    want = '\\n'.join(_o.path.basename(w) for w in srcs)\n    with open(_o.path.join(site, '.ice_site_ok'), 'w') as f:\n        f.write(want)\n    print('[claw] site ready; substituted', dropped, '->', [_o.path.basename(w) for w in new[:1]], flush=True)\n    return want\n\n\ntry:\n    ICE_PKG = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))\n    sys.path.append(ICE_PKG)\n    import fuse as ice_fuse\n    del V, E, bank; gc.collect(); torch.cuda.empty_cache()\n    # priority: molecules without a library-level match first (class 1 rarely changes), then the rest\n    mids = sorted(BASE, key=lambda m: BASE[m][2])\n    cands = {m: (ice_fuse.ice_candidates(BASE[m][0], BASE[m][4], TOPN) if BASE_ICE else []) for m in mids}\n    if ENG:                              # ADDED (ICE_UNION): also score our engine's candidates (same-formula groups\n        from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF   # formed over the v4g top-N + engine top-40)\n        from rdkit import Chem as _Ch\n        def _f2(s):\n            try:\n                return _CMF(_Ch.MolFromSmiles(s))\n            except Exception:\n                return s\n        n_add = 0\n        for m in mids:\n            e = ENG.get(str(m))\n            if not e or not e.get('smiles'):\n                continue\n            bs, bk, bf = list(BASE[m][0][:TOPN]), set(BASE[m][1][:TOPN]), list(BASE[m][4][:TOPN])\n            es = [s for s, k in zip(e['smiles'][:40], e['keys'][:40]) if k not in bk]\n            ef = [_f2(s) for s in es]\n            extra = ice_fuse.ice_candidates(bs + es, bf + ef, len(bs) + len(es))\n            before = len(cands.get(m, []))\n            cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))\n            n_add += len(cands[m]) - before\n        print('ICE_UNION: engine candidates added to ICE input:', n_add, flush=True)\n    if ICE_PC:                          # also the PubChem-only list of molecules the v3.6 gate may use (lib_max < LIB_TAU)\n        from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n        from rdkit import Chem as _Chem\n        def _form(s):\n            try:\n                return CalcMolFormula(_Chem.MolFromSmiles(s))\n            except Exception:\n                return s\n        for m in mids:\n            p = PC.get(m) if isinstance(PC, dict) else None\n            if p and p.get('pc') and BASE[m][2] < LIB_TAU:\n                p['pc_form'] = [_form(x) for x in p['pc']]\n                extra = ice_fuse.ice_candidates(p['pc'], p['pc_form'], 25)\n                cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))\n    cands = {m: c for m, c in cands.items() if c}\n    items = ice_fuse.build_ice_input(te, cands)\n    print('ICE input molecules', len(items), 'candidates', sum(len(it['cands']) for it in items), f'{time.time()-T0:.0f}s', flush=True)\n    _claw_preseed_site(ICE_PKG, '/kaggle/working/ice_site')\n    ICE_SCORES = ice_fuse.run_ice(ICE_PKG, items, workdir='/kaggle/working/ice_work', device='cuda', budget_s=ICE_BUDGET,\n                                  site='/kaggle/working/ice_site', extra_args=['--any-rdkit'])\n    meta_f = '/kaggle/working/ice_work/ice_out.json.meta.json'\n    if os.path.exists(meta_f):\n        print('ICE meta', open(meta_f).read()[:1500])\nexcept Exception as e:\n    CLAW_ICE_HEALTH = f'ICE FAILED: {e!r}'\n    print('ICE FAILED -> ranker order kept:', repr(e), flush=True)\n'''v4h: GLACIER third re-scoring term (casmi26-glacier) on the SAME runner input; any GLACIER failure -> v4g (ICE only)'''\nGL_SCORES, gl_stats, GL_LAM, GL_BUDGET = {}, dict(molecules=0, changed_vs_ice_top25=0, changed_vs_ice_top1=0), 1.0, 4000\ntry:\n    GL_PKG = os.path.dirname(find('casmi26-glacier/**/gl_runner.py'))\n    sys.path.append(GL_PKG)\n    import gl_fuse\n    gc.collect(); torch.cuda.empty_cache()   # ICE ran in its own (finished) process; free the notebook's cached GPU memory\n    gl_items = [dict(it, spectra=[s for s in it['spectra'] if s.get('adduct') == '[M+H]+']) for it in items]   # v29: GLACIER on [M+H]+ only (v27: +0.006)\n    gl_items = [it for it in gl_items if it['spectra']]\n    print('v29 GLACIER [M+H]+ only: molecules', len(gl_items), 'of', len(items))\n    GL_SCORES = gl_fuse.run_gl(GL_PKG, gl_items, workdir='/kaggle/working/gl_work', device='cuda', budget_s=GL_BUDGET,\n                               site='/kaggle/working/ice_site', wheels=os.path.join(ICE_PKG, 'wheels'), extra_args=['--any-rdkit'])\n    meta_f = '/kaggle/working/gl_work/gl_out.json.meta.json'\n    print('GL meta', open(meta_f).read()[:1500] if os.path.exists(meta_f) else 'missing')\nexcept Exception as e:\n    GL_SCORES = {}\n    CLAW_GL_HEALTH = f'GL FAILED: {e!r}'\n    print('GL FAILED -> v4g behaviour (ICE only):', repr(e), flush=True)\n# N1A complement: Seyit Kaan Gunes; integration as in Ahmed Berat Ozer v15 / guhongbin v465.\n# ---- N1A_MCE (gold roadmap 2026-10-09 section 3.2: N1 arm A + I14b merged CE on the proxies only) ---------------\n# Molecules WITHOUT a real [M+H]+ spectrum are GLACIER-blind in H1. N1 arm A scores their [M+NH4]+ / [M-H2O+H]+\n# spectra as [M+H]+ proxies (NH4: M = prec - 18.033823, peaks >= M + 1.007276 - 1.5 dropped; water loss:\n# M = prec + 18.010565 - 1.007276, peaks < prec - 1.5 kept; precursor -> M + 1.007276, relabelled [M+H]+;\n# [M-2H2O+H]+ / Na / K never), and a merged-CE proxy (>= 2 distinct CE buckets) is predicted at EVERY bucket and the\n# predictions merge()d like the SUM_CE path (I14b). = bench kdev_glbench_n1 arm N1A_D on the blind molecules (521\n# bench molecules: 21 blind molecules with a proxy, lift over random +0.270 [0.117, 0.443]; vs H1 on all np molecules\n# +0.027 [0.010, 0.047]; merged vs mean CE on those 21 +0.0026). Arm B (proxies also where [M+H]+ exists, -0.047) and\n# merged CE for real [M+H]+ spectra (no gain) are NOT done: molecules WITH a real [M+H]+ keep the H1 run above\n# untouched (stock gl_runner, mean CE, byte-identical input). The bench's own code runs here, embedded verbatim\n# (sha256-checked): glb_core.py (n1_proxy / build_variants / mol_flags / arm_tags / arm_scores) in this process,\n# glb_driver.py (driver around the attached gl_runner.py: same env, model, Predictor, canonicalisation, CE bucketing,\n# merge, entropy similarity) in its own process like gl_runner. N1 scores are added to GL_SCORES (-> the PubChem-list\n# re-rank below and the fused re-rank of cell 19, whose `if ice:` gate is opened ONLY for N1 molecules with scores).\n# Any failure -> no N1 scores = H1 behaviour (the commit raises instead). The 12 dummy commit molecules never\n# trigger N1, so the commit adds a forced exercise: synthetic [M+H]+-less copies 'n1x:<mid>' ([M+H]+ -> [M+NH4]+ with\n# an injected NH4 precursor peak, the first one also -> [M-H2O+H]+) run through the same code + asserts.\nN1_ARM = 'N1A_D'                 # bench arm: blind molecule -> spectrum tags (NH4p, WLp), CE mode 'merged'\nN1_BUDGET = 1200.0 if IS_RERUN else 600.0\nN1_WORK, N1_SITE, N1_DEVICE, N1_DRIVER_EXTRA = '/kaggle/working/n1_work', '/kaggle/working/ice_site', 'cuda', []\nN1_MIDS, N1_GL_MIDS, N1_STATS, N1_DRV, N1_HEALTH = set(), set(), {}, {}, 'N1 NOT RUN'\nN1X_MIDS, N1X_GL, N1X_STATS = set(), {}, {}\n# k2 (E006 port), variant A = N1-complement. E006 (cell 31) re-ranks a GLACIER-blind molecule with ICEBERG when its ICE\n# dict has finite scores (in practice [M+Na]+ without [M+H]+: `elif ice_fallback and ice and not gl`). N1 GL scores\n# would move such a molecule to the GLACIER branch, so they are kept ONLY for N1 molecules WITHOUT any finite ICE\n# score (E006 has no forward-model evidence there: its ICE fallback on an all-None dict is a no-op). False = the plain\n# port (N1 GL scores for every N1 molecule, E2 behaviour). Molecules with a real [M+H]+ are never touched either way.\n# Challenger route: flexonafft, casmi26-closing-the-evidence-gap v47.\nN1_COMPLEMENT = False\nprint('[N1 challenger] configuration: N1_COMPLEMENT=False; real [M+H]+ stays on H1', flush=True)\nN1_ICE_DROPPED = set()           # N1 molecules left to E006's ICE fallback by the complement filter\nN1_CORE_SHA256 = 'f7a3b8cd8802a7ba3b69a297603241784a63c069044cf7bd6efda7d68cc8fdfa'      # = subs/out/kdev_glbench_n1_full/glb_core.py\nN1_DRIVER_SHA256 = 'e919a154b57bda5a52573481348e38d8b4e7c97883532ea5b385bc3cb6f98502'    # = subs/out/kdev_glbench_n1_full/glb_driver.py\nN1_GL_RUNNER_SHA256 = '5c4f74313ac54263d05d987c39d012d81ec57c5f35e1095ecd49a5bcb6440fca'   # casmi26-glacier gl_runner.py (B1 MANIFEST, bench)\n# kdev_glbench_n1 glb_core.py, verbatim (proxy construction, variants, arm resolution, aggregation)\nN1_CORE_SRC = ''.join([\n    '\"\"\"casmi26 kdev_glbench_n1 -- pure-python core of the roadmap 3.2 GLACIER bench (no torch / rdkit import).\\n',\n    '\\n',\n    'Query handling (N1, roadmap 3.2), on fuse.build_ice_input(..., covered_only=False) spectrum dicts:\\n',\n    '  [M+NH4]+    M = prec - 18.033823; prec -> M + 1.007276; drop peaks mz >= M + 1.007276 - 1.5; relabel [M+H]+\\n',\n    '  [M-H2O+H]+  M = prec + 18.010565 - 1.007276; prec -> M + 1.007276; keep peaks mz < original prec - 1.5; relabel\\n',\n    '  (a proxy left without peaks is dropped; [M-2H2O+H]+, [M+Na]+, [M+K]+ are never proxied)\\n',\n    '  Hcut (diagnostic): a real [M+H]+ spectrum with the same precursor-region cut (peaks mz >= prec - 1.5 dropped), the\\n',\n    '  like-for-like reference for the proxies (separates \"different adduct\" from \"precursor region removed\").\\n',\n    '\\n',\n    'Scoring: the driver returns per-spectrum GLACIER similarities for the frozen bench candidates in two CE modes\\n',\n    \"('mean' = production gl_runner: one prediction at the bucket of mean(ce_ev); 'merged' = I14b: one prediction per\\n\",\n    \"distinct CE bucket of a merged-CE spectrum, merged like the SUM_CE path). A molecule's arm score per candidate =\\n\",\n    \"mean over the arm's spectra with a similarity = gl_ex_mean restricted to those spectra.\\n\",\n    '\\n',\n    'Ranking metric: expected reciprocal rank of the truth (cands[mid][0]) under a UNIFORM RANDOM tie-break (no truth-first\\n',\n    'bias: a truth tied with k decoys gets mean(1/(r..r+k))); unscored candidates rank below every scored one (tied among\\n',\n    'themselves). Baseline = a random order of all frozen candidates, mean(1/1..1/n). An arm \"covers\" a molecule when the\\n',\n    'truth and >= 2 candidates are scored; elsewhere the arm has no evidence (= the random baseline in fill-in totals).\\n',\n    '\"\"\"\\n',\n    'import math\\n',\n    'from collections import Counter, OrderedDict\\n',\n    '\\n',\n    'import numpy as np\\n',\n    '\\n',\n    'PROTON = 1.007276\\n',\n    'NH4_ADD = 18.033823           # [M+NH4]+ precursor m/z - M\\n',\n    'H2O = 18.010565\\n',\n    'CUT_DA = 1.5\\n',\n    'MIN_PEAKS = 1\\n',\n    \"PROXY_TAG = OrderedDict([('[M+NH4]+', 'NH4p'), ('[M-H2O+H]+', 'WLp')])\\n\",\n    'EPS_TIE = 1e-12\\n',\n    'N_BOOT = 2000\\n',\n    'BOOT_SEED = 12345\\n',\n    '\\n',\n    \"# arm -> (spectrum tags, CE mode, description); 'N1A' is resolved per molecule (see arm_tags)\\n\",\n    'ARMS = OrderedDict([\\n',\n    \"    ('A_H', (('H',), 'mean', '(a) [M+H]+ only, mean CE = B1 production GL')),\\n\",\n    \"    ('B_NH4', (('NH4p',), 'mean', '(b) [M+NH4]+ proxies relabelled [M+H]+')),\\n\",\n    \"    ('C_WL', (('WLp',), 'mean', '(c) [M-H2O+H]+ proxies relabelled [M+H]+')),\\n\",\n    \"    ('D_Hmerged', (('H',), 'merged', '(d) [M+H]+ only, merged-CE per-bucket prediction (I14b)')),\\n\",\n    \"    ('A_Hcut', (('Hcut',), 'mean', 'diag: [M+H]+ with the proxy-style precursor cut')),\\n\",\n    \"    ('B_NH4m', (('NH4p',), 'merged', 'diag: NH4 proxies, merged CE')),\\n\",\n    \"    ('C_WLm', (('WLp',), 'merged', 'diag: water-loss proxies, merged CE')),\\n\",\n    \"    ('HNa', (('H', 'Na'), 'mean', 'diag: [M+H]+ and [M+Na]+ (old fwdbench COV; P4 analogue)')),\\n\",\n    \"    ('AB', (('H', 'NH4p', 'WLp'), 'mean', 'diag: N1 arm B = [M+H]+ plus all proxies')),\\n\",\n    \"    ('N1A', (None, 'mean', 'N1 arm A: [M+H]+ if the molecule has one, else its proxies')),\\n\",\n    \"    ('N1A_D', (None, 'merged', 'diag: N1 arm A with merged CE (N1A + I14b)')),\\n\",\n    '])\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------ query handling --\\n',\n    'def _keep_below(sp, cut, min_peaks):\\n',\n    \"    mz, it = sp.get('mz') or [], sp.get('it') or []\\n\",\n    '    keep = [(float(m), float(i)) for m, i in zip(mz, it) if float(m) < cut]\\n',\n    '    if len(keep) < min_peaks or not any(i > 0 for _, i in keep):\\n',\n    '        return None\\n',\n    '    return [m for m, _ in keep], [i for _, i in keep]\\n',\n    '\\n',\n    '\\n',\n    'def n1_proxy(sp, min_peaks=MIN_PEAKS):\\n',\n    '    \"\"\"[M+NH4]+ / [M-H2O+H]+ positive spectrum dict -> relabelled [M+H]+ proxy dict, or None.\"\"\"\\n',\n    \"    if not isinstance(sp, dict) or sp.get('mode') != 'positive':\\n\",\n    '        return None\\n',\n    \"    ad = sp.get('adduct')\\n\",\n    '    try:\\n',\n    \"        prec = float(sp['prec'])\\n\",\n    '    except Exception:\\n',\n    '        return None\\n',\n    '    if not math.isfinite(prec):\\n',\n    '        return None\\n',\n    \"    if ad == '[M+NH4]+':\\n\",\n    '        M = prec - NH4_ADD\\n',\n    '        cut = M + PROTON - CUT_DA\\n',\n    \"    elif ad == '[M-H2O+H]+':\\n\",\n    '        M = prec + H2O - PROTON\\n',\n    '        cut = prec - CUT_DA\\n',\n    '    else:\\n',\n    '        return None\\n',\n    '    k = _keep_below(sp, cut, min_peaks)\\n',\n    '    if k is None:\\n',\n    '        return None\\n',\n    '    out = dict(sp)\\n',\n    \"    out.update(mz=k[0], it=k[1], prec=M + PROTON, adduct='[M+H]+', mode='positive', n1_src=ad)\\n\",\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    'def h_cut(sp, min_peaks=MIN_PEAKS):\\n',\n    '    \"\"\"Real [M+H]+ spectrum with the proxy-style cut (peaks mz >= prec - 1.5 dropped); None if nothing is left.\"\"\"\\n',\n    \"    if not isinstance(sp, dict) or sp.get('mode') != 'positive' or sp.get('adduct') != '[M+H]+':\\n\",\n    '        return None\\n',\n    \"    k = _keep_below(sp, float(sp['prec']) - CUT_DA, min_peaks)\\n\",\n    '    if k is None:\\n',\n    '        return None\\n',\n    '    return dict(sp, mz=k[0], it=k[1])\\n',\n    '\\n',\n    '\\n',\n    'def build_variants(items_all):\\n',\n    '    \"\"\"fuse.build_ice_input(Q, cands, covered_only=False) items -> driver items whose spectra carry sid / tag\\n',\n    '    (tags H, Hcut, Na, NH4p, WLp), plus counts. sid = \\'<index of the spectrum in the molecule>:<tag>\\'.\"\"\"\\n',\n    '    out, st = [], Counter()\\n',\n    '    for it in items_all:\\n',\n    '        sps = []\\n',\n    \"        for j, sp in enumerate(it.get('spectra') or []):\\n\",\n    \"            if sp.get('mode') != 'positive':\\n\",\n    '                continue\\n',\n    \"            ad = sp.get('adduct')\\n\",\n    \"            if ad == '[M+H]+':\\n\",\n    \"                sps.append(dict(sp, sid=f'{j}:H', tag='H')); st['H'] += 1\\n\",\n    '                c = h_cut(sp)\\n',\n    '                if c is None:\\n',\n    \"                    st['Hcut_empty'] += 1\\n\",\n    '                else:\\n',\n    \"                    sps.append(dict(c, sid=f'{j}:Hcut', tag='Hcut')); st['Hcut'] += 1\\n\",\n    \"            elif ad == '[M+Na]+':\\n\",\n    \"                sps.append(dict(sp, sid=f'{j}:Na', tag='Na')); st['Na'] += 1\\n\",\n    '            elif ad in PROXY_TAG:\\n',\n    '                tag = PROXY_TAG[ad]\\n',\n    '                p = n1_proxy(sp)\\n',\n    '                if p is None:\\n',\n    \"                    st[tag + '_empty'] += 1\\n\",\n    '                else:\\n',\n    \"                    sps.append(dict(p, sid=f'{j}:{tag}', tag=tag)); st[tag] += 1\\n\",\n    \"        out.append(dict(mid=str(it['mid']), cands=list(it['cands']), spectra=sps))\\n\",\n    '    return out, dict(st)\\n',\n    '\\n',\n    '\\n',\n    'def proxy_prec_check(items):\\n',\n    '    \"\"\"Real-data check of the adduct arithmetic: proxy precursor (M + 1.007276) vs the molecule\\'s real [M+H]+\\n',\n    '    precursor m/z (median over its [M+H]+ spectra), in mDa, per proxy tag.\"\"\"\\n',\n    '    out = {}\\n',\n    '    for tag in PROXY_TAG.values():\\n',\n    '        d = []\\n',\n    '        for it in items:\\n',\n    \"            H = [float(sp['prec']) for sp in it['spectra'] if sp.get('tag') == 'H']\\n\",\n    '            if H:\\n',\n    \"                d += [1000.0 * (float(sp['prec']) - float(np.median(H))) for sp in it['spectra'] if sp.get('tag') == tag]\\n\",\n    '        out[tag] = dict(n=len(d), median_mda=float(np.median(d)) if d else None,\\n',\n    '                        max_abs_mda=float(np.max(np.abs(d))) if d else None)\\n',\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    '# ---------------------------------------------------------------------------------------------------- ranking ----\\n',\n    'def rr_random_tb(vals, ti=0, eps=EPS_TIE):\\n',\n    '    \"\"\"Expected reciprocal rank of vals[ti] under a uniform random tie-break; None entries rank below all scored\\n',\n    '    ones. None if vals[ti] is None.\"\"\"\\n',\n    '    t = vals[ti]\\n',\n    '    if t is None:\\n',\n    '        return None\\n',\n    '    gt = eq = 0\\n',\n    '    for j, v in enumerate(vals):\\n',\n    '        if j == ti or v is None:\\n',\n    '            continue\\n',\n    '        if v > t + eps:\\n',\n    '            gt += 1\\n',\n    '        elif abs(v - t) <= eps:\\n',\n    '            eq += 1\\n',\n    '    return sum(1.0 / (gt + 1 + k) for k in range(eq + 1)) / (eq + 1)\\n',\n    '\\n',\n    '\\n',\n    'def rr_truth_first(vals, ti=0, eps=EPS_TIE):\\n',\n    '    \"\"\"Old-bench convention with the truth winning every tie (the bias this bench removes).\"\"\"\\n',\n    '    t = vals[ti]\\n',\n    '    if t is None:\\n',\n    '        return None\\n',\n    '    return 1.0 / (1 + sum(1 for j, v in enumerate(vals) if j != ti and v is not None and v > t + eps))\\n',\n    '\\n',\n    '\\n',\n    'def rr_split(vals, ti=0, eps=EPS_TIE):\\n',\n    '    \"\"\"fwdbench bench_core.mrr_table convention (ties count half).\"\"\"\\n',\n    '    t = vals[ti]\\n',\n    '    if t is None:\\n',\n    '        return None\\n',\n    '    gt = sum(1 for j, v in enumerate(vals) if j != ti and v is not None and v > t + eps)\\n',\n    '    eq = sum(1 for j, v in enumerate(vals) if j != ti and v is not None and abs(v - t) <= eps)\\n',\n    '    return 1.0 / (1 + gt + 0.5 * eq)\\n',\n    '\\n',\n    '\\n',\n    'def rr_rand(n):\\n',\n    \"    return sum(1.0 / k for k in range(1, n + 1)) / n if n > 0 else float('nan')\\n\",\n    '\\n',\n    '\\n',\n    'def covered(vals, ti=0):\\n',\n    '    return vals[ti] is not None and sum(v is not None for v in vals) >= 2\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------ aggregation ----\\n',\n    'def mol_flags(item):\\n',\n    \"    tags = Counter(sp['tag'] for sp in item['spectra'])\\n\",\n    \"    return dict(has_H=tags['H'] > 0, has_NH4p=tags['NH4p'] > 0, has_WLp=tags['WLp'] > 0, has_Na=tags['Na'] > 0)\\n\",\n    '\\n',\n    '\\n',\n    'def arm_tags(arm, flags):\\n',\n    '    tags, mode, _ = ARMS[arm]\\n',\n    '    if tags is None:                                        # N1 arm A: proxies only where no real [M+H]+ exists\\n',\n    \"        tags = ('H',) if flags['has_H'] else ('NH4p', 'WLp')\\n\",\n    '    return tags, mode\\n',\n    '\\n',\n    '\\n',\n    'def arm_scores(mol_out, tags, mode):\\n',\n    '    \"\"\"mol_out = driver output of one molecule {\\'cands\\': [...], \\'spec\\': {sid: {\\'tag\\',\\'nb\\',\\'mean\\',\\'merged\\'}}} ->\\n',\n    '    per-candidate list (aligned with mol_out[\\'cands\\']) of the mean similarity over the arm\\'s spectra (None if none).\"\"\"\\n',\n    \"    cands = mol_out.get('cands') or []\\n\",\n    \"    cols = [s[mode] for sid, s in sorted((mol_out.get('spec') or {}).items(), key=lambda kv: _sid_key(kv[0]))\\n\",\n    \"            if s.get('tag') in tags and s.get(mode) is not None]\\n\",\n    '    out = []\\n',\n    '    for i in range(len(cands)):\\n',\n    '        xs = [c[i] for c in cols if i < len(c) and c[i] is not None]\\n',\n    '        out.append(float(np.mean(xs)) if xs else None)\\n',\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    'def _sid_key(sid):\\n',\n    \"    a, _, b = str(sid).partition(':')\\n\",\n    '    try:\\n',\n    '        return (int(a), b)\\n',\n    '    except ValueError:\\n',\n    '        return (10 ** 9, str(sid))\\n',\n    '\\n',\n    '\\n',\n    'def molecule_table(items, out, done):\\n',\n    '    \"\"\"-> list of row dicts, one per molecule the driver finished: rr per arm (None = arm does not cover it).\"\"\"\\n',\n    '    rows = []\\n',\n    '    for it in items:\\n',\n    \"        mid = it['mid']\\n\",\n    '        if mid not in done:\\n',\n    '            continue\\n',\n    \"        mo = out.get(mid) or {'cands': it['cands'], 'spec': {}}\\n\",\n    \"        assert list(mo.get('cands') or it['cands']) == list(it['cands']), ('candidate order changed', mid)\\n\",\n    '        fl = mol_flags(it)\\n',\n    \"        n = len(it['cands'])\\n\",\n    \"        nb_h = [s.get('nb', 1) for s in (mo.get('spec') or {}).values() if s.get('tag') == 'H']\\n\",\n    \"        r = dict(mid=mid, set=mid.split(':')[0], n=n, rand=rr_rand(n), has_merged_H=any(b >= 2 for b in nb_h), **fl)\\n\",\n    '        for arm in ARMS:\\n',\n    '            tags, mode = arm_tags(arm, fl)\\n',\n    '            v = arm_scores(mo, tags, mode)\\n',\n    \"            r['rr_' + arm] = rr_random_tb(v) if covered(v) else None\\n\",\n    \"            if arm == 'A_H':\\n\",\n    \"                r['rrtf_A_H'] = rr_truth_first(v) if covered(v) else None\\n\",\n    \"                r['rrsp_A_H'] = rr_split(v) if covered(v) else None\\n\",\n    \"                r['top1_A_H'] = (None if not covered(v) else float(rr_random_tb(v) == 1.0))\\n\",\n    '        rows.append(r)\\n',\n    '    return rows\\n',\n    '\\n',\n    '\\n',\n    '# ----------------------------------------------------------------------------------------------------- stats -----\\n',\n    'def _boot_idx(n, seed=BOOT_SEED, nb=N_BOOT):\\n',\n    '    return np.random.default_rng(seed).integers(0, n, (nb, n))\\n',\n    '\\n',\n    '\\n',\n    \"def paired(rows, a, b, sel=None, label=''):\\n\",\n    '    \"\"\"Paired comparison of arms a and b over the rows both cover (and sel(row) if given).\"\"\"\\n',\n    \"    R = [r for r in rows if r['rr_' + a] is not None and r['rr_' + b] is not None and (sel is None or sel(r))]\\n\",\n    '    n = len(R)\\n',\n    '    res = dict(label=label, a=a, b=b, n=n)\\n',\n    '    if n == 0:\\n',\n    '        return res\\n',\n    \"    ra = np.array([r['rr_' + a] for r in R]); rb = np.array([r['rr_' + b] for r in R])\\n\",\n    \"    rn = np.array([r['rand'] for r in R])\\n\",\n    '    d = rb - ra\\n',\n    '    idx = _boot_idx(n)\\n',\n    '    la, lb = ra - rn, rb - rn\\n',\n    \"    with np.errstate(divide='ignore', invalid='ignore'):\\n\",\n    '        bo_ratio = lb[idx].mean(1) / la[idx].mean(1)\\n',\n    \"    ratio = float(lb.mean() / la.mean()) if la.mean() > 0 else float('nan')\\n\",\n    '    res.update(mrr_a=float(ra.mean()), mrr_b=float(rb.mean()), rand=float(rn.mean()), lift_a=float(la.mean()),\\n',\n    '               lift_b=float(lb.mean()), delta=float(d.mean()),\\n',\n    '               delta_ci=[float(x) for x in np.percentile(d[idx].mean(1), [2.5, 97.5])],\\n',\n    '               ratio=ratio, ratio_ci=[float(x) for x in np.nanpercentile(bo_ratio, [2.5, 97.5])] if n > 1 else [ratio, ratio],\\n',\n    '               wins=int((d > 1e-9).sum()), ties=int((np.abs(d) <= 1e-9).sum()), losses=int((d < -1e-9).sum()),\\n',\n    \"               by_set={s: int(sum(r['set'] == s for r in R)) for s in sorted({r['set'] for r in R})})\\n\",\n    '    return res\\n',\n    '\\n',\n    '\\n',\n    \"def arm_table(rows, arm, sets=('np', 'e180', 'all')):\\n\",\n    '    out = {}\\n',\n    '    for s in sets:\\n',\n    \"        R = [r for r in rows if r['rr_' + arm] is not None and (s == 'all' or r['set'] == s)]\\n\",\n    '        if not R:\\n',\n    '            out[s] = dict(n=0)\\n',\n    '            continue\\n',\n    \"        rr = np.array([r['rr_' + arm] for r in R]); rn = np.array([r['rand'] for r in R])\\n\",\n    '        out[s] = dict(n=len(R), mrr=float(rr.mean()), rand=float(rn.mean()), lift=float((rr - rn).mean()),\\n',\n    '                      top1=float(np.mean(rr >= 1.0 - 1e-12)))\\n',\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    \"def filled_delta(rows, a, b, sel=None, label=''):\\n\",\n    '    \"\"\"Mean over ALL selected molecules with a missing arm filled by the random baseline (no evidence).\"\"\"\\n',\n    '    R = [r for r in rows if sel is None or sel(r)]\\n',\n    '    n = len(R)\\n',\n    '    res = dict(label=label, a=a, b=b, n=n)\\n',\n    '    if n == 0:\\n',\n    '        return res\\n',\n    \"    fa = np.array([r['rr_' + a] if r['rr_' + a] is not None else r['rand'] for r in R])\\n\",\n    \"    fb = np.array([r['rr_' + b] if r['rr_' + b] is not None else r['rand'] for r in R])\\n\",\n    '    d = fb - fa\\n',\n    '    idx = _boot_idx(n)\\n',\n    '    res.update(mrr_a=float(fa.mean()), mrr_b=float(fb.mean()), delta=float(d.mean()),\\n',\n    '               delta_ci=[float(x) for x in np.percentile(d[idx].mean(1), [2.5, 97.5])],\\n',\n    '               n_changed=int((np.abs(d) > 1e-9).sum()))\\n',\n    '    return res\\n',\n    '\\n',\n    '\\n',\n    'def fmt_p(p):\\n',\n    \"    if not p.get('n'):\\n\",\n    '        return f\"{p.get(\\'label\\', \\'\\')}: {p[\\'a\\']} vs {p[\\'b\\']} n=0\"\\n',\n    '    s = (f\"{p.get(\\'label\\', \\'\\')}: n={p[\\'n\\']} {p[\\'a\\']} {p[\\'mrr_a\\']:.4f} vs {p[\\'b\\']} {p[\\'mrr_b\\']:.4f} (rand {p[\\'rand\\']:.4f}) \"\\n',\n    '         f\"delta {p[\\'delta\\']:+.4f} CI [{p[\\'delta_ci\\'][0]:+.4f}, {p[\\'delta_ci\\'][1]:+.4f}] \"\\n',\n    '         f\"lift {p[\\'lift_a\\']:.4f} -> {p[\\'lift_b\\']:.4f} ratio {p[\\'ratio\\']:.3f} CI [{p[\\'ratio_ci\\'][0]:.3f}, {p[\\'ratio_ci\\'][1]:.3f}] \"\\n',\n    '         f\"W/T/L {p[\\'wins\\']}/{p[\\'ties\\']}/{p[\\'losses\\']} sets {p[\\'by_set\\']}\")\\n',\n    '    return s\\n',\n    '\\n',\n    '\\n',\n    'def gates(rows):\\n',\n    '    \"\"\"Roadmap 3.2 gates. N1-NH4 / N1-WL: kill if the proxy lift < 50% of the paired [M+H]+ lift (same molecules).\\n',\n    '    I14b: adopt only if delta >= +0.01 on merged-CE e180 molecules and >= 0 on np merged-CE molecules.\"\"\"\\n',\n    '    g = OrderedDict()\\n',\n    \"    for key, arm in (('N1_NH4', 'B_NH4'), ('N1_WL', 'C_WL')):\\n\",\n    \"        p = paired(rows, 'A_H', arm, label=f'{key} paired [M+H]+ vs proxy')\\n\",\n    \"        pc = paired(rows, 'A_Hcut', arm, label=f'{key} paired [M+H]+cut vs proxy (diag)')\\n\",\n    \"        if not p.get('n'):\\n\",\n    \"            v = 'NO DATA'\\n\",\n    \"        elif not math.isfinite(p['ratio']):\\n\",\n    \"            v = 'UNDEFINED ([M+H]+ lift <= 0)'\\n\",\n    '        else:\\n',\n    \"            v = 'PASS' if p['ratio'] >= 0.5 else 'KILL'\\n\",\n    \"        g[key] = dict(verdict=v, rule='proxy lift >= 0.5 x paired [M+H]+ lift', paired=p, paired_cut=pc)\\n\",\n    \"    pe = paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'e180' and r['has_merged_H'], label='I14b e180 merged-CE')\\n\",\n    \"    pn = paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'np' and r['has_merged_H'], label='I14b np merged-CE')\\n\",\n    \"    if not pe.get('n') or not pn.get('n'):\\n\",\n    \"        v = 'NO DATA'\\n\",\n    '    else:\\n',\n    \"        v = 'ADOPT' if (pe['delta'] >= 0.01 and pn['delta'] >= 0.0) else 'REJECT'\\n\",\n    \"    g['I14b'] = dict(verdict=v, rule='delta >= +0.01 on e180 merged-CE and >= 0 on np merged-CE', e180=pe, np=pn,\\n\",\n    \"                     np_all=paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'np', label='I14b np all'),\\n\",\n    \"                     e180_all=paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'e180', label='I14b e180 all'))\\n\",\n    '    return g\\n',\n    '\\n',\n    '\\n',\n    'def blind_report(rows):\\n',\n    '    \"\"\"Molecules with NO real [M+H]+ spectrum (GLACIER-blind in B1): what N1 arm A recovers there.\"\"\"\\n',\n    \"    B = [r for r in rows if not r['has_H']]\\n\",\n    \"    res = OrderedDict(n_blind=len(B), n_blind_np=sum(r['set'] == 'np' for r in B),\\n\",\n    \"                      n_blind_with_proxy=sum(r['has_NH4p'] or r['has_WLp'] for r in B),\\n\",\n    \"                      n_blind_proxy_covered=sum(r['rr_N1A'] is not None for r in B))\\n\",\n    \"    C = [r for r in B if r['rr_N1A'] is not None]\\n\",\n    '    if C:\\n',\n    \"        rr = np.array([r['rr_N1A'] for r in C]); rn = np.array([r['rand'] for r in C])\\n\",\n    '        idx = _boot_idx(len(C))\\n',\n    '        res.update(mrr_proxy=float(rr.mean()), rand=float(rn.mean()), lift=float((rr - rn).mean()),\\n',\n    '                   lift_ci=[float(x) for x in np.percentile((rr - rn)[idx].mean(1), [2.5, 97.5])],\\n',\n    '                   top1=float(np.mean(rr >= 1.0 - 1e-12)),\\n',\n    \"                   by_src={k: int(sum(1 for r in C if r[k])) for k in ('has_NH4p', 'has_WLp')},\\n\",\n    \"                   mids=[r['mid'] for r in C])\\n\",\n    \"    res['overall_np'] = filled_delta(rows, 'A_H', 'N1A', sel=lambda r: r['set'] == 'np', label='N1A vs A, all np (blind filled with rand)')\\n\",\n    \"    res['overall_all'] = filled_delta(rows, 'A_H', 'N1A', label='N1A vs A, all molecules')\\n\",\n    \"    res['overall_np_D'] = filled_delta(rows, 'A_H', 'N1A_D', sel=lambda r: r['set'] == 'np', label='N1A+I14b vs A, all np')\\n\",\n    '    return res\\n',\n    '\\n',\n    '\\n',\n    'def score_corr(new, old):\\n',\n    '    \"\"\"new / old: {mid: {smiles: score|None}} -> pooled Pearson + mean per-molecule Spearman over common scored.\"\"\"\\n',\n    '    xs, ys, sp = [], [], []\\n',\n    '    for mid, d in new.items():\\n',\n    '        o = old.get(mid) or {}\\n',\n    '        pairs = [(v, o.get(s)) for s, v in d.items() if v is not None and o.get(s) is not None]\\n',\n    '        if len(pairs) < 3:\\n',\n    '            continue\\n',\n    '        a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])\\n',\n    '        xs.extend(a); ys.extend(b)\\n',\n    '        ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))\\n',\n    '        if ra.std() > 0 and rb.std() > 0:\\n',\n    '            sp.append(float(np.corrcoef(ra, rb)[0, 1]))\\n',\n    '    if len(xs) < 3:\\n',\n    '        return dict(n_mols=0)\\n',\n    '    return dict(n_mols=len(sp), n_pairs=len(xs), pearson=float(np.corrcoef(xs, ys)[0, 1]),\\n',\n    \"                spearman_mean=float(np.mean(sp)) if sp else float('nan'))\\n\",\n    '\\n',\n    '\\n',\n    '# -------------------------------------------------------------------------------------------------- self-test ----\\n',\n    'def selftest():\\n',\n    '    \"\"\"Forced unit tests of the pure functions (run on Kaggle before any GPU work and locally by the build).\"\"\"\\n',\n    '    M = 300.0\\n',\n    '    nh4 = dict(mz=[100.0, 250.0, 299.5, 300.0, 301.0, 318.03], it=[0.1, 1.0, 0.2, 0.3, 0.4, 0.5], prec=M + NH4_ADD,\\n',\n    \"               adduct='[M+NH4]+', mode='positive', instrument='timsTOF', ce_ev=[20.0], ce_orig='20', ce_units='eV')\\n\",\n    '    p = n1_proxy(nh4)\\n',\n    \"    assert p is not None and p['mz'] == [100.0, 250.0, 299.5] and p['it'] == [0.1, 1.0, 0.2], p\\n\",\n    \"    assert abs(p['prec'] - (M + PROTON)) < 1e-9 and p['adduct'] == '[M+H]+' and p['n1_src'] == '[M+NH4]+'\\n\",\n    \"    assert p['ce_ev'] == [20.0] and p['instrument'] == 'timsTOF' and nh4['adduct'] == '[M+NH4]+'   # input untouched\\n\",\n    \"    wl = dict(nh4, mz=[100.0, 281.4, 281.5, 283.0], it=[1.0, 0.5, 0.5, 0.9], prec=M + PROTON - H2O, adduct='[M-H2O+H]+')\\n\",\n    '    p = n1_proxy(wl)\\n',\n    \"    assert p is not None and p['mz'] == [100.0, 281.4] and abs(p['prec'] - (M + PROTON)) < 1e-9, p\\n\",\n    \"    for ad in ('[M+Na]+', '[M+K]+', '[M-2H2O+H]+', '[M+H]+', '[M-H]-'):\\n\",\n    '        assert n1_proxy(dict(nh4, adduct=ad)) is None, ad\\n',\n    \"    assert n1_proxy(dict(nh4, mode='negative')) is None\\n\",\n    '    assert n1_proxy(dict(nh4, mz=[310.0, 315.0], it=[1.0, 1.0])) is None             # nothing below the cut\\n',\n    '    assert n1_proxy(dict(nh4, mz=[100.0], it=[0.0])) is None                           # no positive intensity\\n',\n    \"    h = dict(nh4, adduct='[M+H]+', prec=M + PROTON, mz=[100.0, 299.5, 299.6, 301.0], it=[1.0, 1.0, 1.0, 1.0])\\n\",\n    '    c = h_cut(h)\\n',\n    \"    assert c is not None and c['mz'] == [100.0, 299.5] and c['prec'] == h['prec'] and c['adduct'] == '[M+H]+', c\\n\",\n    '    assert h_cut(nh4) is None\\n',\n    \"    items = [dict(mid='np:X', cands=['A', 'B'], spectra=[h, nh4, wl, dict(h, mode='negative', adduct='[M-H]-'),\\n\",\n    \"                                                        dict(h, adduct='[M+Na]+'), dict(h, adduct='[M+K]+')])]\\n\",\n    '    v, st = build_variants(items)\\n',\n    \"    assert [s['sid'] for s in v[0]['spectra']] == ['0:H', '0:Hcut', '1:NH4p', '2:WLp', '4:Na'], v\\n\",\n    '    assert st == dict(H=1, Hcut=1, NH4p=1, WLp=1, Na=1), st\\n',\n    '    # random tie-break expected RR\\n',\n    '    assert abs(rr_random_tb([0.5, 0.9, 0.5, 0.1]) - (1 / 2 + 1 / 3) / 2) < 1e-12\\n',\n    '    assert rr_random_tb([None, 0.2, 0.1]) is None and rr_random_tb([0.2, None, 0.1]) == 1.0\\n',\n    '    assert abs(rr_random_tb([0.0, 0.0, 0.0, 0.0]) - rr_rand(4)) < 1e-12\\n',\n    '    assert rr_truth_first([0.0, 0.0, 0.0, 0.0]) == 1.0 and abs(rr_split([0.0, 0.0, 0.0]) - 0.5) < 1e-12\\n',\n    '    assert abs(rr_rand(3) - (1 + 0.5 + 1 / 3) / 3) < 1e-12\\n',\n    '    assert covered([0.1, None, 0.2]) and not covered([0.1, None]) and not covered([None, 0.1, 0.2])\\n',\n    \"    # aggregation: mean over the arm's spectra, None-aware; N1A tag resolution\\n\",\n    \"    mo = {'cands': ['A', 'B', 'C'], 'spec': {\\n\",\n    \"        '0:H': dict(tag='H', nb=3, mean=[0.2, 0.4, None], merged=[0.3, 0.1, None]),\\n\",\n    \"        '1:H': dict(tag='H', nb=1, mean=[0.4, None, None], merged=[0.4, None, None]),\\n\",\n    \"        '2:NH4p': dict(tag='NH4p', nb=1, mean=[0.9, 0.1, 0.5], merged=[0.9, 0.1, 0.5])}}\\n\",\n    \"    am = arm_scores(mo, ('H',), 'mean')\\n\",\n    '    assert len(am) == 3 and np.allclose(am[:2], [0.3, 0.4]) and am[2] is None, am\\n',\n    \"    assert np.allclose(arm_scores(mo, ('H',), 'merged')[:2], [0.35, 0.1])\\n\",\n    \"    assert arm_tags('N1A', dict(has_H=True)) == (('H',), 'mean')\\n\",\n    \"    assert arm_tags('N1A', dict(has_H=False)) == (('NH4p', 'WLp'), 'mean')\\n\",\n    \"    assert arm_tags('N1A_D', dict(has_H=False)) == (('NH4p', 'WLp'), 'merged')\\n\",\n    '    # end-to-end on two synthetic molecules\\n',\n    \"    its = [dict(mid='np:X', cands=['A', 'B', 'C'], spectra=[dict(tag='H'), dict(tag='NH4p')]),\\n\",\n    \"           dict(mid='np:Y', cands=['A', 'B', 'C'], spectra=[dict(tag='NH4p')]),\\n\",\n    \"           dict(mid='e180:Z', cands=['A', 'B'], spectra=[dict(tag='H')])]\\n\",\n    \"    out = {'np:X': mo, 'np:Y': {'cands': ['A', 'B', 'C'], 'spec': {'0:NH4p': dict(tag='NH4p', nb=1, mean=[0.1, 0.5, 0.0], merged=[0.1, 0.5, 0.0])}}}\\n\",\n    \"    rows = molecule_table(its, out, done={'np:X', 'np:Y'})\\n\",\n    \"    assert [r['mid'] for r in rows] == ['np:X', 'np:Y']\\n\",\n    '    x, y = rows\\n',\n    \"    assert abs(x['rr_A_H'] - 0.5) < 1e-12 and x['rr_B_NH4'] == 1.0 and x['rr_N1A'] == x['rr_A_H'] and x['has_merged_H']\\n\",\n    \"    assert y['rr_A_H'] is None and abs(y['rr_N1A'] - 0.5) < 1e-12 and not y['has_H']\\n\",\n    \"    p = paired(rows, 'A_H', 'B_NH4')\\n\",\n    \"    assert p['n'] == 1 and abs(p['delta'] - 0.5) < 1e-12\\n\",\n    \"    f = filled_delta(rows, 'A_H', 'N1A')\\n\",\n    \"    assert f['n'] == 2 and abs(f['delta'] - (0.5 - rr_rand(3)) / 2) < 1e-12\\n\",\n    '    g = gates(rows)                                          # A lift < 0 here (0.5 vs rand 0.611)\\n',\n    \"    assert g['N1_NH4']['verdict'].startswith('UNDEFINED') and g['N1_WL']['verdict'] == 'NO DATA', g['N1_NH4']['verdict']\\n\",\n    \"    assert g['I14b']['verdict'] == 'NO DATA'\\n\",\n    '\\n',\n    '    def row(s, a, b, d, mh=True):\\n',\n    \"        return dict(mid=f'{s}:{a}{b}{d}', set=s, rand=0.2, rr_A_H=a, rr_B_NH4=b, rr_A_Hcut=None, rr_C_WL=None,\\n\",\n    '                    rr_D_Hmerged=d, has_merged_H=mh)\\n',\n    \"    gk = gates([row('np', 1.0, 0.5, 1.0), row('np', 0.6, 0.4, 0.6), row('e180', 1.0, None, 1.0)])   # ratio 0.25/0.6\\n\",\n    \"    assert gk['N1_NH4']['verdict'] == 'KILL' and abs(gk['N1_NH4']['paired']['ratio'] - 0.25 / 0.6) < 1e-12\\n\",\n    \"    assert gk['I14b']['verdict'] == 'REJECT' and gk['I14b']['e180']['delta'] == 0.0      # +0.01 needed on e180\\n\",\n    \"    gp = gates([row('np', 1.0, 0.9, 1.0), row('np', 0.6, 0.6, 0.5), row('e180', 0.5, None, 1.0)])   # ratio 1.1/1.2\\n\",\n    \"    assert gp['N1_NH4']['verdict'] == 'PASS' and gp['I14b']['verdict'] == 'REJECT'          # np merged delta < 0\\n\",\n    \"    ga = gates([row('np', 0.5, 0.5, 0.5), row('e180', 0.5, None, 0.52)])\\n\",\n    \"    assert ga['I14b']['verdict'] == 'ADOPT' and abs(ga['I14b']['e180']['delta'] - 0.02) < 1e-12\\n\",\n    '    b = blind_report(rows)\\n',\n    \"    assert b['n_blind'] == 1 and b['n_blind_proxy_covered'] == 1\\n\",\n    \"    sc = score_corr({'m': {'a': 1.0, 'b': 2.0, 'c': 3.0}}, {'m': {'a': 2.0, 'b': 4.0, 'c': 6.5}})\\n\",\n    \"    assert sc['n_mols'] == 1 and sc['pearson'] > 0.99 and abs(sc['spearman_mean'] - 1.0) < 1e-12\\n\",\n    \"    return 'glb_core selftest ok (proxy NH4/WL/Hcut, variants, random tie-break RR, aggregation, paired, gates, blind)'\\n\",\n])\n# kdev_glbench_n1 glb_driver.py, verbatim (GLACIER driver: mean + merged-CE per spectrum, own process)\nN1_DRIVER_SRC = ''.join([\n    '#!/usr/bin/env python\\n',\n    '\"\"\"casmi26 kdev_glbench_n1 -- GLACIER bench driver (runs in its OWN process, like gl_runner.py).\\n',\n    '\\n',\n    'Imports the attached casmi26-glacier gl_runner.py as a module and reuses ITS environment setup (offline wheel site,\\n',\n    'shims, RDKit 2025.03 check), model loading, batched Predictor (canonical-SMILES dedupe, OOM split, fail-fast), query\\n',\n    \"cleaning, CE bucketing, summed-CE merge() and entropy similarity. The 'mean' mode calls gl_runner.score_molecule on\\n\",\n    \"one spectrum at a time, so a molecule's mean over its [M+H]+ spectra is exactly the production gl_ex_mean.\\n\",\n    \"Added: per-spectrum output, and the I14b 'merged' mode (a merged-CE eV spectrum = >= 2 distinct CE buckets is\\n\",\n    'predicted at each bucket and the predictions are merge()d like the SUM_CE path; single-CE / NCE / no-CE spectra:\\n',\n    'merged = mean).\\n',\n    '\\n',\n    'usage: python glb_driver.py <gl_pkg> <in.json> <out.json> [--device cuda] [--budget S] [--site DIR] [--wheels DIR]\\n',\n    '                            [--bs 32] [--chunk-jobs 3000] [--fake]\\n',\n    'in.json : [{\"mid\", \"cands\": [smiles], \"spectra\": [{\"sid\", \"tag\", \"mz\", \"it\", \"prec\", \"adduct\", \"mode\", \"instrument\",\\n',\n    '            \"ce_ev\", \"ce_orig\", \"ce_units\"}]}]\\n',\n    'out.json: {mid: {\"cands\": [smiles], \"spec\": {sid: {\"tag\", \"nb\", \"mean\": [sim|null], \"merged\": [sim|null]}}}}\\n',\n    '          (lists aligned with \"cands\"; null = rejected SMILES / failed prediction)\\n',\n    'out.json.meta.json: status, counts, timings, versions, errors, \"done\" = molecules fully scored (input order).\\n',\n    '--fake: no torch / model / wheel site (local dry run): deterministic pseudo-random predictions, SMILES accepted as is.\\n',\n    '\"\"\"\\n',\n    'import os, sys, json, time, math, argparse, traceback, hashlib, importlib.util\\n',\n    '\\n',\n    'sys.dont_write_bytecode = True\\n',\n    \"NEED = ('setup_env', 'load_model', 'Predictor', 'smiles_ok', '_CANON', 'molecule_jobs', 'score_molecule',\\n\",\n    \"        'prep_spectrum', 'prep_query', 'entropy_sim', 'merge', 'post', 'ce_bucket', 'spectrum_ce', 'is_covered',\\n\",\n    \"        'instr_token', 'SUM_CE', 'FLOOR', 'TOPK', 'POWER', 'ENTW', 'TOL', 'PPM', 'write_json', 'Log')\\n\",\n    '\\n',\n    '\\n',\n    'def sha256(path):\\n',\n    '    h = hashlib.sha256()\\n',\n    \"    with open(path, 'rb') as f:\\n\",\n    \"        for b in iter(lambda: f.read(1 << 20), b''):\\n\",\n    '            h.update(b)\\n',\n    '    return h.hexdigest()\\n',\n    '\\n',\n    '\\n',\n    'def load_gl(pkg):\\n',\n    '    \"\"\"gl_runner.py of the package as a module (no side effects beyond its imports: numpy, numba).\"\"\"\\n',\n    \"    spec = importlib.util.spec_from_file_location('gl_runner', os.path.join(pkg, 'gl_runner.py'))\\n\",\n    '    G = importlib.util.module_from_spec(spec)\\n',\n    \"    sys.modules['gl_runner'] = G\\n\",\n    '    spec.loader.exec_module(G)\\n',\n    '    miss = [n for n in NEED if not hasattr(G, n)]\\n',\n    '    if miss:\\n',\n    \"        raise RuntimeError(f'gl_runner API differs from the one this driver was written for: missing {miss}')\\n\",\n    '    return G\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------- per-spectrum plan ----\\n',\n    'def merged_buckets(G, sp):\\n',\n    '    \"\"\"I14b: sorted distinct CE buckets of a merged-CE eV spectrum when there are >= 2, else None.\"\"\"\\n',\n    \"    if sp.get('ce_units') == 'NCE':\\n\",\n    '        return None\\n',\n    '    b = set()\\n',\n    \"    for x in sp.get('ce_ev') or []:\\n\",\n    '        try:\\n',\n    '            x = float(x)\\n',\n    '        except (TypeError, ValueError):\\n',\n    '            continue\\n',\n    '        if math.isfinite(x):\\n',\n    '            cb = G.ce_bucket(x)\\n',\n    '            if math.isfinite(cb):\\n',\n    '                b.add(float(cb))\\n',\n    '    return sorted(b) if len(b) >= 2 else None\\n',\n    '\\n',\n    '\\n',\n    'def spec_plan(G, sp):\\n',\n    '    \"\"\"None if gl_runner would not score this spectrum, else {\\'entry\\': molecule_plan entry, \\'mb\\': buckets|None}.\"\"\"\\n',\n    '    if not G.is_covered(sp):\\n',\n    '        return None\\n',\n    '    qm, qp = G.prep_spectrum(sp)\\n',\n    \"    return dict(entry=(qm, qp, sp['adduct'], G.instr_token(sp), G.ce_bucket(G.spectrum_ce(sp))), mb=merged_buckets(G, sp))\\n\",\n    '\\n',\n    '\\n',\n    'def spec_jobs(G, plan, cands):\\n',\n    \"    J = set(G.molecule_jobs([plan['entry']], cands))\\n\",\n    \"    if plan['mb']:\\n\",\n    \"        _, _, ad, ins, _ = plan['entry']\\n\",\n    '        for c in cands:\\n',\n    \"            for b in plan['mb']:\\n\",\n    '                J.add((c, ad, ins, b))\\n',\n    '    return J\\n',\n    '\\n',\n    '\\n',\n    'def spec_scores(G, plan, cands, cache):\\n',\n    '    \"\"\"-> (mean {c: sim|None}, merged {c: sim|None}, n_buckets).\"\"\"\\n',\n    \"    mean = G.score_molecule([plan['entry']], cands, cache)\\n\",\n    \"    if not plan['mb']:\\n\",\n    '        return mean, dict(mean), 1\\n',\n    \"    qm, qp, ad, ins, _ = plan['entry']\\n\",\n    '    merged = {}\\n',\n    '    for c in cands:\\n',\n    \"        parts = [cache.get((c, ad, ins, b)) for b in plan['mb']]\\n\",\n    '        if any(p is None for p in parts):            # = the SUM_CE path: every component must be predicted\\n',\n    '            merged[c] = None\\n',\n    '            continue\\n',\n    '        pr = G.merge(parts)\\n',\n    '        pm, pp = G.prep_query(pr[0], pr[1], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\\n',\n    '        s = float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))\\n',\n    '        merged[c] = s if math.isfinite(s) else None\\n',\n    \"    return mean, merged, len(plan['mb'])\\n\",\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------ fake mode ------\\n',\n    'class FakePredictor:\\n',\n    '    \"\"\"Same interface as gl_runner.Predictor (predict(jobs, deadline) -> bool, .cache); deterministic random peaks.\"\"\"\\n',\n    '\\n',\n    '    def __init__(self, G):\\n',\n    '        self.G, self.cache = G, {}\\n',\n    '        self.n_pred = self.n_fail = self.n_feat_fail = 0\\n',\n    '        self.t_pred, self.bs = 0.0, 0\\n',\n    '\\n',\n    '    def predict(self, jobs, deadline):\\n',\n    '        import numpy as np\\n',\n    '        t0 = time.time()\\n',\n    '        for j in sorted(jobs):\\n',\n    '            if j in self.cache:\\n',\n    '                continue\\n',\n    '            if time.time() > deadline:\\n',\n    '                self.t_pred += time.time() - t0\\n',\n    '                return False\\n',\n    '            c = self.G._CANON.get(j[0], (None,))[0]\\n',\n    '            if c is None:\\n',\n    '                self.cache[j] = None\\n',\n    '                continue\\n',\n    '            r = np.random.default_rng(int(hashlib.md5(repr((c, j[1], j[2], float(j[3]))).encode()).hexdigest()[:8], 16))\\n',\n    '            mz = np.round(r.uniform(40.0, 400.0, 25), 2)\\n',\n    '            o = self.G.post(np.stack([mz, r.uniform(0.05, 1.0, 25)], 1).astype(np.float32))\\n',\n    '            self.cache[j] = (o[0].astype(np.float64), o[1].astype(np.float64))\\n',\n    '            self.n_pred += 1\\n',\n    '        self.t_pred += time.time() - t0\\n',\n    '        return True\\n',\n    '\\n',\n    '\\n',\n    'def fake_smiles_ok(G):\\n',\n    '    def ok(smi):\\n',\n    '        good = isinstance(smi, str) and bool(smi)\\n',\n    \"        G._CANON[smi] = (smi if good else None, max(1, len(smi) // 2) if good else 0, '' if good else 'empty')\\n\",\n    \"        return good, ('' if good else 'empty')\\n\",\n    '    return ok\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------- self-test -----\\n',\n    'def selftest(G):\\n',\n    '    \"\"\"Forced unit test of the merged-CE logic on a synthetic cache (no model); run by the notebook before the GPU\\n',\n    '    job and by the local build.\"\"\"\\n',\n    '    import numpy as np\\n',\n    \"    sp = dict(mz=[50.0, 80.0, 120.0, 150.5], it=[0.2, 1.0, 0.5, 0.3], prec=151.0, adduct='[M+H]+', mode='positive',\\n\",\n    \"              instrument='timsTOF', ce_ev=[20.0, 40.0, 60.0], ce_orig='20,40,60', ce_units='eV')\\n\",\n    '    p = spec_plan(G, sp)\\n',\n    \"    assert p is not None and p['mb'] == [20.0, 40.0, 60.0], p\\n\",\n    \"    assert p['entry'][2:] == ('[M+H]+', 'QTOF', 40.0), p['entry'][2:]\\n\",\n    \"    cands = ['CCO', 'CCC']\\n\",\n    '    J = spec_jobs(G, p, cands)\\n',\n    \"    assert J == {(c, '[M+H]+', 'QTOF', b) for c in cands for b in (20.0, 40.0, 60.0)}, J\\n\",\n    '    rng = np.random.default_rng(0)\\n',\n    '    cache = {}\\n',\n    '    for j in sorted(J):\\n',\n    '        mz = np.sort(rng.choice([50.0, 80.0, 120.0, 150.5, 60.0, 99.0], 4, replace=False))\\n',\n    '        cache[j] = (mz.astype(np.float64), rng.uniform(0.1, 1.0, 4).astype(np.float64))\\n',\n    \"    cache[('CCC', '[M+H]+', 'QTOF', 60.0)] = None\\n\",\n    '    mean, merged, nb = spec_scores(G, p, cands, cache)\\n',\n    \"    assert nb == 3 and merged['CCC'] is None and mean['CCC'] is not None and mean['CCO'] is not None, (mean, merged)\\n\",\n    \"    pr = G.merge([cache[('CCO', '[M+H]+', 'QTOF', b)] for b in (20.0, 40.0, 60.0)])\\n\",\n    '    qm, qp = G.prep_spectrum(sp)\\n',\n    '    pm, pp = G.prep_query(pr[0], pr[1], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\\n',\n    \"    assert abs(merged['CCO'] - float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))) < 1e-12\\n\",\n    \"    pm, pp = G.prep_query(*cache[('CCO', '[M+H]+', 'QTOF', 40.0)], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\\n\",\n    \"    assert abs(mean['CCO'] - float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))) < 1e-12\\n\",\n    '    p1 = spec_plan(G, dict(sp, ce_ev=[40.0]))                         # single CE: merged == mean\\n',\n    '    m1, g1, n1 = spec_scores(G, p1, cands, cache)\\n',\n    \"    assert p1['mb'] is None and n1 == 1 and m1 == g1 and abs(m1['CCO'] - mean['CCO']) < 1e-12\\n\",\n    '    assert merged_buckets(G, dict(sp, ce_ev=[20.0, 21.0])) is None    # same bucket twice\\n',\n    '    assert merged_buckets(G, dict(sp, ce_ev=[0.0, 20.0])) == [5.0, 20.0]\\n',\n    \"    assert merged_buckets(G, dict(sp, ce_units='NCE', ce_orig='20,40')) is None\\n\",\n    '    assert merged_buckets(G, dict(sp, ce_ev=[])) is None\\n',\n    \"    assert spec_plan(G, dict(sp, adduct='[M+NH4]+')) is None and spec_plan(G, dict(sp, mode='negative')) is None\\n\",\n    '    pn = spec_plan(G, dict(sp, ce_ev=[]))                              # no CE: SUM_CE jobs, merged == mean\\n',\n    \"    assert pn['mb'] is None and spec_jobs(G, pn, ['CCO']) == {('CCO', '[M+H]+', 'QTOF', c) for c in G.SUM_CE}\\n\",\n    \"    return 'glb_driver selftest ok (merged-CE buckets, jobs, merge == SUM_CE path, single-CE identity, coverage)'\\n\",\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------------ main -----\\n',\n    'def parse(argv=None):\\n',\n    '    ap = argparse.ArgumentParser()\\n',\n    \"    ap.add_argument('pkg'); ap.add_argument('inp'); ap.add_argument('out')\\n\",\n    \"    ap.add_argument('--device', default='cuda')\\n\",\n    \"    ap.add_argument('--budget', type=float, default=0.0)\\n\",\n    \"    ap.add_argument('--site', default='/tmp/glb_site')\\n\",\n    \"    ap.add_argument('--wheels', default=None)\\n\",\n    \"    ap.add_argument('--bs', type=int, default=32)\\n\",\n    \"    ap.add_argument('--chunk-jobs', type=int, default=3000)\\n\",\n    \"    ap.add_argument('--fake', action='store_true')\\n\",\n    '    return ap.parse_args(argv)\\n',\n    '\\n',\n    '\\n',\n    'def main(argv=None):\\n',\n    '    a = parse(argv)\\n',\n    '    if not a.fake:\\n',\n    \"        os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # = gl_runner.main (deterministic torch)\\n\",\n    '    t_start = time.time()\\n',\n    \"    deadline = t_start + a.budget if a.budget > 0 else float('inf')\\n\",\n    \"    meta = dict(model='glacier-bench', status='init', fake=a.fake, n_mols=0, n_mols_done=0, n_spec=0, n_spec_covered=0,\\n\",\n    '                n_spec_merged=0, n_jobs=0, errors=[], done=[])\\n',\n    '    G = None\\n',\n    '    result = {}\\n',\n    '\\n',\n    '    def write(status=None):\\n',\n    '        if status:\\n',\n    \"            meta['status'] = status\\n\",\n    \"        meta['seconds'] = round(time.time() - t_start, 2)\\n\",\n    \"        tmp = a.out + '.tmp'\\n\",\n    \"        with open(tmp, 'w') as f:\\n\",\n    '            json.dump(result, f, allow_nan=False)\\n',\n    '        os.replace(tmp, a.out)\\n',\n    \"        with open(a.out + '.meta.json.tmp', 'w') as f:          # atomic too: a kill mid-write never leaves bad JSON\\n\",\n    '            json.dump(meta, f, allow_nan=False)\\n',\n    \"        os.replace(a.out + '.meta.json.tmp', a.out + '.meta.json')\\n\",\n    '\\n',\n    '    def log(*x):\\n',\n    \"        print(f'[glb {time.time() - t_start:7.1f}s]', *x, file=sys.stderr, flush=True)\\n\",\n    '\\n',\n    '    try:\\n',\n    '        mols = json.load(open(a.inp))\\n',\n    \"        meta['n_mols'] = len(mols)\\n\",\n    '        write()\\n',\n    '        G = load_gl(a.pkg)\\n',\n    \"        meta['gl_runner_sha256'] = sha256(os.path.join(a.pkg, 'gl_runner.py'))\\n\",\n    '        if a.fake:\\n',\n    \"            meta['rdkit'] = 'fake'\\n\",\n    '            G.smiles_ok = fake_smiles_ok(G)\\n',\n    '            pred = FakePredictor(G)\\n',\n    \"            log('FAKE mode: pseudo-random predictions, no model')\\n\",\n    '        else:\\n',\n    '            ns = argparse.Namespace(pkg=a.pkg, site=a.site, wheels=a.wheels, any_rdkit=False)\\n',\n    \"            meta['rdkit'] = G.setup_env(ns, log)\\n\",\n    '            import torch\\n',\n    \"            meta['torch'] = torch.__version__\\n\",\n    '            dev = a.device\\n',\n    \"            if dev.startswith('cuda') and not torch.cuda.is_available():\\n\",\n    \"                log('cuda unavailable -> cpu'); dev = 'cpu'\\n\",\n    \"            meta['device'] = dev\\n\",\n    '            torch.use_deterministic_algorithms(True, warn_only=True)\\n',\n    '            torch.backends.cudnn.benchmark = False\\n',\n    '            model = G.load_model(os.path.abspath(a.pkg), dev, log)\\n',\n    '            pred = G.Predictor(model, dev, a.bs, log)\\n',\n    '            log(f\\'model loaded on {dev} (rdkit {meta[\"rdkit\"]}, torch {torch.__version__}, numba {G.HAVE_NUMBA})\\')\\n',\n    \"        meta['t_load'] = round(time.time() - t_start, 2)\\n\",\n    '        okc = {}\\n',\n    '        chunk, chunk_jobs = [], set()\\n',\n    '\\n',\n    '        def flush():\\n',\n    '            nonlocal chunk, chunk_jobs\\n',\n    '            if not chunk:\\n',\n    '                return True\\n',\n    '            t0 = time.time()\\n',\n    '            if not pred.predict(chunk_jobs, deadline):\\n',\n    '                return False\\n',\n    '            for mid, cands, plans in chunk:\\n',\n    '                spec = {}\\n',\n    '                for sid, tag, pl in plans:\\n',\n    '                    try:\\n',\n    '                        mean, merged, nb = spec_scores(G, pl, cands, pred.cache)\\n',\n    '                    except Exception as e:\\n',\n    \"                        meta['errors'].append(f'{mid} {sid}: score {repr(e)[:200]}')\\n\",\n    '                        continue\\n',\n    \"                    spec[sid] = dict(tag=tag, nb=nb, mean=[mean.get(c) for c in result[mid]['cands']],\\n\",\n    \"                                     merged=[merged.get(c) for c in result[mid]['cands']])\\n\",\n    \"                result[mid]['spec'] = spec\\n\",\n    \"                meta['done'].append(mid)\\n\",\n    \"            meta['n_mols_done'] = len(meta['done'])\\n\",\n    '            write()\\n',\n    \"            log(f'chunk: {len(chunk)} mols {len(chunk_jobs)} jobs {time.time() - t0:.1f}s '\\n\",\n    '                f\\'({meta[\"n_mols_done\"]}/{meta[\"n_mols\"]} mols done, {pred.n_pred} predictions)\\')\\n',\n    '            chunk, chunk_jobs = [], set()\\n',\n    '            return True\\n',\n    '\\n',\n    '        stopped = False\\n',\n    '        for mol in mols:\\n',\n    \"            mid = str(mol['mid'])\\n\",\n    \"            allc = list(dict.fromkeys(mol.get('cands') or []))\\n\",\n    '            result[mid] = dict(cands=allc, spec={})\\n',\n    '            for c in allc:\\n',\n    '                if c not in okc:\\n',\n    '                    okc[c], why = G.smiles_ok(c)\\n',\n    '                    if not okc[c] and sum(1 for v in okc.values() if not v) <= 5:\\n',\n    \"                        meta['errors'].append(f'smiles rejected ({why}): {str(c)[:100]}')\\n\",\n    '            cands = [c for c in allc if okc[c]]\\n',\n    '            plans = []\\n',\n    \"            for sp in mol.get('spectra') or []:\\n\",\n    \"                meta['n_spec'] += 1\\n\",\n    '                try:\\n',\n    '                    pl = spec_plan(G, sp)\\n',\n    '                except Exception as e:\\n',\n    '                    meta[\\'errors\\'].append(f\\'{mid} {sp.get(\"sid\")}: plan {repr(e)[:200]}\\')\\n',\n    '                    continue\\n',\n    '                if pl is None:\\n',\n    '                    continue\\n',\n    \"                meta['n_spec_covered'] += 1\\n\",\n    \"                meta['n_spec_merged'] += int(bool(pl['mb']))\\n\",\n    \"                plans.append((sp['sid'], sp.get('tag'), pl))\\n\",\n    '            if not cands or not plans:\\n',\n    \"                meta['done'].append(mid)                         # nothing GLACIER could score: done, empty\\n\",\n    '                continue\\n',\n    '            J = set()\\n',\n    '            for _, _, pl in plans:\\n',\n    '                J |= spec_jobs(G, pl, cands)\\n',\n    \"            meta['n_jobs'] += len(J)\\n\",\n    '            chunk.append((mid, cands, plans))\\n',\n    '            chunk_jobs |= J\\n',\n    '            if len(chunk_jobs) >= a.chunk_jobs and not flush():\\n',\n    '                stopped = True\\n',\n    '                break\\n',\n    '        if not stopped and not flush():\\n',\n    '            stopped = True\\n',\n    '        meta.update(n_pred=pred.n_pred, n_pred_fail=pred.n_fail, n_feat_fail=pred.n_feat_fail, t_pred=round(pred.t_pred, 2),\\n',\n    '                    pred_per_s=round(pred.n_pred / max(pred.t_pred, 1e-9), 2), bs_final=pred.bs,\\n',\n    \"                    n_bad_smiles=sum(1 for v in okc.values() if not v), n_mols_done=len(meta['done']))\\n\",\n    \"        if not a.fake and meta.get('device', '').startswith('cuda'):\\n\",\n    '            import torch\\n',\n    \"            meta['max_mem_gb'] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)\\n\",\n    \"        write('budget' if stopped else 'ok')\\n\",\n    \"        log('status', meta['status'], {k: meta[k] for k in ('n_mols', 'n_mols_done', 'n_spec_covered', 'n_spec_merged',\\n\",\n    \"                                                            'n_pred', 'n_pred_fail', 'pred_per_s')})\\n\",\n    '        return 0\\n',\n    '    except BaseException as e:                                   # never crash the notebook: status error + partial out\\n',\n    \"        meta['errors'].append(traceback.format_exc()[-3000:])\\n\",\n    \"        log('ERROR', repr(e)[:300])\\n\",\n    '        try:\\n',\n    \"            write('error')\\n\",\n    '        except Exception:\\n',\n    '            return 2\\n',\n    '        return 0\\n',\n    '\\n',\n    '\\n',\n    \"if __name__ == '__main__':\\n\",\n    '    sys.exit(main())\\n',\n])\n\n\ndef _n1_mod(name, path):\n    import importlib.util as _iu\n    spec = _iu.spec_from_file_location(name, path)\n    mod = _iu.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef n1_select(items_all, K):\n    \"\"\"N1 arm A (bench glb_core) on fuse.build_ice_input(..., covered_only=False) items -> (driver items, counts):\n    the molecules WITHOUT a real positive [M+H]+ spectrum that have >= 1 [M+NH4]+ / [M-H2O+H]+ proxy, carrying only\n    those proxies (bench tags NH4p / WLp, sid '<spectrum index>:<tag>'), in input (= GLACIER priority) order.\"\"\"\n    var, _ = K.build_variants(items_all)\n    raw = {str(it['mid']): sum(1 for s in it.get('spectra') or [] if s.get('mode') == 'positive'\n                               and s.get('adduct') in K.PROXY_TAG) for it in items_all}\n    out = []\n    st = dict(molecules=0, with_H=0, blind=0, blind_nh4_wl=0, n1=0, proxies=0, NH4p=0, WLp=0, proxy_empty=0)\n    for it in var:\n        fl = K.mol_flags(it)\n        st['molecules'] += 1\n        if fl['has_H']:                  # arm A: a real [M+H]+ exists -> the H1 run above scores this molecule\n            st['with_H'] += 1\n            continue\n        st['blind'] += 1\n        tags, _mode = K.arm_tags(N1_ARM, fl)\n        sps = [s for s in it['spectra'] if s.get('tag') in tags]\n        st['blind_nh4_wl'] += int(raw.get(it['mid'], 0) > 0)\n        st['proxy_empty'] += raw.get(it['mid'], 0) - len(sps)\n        if not sps or not it['cands']:\n            continue\n        st['n1'] += 1\n        st['proxies'] += len(sps)\n        for s in sps:\n            st[s['tag']] += 1\n        out.append(dict(mid=it['mid'], cands=list(it['cands']), spectra=sps))\n    return out, st\n\n\ndef n1_run(items, budget):\n    \"\"\"The bench glb_driver.py in its own process (= gl_fuse.run_gl pattern) -> (out, meta, done). Never raises for a\n    driver failure: the driver checkpoints after every chunk, so a budget stop / crash / kill keeps the molecules it\n    finished; unreadable output -> nothing done.\"\"\"\n    fin, fout = os.path.join(N1_WORK, 'n1_in.json'), os.path.join(N1_WORK, 'n1_out.json')\n    for f in (fout, fout + '.meta.json'):\n        if os.path.exists(f):\n            os.remove(f)\n    with open(fin, 'w') as f:\n        json.dump(items, f)\n    cmd = [sys.executable, os.path.join(N1_WORK, 'glb_driver.py'), GL_PKG, fin, fout, '--device', N1_DEVICE,\n           '--budget', str(float(budget)), '--site', N1_SITE, '--wheels', os.path.join(ICE_PKG, 'wheels')] + list(N1_DRIVER_EXTRA)\n    t0 = time.time()\n    try:\n        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace',\n                           timeout=float(budget) + 600)\n        tail = (r.stdout or '')[-3000:]\n    except subprocess.TimeoutExpired as e:\n        tail = f'TIMEOUT after {time.time() - t0:.0f}s ' + (e.stdout.decode(errors='replace') if isinstance(e.stdout, bytes)\n                                                            else str(e.stdout or ''))[-2000:]\n    print(tail, flush=True)\n    meta, out = {}, {}\n    try:\n        with open(fout + '.meta.json') as f:\n            meta = json.load(f)\n    except Exception as e:\n        print('[N1] driver meta unreadable:', repr(e), flush=True)\n    try:\n        with open(fout) as f:\n            out = json.load(f)\n    except Exception as e:\n        print('[N1] driver output unreadable:', repr(e), flush=True)\n    done = set(meta.get('done') or []) if meta else {m for m, v in out.items() if isinstance(v, dict) and v.get('spec')}\n    return out, meta, done\n\n\ndef n1_scores(out, done, items, K, mode=None):\n    \"\"\"Driver output -> {mid: {smiles: score | None}} = bench arm_scores of N1_ARM (mean over the molecule's proxy\n    spectra of the per-spectrum similarity in the arm's CE mode, or `mode`); finished molecules only, candidate order\n    as sent; non-finite / outside [0, 1] -> None; molecules without any score are left out.\"\"\"\n    tags, amode = K.arm_tags(N1_ARM, dict(has_H=False))\n    res = {}\n    for it in items:\n        mid = str(it['mid'])\n        mo = out.get(mid)\n        if mid not in done or not isinstance(mo, dict):\n            continue\n        cands = [str(c) for c in mo.get('cands') or []]\n        if cands != [str(c) for c in dict.fromkeys(it['cands'])]:\n            print('[N1] candidate order changed -> molecule dropped', mid, flush=True)\n            continue\n        d = {}\n        for c, x in zip(cands, K.arm_scores(mo, tags, mode or amode)):\n            d[c] = float(x) if x is not None and math.isfinite(x) and -1e-9 <= x <= 1.0 + 1e-9 else None\n        if any(x is not None for x in d.values()):\n            res[mid] = d\n    return res\n\n\ndef n1_synthetic_te(te_df, mids, K):\n    \"\"\"Commit-only forced exercise input: for every molecule in `mids` with a positive [M+H]+ spectrum a copy\n    'n1x:<mid>' WITHOUT any [M+H]+: each positive [M+H]+ row -> an [M+NH4]+ row (precursor + 18.033823 - 1.007276,\n    plus an injected peak AT the NH4 precursor that the proxy cut must drop); the first one also -> an [M-H2O+H]+ row\n    (precursor - 18.010565, peaks unchanged); original NH4 / water-loss rows are left out; every other row (negative,\n    Na, K, 2M, ...) is copied unchanged and must never be proxied.\n    -> (DataFrame, {xmid: [(kind, original [M+H]+ precursor, mz list, intensity list) per row]})\"\"\"\n    rows, ref = [], {}\n    want = {str(m) for m in mids}\n    for mid, sub in te_df.groupby(te_df.molecule_id.astype(str), sort=False):\n        if mid not in want:\n            continue\n        recs = sub.to_dict('records')\n        if not any(r.get('ionization_mode') == 'positive' and r.get('adduct') == '[M+H]+' for r in recs):\n            continue\n        xm, first, rf = f'n1x:{mid}', True, []\n        for r in recs:\n            r = dict(r, molecule_id=xm)\n            if r.get('ionization_mode') == 'positive' and r.get('adduct') == '[M+H]+':\n                prec = float(r['precursor_mz'])\n                mz = [float(x) for x in r['ms2_mzs']]\n                it = [float(x) for x in r['ms2_normalized_intensities']]\n                if first:\n                    rows.append(dict(r, adduct='[M-H2O+H]+', precursor_mz=prec - K.H2O, ms2_mzs=np.asarray(mz),\n                                     ms2_normalized_intensities=np.asarray(it)))\n                    rf.append(('WL', prec, mz, it))\n                    first = False\n                p2 = prec - K.PROTON + K.NH4_ADD\n                rows.append(dict(r, adduct='[M+NH4]+', precursor_mz=p2, ms2_mzs=np.asarray(mz + [p2]),\n                                 ms2_normalized_intensities=np.asarray(it + [1.0])))\n                rf.append(('NH4', prec, mz, it))\n            elif r.get('ionization_mode') == 'positive' and r.get('adduct') in K.PROXY_TAG:\n                continue                 # an original NH4 / water-loss row: left out, so every proxy traces to a converted row\n            else:\n                rows.append(r)\n                rf.append(('keep', None, None, None))\n        ref[xm] = rf\n    return pd.DataFrame(rows), ref\n\n\ndef n1_check_proxies(x_items, ref, K):\n    \"\"\"Commit asserts on the synthetic proxies: relabel, precursor back at the original [M+H]+ m/z (< 1e-6 Da), cut =\n    exactly the original peaks below the bench cut (injected NH4 precursor peak gone), one proxy per converted row.\"\"\"\n    n = 0\n    for it in x_items:\n        rf = ref[it['mid']]\n        for s in it['spectra']:\n            j, tag = s['sid'].split(':')\n            kind, prec0, mz0, it0 = rf[int(j)]\n            assert (tag, kind) in (('NH4p', 'NH4'), ('WLp', 'WL')), (it['mid'], s['sid'], kind)\n            assert s['adduct'] == '[M+H]+' and s['mode'] == 'positive' and s['n1_src'] == (\n                '[M+NH4]+' if kind == 'NH4' else '[M-H2O+H]+'), (it['mid'], s['sid'])\n            assert abs(float(s['prec']) - prec0) < 1e-6, (it['mid'], s['sid'], s['prec'], prec0)\n            if kind == 'NH4':\n                p2 = prec0 - K.PROTON + K.NH4_ADD\n                cut = (p2 - K.NH4_ADD) + K.PROTON - K.CUT_DA\n            else:\n                cut = (prec0 - K.H2O) - K.CUT_DA\n            keep = [(m, i) for m, i in zip(mz0, it0) if m < cut]\n            assert s['mz'] == [m for m, _ in keep] and s['it'] == [i for _, i in keep], (it['mid'], s['sid'])\n            assert max(s['mz']) < float(s['prec']) - K.CUT_DA + 1e-6, (it['mid'], s['sid'])\n            n += 1\n        assert {s['tag'] for s in it['spectra']} <= {'NH4p', 'WLp'}, it['mid']\n    return n\n\n\ndef _n1_spearman(a, b):\n    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)\n    ra, rb = np.argsort(np.argsort(a)), np.argsort(np.argsort(b))\n    return float(np.corrcoef(ra, rb)[0, 1]) if ra.std() > 0 and rb.std() > 0 else float('nan')\n\n\ndef n1_ice_finite(ice):\n    \"\"\"Number of finite ICEBERG scores in one molecule's ICE dict (None / NaN / inf / non-numeric = no evidence; the\n    per-value test of fuse.rerank and of E006's _finite_coverage).\"\"\"\n    import math as _m\n    n = 0\n    for v in (ice.values() if isinstance(ice, dict) else ()):\n        try:\n            n += v is not None and _m.isfinite(float(v))\n        except (TypeError, ValueError):\n            pass\n    return n\n\n\ndef n1_complement(scores, ice_scores):\n    \"\"\"k2 variant A: drop the N1 GL scores of the molecules whose ICE dict (ice_scores[str(mid)]) holds a finite\n    score -> (kept {mid: same score dict}, dropped {mid}). Kept dicts are passed through unchanged.\"\"\"\n    ice_scores = ice_scores if isinstance(ice_scores, dict) else {}\n    drop = {m for m in scores if n1_ice_finite(ice_scores.get(str(m))) > 0}\n    return {m: v for m, v in scores.items() if m not in drop}, drop\n\n\ntry:\n    import math\n    _n1_h1_gl = isinstance(GL_SCORES, dict) and any(isinstance(g, dict) and any(v is not None for v in g.values())\n                                                  for g in GL_SCORES.values())\n    if not _n1_h1_gl:\n        raise RuntimeError('H1 GLACIER produced no scores (GL failed / not run) -> N1 not attempted')\n    os.makedirs(N1_WORK, exist_ok=True)\n    for _n1_fn, _n1_s, _n1_h in (('glb_core.py', N1_CORE_SRC, N1_CORE_SHA256), ('glb_driver.py', N1_DRIVER_SRC, N1_DRIVER_SHA256)):\n        assert hashlib.sha256(_n1_s.encode('utf-8')).hexdigest() == _n1_h, ('embedded bench source changed', _n1_fn)\n        with open(os.path.join(N1_WORK, _n1_fn), 'w', encoding='utf-8') as _n1_f:\n            _n1_f.write(_n1_s)\n    N1K = _n1_mod('glb_core', os.path.join(N1_WORK, 'glb_core.py'))\n    N1DR = _n1_mod('glb_driver', os.path.join(N1_WORK, 'glb_driver.py'))\n    print('[N1]', N1K.selftest(), flush=True)\n    _n1_G = N1DR.load_gl(GL_PKG)            # attached gl_runner.py as a module (numpy / numba only; no torch here)\n    print('[N1]', N1DR.selftest(_n1_G), flush=True)\n    _n1_sg = sha256(os.path.join(GL_PKG, 'gl_runner.py'))\n    print(f'[N1] code: glb_core {N1_CORE_SHA256[:12]} glb_driver {N1_DRIVER_SHA256[:12]} (= kdev_glbench_n1) | '\n          f'gl_runner.py {_n1_sg[:12]} matches the B1/bench MANIFEST: {_n1_sg == N1_GL_RUNNER_SHA256} | arm {N1_ARM} '\n          f'(tags, mode) {N1K.arm_tags(N1_ARM, dict(has_H=False))} | budget {N1_BUDGET:.0f}s', flush=True)\n    _n1_items_all = ice_fuse.build_ice_input(te, cands, covered_only=False)\n    _n1_items, N1_STATS = n1_select(_n1_items_all, N1K)\n    print(f\"[N1] molecules: GL/ICE input {N1_STATS['molecules']} | with real [M+H]+ {N1_STATS['with_H']} (H1 run, \"\n          f\"unchanged) | blind {N1_STATS['blind']} (with NH4/WL spectra {N1_STATS['blind_nh4_wl']}) | N1 molecules \"\n          f\"{N1_STATS['n1']} | proxies built {N1_STATS['proxies']} (NH4p {N1_STATS['NH4p']}, WLp {N1_STATS['WLp']}; \"\n          f\"empty after the cut {N1_STATS['proxy_empty']})\", flush=True)\n    _n1_x_items, _n1_x_ref = [], {}\n    if not IS_RERUN:                     # forced commit exercise: synthetic blind copies through the same code\n        _n1_tx, _n1_x_ref = n1_synthetic_te(te, list(cands), N1K)\n        _n1_x_all = ice_fuse.build_ice_input(_n1_tx, {f'n1x:{m}': c for m, c in cands.items() if f'n1x:{m}' in _n1_x_ref},\n                                          covered_only=False)\n        _n1_x_items, N1X_STATS = n1_select(_n1_x_all, N1K)\n        N1X_STATS['proxy_checked'] = n1_check_proxies(_n1_x_items, _n1_x_ref, N1K)\n        assert N1X_STATS['blind'] == N1X_STATS['molecules'] == len(_n1_x_ref) >= 1 and N1X_STATS['with_H'] == 0, N1X_STATS\n        assert N1X_STATS['n1'] >= 1 and N1X_STATS['proxy_checked'] == N1X_STATS['proxies'] >= N1X_STATS['n1'], N1X_STATS\n        assert N1X_STATS['proxies'] + N1X_STATS['proxy_empty'] == sum(k != 'keep' for v in _n1_x_ref.values() for k, *_ in v)\n        # arm-B guard: a molecule WITH a real [M+H]+ plus NH4 / water-loss spectra must NOT become an N1 molecule\n        _n1_src = sorted(_n1_x_ref)[0].split(':', 1)[1]\n        _n1_gdf = pd.concat([te[te.molecule_id.astype(str) == _n1_src], _n1_tx[_n1_tx.molecule_id == f'n1x:{_n1_src}']],\n                            ignore_index=True).assign(molecule_id=f'n1b:{_n1_src}')\n        _n1_gi, _n1_gs = n1_select(ice_fuse.build_ice_input(_n1_gdf, {f'n1b:{_n1_src}': {str(m): c for m, c in cands.items()}[_n1_src]},\n                                                           covered_only=False), N1K)\n        assert _n1_gi == [] and _n1_gs['with_H'] == 1 and _n1_gs['n1'] == 0, ('arm-B guard: [M+H]+ molecule selected', _n1_gs)\n        N1X_STATS['armB_guard'] = 'ok'\n        N1X_MIDS = {it['mid'] for it in _n1_x_items}\n    _n1_run = _n1_items + _n1_x_items\n    _n1_exp_merged = sum(bool(N1DR.merged_buckets(_n1_G, s)) for it in _n1_run for s in it['spectra'])\n    _n1_out, N1_DRV, _n1_done = n1_run(_n1_run, N1_BUDGET) if _n1_run else ({}, {'status': 'nothing to score'}, set())\n    print(f\"[N1] driver: status {N1_DRV.get('status')} | molecules sent {len(_n1_run)} (real {len(_n1_items)}, synthetic \"\n          f\"{len(_n1_x_items)}) done {len(_n1_done)} | spectra covered {N1_DRV.get('n_spec_covered')} (merged-CE \"\n          f\"{N1_DRV.get('n_spec_merged')}, expected {_n1_exp_merged}) | predictions {N1_DRV.get('n_pred')} (fail \"\n          f\"{N1_DRV.get('n_pred_fail')}) | {N1_DRV.get('seconds')}s | errors {(N1_DRV.get('errors') or [])[:2]}\", flush=True)\n    _n1_sc = n1_scores(_n1_out, _n1_done, _n1_run, N1K)\n    _n1_real = {m: v for m, v in _n1_sc.items() if not m.startswith('n1x:')}\n    _n1_clash = sorted(m for m in _n1_real if any(x is not None for x in (GL_SCORES.get(m) or {}).values()))\n    if not IS_RERUN:\n        assert not _n1_clash, ('N1 molecule already scored by the H1 run', _n1_clash[:5])\n        N1X_GL = {m: v for m, v in _n1_sc.items() if m.startswith('n1x:')}\n        assert N1_DRV.get('status') == 'ok' and N1X_MIDS <= _n1_done, (N1_DRV.get('status'), sorted(N1X_MIDS - _n1_done)[:5])\n        assert len(N1X_GL) >= 1 and N1_DRV.get('n_spec_merged') == _n1_exp_merged, (len(N1X_GL), N1_DRV.get('n_spec_merged'))\n        _n1_nb1 = _n1_nbm = 0\n        for _n1_m in N1X_MIDS:              # single-CE spectra: merged-CE mode == mean (production gl_runner) exactly;\n            for _n1_sp in (_n1_out[_n1_m].get('spec') or {}).values():     # merged-CE ones: the I14b path acts\n                if _n1_sp.get('nb') == 1:\n                    assert _n1_sp['mean'] == _n1_sp['merged'], _n1_m\n                    _n1_nb1 += 1\n                else:\n                    _n1_nbm += int(_n1_sp['mean'] != _n1_sp['merged'])\n        # the dummy commit molecules carry 20,40,60 eV [M+H]+ spectra -> merged-CE proxies (8 in the local dry run): the\n        # per-bucket prediction + merge() must run on them with the real model and change their scores vs mean CE\n        assert _n1_exp_merged >= 1 and _n1_nbm >= 1, ('merged-CE path not exercised on the commit', _n1_exp_merged, _n1_nbm)\n        _n1_rho = []                        # proxies (= the [M+H]+ peaks, cut) vs the real H1 GL scores of the source\n        _n1_mean = n1_scores(_n1_out, _n1_done, _n1_x_items, N1K, mode='mean')\n        for _n1_m, _n1_d in _n1_mean.items():\n            _n1_h = GL_SCORES.get(_n1_m.split(':', 1)[1]) or {}\n            _n1_p = [(v, _n1_h.get(c)) for c, v in _n1_d.items() if v is not None and _n1_h.get(c) is not None]\n            if len(_n1_p) >= 4:\n                _n1_r = _n1_spearman([a for a, _ in _n1_p], [b for _, b in _n1_p])\n                if math.isfinite(_n1_r):\n                    _n1_rho.append(_n1_r)\n        N1X_STATS.update(gl_scored=len(N1X_GL), single_ce_identical=_n1_nb1, merged_changed=_n1_nbm, rho_n=len(_n1_rho),\n                         rho_median=float(np.median(_n1_rho)) if _n1_rho else float('nan'))\n        print(f\"[N1X] synthetic exercise: molecules {N1X_STATS['molecules']} | N1 molecules {N1X_STATS['n1']} | proxies \"\n              f\"built {N1X_STATS['proxies']} (NH4p {N1X_STATS['NH4p']}, WLp {N1X_STATS['WLp']}; empty \"\n              f\"{N1X_STATS['proxy_empty']}) | proxy checks OK {N1X_STATS['proxy_checked']} | arm-B guard \"\n              f\"{N1X_STATS['armB_guard']} | merged-CE spectra \"\n              f\"{N1_DRV.get('n_spec_merged')} (expected {_n1_exp_merged}; merged != mean CE {_n1_nbm}) | single-CE \"\n              f\"merged==mean {_n1_nb1} | GL-scored \"\n              f\"synthetic molecules {len(N1X_GL)} | proxy (mean CE) vs H1 GL Spearman median \"\n              f\"{N1X_STATS['rho_median']:.3f} over {len(_n1_rho)}\", flush=True)\n        assert not _n1_rho or N1X_STATS['rho_median'] >= 0.3, ('proxy scores do not track the H1 GL scores', _n1_rho)\n    else:\n        _n1_real = {m: v for m, v in _n1_real.items() if m not in _n1_clash}\n    if N1_COMPLEMENT:                    # k2 variant A: E006's ICE fallback keeps the blind molecules ICEBERG covers\n        _n1_n0 = len(_n1_real)\n        _n1_real, N1_ICE_DROPPED = n1_complement(_n1_real, ICE_SCORES)\n        if not IS_RERUN:                 # forced commit exercise of the filter on THIS commit's real ICE output: a\n            #                              synthetic copy 'n1x:<mid>' stands for a blind molecule with <mid>'s ICE dict\n            _n1_xi = {m: (ICE_SCORES.get(m.split(':', 1)[1]) if isinstance(ICE_SCORES, dict) else None) for m in N1X_GL}\n            _n1_xk, _n1_xd = n1_complement(N1X_GL, _n1_xi)\n            assert _n1_xd == {m for m, v in _n1_xi.items() if n1_ice_finite(v) > 0}, _n1_xd\n            assert set(_n1_xk) == set(N1X_GL) - _n1_xd and all(_n1_xk[m] is N1X_GL[m] for m in _n1_xk)\n            assert _n1_xd, ('complement filter not exercised: no synthetic N1 molecule whose source has a finite ICE '\n                            'score', len(N1X_GL), sum(n1_ice_finite(v) for v in _n1_xi.values()))\n            _n1_t = n1_complement({'f': {}, 'n': {}, 'x': {}, 'b': {}, 'm': {}, 'e': {}},\n                                  {'f': {'p': 0.5, 'q': None}, 'n': {'p': None}, 'x': {'p': float('nan'), 'q': float('inf')},\n                                   'b': {'p': 'bad', 'q': [1]}, 'e': {}})\n            assert set(_n1_t[0]) == {'n', 'x', 'b', 'm', 'e'} and _n1_t[1] == {'f'}, _n1_t\n            assert n1_complement({}, ICE_SCORES) == ({}, set()) and n1_complement({'f': {}}, None) == ({'f': {}}, set())\n            N1X_STATS['complement'] = dict(synthetic=len(N1X_GL), dropped=len(_n1_xd), kept=len(_n1_xk))\n            print(f\"[N1X] complement filter exercise: synthetic N1 molecules with GL {len(N1X_GL)} | source ICE dict has a \"\n                  f\"finite score -> dropped {len(_n1_xd)} | kept {len(_n1_xk)} | unit cases OK\", flush=True)\n        print(f\"[N1] complement (k2 variant A): N1 molecules with GL scores {_n1_n0} | dropped (finite ICE score, E006 ICE \"\n              f\"fallback kept) {len(N1_ICE_DROPPED)} | kept {len(_n1_real)}\", flush=True)\n    _n1_health = (f\"N1 OK: {len(_n1_real)} of {len(_n1_items)} N1 molecules GL-scored \"\n                  f\"({sum(sum(v is not None for v in d.values()) for d in _n1_real.values())} candidates), driver \"\n                  f\"{N1_DRV.get('status')}, clashes dropped {len(_n1_clash)}\")\n    N1_MIDS, N1_GL_MIDS = {str(it['mid']) for it in _n1_items}, set(_n1_real)\n    GL_SCORES.update(_n1_real)           # last statement: N1 scores exist only if everything above succeeded\n    N1_HEALTH = _n1_health\nexcept Exception as e:\n    for _n1_m in list(N1_GL_MIDS):\n        GL_SCORES.pop(_n1_m, None)\n    N1_MIDS, N1_GL_MIDS = set(), set()\n    N1_HEALTH = f'N1 FAILED: {e!r}'\n    print('[N1] FAILED -> H1 behaviour (no N1 GL scores):', repr(e)[:600], flush=True)\n    if not IS_RERUN:\n        raise\nprint(f'[N1] GL-scored N1 molecules: {len(N1_GL_MIDS)} of {len(N1_MIDS)} | health: {N1_HEALTH} | '\n      f'{time.time()-T0:.0f}s', flush=True)\n# Counts distinguish selected proxies from completed, retained forward scores.\nN1_CHALLENGER_COVERAGE = dict(\n    complement=bool(N1_COMPLEMENT), eligible_molecules=len(N1_MIDS),\n    proxy_covered_molecules=len(N1_GL_MIDS),\n    proxy_covered_with_finite_ice=sum(n1_ice_finite(ICE_SCORES.get(str(m), {})) > 0\n                                       for m in N1_GL_MIDS),\n    real_h_molecules_skipped=int(N1_STATS.get('with_H', 0)), health=N1_HEALTH)\nprint('N1_CHALLENGER_COVERAGE', json.dumps(N1_CHALLENGER_COVERAGE), flush=True)\n# ---- /N1A_MCE ---------------------------------------------------------------------------------------------------\ndef gl_of(m):\n    g = GL_SCORES.get(str(m), {}) if isinstance(GL_SCORES, dict) else {}\n    return g if isinstance(g, dict) and any(v is not None for v in g.values()) else {}\ndef gl_rerank(order0, smis, keys, scs, forms, ice, gl, top_n, tag=''):\n    # z(scs) + ICE_LAM z(ice) + GL_LAM z(gl) in the same slots; no GL scores / any failure -> order0 (= v4g order)\n    if not gl:\n        return order0\n    try:\n        o = gl_fuse.rerank_multi(smis, keys, scs, forms, [ice, gl], [ICE_LAM, GL_LAM], top_n=top_n)\n    except Exception as e:\n        print('gl rerank failed', tag, repr(e)); return order0\n    for k, v in (('molecules', 1), ('changed_vs_ice_top25', int(o[:25] != order0[:25])), ('changed_vs_ice_top1', int(o[:1] != order0[:1]))):\n        gl_stats[tag + k] = gl_stats.get(tag + k, 0) + v\n    return o\nfor m in list(BASE):\n    smis, keys, lib_max, scs, forms = BASE[m]\n    ice = ICE_SCORES.get(str(m), {}) if ICE_SCORES else {}\n    gl = gl_of(m)\n    if BASE_ICE and ice and any(v is not None for v in ice.values()):\n        try:\n            order = ice_fuse.rerank(smis, keys, scs, forms, ice, lam=ICE_LAM, top_n=TOPN)\n            order = gl_rerank(order, smis, keys, scs, forms, ice, gl, TOPN)\n            s2 = [smis[i] for i in order]; k2 = [keys[i] for i in order]\n            ice_stats['molecules'] += 1\n            ice_stats['changed_top25'] += int(s2[:25] != smis[:25]); ice_stats['changed_top1'] += int(bool(s2) and s2[0] != smis[0])\n            smis, keys = s2, k2\n        except Exception as e:\n            print('rerank failed', m, repr(e))\n    BASE[m] = (smis[:25], keys[:25], lib_max)\n    p = PC.get(m) if (ICE_PC and isinstance(PC, dict)) else None\n    if p and p.get('pc_form') and ice:\n        try:                             # re-order the PubChem list by z(f.z) + lam z(ice) inside same-formula groups;\n            o = ice_fuse.rerank(p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, lam=ICE_LAM, top_n=25)\n            o = gl_rerank(o, p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, gl, 25, 'pc_')\n            fz0 = p['pc_fz'][0]          # the gate keeps using the ORIGINAL top f.z (rel), so gate decisions are unchanged\n            p['pc'] = [p['pc'][i] for i in o]; p['pc_keys'] = [p['pc_keys'][i] for i in o]\n            p['pc_fz'] = [fz0] + [p['pc_fz'][i] for i in o][1:]\n            ice_stats['pc_changed'] = ice_stats.get('pc_changed', 0) + int(o[:1] != [0])\n        except Exception as e:\n            print('pc rerank failed', m, repr(e))\nprint('ICE rerank stats', ice_stats, f'{time.time()-T0:.0f}s')\nprint('GL rerank stats', gl_stats, f'{time.time()-T0:.0f}s')\n\n\n# ===== cell 32 =====\n'''## 8. Gated PubChem merge\n\nPubChem-only candidates enter the main list only when there is no strong library match (`lib_max < 0.9`). When the PubChem fingerprint fit beats the best pool candidate by a wide margin, they take slots 2, 4, 6, 8 and 10. Otherwise they take slots 4, 8, 12, 16 and 20.\n'''\n\n\n# ===== cell 33 =====\n'''gated PubChem merge (v3.6, unchanged) -> submission.csv'''\ndef merge(base, base_keys, pc, pc_keys, slots, n=25):\n    bk = set(base_keys)\n    pcs = [s for s, k in zip(pc, pc_keys) if k not in bk]\n    out, bi, pj = [], 0, 0\n    for pos in range(1, n + 1):\n        if pos in slots and pj < len(pcs):\n            out.append(pcs[pj]); pj += 1\n        elif bi < len(base):\n            out.append(base[bi]); bi += 1\n        elif pj < len(pcs):\n            out.append(pcs[pj]); pj += 1\n    return out\n'''---- CLAW PROMOTION RULE (b417) --------------------------------------------------'''\n'''The shipped gate only INTERLEAVES PubChem-only structures into fixed slots, so the'''\n'''main list always keeps rank 1. When the best PubChem proposal is a strong outlier --'''\n'''S = z(f.z over the pass-2 set) + 0.25*pop, carried out of the worker -- AND is well'''\n'''documented (pop >= 5), let it lead; everything else keeps its usual slot.'''\n'''NOTE: S is the Z-SCORED quantity (median ~6). The raw p['pc_fz'][0] (median -1437)'''\n'''must never be compared to S_TAU; that would fire on ~71% of molecules.'''\n'''EVIDENCE: b416 (promotion, ICEBERG broken) = 0.417 vs base415 (ICE broken, no'''\n'''promotion) 0.374 and b415c (ICE working, no promotion) 0.393. The promotion overcame'''\n'''losing ICE, so its marginal value is >= +0.024 (est. ~+0.037 using the notebook's own'''\n'''ICE +0.013 / GL +0.006 measurements). b417 isolates it on the WORKING-ICE base.'''\nS_TAU, POP_TAU = 6.0, 5.0\n\n\ndef promote(base, base_keys, pc, pc_keys, slots, n=25):\n    \"\"\"Confident PubChem proposal leads the usual list; every other entry keeps its slot.\"\"\"\n    usual = merge(base, base_keys, pc, pc_keys, slots, n=n)\n    if not pc:\n        return usual\n    top = pc[0]\n    return ([top] + [x for x in usual if x != top])[:n]\n\n\n'''self-test of both pure functions (this also runs in the cheap smoke pass)'''\n_b, _bk = ['A', 'B', 'C'], ['ka', 'kb', 'kc']\n_p, _pk = ['X', 'Y'], ['kx', 'ky']\nassert merge(_b, _bk, _p, _pk, [2, 4]) == ['A', 'X', 'B', 'Y', 'C'], merge(_b, _bk, _p, _pk, [2, 4])\nassert promote(_b, _bk, _p, _pk, [2, 4]) == ['X', 'A', 'B', 'Y', 'C'], promote(_b, _bk, _p, _pk, [2, 4])\nassert merge(_b, _bk, _p, _pk, [2, 4])[0] == 'A', 'shipped gate must keep rank 1'\nassert merge(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']\nassert promote(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']\nassert promote(_b, _bk, _p, _pk, [2, 4])[:1] == ['X'] and len(promote(_b, _bk, _p, _pk, [2, 4])) == 5\nprint('merge/promote self-test OK')\n\ndef candidate_promotion_evidence(p):\n    \"\"\"Look up the current leader by its immutable retrieval identity.\"\"\"\n    key = p['pc_keys'][0]\n    evidence = p.get('promotion_evidence', {}).get(key)\n    assert evidence is not None, 'Missing candidate promotion evidence; rerun PubChem: ' + str(key)\n    assert evidence['smiles'] == p['pc'][0], 'PubChem identity/representative mismatch'\n    S, pop, raw = (float(evidence[x]) for x in ('S', 'pop', 'raw_fz'))\n    assert all(np.isfinite(v) for v in (S, pop, raw)), 'Nonfinite promotion evidence'\n    # Preserve the baseline's no-pool-evidence convention.\n    rel = raw - p['best_pool_fz'] if np.isfinite(p['best_pool_fz']) else 1e9\n    return S, pop, rel\n\n\npromotion_audit = []\nrows, stats = [], dict(untouched=0, gentle=0, aggressive=0, no_pc=0, promoted=0)\nfor mid in te.molecule_id.unique():\n    smis, keys, lib_max = BASE.get(mid, ([], [], 0.0))\n    p = PC.get(mid)\n    if p is None or not p['pc']:\n        stats['no_pc'] += 1; final = smis\n    elif lib_max >= LIB_TAU:\n        stats['untouched'] += 1; final = smis\n    else:\n        rel = (p['fz_top'] if p.get('fz_top') is not None else p['pc_fz'][0]) - p['best_pool_fz'] if np.isfinite(p['best_pool_fz']) else 1e9\n        slots = SLOTS_AGG if rel > REL_TH else SLOTS_GENTLE\n        stats['aggressive' if rel > REL_TH else 'gentle'] += 1\n        old_S = p.get('S'); old_pop = float(p.get('top_pop') or 0.0)\n        old_promote = bool((old_S is not None and float(old_S) > S_TAU and old_pop >= POP_TAU) or rel > 150.0)\n        S, pop_top, candidate_rel = candidate_promotion_evidence(p)\n        new_promote = bool((S > S_TAU and pop_top >= POP_TAU) or candidate_rel > 150.0)\n        promotion_audit.append(dict(molecule_id=mid, leader_key=p['pc_keys'][0],\n            leader_smiles=p['pc'][0], original_S=old_S, original_pop=old_pop,\n            channel_rel=rel, candidate_S=S, candidate_pop=pop_top, candidate_rel=candidate_rel,\n            old_promote=old_promote, new_promote=new_promote, changed=old_promote != new_promote))\n        # Only promotion uses candidate-specific evidence; interleaving slots retain channel rel.\n        if new_promote:\n            stats['promoted'] += 1\n            final = promote(smis, keys, p['pc'], p['pc_keys'], slots)\n        else:\n            final = merge(smis, keys, p['pc'], p['pc_keys'], slots)\n    if not final: final = ['CCO']\n    rows.append((mid, ';'.join(final[:25])))\nprint('merge stats', stats)\npd.DataFrame(promotion_audit).to_csv('pubchem_promotion_audit.csv', index=False)\nprint('CANDIDATE PROMOTION:', len(promotion_audit), 'eligible rows;',\n      sum(r['changed'] for r in promotion_audit), 'decisions changed')\nsub = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])\nsamp = pd.read_csv(os.path.join(COMP, 'sample_submission.csv'))\nsub = samp[['molecule_id']].merge(sub, on='molecule_id', how='left')\nsub['smiles'] = sub['smiles'].fillna('CCO')\nassert len(sub) == len(samp) and sub.molecule_id.is_unique and sub.smiles.notna().all()\nassert sub.smiles.map(lambda s: len(s.split(';'))).max() <= 25\nsub.to_csv('submission.csv', index=False)\nprint('CLAW STAGE HEALTH |', CLAW_ICE_HEALTH, '|', CLAW_GL_HEALTH, flush=True)\ntry:\n    _ice_n = json.load(open('/kaggle/working/ice_work/ice_out.json.meta.json')).get('n_mols_scored', 0)\n    _gl_n = json.load(open('/kaggle/working/gl_work/gl_out.json.meta.json')).get('n_mols_scored', 0)\nexcept Exception:\n    _ice_n = _gl_n = -1\nprint(f'CLAW STAGE SCORED | ice={_ice_n} gl={_gl_n}', flush=True)\njson.dump(dict(secs=round(time.time() - T0), merge=stats, n_errors=n_err,\n               ice_health=CLAW_ICE_HEALTH, gl_health=CLAW_GL_HEALTH,\n               ice_n_scored=_ice_n, gl_n_scored=_gl_n,\n               families=MAN['families'],\n               gpu=torch.cuda.get_device_name(0) if dev == 'cuda' else None), open('run_manifest.json', 'w'), indent=1)\nprint('done', sub.shape, f'{time.time()-T0:.0f}s')\nsub.head()\n\n\n# ===== cell 34 =====\n'''## 9. Two-engine fusion + final simulator re-rank\n\nThe two lists are combined by weighted reciprocal-rank fusion. A candidate's score is $\\\\frac{1}{3 + r_{\\\\text{main}}} + \\\\frac{0.6}{3 + r_{\\\\text{engine 2}}}$, and the top 40 are kept. ICEBERG and GLACIER then re-rank same-formula groups of the fused list one last time, and the top 25 become the submission.\n'''\n\n\n# ===== cell 35 =====\nALPHA, KRR = 0.6, 3.0\nCELL19_ICE = False       # b421: cell-19 ICE contribution EXPLICITLY off.\n'''Reproduces BY DESIGN what b416 got by accident: when ICE errored it still wrote a'''\n'''shell dict of all-None values, cell-19's `if ice:` gate passed, and ICE's column then'''\n'''contributed nothing (covered count 0) -- leaving GLACIER alone. b420 (ICE real, both'''\n'''applied) scored 0.402; b416 (GLACIER only) scored 0.417. This isolates that one'''\n'''arithmetic difference on the healthy base.'''\nfrom rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF2\nfrom rdkit import Chem as _Ch2\ndef _form2(s):\n    try:\n        return _CMF2(_Ch2.MolFromSmiles(s))\n    except Exception:\n        return s\ndef fuse2(v_smis, e_smis, e_keys, n=40):\n    sc, smi_of = {}, {}\n    for r, s in enumerate(v_smis, 1):\n        k = chem.score_key(s) or s\n        sc[k] = sc.get(k, 0.0) + 1.0 / (KRR + r); smi_of.setdefault(k, s)\n    for r, (s, k) in enumerate(zip(e_smis, e_keys), 1):\n        if not k:\n            continue\n        sc[k] = sc.get(k, 0.0) + ALPHA / (KRR + r); smi_of.setdefault(k, s)\n    order = sorted(sc, key=lambda k: -sc[k])[:n]\n    return [smi_of[k] for k in order], order, [sc[k] for k in order]\n\nPREFUSION_LISTS = {str(r.molecule_id): str(r.smiles).split(';') for r in pd.read_csv('submission.csv').itertuples(index=False)}\nFUSION_OVERFLOW = {}\nif ENG:\n    v4 = pd.read_csv('submission.csv')\n    changed = n_ice = 0; out = []\n    for mid, s in zip(v4.molecule_id, v4.smiles):\n        vs = [x for x in s.split(';') if x and x != 'CCO']\n        e = ENG.get(str(mid))\n        if e and e['smiles']:\n            fsm, fk, fsc = fuse2(vs, e['smiles'], e['keys'])\n            ice = ICE_SCORES.get(str(mid), {}) if isinstance(ICE_SCORES, dict) else {}\n            if ice:\n                try:\n                    o = ice_fuse.rerank(fsm, fk, fsc, [_form2(x) for x in fsm], (ice if CELL19_ICE else {}), lam=ICE_LAM, top_n=len(fsm))\n                    _g = gl_of(mid)\n                    if _g:\n                        o = gl_fuse.rerank_multi(fsm, fk, fsc, [_form2(x) for x in fsm], [(ice if CELL19_ICE else {}), _g], [ICE_LAM, GL_LAM], top_n=len(fsm))\n                    n_ice += int(o[:25] != list(range(min(25, len(o)))))\n                    fsm = [fsm[i] for i in o]\n                except Exception as ex:\n                    print('fused ICE rerank failed', mid, repr(ex))\n            changed += int(bool(vs) and fsm[:1] != vs[:1])\n            FUSION_OVERFLOW[str(mid)] = list(fsm[25:])\n            vs = fsm[:25]\n        out.append((mid, ';'.join(vs[:25]) if vs else 'CCO'))\n    fs = pd.DataFrame(out, columns=['molecule_id', 'smiles'])\n    assert len(fs) == len(v4) and fs.molecule_id.is_unique and fs.smiles.notna().all()\n    assert fs.smiles.map(lambda s: len(s.split(';'))).max() <= 25\n    fs.to_csv('submission.csv', index=False)\n\n\nprint('V1 inference complete; applying separately audited final pass next.')\n\n\n# ===== cell 36 =====\n'''## Final metric audit\n\nThe final pass cannot demote any existing metric-equivalent answer: it retains the first\noccurrence of every valid identity in order, then appends new identities up to 25 candidates.\nRuntime assertions check this for every active molecule. This guarantee applies to the\nsame predictions and current metric; it does not predict the score of a fresh inference run.\n\nOutputs: `submission.csv`, `baseline_submission.csv`, `candidate_audit.csv`,\n`candidate_audit.json`, and stage health in `run_manifest.json`.\n'''\n\n\n# ===== cell 37 =====\nfrom functools import lru_cache\nfrom rdkit import Chem, RDLogger\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\nRDLogger.DisableLog('rdApp.*')\n_tautomer_canonicalizer = rdMolStandardize.TautomerEnumerator()\ndef canonicalize_smiles_to_ik14(smiles: str) -> str:\n    \"\"\"Canonicalize SMILES to InChIKey14, or None if parsing fails.\n\n    Stereochemistry is removed *before* the tautomer canonicalization, not left to fall out of\n    the InChIKey14 truncation afterwards. Truncating to InChIKey14 discards the stereo block, so\n    it looks like it should already make stereochemistry free -- but the canonicalization runs\n    first, and `TautomerEnumerator` will not move a double bond whose position a neighbouring\n    stereocentre constrains. The same skeleton drawn with and without that stereocentre can\n    therefore land on two different canonical tautomers, and different tautomers are different\n    connectivity, which is the part of the InChIKey14 that survives truncation. Stripping first\n    makes the canonical form depend only on the skeleton, which is what the competition promises.\n\n    Both the answer key and the submission go through this function, so the two sides can never\n    be treated asymmetrically.\n    \"\"\"\n    if not isinstance(smiles, str):\n        return None\n    try:\n        mol = Chem.MolFromSmiles(smiles)\n        if mol is None:\n            return None\n        Chem.RemoveStereochemistry(mol)\n        canonical_mol = _tautomer_canonicalizer.Canonicalize(mol)\n        inchikey = Chem.MolToInchiKey(canonical_mol)\n        return inchikey[:14]\n    # RDKit's MolSanitizeException family subclasses ValueError; RuntimeError covers the C++\n    # layer and TypeError covers Boost.Python argument errors. One participant's malformed\n    # SMILES must never take the whole scoring run down, so a failure here is just a non-match.\n    except (ValueError, RuntimeError, TypeError):\n        return None\n\ndef heavy_atom_composition(smiles: str):\n    \"\"\"Heavy-atom element counts (excluding H, ignoring charge) as a hashable key, or None\n    if parsing fails.\n\n    Cheap (no tautomer enumeration) -- used to gate the expensive canonicalization. Anything\n    that shares an InChIKey14 with an accepted structure has the same heavy-atom skeleton, so\n    a heavy-atom-composition mismatch lets us skip canonicalization with no change to any\n    score. We deliberately do NOT gate on full molecular formula: protomers / charge variants\n    (e.g. a zwitterion C(=O)[O-]...[N+] vs its neutral-acid protomer C(=O)O...[N+]) share an\n    InChIKey14 but differ in H count and charge, so a formula gate would wrongly reject them.\n    \"\"\"\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\n    if mol is None:\n        return None\n    counts = {}\n    for atom in mol.GetAtoms():\n        sym = atom.GetSymbol()\n        if sym != 'H':\n            counts[sym] = counts.get(sym, 0) + 1\n    return tuple(sorted(counts.items()))\n\n@lru_cache(maxsize=150000)\ndef metric_identity(smiles):\n    key = canonicalize_smiles_to_ik14(smiles)\n    return (key, heavy_atom_composition(smiles)) if key else None\n\ndef safe_extend(original, extra, limit=25):\n    \"\"\"Preserve first valid metric identities; fill only unoccupied ranks.\"\"\"\n    out, seen = [], set()\n    for smi in list(original) + list(extra):\n        smi = str(smi).strip()\n        identity = metric_identity(smi)\n        if identity is not None and identity not in seen:\n            seen.add(identity)\n            out.append(smi)\n            if len(out) == limit:\n                break\n    return out\n\ndef first_ranks(smiles):\n    ranks = {}\n    for rank, smi in enumerate(smiles, 1):\n        identity = metric_identity(smi)\n        if identity is not None:\n            ranks.setdefault(identity, rank)\n    return ranks\n\ndef assert_monotone(original, revised):\n    old, new = first_ranks(original), first_ranks(revised)\n    assert all(identity in new and new[identity] <= rank for identity, rank in old.items())\n\n\n# ===== cell 38 =====\nMETRIC_SOURCE_SHA256 = 'c61d163f1bb768313fd0863822952f035f343924f37e3f5e4b5883ccef7e6201'\n'''Final pass: no change to any V1 model, fusion weight, promotion gate or simulator.'''\nimport collections\nimport hashlib\nfrom pathlib import Path\nassert rdkit.__version__ == '2026.03.3'\nassert n_err == 0, f'Base inference errors: {n_err}'\n_before = pd.read_csv('submission.csv')\n_before.to_csv('baseline_submission.csv', index=False)\n_active = set(te.molecule_id.astype(str))\nassert _active <= set(map(str, ENG)), 'Engine 2 did not cover all active molecules'\n_stage_meta = {}\nfor _stage, _path in [('ice', 'ice_work/ice_out.json.meta.json'), ('glacier', 'gl_work/gl_out.json.meta.json')]:\n    if Path(_path).is_file():\n        _stage_meta[_stage] = json.loads(Path(_path).read_text())\n    else:\n        _stage_meta[_stage] = {'status': 'missing'}\nassert _stage_meta['ice'].get('status') == 'ok', _stage_meta['ice']\nif (te.adduct == '[M+H]+').any():\n    assert _stage_meta['glacier'].get('status') == 'ok', _stage_meta['glacier']\n\n_dm_smiles = _dm_mass = _dm_order = None\n_dm_paths = sorted(glob.glob('/kaggle/input/**/dreams_np_mass.npy', recursive=True))\n_dreams_used = 0\n_audit_rows, _out = [], []\n_targets = te.groupby('molecule_id').nm.median().to_dict()\nfor _row in _before.itertuples(index=False):\n    _mid = str(_row.molecule_id)\n    _original = [s.strip() for s in str(_row.smiles).split(';') if s.strip()]\n    if _mid not in _active:  # Visible smoke placeholders; hidden rerun processes every molecule.\n        _out.append((_row.molecule_id, _row.smiles))\n        continue\n    _ranks = first_ranks(_original)\n    _clean = safe_extend(_original, [])\n    _extras = list(FUSION_OVERFLOW.get(_mid, [])) + list(BASE_TAIL_CANDIDATES.get(_mid, []))\n    _engine = ENG.get(_mid, {})\n    _extras.extend(_engine.get('smiles', []))\n    _pc = PC.get(_row.molecule_id, PC.get(_mid, {}))\n    _extras.extend(_pc.get('pc', []))\n    _revised = safe_extend(_clean, _extras)\n    _after_ranked = len(_revised)\n    _target = float(_targets.get(_row.molecule_id, _targets.get(_mid, np.nan)))\n    if len(_revised) < 25 and np.isfinite(_target) and _target > 0 and _dm_paths:\n        if _dm_mass is None:\n            _dd = Path(_dm_paths[0]).parent\n            _dm_mass = np.load(_dd / 'dreams_np_mass.npy').astype(np.float64)\n            _dm_smiles = np.load(_dd / 'dreams_np_smiles.npy', allow_pickle=True)\n            assert len(_dm_mass) == len(_dm_smiles)\n            _dm_order = np.argsort(_dm_mass, kind='stable')\n            _dm_sorted = _dm_mass[_dm_order]\n        _lo, _hi = np.searchsorted(_dm_sorted, [_target * (1 - 10e-6), _target * (1 + 10e-6)])\n        _ix = _dm_order[_lo:_hi]\n        _ix = _ix[np.argsort(np.abs(_dm_mass[_ix] - _target), kind='stable')]\n        _mass_candidates = [s.decode() if isinstance(s, bytes) else str(s) for s in _dm_smiles[_ix]]\n        _revised = safe_extend(_revised, _mass_candidates)\n        _dreams_used += len(_revised) - _after_ranked\n    assert _revised, f'No valid candidates for {_mid}'\n    assert len(_revised) <= 25 and len(first_ranks(_revised)) == len(_revised)\n    assert_monotone(_original, _revised)\n    _new_ranks = first_ranks(_revised)\n    _audit_rows.append(dict(molecule_id=_mid, original_count=len(_original),\n        original_unique=len(_ranks), invalid_count=sum(metric_identity(s) is None for s in _original),\n        duplicate_count=sum(metric_identity(s) is not None for s in _original) - len(_ranks),\n        revised_count=len(_revised), ranked_added=_after_ranked - len(_clean),\n        dreams_added=len(_revised) - _after_ranked,\n        promoted_original=sum(_new_ranks[k] < v for k, v in _ranks.items()),\n        changed=_original != _revised))\n    _out.append((_row.molecule_id, ';'.join(_revised)))\n_final = pd.DataFrame(_out, columns=['molecule_id', 'smiles'])\n_expected = pd.read_csv(os.path.join(COMP, 'sample_submission.csv')).molecule_id\nassert _final.molecule_id.is_unique and set(_final.molecule_id) == set(_expected)\nassert _final.smiles.notna().all() and _final.smiles.str.split(';').map(len).between(1, 25).all()\nif IS_RERUN:\n    assert len(_active) == len(_final), 'A hidden molecule was left unprocessed'\n_final.to_csv('submission.csv', index=False)\n_audit_df = pd.DataFrame(_audit_rows)\n_audit_df.to_csv('candidate_audit.csv', index=False)\n_summary = dict(experiment='V1.1 metric-safe tail', rdkit=rdkit.__version__,\n    hidden_rerun=IS_RERUN, active_molecules=len(_active), output_rows=len(_final),\n    changed_molecules=int(_audit_df.changed.sum()),\n    invalid_removed=int(_audit_df.invalid_count.sum()),\n    equivalent_duplicates_removed=int(_audit_df.duplicate_count.sum()),\n    existing_candidates_promoted=int(_audit_df.promoted_original.sum()),\n    ranked_candidates_added=int(_audit_df.ranked_added.sum()),\n    dreams_candidates_added=int(_audit_df.dreams_added.sum()),\n    monotone_rank_assertions_passed=True,\n    metric_source_sha256=METRIC_SOURCE_SHA256,\n    baseline_sha256=hashlib.sha256(Path('baseline_submission.csv').read_bytes()).hexdigest(),\n    submission_sha256=hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest(),\n    stages=_stage_meta, runtime_seconds=round(time.time()-T0, 1),\n    validation_note='Visible test is a smoke check only and overlaps training; no generalization score claimed.')\nPath('candidate_audit.json').write_text(json.dumps(_summary, indent=2, default=str))\nprint(json.dumps({k:v for k,v in _summary.items() if k != 'stages'}, indent=2))\ndisplay(_audit_df)\n\n\n# ===== cell 39 =====\n'''Independent experimental overlay\nOne vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. Preserve the existing PubChem gate and same-formula GLACIER logic. This tests duplicate-vote bias, not a coefficient search. It can change top-1 and reduce the score.'''\n\ndef metric_fusion(first, second, identity, alpha=0.6, offset=3.0, limit=40):\n    scores, representatives = {}, {}\n    collapsed = 0\n    for candidates, weight in ((first, 1.0), (second, alpha)):\n        seen = set()\n        for candidate in candidates:\n            key = identity(candidate)\n            if key is None:\n                continue\n            if key in seen:\n                collapsed += 1\n                continue\n            seen.add(key)\n            scores[key] = scores.get(key, 0.0) + weight / (offset + len(seen))\n            representatives.setdefault(key, candidate)\n    ordered = sorted(scores, key=lambda key: -scores[key])[:limit]\n    return [representatives[key] for key in ordered], [scores[key] for key in ordered], collapsed\n\nEXPERIMENT = 'metric_identity_fusion'\n_reference = pd.read_csv('submission.csv')\n_reference.to_csv('submission_reference.csv', index=False)\n_rows, _audit = [], []\nfor _row in _reference.itertuples(index=False):\n    _mid = str(_row.molecule_id)\n    _original = str(_row.smiles).split(';')\n    _revised, _collapsed = list(_original), 0\n    _engine = ENG.get(_mid)\n    if _mid in PREFUSION_LISTS and _engine and _engine.get('smiles'):\n        _first = [s for s in PREFUSION_LISTS[_mid] if s and s != 'CCO']\n        _candidates, _scores, _collapsed = metric_fusion(_first, _engine['smiles'], metric_identity,\n                                                       alpha=ALPHA, offset=KRR)\n        _keys = [chem.score_key(s) or s for s in _candidates]\n        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}\n        _gl = gl_of(_mid)\n        if _ice and _gl:\n            _order = gl_fuse.rerank_multi(_candidates, _keys, _scores,\n                [_form2(s) for s in _candidates], [{}, _gl], [ICE_LAM, GL_LAM], top_n=len(_candidates))\n            _candidates = [_candidates[i] for i in _order]\n        if _candidates:\n            _revised = safe_extend(_candidates[:25], _candidates[25:] + _original)\n    _rows.append((_row.molecule_id, ';'.join(_revised)))\n    _audit.append(dict(molecule_id=_mid, collapsed_source_votes=_collapsed,\n        changed=_revised != _original, changed_top1=_revised[:1] != _original[:1]))\n\n_result = pd.DataFrame(_rows, columns=['molecule_id', 'smiles'])\nassert list(_result.molecule_id) == list(_reference.molecule_id)\nassert _result.molecule_id.is_unique and _result.smiles.notna().all()\nassert _result.smiles.map(lambda value: 1 <= len(value.split(';')) <= 25).all()\n_result.to_csv('submission.csv', index=False)\npd.DataFrame(_audit).to_csv('experiment_audit.csv', index=False)\n_changed = sum(row['changed'] for row in _audit)\n_report = dict(experiment=EXPERIMENT, changed_molecules=_changed,\n               changed_top1=sum(row['changed_top1'] for row in _audit),\n               reference_sha256=sha256('submission_reference.csv'),\n               submission_sha256=sha256('submission.csv'), measured_score=None)\njson.dump(_report, open('experiment_audit.json', 'w'), indent=2)\nprint(_report)\nif not _changed:\n    print('NO-OP: predictions match the reference; do not spend a second submission on this file.')\n\n\n# ===== cell 40 =====\n\"\"\"Observe the frozen source output and preserve same-run fusion inputs for later work.\"\"\"\n\n\ndef _e004_postflight(scope):\n    import hashlib\n    import json\n    import time\n    from pathlib import Path\n    import pandas as pd\n    import rdkit\n\n    pre = scope['E004_PREFLIGHT']\n    ids = set(pre['input_ids'])\n    processed = set(scope['te'].molecule_id.astype(str))\n    full = bool(scope['IS_RERUN'])\n    assert processed == (ids if full else set(sorted(ids)[:scope['SMOKE_N']]))\n    frame = pd.read_csv('submission.csv', dtype=str, keep_default_na=False)\n    assert list(frame.columns) == ['molecule_id', 'smiles']\n    assert frame.molecule_id.is_unique and set(frame.molecule_id) == ids\n    assert frame.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25).all()\n    base, eng, pc = scope['BASE'], scope['ENG'], scope['PC']\n    assert set(map(str, base)) == processed\n    assert processed <= set(map(str, eng))\n    assert scope['n_err'] == 0\n    if not full:\n        assert all(v[0] for v in base.values())\n        assert all(eng[mid]['smiles'] and pc.get(mid, {}).get('pc') for mid in processed)\n    stages = {}\n    for stage in ('ice', 'gl'):\n        path = Path(f'{stage}_work/{stage}_out.json.meta.json')\n        meta = json.loads(path.read_text()) if path.is_file() else {'status': 'missing'}\n        # The frozen source owns full-run failure policy. Extra acceptance checks\n        # only gate the visible smoke, where all expected stages are exercised.\n        if not full:\n            assert meta['status'] == 'ok' and meta['device'].startswith('cuda'), (stage, meta)\n            assert meta['rdkit'] == '2025.03.6'\n            assert not meta.get('errors') and meta.get('n_pred_fail', 0) == 0\n            assert meta.get('n_feat_fail', 0) == 0\n            if meta.get('n_mols_covered', 0):\n                assert meta['n_mols_scored'] > 0 and meta['n_scored'] > 0\n        stages[stage] = meta\n    payload = {\n        'prefusion': {mid: scope['PREFUSION_LISTS'][mid] for mid in sorted(processed)},\n        'engine2': {mid: eng[mid] for mid in sorted(processed)},\n        'pubchem': {mid: pc.get(mid, {}) for mid in sorted(processed)},\n        'base_tail': {mid: scope['BASE_TAIL_CANDIDATES'][mid] for mid in sorted(processed)},\n        'fusion_overflow': {mid: scope['FUSION_OVERFLOW'].get(mid, []) for mid in sorted(processed)},\n    }\n    Path('e004_candidate_inputs.json').write_text(json.dumps(payload, indent=2, default=str) + '\\n')\n    hashes = {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in (\n        'submission.csv', 'baseline_submission.csv', 'submission_reference.csv',\n        'candidate_audit.json', 'experiment_audit.json', 'e004_candidate_inputs.json')}\n    report = dict(experiment='E004', source_version_id=356740494, source_version=80,\n                  source_code_sha256='eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2',\n                  **pre, is_full_run=full, processed_ids=sorted(processed),\n                  processed_molecules=len(processed), input_molecules=len(ids),\n                  output_rows=len(frame), placeholder_fill_rows=len(ids - processed),\n                  base_errors=int(scope['n_err']), base_nonempty=sum(bool(v[0]) for v in base.values()),\n                  engine2_nonempty=sum(bool(eng[mid]['smiles']) for mid in processed),\n                  pubchem_nonempty=sum(bool(pc.get(mid, {}).get('pc')) for mid in processed),\n                  main_rdkit=rdkit.__version__, stages=stages, hashes=hashes,\n                  elapsed_seconds=round(time.time() - scope['T0'], 2),\n                  limitations=['Visible smoke is not independent validation.',\n                               'Author 0.436 is not this reproduction score.',\n                               'Whole v80 recipe differs from E003 in multiple factors.'])\n    Path('e004_verification.json').write_text(json.dumps(report, indent=2) + '\\n')\n    print('E004_VERIFICATION', json.dumps(report))\n    return report\n\n\nE004_REPORT = _e004_postflight(globals())\ndel _e004_postflight\n\n\n# ===== cell 41 =====\n# Fragment score for molecules without forward-model scores (since v46), copied verbatim from\n# ahmedberatozer/casmi26-v7c-e006-fraggap version 1; its frag_rescore.py is attributed there to\n# Huseyin Emre Aksoy (Apache-2.0).\n# v7c: MetFrag-lite fragment evidence for molecules with NO forward-model coverage (no GLACIER, no finite ICEBERG)\nFRAG_GAP = {}\ntry:\n    FRAG_SRC = '\"\"\"Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\\'s <=2-bond-cleavage fragments explain.\\n\\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\\nremember the fewest cleavages that give each mass, then match every peak against\\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\\nA peak\\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\\nand a spectrum\\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\\n\\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\\n\\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\\n\"\"\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\\'s periodic table\\nAMU = {\\'C\\': 12.0, \\'H\\': 1.00782503207, \\'N\\': 14.0030740048, \\'O\\': 15.9949146196, \\'P\\': 30.97376163,\\n       \\'S\\': 31.97207100, \\'F\\': 18.99840322, \\'Cl\\': 34.96885268, \\'Br\\': 78.9183371, \\'I\\': 126.904473,\\n       \\'Na\\': 22.9897692809, \\'K\\': 38.96370668, \\'Si\\': 27.9769265325, \\'B\\': 11.0093054, \\'Se\\': 79.9165213}\\nH = AMU[\\'H\\']\\nE = 0.00054857990\\nPROTON = H - E\\nH2O = 2 * H + AMU[\\'O\\']\\nNH4 = AMU[\\'N\\'] + 4 * H\\nFORMATE = AMU[\\'C\\'] + 2 * H + 2 * AMU[\\'O\\']\\n\\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\\nCARRIERS = {\\n    \\'[M+H]+\\': (PROTON,), \\'[M-H2O+H]+\\': (PROTON,), \\'[M-2H2O+H]+\\': (PROTON,),\\n    \\'[M+NH4]+\\': (PROTON, NH4 - E), \\'[M+Na]+\\': (PROTON, AMU[\\'Na\\'] - E), \\'[M+K]+\\': (PROTON, AMU[\\'K\\'] - E),\\n    \\'[M-H]-\\': (-PROTON,), \\'[M-H2O-H]-\\': (-PROTON,), \\'[M+CH2O2-H]-\\': (-PROTON, FORMATE - PROTON),\\n    \\'[M+Cl]-\\': (-PROTON, AMU[\\'Cl\\'] + E),\\n}\\n\\nDEFAULT_CFG = dict(\\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\\n    intensity=\\'linear\\',          # \\'linear\\' | \\'sqrt\\' | a float power applied to `it`\\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\\n    tol=0.005,                   # absolute m/z tolerance, Da\\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\\n    max_breaks=2,                # bond cleavages per fragment (1..3)\\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\\n)\\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\\'sqrt\\', tol=0.01, h_shifts=tuple(range(-2, 3)))\\n\\n\\ndef resolve_cfg(cfg=None):\\n    \"\"\"DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently).\"\"\"\\n    out = dict(DEFAULT_CFG)\\n    for k, v in (cfg or {}).items():\\n        if k not in out:\\n            raise KeyError(f\\'frag_rescore: unknown cfg key {k!r}\\')\\n        out[k] = v\\n    out[\\'h_shifts\\'] = tuple(int(d) for d in out[\\'h_shifts\\'])\\n    if not 1 <= int(out[\\'max_breaks\\']) <= 3:\\n        raise ValueError(\\'frag_rescore: max_breaks must be 1, 2 or 3\\')\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------------------------------------------\\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\\n# ---------------------------------------------------------------------------------------------------------------\\ndef _find(parent, a):\\n    while parent[a] != a:\\n        parent[a] = parent[parent[a]]\\n        a = parent[a]\\n    return a\\n\\n\\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\\n    \"\"\"Union the kept bonds; if the graph splits, append each piece\\'s mass with cleavage count k.\"\"\"\\n    for a in range(n):\\n        parent[a] = a\\n        acc[a] = 0.0\\n    ncomp = n\\n    for e in range(ea.shape[0]):\\n        if removed[e]:\\n            continue\\n        ra = _find(parent, ea[e])\\n        rb = _find(parent, eb[e])\\n        if ra != rb:\\n            parent[ra] = rb\\n            ncomp -= 1\\n    if ncomp < 2:\\n        return cnt\\n    for a in range(n):\\n        r = _find(parent, a)\\n        acc[r] += w[a]\\n    for a in range(n):\\n        if parent[a] == a:\\n            out_m[cnt] = acc[a]\\n            out_k[cnt] = k\\n            cnt += 1\\n    return cnt\\n\\n\\ndef _enumerate(n, w, ea, eb, max_breaks, base_components):\\n    \"\"\"All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces.\"\"\"\\n    nb = ea.shape[0]\\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\\n    out_m = np.empty(cap, np.float64)\\n    out_k = np.empty(cap, np.int8)\\n    parent = np.empty(n, np.int64)\\n    acc = np.empty(n, np.float64)\\n    removed = np.zeros(nb, np.bool_)\\n    cnt = 0\\n    for i in range(nb):\\n        removed[i] = True\\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\\n        if max_breaks >= 2:\\n            for j in range(i + 1, nb):\\n                removed[j] = True\\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 2)\\n                if max_breaks >= 3:\\n                    for l in range(j + 1, nb):\\n                        removed[l] = True\\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\\n                        removed[l] = False\\n                removed[j] = False\\n        removed[i] = False\\n    return out_m[:cnt], out_k[:cnt]\\n\\n\\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\\n    from numba import njit\\n    _find = njit(cache=True, nogil=True)(_find)\\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\\n    HAVE_NUMBA = True\\nexcept Exception:  # pragma: no cover - fallback path\\n    HAVE_NUMBA = False\\n\\n\\ndef mol_graph(smi):\\n    \"\"\"(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable.\"\"\"\\n    from rdkit import Chem, RDLogger\\n    RDLogger.DisableLog(\\'rdApp.*\\')\\n    try:\\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\\n    except Exception:\\n        m = None\\n    if m is None or m.GetNumAtoms() == 0:\\n        return None\\n    pt = Chem.GetPeriodicTable()\\n    w = np.empty(m.GetNumAtoms(), np.float64)\\n    for a in m.GetAtoms():\\n        s = a.GetSymbol()\\n        am = AMU.get(s)\\n        if am is None:\\n            am = pt.GetMostCommonIsotopeMass(s)\\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\\n    return w, ea, eb, len(Chem.GetMolFrags(m))\\n\\n\\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40):\\n    \"\"\"Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\\n\\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\\n    SMILES does not parse.\\n    \"\"\"\\n    g = mol_graph(smi)\\n    if g is None:\\n        return None\\n    w, ea, eb, npieces = g\\n    nb = len(ea)\\n    mb = int(max_breaks)\\n    if mb >= 3 and nb > max_bonds_3:\\n        mb = 2\\n    if mb >= 2 and nb > max_bonds_2:\\n        mb = 1\\n    levels = [np.array([w.sum()])]\\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\\n        from rdkit import Chem\\n        m = Chem.MolFromSmiles(smi)\\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\\n    if nb == 0:\\n        return levels + [np.zeros(0) for _ in range(int(max_breaks))]\\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces)\\n    fm = np.concatenate([levels[0], fm])\\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\\n    key = np.round(fm, 5)\\n    order = np.lexsort((fk, key))                  # by mass, then fewest cleavages first\\n    key, fm, fk = key[order], fm[order], fk[order]\\n    first = np.ones(len(key), bool)\\n    first[1:] = key[1:] != key[:-1]\\n    fm, fk = fm[first], fk[first]\\n    return [np.sort(fm[fk == k]) for k in range(int(max_breaks) + 1)]\\n\\n\\n# ---------------------------------------------------------------------------------------------------------------\\n# scoring\\n# ---------------------------------------------------------------------------------------------------------------\\ndef _carriers(adduct, mode):\\n    c = CARRIERS.get(adduct)\\n    if c is not None:\\n        return c\\n    if isinstance(adduct, str) and adduct.endswith(\\'-\\'):\\n        return (-PROTON,)\\n    if isinstance(adduct, str) and adduct.endswith(\\'+\\'):\\n        return (PROTON,)\\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\\n\\n\\ndef _prep_spectrum(s, c):\\n    \"\"\"Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair.\"\"\"\\n    mz = np.asarray(s.get(\\'mz\\', ()), np.float64).ravel()\\n    it = np.asarray(s.get(\\'it\\', ()), np.float64).ravel()\\n    n = min(len(mz), len(it))\\n    mz, it = mz[:n], it[:n]\\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\\n    mz, it = mz[ok], it[ok]\\n    if len(mz) == 0:\\n        return None\\n    p = c[\\'intensity\\']\\n    iw = it if p == \\'linear\\' else np.sqrt(it) if p == \\'sqrt\\' else it ** float(p)\\n    if c[\\'mz_power\\']:\\n        prec = float(s.get(\\'prec\\') or mz.max())\\n        iw = iw * (mz / prec) ** float(c[\\'mz_power\\'])\\n    tot = iw.sum()\\n    if not tot > 0:\\n        return None\\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\\'adduct\\'), s.get(\\'mode\\')) for dh in c[\\'h_shifts\\']])\\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\\n    return iw / tot, q, len(shifts), len(mz)\\n\\n\\ndef _score_spectrum(levels, weights, prep, tol):\\n    iw, q, ns, npk = prep\\n    best = np.zeros(npk)\\n    for arr, wt in zip(levels, weights):\\n        if len(arr) == 0 or wt <= 0:\\n            continue\\n        a = np.searchsorted(arr, q - tol, \\'left\\')\\n        b = np.searchsorted(arr, q + tol, \\'right\\')\\n        hit = (b > a).reshape(ns, npk).any(axis=0)\\n        best = np.where(hit & (wt > best), wt, best)\\n    return float(iw @ best)\\n\\n\\ndef candidate_levels(smi, cfg=None, cache=None):\\n    \"\"\"Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter).\"\"\"\\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\\n\\n\\ndef _levels_cached(smi, c, cache):\\n    key = (smi, int(c[\\'max_breaks\\']), int(c[\\'max_bonds_2\\']), int(c[\\'max_bonds_3\\']))\\n    if cache is not None and key in cache:\\n        return cache[key]\\n    try:\\n        lv = fragment_levels(smi, c[\\'max_breaks\\'], c[\\'max_bonds_2\\'], c[\\'max_bonds_3\\'])\\n    except Exception:\\n        lv = None\\n    if cache is not None:\\n        cache[key] = lv\\n    return lv\\n\\n\\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\\n    \"\"\"One score per candidate: mean explained-intensity share over the molecule\\'s spectra (None: unparsable).\"\"\"\\n    c = resolve_cfg(cfg)\\n    cache = {} if cache is None else cache\\n    weights = [1.0, float(c[\\'w1\\']), float(c[\\'w2\\']), float(c[\\'w3\\'])][: int(c[\\'max_breaks\\']) + 1]\\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\\n    tol = float(c[\\'tol\\'])\\n    out = []\\n    for smi in cand_smiles:\\n        lv = _levels_cached(smi, c, cache)\\n        if lv is None:\\n            out.append(None)\\n            continue\\n        if not preps:\\n            out.append(0.0)\\n            continue\\n        out.append(float(np.mean([_score_spectrum(lv, weights, p, tol) for p in preps])))\\n    return out\\n\\n\\ndef warmup():\\n    \"\"\"Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it.\"\"\"\\n    fragment_levels(\\'CCOC(=O)c1ccccc1O\\')\\n    return HAVE_NUMBA\\n'\n    open('/kaggle/working/frag_rescore.py', 'w').write(FRAG_SRC)\n    if '/kaggle/working' not in sys.path:\n        sys.path.insert(0, '/kaggle/working')\n    import frag_rescore\n    # ==== ours (yasunorim, build.py lfrag=1): learned peak credits (ours_lfrag) replace frag_rescore.score_molecule ====\n    open('/kaggle/working/ours_lfrag.py', 'w').write('\"\"\"ours_lfrag (yasunorim): learned peak credits for the base\\'s MetFrag-lite fragment scorer (frag_rescore).\\n\\nfrag_rescore credits a peak 1.0 when its cheapest explanation is the precursor or a 1-bond cleavage and 0.6 for a\\n2-bond cleavage, whatever hydrogen shift dh (-2..+3) it needs, on linear intensity. Here each peak falls in one bucket\\n(cleavage level 0/1 vs 2) x (dh of its cheapest explanation: fewest cleavages, then smallest |dh|), on linear and on\\nsqrt intensity, and the candidate score is the learned linear combination of the bucket shares (mean over spectra).\\nWeights: listwise softmax over 20 same-formula train isomers, fitted on 4,432 train molecules (3,000 enveda-180 +\\n1,432 gnps / mona / massbank / riken / msdial / pluskal), L2 0.01 towards the base credits. Held out: enveda-180\\n500 molecules MRR 0.723 -> 0.748, enveda-np-examples 171 molecules 0.567 -> 0.584 (truth among 20 isomers).\\n\"\"\"\\nimport numpy as np\\n\\nW = np.array([0.78009, 1.313512, 1.016127, 1.228675, 0.670753, 0.58122, 0.632298, 0.4081, 0.905728, 0.542433, 0.588673, 0.17127, -0.22641, 1.224142, 0.419948, 0.73033, -0.094456, -0.409913, 0.293189, 0.068881, 0.430604, 0.013667, -0.279784, -0.495254, 0.0])\\n\\n\\ndef install(fr):\\n    \"\"\"Replace fr.score_molecule (fr = the imported frag_rescore module) by the learned scorer; same signature.\"\"\"\\n    C = fr.resolve_cfg(None)\\n    HS = C[\\'h_shifts\\']\\n    NH = len(HS)\\n    assert len(W) == 4 * NH + 1, (len(W), NH)\\n\\n    def spec_feats(lv, s):\\n        mz = np.asarray(s.get(\\'mz\\', ()), np.float64).ravel()\\n        it = np.asarray(s.get(\\'it\\', ()), np.float64).ravel()\\n        n = min(len(mz), len(it))\\n        mz, it = mz[:n], it[:n]\\n        ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\\n        mz, it = mz[ok], it[ok]\\n        if len(mz) == 0:\\n            return None\\n        car = fr._carriers(s.get(\\'adduct\\'), s.get(\\'mode\\'))\\n        shifts = np.array([c + dh * fr.H for c in car for dh in HS])\\n        q = mz[None, :] - shifts[:, None]\\n        best_l = np.full(len(mz), 9)\\n        best_d = np.zeros(len(mz), int)\\n        best_a = np.full(len(mz), 99)\\n        for L, arr in enumerate(lv[:3]):\\n            if len(arr) == 0:\\n                continue\\n            a = np.searchsorted(arr, q - C[\\'tol\\'], \\'left\\')\\n            b = np.searchsorted(arr, q + C[\\'tol\\'], \\'right\\')\\n            hit = b > a\\n            lvl = 0 if L <= 1 else 1\\n            for si in range(len(shifts)):\\n                di = si % NH\\n                ad = abs(HS[di])\\n                m = hit[si] & ((lvl < best_l) | ((lvl == best_l) & (ad < best_a)))\\n                best_l = np.where(m, lvl, best_l)\\n                best_d = np.where(m, di, best_d)\\n                best_a = np.where(m, ad, best_a)\\n        f = np.zeros(4 * NH + 1)\\n        hk = best_l < 9\\n        for w_i, wt in enumerate((it / it.sum(), np.sqrt(it) / np.sqrt(it).sum())):\\n            np.add.at(f, w_i * 2 * NH + best_l[hk] * NH + best_d[hk], wt[hk])\\n        f[-1] = 1.0\\n        return f\\n\\n    def score_molecule(spectra, cand_smiles, cfg=None, cache=None):\\n        c = fr.resolve_cfg(cfg)\\n        cache = {} if cache is None else cache\\n        out = []\\n        for smi in cand_smiles:\\n            lv = fr._levels_cached(smi, c, cache)\\n            if lv is None:\\n                out.append(None)\\n                continue\\n            fs = [x for x in (spec_feats(lv, s) for s in (spectra or [])) if x is not None]\\n            out.append(float(np.mean(fs, 0) @ W) if fs else 0.0)\\n        return out\\n\\n    fr.score_molecule_base = fr.score_molecule\\n    fr.score_molecule = score_molecule\\n    return fr\\n')\n    import ours_lfrag\n    ours_lfrag.install(frag_rescore)\n    print('ours_lfrag installed: learned fragment credits', flush=True)\n    frag_rescore.warmup()\n    _fcache = {}\n    _by = {str(m): g for m, g in te.groupby('molecule_id', sort=False)}\n    _t0 = time.time(); _nf = 0\n    for _mid, _lst in PREFUSION_LISTS.items():\n        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}\n        if gl_of(_mid) or any(v is not None for v in (_ice or {}).values()):\n            continue\n        _e = ENG.get(_mid) or {}\n        _c = [s for s in dict.fromkeys(list(_lst) + list(_e.get('smiles', []))) if s and s != 'CCO']\n        _g = _by.get(_mid)\n        if _g is None or len(_c) < 2:\n            continue\n        _spec = [dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                      adduct=r.adduct, prec=float(r.precursor_mz), mode=1 if r.ionization_mode == 'positive' else -1)\n                 for r in _g.itertuples()][:16]\n        try:\n            _sc = frag_rescore.score_molecule(_spec, _c, cache=_fcache)\n            FRAG_GAP[_mid] = {s: float(v) for s, v in zip(_c, _sc) if v is not None}\n            _nf += 1\n        except Exception as _ex:\n            print('frag failed', _mid, repr(_ex))\n    print('v7c FRAG gap molecules', _nf, 'of', len(PREFUSION_LISTS), f'{time.time()-_t0:.0f}s', flush=True)\nexcept Exception as _e:\n    FRAG_GAP = {}\n    print('v7c FRAG FAILED -> v7a behaviour:', repr(_e), flush=True)\n\n\n# ===== cell 42 =====\ndef _factorial_entry(scope):\n    \"\"\"Append-only E005\u2013E007 overlay; original v80 functions remain authoritative.\n\n    Embedded function-locally after the complete frozen E004 notebook. This module\n    never installs dependencies, changes model scores, or recanonicalizes SMILES.\n    \"\"\"\n\n    import hashlib\n    import json\n    import math\n    from pathlib import Path\n\n\n    def _finite_coverage(scores, candidates, keys):\n        count = 0\n        for smi, key in zip(candidates, keys):\n            value = scores.get(smi)\n            if value is None:\n                value = scores.get(key)\n            try:\n                count += value is not None and math.isfinite(float(value))\n            except (TypeError, ValueError):\n                pass\n        return count\n\n\n    def rerank_row(scope, mid, original, *, window, ice_fallback, processed):\n        \"\"\"Rebuild final metric fusion from same-run prefusion and engine lists.\n\n        ``original`` is the metric-safe reference BEFORE the frozen final overlay,\n        not the E004 final output. The original identity, representative selection,\n        same-formula slot rerank and tail fill are reused without modification.\n        \"\"\"\n        if window not in (40, 80):\n            raise ValueError(\"Only the prespecified 40/80 candidate windows are supported\")\n        original = list(original)\n        audit = {\n            \"molecule_id\": mid,\n            \"processed\": bool(processed),\n            \"branch\": \"placeholder\",\n            \"candidate_count\": 0,\n            \"collapsed_source_votes\": 0,\n            \"ice_finite_count\": 0,\n            \"gl_finite_count\": 0,\n            \"fallback_changed\": False,\n            \"fallback_changed_top1\": False,\n            \"n1_proxy\": mid in scope.get(\"N1_GL_MIDS\", set()),\n            \"joint_ice_proxy_gl\": False,\n        }\n        if not processed:\n            return original, audit\n        audit[\"branch\"] = \"no_engine\"\n        engine = scope[\"ENG\"].get(mid)\n        if mid not in scope[\"PREFUSION_LISTS\"] or not engine or not engine.get(\"smiles\"):\n            return original, audit\n        first = [s for s in scope[\"PREFUSION_LISTS\"][mid] if s and s != \"CCO\"]\n        candidates, scores, collapsed = scope[\"metric_fusion\"](\n            first,\n            engine[\"smiles\"],\n            scope[\"metric_identity\"],\n            alpha=scope[\"ALPHA\"],\n            offset=scope[\"KRR\"],\n            limit=window,\n        )\n        keys = [scope[\"chem\"].score_key(s) or s for s in candidates]\n        ice_scores = scope[\"ICE_SCORES\"]\n        ice = ice_scores.get(mid, {}) if isinstance(ice_scores, dict) else {}\n        gl = scope[\"gl_of\"](mid)\n        audit.update(\n            branch=\"unrescored\",\n            candidate_count=len(candidates),\n            collapsed_source_votes=collapsed,\n            ice_finite_count=_finite_coverage(ice, candidates, keys),\n            gl_finite_count=_finite_coverage(gl, candidates, keys),\n        )\n        before = list(candidates)\n        if ice and gl:\n            # flexonafft v47: joint ICE + proxy GL only on N1 treatment rows.\n            # ice_fallback=False still rebuilds the exact E004 GL-only control.\n            order = scope[\"gl_fuse\"].rerank_multi(\n                candidates,\n                keys,\n                scores,\n                [scope[\"_form2\"](s) for s in candidates],\n                [ice if ice_fallback and mid in scope.get(\"N1_GL_MIDS\", set()) else {}, gl],\n                [scope[\"ICE_LAM\"], scope[\"GL_LAM\"]],\n                top_n=len(candidates),\n            )\n            candidates = [candidates[i] for i in order]\n            audit[\"branch\"] = \"glacier\"\n            audit[\"joint_ice_proxy_gl\"] = bool(ice_fallback and audit[\"n1_proxy\"])\n        elif (ice_fallback and not gl and scope.get(\"FRAG_GAP\", {}).get(mid)\n              and not any(v is not None for v in (ice or {}).values())):\n            # v7c: no forward-model evidence at all -> MetFrag-lite gap re-rank (lambda 0.5), once\n            order = scope[\"gl_fuse\"].rerank_multi(\n                candidates, keys, scores, [scope[\"_form2\"](s) for s in candidates],\n                [scope[\"FRAG_GAP\"][mid]], [0.5], top_n=len(candidates))\n            candidates = [candidates[i] for i in order]\n            audit[\"branch\"] = \"frag_gap\"\n        elif ice_fallback and ice and not gl:\n            # Public v1 / script 356785434: ICE is used only where gl_of(mid) is empty.\n            # Keep its truthy-dict gate: all-None ICE is an observable harmless no-op.\n            order = scope[\"ice_fuse\"].rerank(\n                candidates,\n                keys,\n                scores,\n                [scope[\"_form2\"](s) for s in candidates],\n                ice,\n                lam=scope[\"ICE_LAM\"],\n                top_n=len(candidates),\n            )\n            candidates = [candidates[i] for i in order]\n            audit[\"branch\"] = \"ice_fallback\"\n        result = (\n            scope[\"safe_extend\"](candidates[:25], candidates[25:] + original)\n            if candidates\n            else original\n        )\n        if audit[\"branch\"] == \"ice_fallback\":\n            without_ice = (\n                scope[\"safe_extend\"](before[:25], before[25:] + original) if before else original\n            )\n            audit[\"fallback_changed\"] = result != without_ice\n            audit[\"fallback_changed_top1\"] = result[:1] != without_ice[:1]\n        return result, audit\n\n\n    def run_overlay(scope, *, experiment, window, ice_fallback, provenance, workdir=\".\"):\n        \"\"\"Save the E004 bytes, validate same-run control, then emit one treatment.\"\"\"\n        import pandas as pd\n\n        root = Path(workdir)\n        path = root / \"submission.csv\"\n        control_bytes = path.read_bytes()\n        control_sha = hashlib.sha256(control_bytes).hexdigest()\n        base_report = json.loads((root / \"e004_verification.json\").read_text())\n        assert base_report[\"experiment\"] == \"E004\"\n        assert base_report[\"hashes\"][\"submission.csv\"] == control_sha\n        # Copy bytes directly; never reserialize or overwrite original E004 audit files.\n        (root / \"submission_e004_control.csv\").write_bytes(control_bytes)\n        control = pd.read_csv(path, dtype=str, keep_default_na=False)\n        reference = pd.read_csv(root / \"submission_reference.csv\", dtype=str, keep_default_na=False)\n        assert list(reference.columns) == list(control.columns) == [\"molecule_id\", \"smiles\"]\n        assert list(reference.molecule_id) == list(control.molecule_id)\n        assert control.molecule_id.is_unique\n        processed = set(base_report[\"processed_ids\"])\n        assert processed == set(scope[\"te\"].molecule_id.astype(str))\n        controls = dict(zip(control.molecule_id, control.smiles))\n        output, audits = [], []\n        for row in reference.itertuples(index=False):\n            mid, original = str(row.molecule_id), str(row.smiles).split(\";\")\n            active = mid in processed\n            # The exact original pre-overlay tail is essential for paired equivalence.\n            rebuilt, _ = rerank_row(\n                scope, mid, original, window=40, ice_fallback=False, processed=active\n            )\n            assert \";\".join(rebuilt) == controls[mid], f\"E004 control reconstruction mismatch: {mid}\"\n            result, audit = rerank_row(\n                scope, mid, original, window=window, ice_fallback=ice_fallback, processed=active\n            )\n            assert 1 <= len(result) <= 25\n            if active:\n                identities = [scope[\"metric_identity\"](s) for s in result]\n                assert None not in identities and len(identities) == len(set(identities)), mid\n            else:\n                assert \";\".join(result) == controls[mid], f\"Placeholder changed: {mid}\"\n            old = controls[mid].split(\";\")\n            audit.update(changed=result != old, changed_top1=result[:1] != old[:1])\n            audits.append(audit)\n            output.append((mid, \";\".join(result)))\n        frame = pd.DataFrame(output, columns=[\"molecule_id\", \"smiles\"])\n        assert list(frame.molecule_id) == list(control.molecule_id)\n        frame.to_csv(path, index=False)\n        report = {\n            \"experiment\": experiment,\n            \"window\": window,\n            \"ice_fallback\": bool(ice_fallback),\n            \"source_experiment\": \"E004\",\n            \"base_report_file\": \"e004_verification.json\",\n            \"control_file\": \"submission_e004_control.csv\",\n            \"control_sha256\": control_sha,\n            \"submission_sha256\": hashlib.sha256(path.read_bytes()).hexdigest(),\n            \"processed_ids\": sorted(processed),\n            \"output_rows\": len(frame),\n            \"changed_rows\": sum(a[\"changed\"] for a in audits),\n            \"changed_top1\": sum(a[\"changed_top1\"] for a in audits),\n            \"fallback_rows\": sum(a[\"branch\"] == \"ice_fallback\" for a in audits),\n            \"fallback_finite_rows\": sum(\n                a[\"branch\"] == \"ice_fallback\" and a[\"ice_finite_count\"] > 0 for a in audits\n            ),\n            \"fallback_changed_rows\": sum(a[\"fallback_changed\"] for a in audits),\n            \"fallback_changed_top1\": sum(a[\"fallback_changed_top1\"] for a in audits),\n            \"same_run_control_reconstructed\": True,\n            \"n1_challenger\": {\n                \"complement\": bool(scope.get(\"N1_COMPLEMENT\", True)),\n                \"proxy_covered_molecules\": len(scope.get(\"N1_GL_MIDS\", set())),\n                \"joint_rerank_rows\": sum(a[\"joint_ice_proxy_gl\"] for a in audits),\n                \"joint_finite_ice_rows\": sum(a[\"joint_ice_proxy_gl\"] and a[\"ice_finite_count\"] > 0 for a in audits),\n                \"joint_finite_both_rows\": sum(a[\"joint_ice_proxy_gl\"] and a[\"ice_finite_count\"] > 0\n                                              and a[\"gl_finite_count\"] > 0 for a in audits),\n            },\n            \"provenance\": provenance,\n            \"limitations\": [\n                \"Visible smoke is not independent validation.\",\n                \"An inactive fallback in visible smoke says nothing about hidden benefit.\",\n            ],\n        }\n        pd.DataFrame(audits).to_csv(root / \"experiment_audit_factorial.csv\", index=False)\n        (root / \"experiment_audit_factorial.json\").write_text(\n            json.dumps(dict(report, rows=audits), indent=2, allow_nan=False) + \"\\n\"\n        )\n        (root / \"experiment_verification.json\").write_text(\n            json.dumps(report, indent=2, allow_nan=False) + \"\\n\"\n        )\n        print(\"FACTORIAL_VERIFICATION\", json.dumps(report))\n        print(\"N1_CHALLENGER_FINAL\", json.dumps(report[\"n1_challenger\"]), flush=True)\n        return report\n\n    return run_overlay(scope, **{'experiment': 'E006', 'window': 40, 'ice_fallback': True, 'provenance': {'base_notebook_sha256': '1f0f9b3026a93ddb5ad6fa6993c8ac0a4bfaddc54594eb1dd6dda7a20d8ef0d6', 'base_source_code_sha256': 'eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2', 'helper_source_sha256': 'a027a69ddda2c2e14639e3ec5196d5f1eb2ab57cba10ef929723148f31fc8f34', 'source_script_version_id': 356740494, 'ice_policy_source_script_version_id': 356785434, 'ice_policy_source': 'samanyu1808/casmi26-iceberg-where-glacier-is-blind-lb-0-438/1'}})\n\nFACTORIAL_REPORT = _factorial_entry(globals())\ndel _factorial_entry\n\n\n# ===== cell 43 =====\n# Rank-2 placement of the second engine's leading structure (since v48), after\n# antriksharya/casmi26-lb-0-443-v7c-engine-2-rank-2-insert (MODE 'rank2'): the first structure of the second-engine\n# list moves to rank 2 unless it already holds rank 1; keys = E006 metric identity; errors keep E006's file unchanged.\ndef _rank2_overlay(scope):\n    import hashlib, json, os\n    from pathlib import Path\n    import pandas as pd\n    path = Path('submission.csv')\n    control = path.read_bytes()\n    report = dict(control_sha256=hashlib.sha256(control).hexdigest(), rows=0, inserted=0, status='fallback')\n    try:\n        identity = scope['metric_identity']\n        engine = scope['ENG'] if isinstance(scope.get('ENG'), dict) else {}\n        frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n        rows = []\n        for mid, answer in frame.itertuples(index=False):\n            report['rows'] += 1\n            toks = [t for t in str(answer).split(';') if t]\n            listed = [s for s in ((engine.get(str(mid)) or {}).get('smiles') or []) if s and s != 'CCO']\n            top = listed[0] if listed else None\n            key = identity(top) if top else None\n            if toks and toks[0] != 'CCO' and key is not None:\n                keys = [identity(t) for t in toks]\n                if key != keys[0]:\n                    rest = [t for t, k in zip(toks, keys) if k != key]\n                    toks = (rest[:1] + [top] + rest[1:])[:25]\n                    report['inserted'] += 1\n            rows.append((mid, ';'.join(toks) if toks else answer))\n        out = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])\n        assert out.molecule_id.tolist() == frame.molecule_id.tolist() and out.smiles.notna().all()\n        assert out.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25).all()\n        for s in out.smiles:\n            ids = [identity(x) for x in s.split(';')]\n            assert s == 'CCO' or (None not in ids and len(ids) == len(set(ids)))\n        Path('submission_e006_control.csv').write_bytes(control)\n        out.to_csv('submission_stage.csv', index=False)\n        os.replace('submission_stage.csv', path)\n        report['status'] = 'ok'\n    except Exception as exc:\n        path.write_bytes(control)\n        report.update(status='fallback', error=f'{type(exc).__name__}: {exc}'[:400], inserted=0)\n    report['submission_sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()\n    Path('rank2_overlay.json').write_text(json.dumps(report, indent=2))\n    print('RANK2_OVERLAY', json.dumps(report), flush=True)\n    return report\n\n\nRANK2_REPORT = _rank2_overlay(globals())\ndel _rank2_overlay\n\n\n# ===== cell 44 =====\nV43_DATASET_SOURCES = ['dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets', 'dmitriigluzdov/casmi26-pubchem-popularity-prior', 'dmitriigluzdov/casmi26-natural-product-panel-benchmark', 'dmitriigluzdov/casmi26-natural-product-knowledge-table', 'dmitriigluzdov/casmi26-fold-safe-fpnet']\nV43_E006_SHA256 = 'ee55b55c7f5bdb08f1a8154396b18fdb0203fb7178b3a52429e0999e69ad7e53'\n# Read the final E006 file; validate it without changing its bytes or candidate order.\ndef _v43_receipt(scope):\n    import hashlib\n    import json\n    import os\n    import time\n    from pathlib import Path\n    import pandas as pd\n    import rdkit\n\n    def digest(path):\n        path = Path(path)\n        h = hashlib.sha256()\n        with path.open('rb') as source:\n            for block in iter(lambda: source.read(8 * 1024 * 1024), b''):\n                h.update(block)\n        return dict(path=str(path), bytes=path.stat().st_size, sha256=h.hexdigest())\n\n    final_path = Path('submission.csv')\n    final_before = digest(final_path)\n    frame = pd.read_csv(final_path, dtype=str, keep_default_na=False)\n    sample = pd.read_csv(Path(scope['COMP']) / 'sample_submission.csv', dtype=str)\n    assert list(frame.columns) == ['molecule_id', 'smiles']\n    assert frame.molecule_id.tolist() == sample.molecule_id.tolist()\n    assert frame.molecule_id.is_unique and frame.smiles.notna().all()\n    processed = set(scope['te'].molecule_id.astype(str))\n    inputs = set(scope['E004_PREFLIGHT']['input_ids'])\n    assert processed == (inputs if scope['IS_RERUN'] else set(sorted(inputs)[:scope['SMOKE_N']]))\n    for mid, answer in frame.itertuples(index=False):\n        guesses = answer.split(';')\n        assert 1 <= len(guesses) <= 25, mid\n        identities = [scope['metric_identity'](s) for s in guesses]\n        assert None not in identities and len(set(identities)) == len(identities), mid\n    assert scope['n_err'] == 0\n    assert scope['FACTORIAL_REPORT']['same_run_control_reconstructed']\n    assert scope['FACTORIAL_REPORT']['submission_sha256'] == scope['RANK2_REPORT']['control_sha256']\n    assert scope['RANK2_REPORT']['submission_sha256'] == final_before['sha256']\n\n    # Hash all files of the mounted public datasets plus the competition files read.\n    # The inventory records content hashes, including model arrays and source modules.\n    roots = {str(Path(scope['E004_PREFLIGHT']['runtime_input_path']).resolve())}\n    input_root = Path('/kaggle/input')\n    dataset_roots = set()\n    for slug in V43_DATASET_SOURCES:\n        owner, name = slug.split('/', 1)\n        direct = input_root / name\n        if direct.is_dir():\n            dataset_roots.add(direct.resolve())\n        else:\n            dataset_roots.update(p.resolve() for p in (input_root / 'datasets' / owner).glob(name) if p.is_dir())\n    mounted = sorted({p.resolve() for root in dataset_roots for p in root.rglob('*') if p.is_file()})\n    comp_root = Path(scope['E004_PREFLIGHT']['runtime_input_path'])\n    mounted = sorted(set(mounted) | {comp_root.joinpath(name).resolve() for name in\n                                    ('train.parquet', 'test.parquet', 'sample_submission.csv')})\n    def quick_digest(path, limit=32 << 20, edge=4 << 20):\n        path = Path(path)\n        size = path.stat().st_size\n        if size <= limit:\n            return dict(digest(path), full=True)\n        h = hashlib.sha256()\n        with path.open('rb') as source:\n            h.update(source.read(edge))\n            source.seek(size - edge)\n            h.update(source.read(edge))\n        return dict(path=str(path), bytes=size, sha256_head_tail=h.hexdigest(), full=False)\n\n    input_hashes = [quick_digest(path) for path in mounted]   # large files: size + head/tail hash (full reads took > 2 h)\n    assert len(dataset_roots) == len(V43_DATASET_SOURCES), (len(dataset_roots), len(V43_DATASET_SOURCES))\n    Path('v43_mounted_input_hashes.json').write_text(json.dumps(input_hashes, indent=2) + '\\n')\n    stages = scope['E004_REPORT']['stages']\n    result = dict(\n        version=49, method='canqiang_E006_window40_ice_fallback_plus_v7c_fragment_gap_plus_e2_rank2_rel150_e2_bags_chebi_lmsd_pubchem_spectral_bonus_np_bonus_pubchemlite_lotus_refs_depth6_poplam015_fold0_n1a_syn050_pcdelta_extlib_n1challenger_jointicegl',\n        source_notebook_sha256=V43_E006_SHA256,\n        determinism=scope.get('V43_DETERMINISM', {'applied': False}),\n        rdkit=rdkit.__version__, output_rows=len(frame), input_molecules=len(inputs),\n        processed_molecules=len(processed), processed_ids=sorted(processed),\n        scoring_rerun=bool(scope['IS_RERUN']), smoke_n=int(scope['SMOKE_N']),\n        placeholder_rows=len(inputs - processed), main_errors=int(scope['n_err']),\n        main_lists=len(scope['BASE']), engine2_lists=len(scope['ENG']), pubchem_lists=len(scope['PC']),\n        engine2_nonempty=sum(bool(scope['ENG'].get(mid, {}).get('smiles')) for mid in processed),\n        pubchem_nonempty=sum(bool(scope['PC'].get(mid, {}).get('pc')) for mid in processed),\n        merge=scope['stats'], stages=stages,\n        final_overlay=scope['FACTORIAL_REPORT'], rank2_overlay=scope['RANK2_REPORT'],\n        pubchem_fpnet_ensemble=dict(models='A+B+fold0', fold0_weight=0.3333333333333333, raw_logit_scale=1.5, fold0_path=scope['FOLD0_PATH']),\n        mounted_datasets=V43_DATASET_SOURCES, mounted_files=len(input_hashes),\n        mounted_input_inventory=digest('v43_mounted_input_hashes.json'),\n        train=digest(comp_root / 'train.parquet'), test=digest(comp_root / 'test.parquet'),\n        processed_test=digest(Path(scope['COMP']) / 'test.parquet'),\n        submission=final_before, source_manifest=digest(Path(scope['V4MOD']) / 'MANIFEST.json'),\n        elapsed_seconds=round(time.time() - scope['T0'], 2),\n        notebook_elapsed_seconds=round(time.monotonic() - scope['NOTEBOOK_STARTED'], 2),\n    )\n    assert digest(final_path)['sha256'] == final_before['sha256']\n    Path('v43_execution_receipt.json').write_text(json.dumps(result, indent=2, default=str) + '\\n')\n    print('V43_EXECUTION_RECEIPT', json.dumps({k: result[k] for k in\n          ('output_rows', 'processed_molecules', 'placeholder_rows', 'main_errors', 'elapsed_seconds')}))\n    return result\n\nV43_RECEIPT = _v43_receipt(globals())\ndel _v43_receipt\n\n\n# ===== cell 45 =====\ntry:\n    # Figure 1 reads the final submission file and never reruns the ranking.\n    example_ids = list(te.molecule_id.drop_duplicates())\n    eligible_ids = [m for m in example_ids if BASE[m][2] < LIB_TAU and PC.get(m, {}).get('pc')]\n    example_mid = eligible_ids[0] if eligible_ids else min(example_ids, key=lambda m: BASE[m][2])\n    example_query = te[te.molecule_id.eq(example_mid)]\n    main_smiles, main_keys, example_lib = BASE[example_mid]\n    proposal = PC.get(example_mid, {})\n    _ft = proposal.get('fz_top')\n    if _ft is None and proposal.get('pc_fz'):\n        _ft = proposal['pc_fz'][0]\n    _bp = proposal.get('best_pool_fz')\n    _delta = (_ft - _bp if _ft is not None and _bp is not None and np.isfinite(_bp) else 1e9)\n    if not proposal.get('pc'):\n        branch = 'no PubChem proposals'\n    elif example_lib >= LIB_TAU:\n        branch = 'strong library match: PubChem proposals not inserted'\n    else:\n        branch = 'PubChem proposals at ranks 2, 4, 6, 8, 10' if _delta > REL_TH else 'PubChem proposals at ranks 4, 8, 12, 16, 20'\n        if (proposal.get('S') is not None and float(proposal['S']) > S_TAU and\n            float(proposal.get('top_pop') or 0.0) >= POP_TAU) or _delta > 150.0:\n            branch += ', leading proposal moved to rank 1'\n    _v43_audits = json.loads(Path('experiment_audit_factorial.json').read_text())['rows']\n    _v43_row = next(row for row in _v43_audits if row['molecule_id'] == str(example_mid))\n    _final_names = {'glacier': 'GLACIER', 'ice_fallback': 'ICEBERG', 'frag_gap': 'fragment score',\n                    'unrescored': 'none', 'no_engine': 'none', 'placeholder': 'none'}\n    _stage = _final_names.get(_v43_row['branch'], _v43_row['branch'])\n    if _v43_row['branch'] == 'ice_fallback' and not _v43_row.get('ice_finite_count'):\n        _stage = 'none (no finite ICEBERG scores)'\n    branch += '; formula-level scoring: ' + _stage\n    delta_text = f' \u00b7 \u0394 {_delta:.0f}' if _delta != 1e9 else ''\n    display(Markdown(f'**{example_mid}** \u00b7 {len(example_query)} spectra \u00b7 '\n        f'neutral mass {np.nanmedian(example_query.nm):.4f} Da \u00b7 library match {example_lib:.3f} \u00b7 '\n        f'**{branch}**' + delta_text))\n    _v43_final = pd.read_csv('submission.csv', dtype=str, keep_default_na=False).set_index('molecule_id')\n    final_smiles = _v43_final.at[str(example_mid), 'smiles'].split(';')\n    known = {name: {metric_identity(s) for s in candidates} for name, candidates in (\n        ('main', BASE_TAIL_CANDIDATES[str(example_mid)]), ('PubChem', proposal.get('pc', [])),\n        ('second engine', ENG.get(str(example_mid), {}).get('smiles', [])))}\n    example_rows = []\n    for rank, smiles in enumerate(final_smiles[:5], 1):\n        identity = metric_identity(smiles)\n        channels = [name for name, identities in known.items() if identity in identities]\n        example_rows.append({'Rank': rank, 'Channel': ' + '.join(channels) or 'list filling',\n                             'Connectivity key': identity[0], 'SMILES': smiles})\n    display(Markdown('**Worked-example candidates.**'))\n    display(pd.DataFrame(example_rows))\n    drawn = [Chem.MolFromSmiles(s) for s in final_smiles[:3]]\n    if all(m is not None for m in drawn):\n        display(Draw.MolsToGridImage(drawn, molsPerRow=3, subImgSize=(360, 280), useSVG=True,\n                                   legends=[f'Rank {i}' for i in range(1, len(drawn) + 1)]))\nexcept Exception as exc:\n    print('worked example unavailable:', type(exc).__name__, exc)\n\n\n# ===== cell 46 =====\n# ==== ours (yasunorim): drop scratch trees from /kaggle/working (the output listing is paged at 500 files) ====\nimport os as _os, shutil as _sh\nfor _d in ('ice_site', 'gl_site', '__pycache__', 'ours_mods'):\n    _p = _os.path.join('/kaggle/working', _d)\n    if _os.path.isdir(_p):\n        _sh.rmtree(_p, ignore_errors=True)\n_n = sum(len(f) for _, _, f in _os.walk('/kaggle/working'))\nprint('files left in /kaggle/working', _n, '| submission.csv', _os.path.isfile('/kaggle/working/submission.csv'), flush=True)\n\n\n# ===== cell 47 =====\n# ==== port (not ours; build.py nn): sanpier 'PubChem + Neural025 - LB 0.466' neural rerank of the final 25 ====\n# Modules copied unchanged (enveda-casmi26/kernels/e3/nn_port/SOURCE.txt). Runs after every base rule and every\n# cell marked ours; any failure keeps the submission as it was.\nimport os as _nn_os, sys as _nn_sys, json as _nn_json, shutil as _nn_sh, subprocess as _nn_sp, gc as _nn_gc\nfrom pathlib import Path as _NNP\nNN_MODULES = {'export_evidence.py': '\"\"\"Executed in the frozen pipeline process AFTER every original final rule.\"\"\"\\nimport math,json,shutil\\nfrom pathlib import Path\\nimport pandas as pd\\n\\ndef export(scope, root):\\n    root=Path(root);frame=pd.read_csv(root/\\'submission.csv\\',dtype=str,keep_default_na=False)\\n    shutil.copy2(root/\\'submission.csv\\',root/\\'submission_baseline0465.csv\\')\\n    processed=set(scope[\\'te\\'].molecule_id.astype(str));identity=scope[\\'metric_identity\\']\\n    def rank_map(smis):\\n        out={}\\n        for rank,s in enumerate(smis,1):\\n            key=identity(s)\\n            if key is not None:out.setdefault(key,rank)\\n        return out\\n    def finite(v):\\n        try:return float(v) if v is not None and math.isfinite(float(v)) else None\\n        except (ValueError,TypeError):return None\\n    rows=[]\\n    for mid,answer in frame.itertuples(index=False):\\n        candidates=answer.split(\\';\\')\\n        assert 1<=len(candidates)<=25 and len(candidates)==len(set(candidates)),mid\\n        active=mid in processed\\n        base=scope[\\'BASE\\'].get(mid,([],[],0.))\\n        maps={\\n            \\'main\\':rank_map(base[0]),\\n            \\'prefusion\\':rank_map(scope.get(\\'PREFUSION_LISTS\\',{}).get(mid,[])),\\n            \\'engine2\\':rank_map(scope.get(\\'ENG\\',{}).get(mid,{}).get(\\'smiles\\',[])),\\n            \\'pubchem\\':rank_map(scope.get(\\'PC\\',{}).get(mid,{}).get(\\'pc\\',[]))}\\n        ice=scope.get(\\'ICE_SCORES\\',{}).get(mid,{}) or {}\\n        gl=scope[\\'gl_of\\'](mid) or {}\\n        for rank,s in enumerate(candidates,1):\\n            ident=identity(s);key=scope[\\'chem\\'].score_key(s)\\n            row=dict(molecule_id=mid,smiles=s,metric_key=ident[0] if ident else None,baseline_rank=rank,\\n                processed=active,library_max=finite(base[2]) or 0.,proxy_gl=mid in scope.get(\\'N1_GL_MIDS\\',set()),\\n                ice=finite(ice.get(s,ice.get(key))),gl=finite(gl.get(s,gl.get(key))))\\n            row.update({name+\\'_rank\\':mapping.get(ident,0) for name,mapping in maps.items()});rows.append(row)\\n    pd.DataFrame(rows).to_parquet(root/\\'frozen_candidate_evidence.parquet\\',index=False)\\n    # Save the effective query spectra used by the pipeline (12 rows\\' molecules in visible smoke, all in rerun).\\n    scope[\\'te\\'].to_parquet(root/\\'effective_query_spectra.parquet\\',index=False)\\n    (root/\\'frozen_run_info.json\\').write_text(json.dumps(dict(processed_molecules=len(processed),\\n        output_molecules=len(frame),is_full_run=bool(scope[\\'IS_RERUN\\']),\\n        note=\\'Baseline file saved after the original Engine2 rank-2 insertion; candidate-specific PubChem promotion fix included.\\'),indent=2))\\n    print(\\'EXPORTED FROZEN FINAL25\\',len(rows),\\'candidate rows\\',flush=True)\\n', 'neural_scores.py': '\"\"\"Runs after the public pipeline process exits, freeing its GPU memory.\"\"\"\\nimport json,sys,importlib.util,types,hashlib\\nfrom pathlib import Path\\nimport numpy as np,pandas as pd,torch\\n\\ndef load_model(bundle,checkpoint,device):\\n    # Private module namespace prevents collision with public engines\\' model/features modules.\\n    package=\\'_casmi_own\\';p=types.ModuleType(package);p.__path__=[str(bundle)];sys.modules[package]=p\\n    for name in [\\'features\\',\\'model\\']:\\n        spec=importlib.util.spec_from_file_location(package+\\'.\\'+name,Path(bundle)/(name+\\'.py\\'))\\n        mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod)\\n    features=sys.modules[package+\\'.features\\'];cls=sys.modules[package+\\'.model\\'].SpectrumCandidateModel\\n    cfg=json.loads((Path(bundle)/\\'config.json\\').read_text());model=cls(cfg).to(device)\\n    ck=torch.load(Path(bundle)/checkpoint,map_location=device,weights_only=True)\\n    model.load_state_dict(ck[\\'state_dict\\']);model.eval()\\n    return model,features,cfg\\n\\n@torch.inference_mode()\\ndef score(bundle,checkpoint,evidence,queries,output,device=\\'cuda\\'):\\n    torch.set_num_threads(4)\\n    if device==\\'cuda\\':assert torch.cuda.is_available(),\\'GPU required for neural scoring\\'\\n    model,feat,cfg=load_model(bundle,checkpoint,device)\\n    frame=pd.read_parquet(evidence);q=pd.read_parquet(queries);q[\\'molecule_id\\']=q.molecule_id.astype(str)\\n    query={mid:g for mid,g in q.groupby(\\'molecule_id\\',sort=False)};cache={};parts=[];errors=[]\\n    for index,(mid,g) in enumerate(frame.groupby(\\'molecule_id\\',sort=False)):\\n        g=g.sort_values(\\'baseline_rank\\',kind=\\'stable\\').copy()\\n        g[\\'neural_ok\\']=False\\n        if not g.processed.iloc[0]:\\n            for c in [\\'neural_cos\\',\\'neural_fp\\',\\'neural_score\\',\\'neural_std\\',\\'n_spectra\\',\\'n_adducts\\']:g[c]=0.\\n            parts.append(g);continue\\n        try:\\n            assert mid in query,mid\\n            records=[]\\n            # Use EVERY observed query spectrum; no adduct merging and no new spectrum selection rule.\\n            for row in query[mid].itertuples(index=False):\\n                try:\\n                    pk,n=feat.peaks(row.ms2_mzs,row.ms2_normalized_intensities,row.precursor_mz,cfg[\\'max_peaks\\'])\\n                    if n<cfg[\\'min_peaks\\']:continue\\n                    m,a=feat.metadata(row.precursor_mz,row.adduct,row.ionization_mode,row.collision_energy_ev)\\n                    records.append((pk,n,m,a))\\n                except (ValueError,TypeError):continue\\n            if not records:raise ValueError(\\'No usable spectra after the training preprocessing filters\\')\\n            fp=[]\\n            for s in g.smiles:\\n                if s not in cache:cache[s]=feat.structure(s,cfg[\\'fp_bits\\'])[-1]\\n                fp.append(cache[s])\\n            F=torch.tensor(np.stack(fp),device=device,dtype=torch.float32);M=model.encode_molecules(F)\\n            cos_all=[];fp_all=[]\\n            for st in range(0,len(records),64):\\n                b=records[st:st+64]\\n                x=torch.tensor(np.stack([v[0] for v in b]),device=device)\\n                lengths=torch.tensor([v[1] for v in b],device=device,dtype=torch.long)\\n                meta=torch.tensor(np.stack([v[2] for v in b]),device=device)\\n                adduct=torch.tensor([v[3] for v in b],device=device,dtype=torch.long)\\n                Z,P=model.encode_spectra(x,lengths,meta,adduct)\\n                cos_all.append((Z@M.T).cpu().numpy());fp_all.append((P@F.T/cfg[\\'fp_bits\\']).cpu().numpy())\\n            cos=np.concatenate(cos_all);fps=np.concatenate(fp_all)\\n            combined=cos/cfg[\\'temperature\\']+cfg[\\'fp_score_weight\\']*fps\\n            assert np.isfinite(combined).all(),mid\\n            g[\\'neural_cos\\']=cos.mean(0);g[\\'neural_fp\\']=fps.mean(0);g[\\'neural_score\\']=combined.mean(0)\\n            g[\\'neural_std\\']=combined.std(0);g[\\'n_spectra\\']=len(records);g[\\'n_adducts\\']=query[mid].adduct.nunique()\\n            g[\\'neural_ok\\']=True\\n            parts.append(g)\\n        except (ValueError,TypeError) as exc:\\n            errors.append(dict(molecule_id=mid,error=str(exc)))\\n            for col in [\\'neural_cos\\',\\'neural_fp\\',\\'neural_score\\',\\'neural_std\\',\\'n_spectra\\',\\'n_adducts\\']:g[col]=0.\\n            parts.append(g)\\n        if (index+1)%25==0:print(\\'NEURAL SCORED\\',index+1,\\'molecules\\',flush=True)\\n    result=pd.concat(parts,ignore_index=True)\\n    assert len(result)==len(frame)\\n    result.to_parquet(output,index=False)\\n    Path(str(output)+\\'.errors.json\\').write_text(json.dumps(errors,indent=2))\\n    print(\\'NEURAL input failures kept at baseline:\\',len(errors),flush=True)\\n    print(\\'NEURAL FEATURES\\',result.shape,checkpoint,flush=True)\\n\\nif __name__==\\'__main__\\':score(*sys.argv[1:])\\n', 'submission.py': 'import json,hashlib\\nfrom pathlib import Path\\nimport pandas as pd,numpy as np\\nfrom ranking import rrf_order\\n\\ndef apply(evidence,baseline,output,weight=.25,offset=3.):\\n    f=pd.read_parquet(evidence);base=pd.read_csv(baseline,dtype=str,keep_default_na=False)\\n    groups={mid:g for mid,g in f.groupby(\\'molecule_id\\',sort=False)};rows=[];audit=[]\\n    assert base.molecule_id.is_unique and set(groups)==set(base.molecule_id)\\n    for mid,text in base.itertuples(index=False):\\n        g=groups[mid].sort_values(\\'baseline_rank\\',kind=\\'stable\\');original=text.split(\\';\\')\\n        assert g.smiles.tolist()==original,mid\\n        active=bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\\n        order=rrf_order(g.neural_score.values,weight,offset) if active else np.arange(len(g))\\n        new=g.smiles.iloc[order].tolist()\\n        assert len(new)==len(original) and set(new)==set(original),mid\\n        assert len(new)<=25 and len(set(new))==len(new)\\n        rows.append((mid,\\';\\'.join(new)))\\n        audit.append(dict(molecule_id=mid,processed=active,changed=new!=original,\\n            changed_top1=new[0]!=original[0],candidate_count=len(new),weight=weight))\\n    result=pd.DataFrame(rows,columns=[\\'molecule_id\\',\\'smiles\\'])\\n    assert result.molecule_id.tolist()==base.molecule_id.tolist()\\n    result.to_csv(output,index=False)\\n    out=Path(output).parent;pd.DataFrame(audit).to_csv(out/\\'neural_rerank_audit.csv\\',index=False)\\n    report=dict(method=\\'baseline rank + neural rank reciprocal-rank fusion AFTER every baseline rule\\',\\n        neural_weight=weight,rrf_offset=offset,weight_is_validated=False, validation_scope=\"Fixed-weight submission experiment; no validation performed\",\\n        candidate_set_preserved=True,changed_rows=sum(x[\\'changed\\'] for x in audit),\\n        changed_top1=sum(x[\\'changed_top1\\'] for x in audit),rows=len(rows),\\n        frozen_sha256=hashlib.sha256(Path(baseline).read_bytes()).hexdigest(),\\n        experimental_sha256=hashlib.sha256(Path(output).read_bytes()).hexdigest())\\n    (out/\\'neural_submission_report.json\\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\\n    return report\\n', 'ranking.py': '\"\"\"Pure deterministic ranking utilities shared by both notebooks.\"\"\"\\nimport numpy as np\\nimport pandas as pd\\n\\ndef stable_rank(values):\\n    values=np.asarray(values,dtype=float)\\n    assert np.isfinite(values).all(), \\'Nonfinite candidate scores\\'\\n    order=np.argsort(-values,kind=\\'stable\\');ranks=np.empty(len(values),int);ranks[order]=np.arange(1,len(values)+1)\\n    return ranks\\n\\ndef rrf_order(neural_scores,weight=0.15,offset=3.0):\\n    assert weight>=0 and offset>=0\\n    n=len(neural_scores);baseline=np.arange(1,n+1)\\n    # Zero weight guarantees exact baseline order even when neural scores are unavailable.\\n    if weight==0:return np.arange(n)\\n    ranks=stable_rank(neural_scores)\\n    return np.argsort(-(1/(offset+baseline)+weight/(offset+ranks)),kind=\\'stable\\')\\n\\ndef metrics(frame,score=\\'prediction\\'):\\n    out=[]\\n    for mid,g in frame.groupby(\\'molecule_id\\',sort=False):\\n        g=g.sort_values(\\'baseline_rank\\',kind=\\'stable\\')\\n        order=np.argsort(-g[score].to_numpy(),kind=\\'stable\\')\\n        labels=g.label.to_numpy()[order];where=np.flatnonzero(labels)\\n        r=int(where[0]+1) if len(where) else None\\n        out.append(dict(molecule_id=mid,mrr25=1/r if r and r<=25 else 0.,top1=int(r==1),\\n                        candidate_recall=int(bool(len(where))),rank=r))\\n    return pd.DataFrame(out)\\n\\nBASE_FEATURES=[\\'baseline_rr\\',\\'main_rr\\',\\'prefusion_rr\\',\\'engine2_rr\\',\\'pubchem_rr\\',\\'engine_agreement\\',\\n    \\'library_max\\',\\'proxy_gl\\',\\'ice_missing\\',\\'gl_missing\\',\\'ice_z\\',\\'gl_z\\',\\n    \\'neural_cos_z\\',\\'neural_fp_z\\',\\'neural_score_z\\',\\'neural_rr\\',\\'neural_std\\',\\n    \\'n_spectra_log\\',\\'n_adducts\\',\\'neural_top_gap\\',\\'neural_agrees_top1\\',\\'engine2_agrees_top1\\',\\n    \\'neural_delta_rank\\',\\'neural_x_library\\',\\'neural_x_agreement\\',\\'neural_x_sim_missing\\',\\n    \\'main_missing\\',\\'engine2_missing\\',\\'pubchem_missing\\']\\n\\ndef make_features(frame):\\n    f=frame.copy()\\n    for src in [\\'baseline\\',\\'main\\',\\'prefusion\\',\\'engine2\\',\\'pubchem\\']:\\n        r=pd.to_numeric(f[src+\\'_rank\\'],errors=\\'coerce\\');f[src+\\'_rr\\']=np.where(r>0,1/(3+r),0)\\n        if src in [\\'main\\',\\'engine2\\',\\'pubchem\\']:f[src+\\'_missing\\']=(~(r>0)).astype(float)\\n    for src in [\\'ice\\',\\'gl\\',\\'neural_cos\\',\\'neural_fp\\',\\'neural_score\\']:\\n        v=pd.to_numeric(f[src],errors=\\'coerce\\');v=v.where(np.isfinite(v))\\n        if src in [\\'ice\\',\\'gl\\']:f[src+\\'_missing\\']=v.isna().astype(float)\\n        grp=v.groupby(f.molecule_id,sort=False)\\n        f[src+\\'_z\\']=((v-grp.transform(\\'mean\\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\\n    f[\\'neural_rank\\']=0.;f[\\'neural_top_gap\\']=0.;f[\\'neural_agrees_top1\\']=0.;f[\\'engine2_agrees_top1\\']=0.\\n    for _,g in f.groupby(\\'molecule_id\\',sort=False):\\n        idx=g.index;rank=stable_rank(g.neural_score.values);f.loc[idx,\\'neural_rank\\']=rank\\n        vals=np.sort(g.neural_score_z.values)[::-1]\\n        f.loc[idx,\\'neural_top_gap\\']=float(vals[0]-vals[1]) if len(vals)>1 else 0.\\n        f.loc[idx,\\'neural_agrees_top1\\']=float(g.iloc[np.argmin(rank)].baseline_rank==1)\\n        f.loc[idx,\\'engine2_agrees_top1\\']=float(((g.baseline_rank==1)&(g.engine2_rank==1)).any())\\n    f[\\'neural_rr\\']=1/(3+f.neural_rank)\\n    f[\\'engine_agreement\\']=((f.main_rank>0).astype(int)+(f.engine2_rank>0).astype(int)+(f.pubchem_rank>0).astype(int))\\n    f[\\'n_spectra_log\\']=np.log1p(f.n_spectra)\\n    f[\\'neural_delta_rank\\']=(f.baseline_rank-f.neural_rank)/25\\n    f[\\'neural_x_library\\']=f.neural_score_z*f.library_max\\n    f[\\'neural_x_agreement\\']=f.neural_score_z*f.engine_agreement\\n    f[\\'neural_x_sim_missing\\']=f.neural_score_z*f.ice_missing*f.gl_missing\\n    X=f[BASE_FEATURES].astype(float).replace([np.inf,-np.inf],np.nan).fillna(0.)\\n    assert np.isfinite(X.values).all()\\n    return X\\n\\ndef residual_scores(frame,probability,beta):\\n    p=np.clip(np.asarray(probability,float),1e-5,1-1e-5)\\n    raw=pd.Series(np.log(p/(1-p)),index=frame.index)\\n    grp=raw.groupby(frame.molecule_id,sort=False)\\n    z=((raw-grp.transform(\\'mean\\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\\n    correction=float(beta)*z.values\\n    if \\'neural_ok\\' in frame:correction=np.where(frame.neural_ok.values,correction,0.)\\n    return -np.log(frame.baseline_rank.values.astype(float))+correction\\n', 'bundle.py': \"import hashlib,json,zipfile\\nfrom pathlib import Path\\n\\ndef sha(path):\\n    h=hashlib.sha256()\\n    with open(path,'rb') as f:\\n        for b in iter(lambda:f.read(8<<20),b''):h.update(b)\\n    return h.hexdigest()\\n\\ndef find_bundle(input_root,output_root,pins):\\n    roots=[]\\n    for path in Path(input_root).rglob('final_model.pt'):\\n        if (path.parent/'config.json').is_file():roots.append(path.parent)\\n    if not roots:\\n        archives=list(Path(input_root).rglob('casmi_spectrum_model_v1.zip'))\\n        assert len(archives)==1, 'Attach the supplied casmi_spectrum_model_v1.zip as a Kaggle dataset (zipped or extracted)'\\n        dest=Path(output_root)/'uploaded_model';dest.mkdir(parents=True,exist_ok=True)\\n        with zipfile.ZipFile(archives[0]) as z:\\n            for item in z.infolist():\\n                target=(dest/item.filename).resolve()\\n                assert target.is_relative_to(dest.resolve()),'Unsafe ZIP member'\\n            z.extractall(dest)\\n        roots=[p.parent for p in dest.rglob('final_model.pt')]\\n    matching=[r for r in roots if all((r/n).is_file() and sha(r/n)==h for n,h in pins.items())]\\n    assert len(matching)==1, f'Expected exactly one matching model bundle, found {len(matching)}. Use the attached v1 artifacts.'\\n    return matching[0]\\n\"}\nNN_PINS = {'features.py': 'a97ad9b3ab3711517864d56e497e1672cb29bb05683c72b52db9f9b0a17b1455', 'model.py': '2547418857386d1c2513fd1ab6c70961e6a96e73d9d495bbfe3052197e69dc39', 'config.json': '563fb2a2a81ca48ea094f9ab78c157a48a6e9a5503e85e6ce8f0fb3615c1cc03', 'final_model.pt': '44436b2daec64373e66f9120dd2389c24fc83dd9430ebeeac18e3cb49dcb9642'}\nNN_WEIGHT, NN_OFFSET = 0.25, 3.0\n_nn_root = _NNP('/kaggle/working'); _nn_code = _nn_root / 'nn_code'; _nn_code.mkdir(exist_ok=True)\n_nn_sh.copy2(_nn_root / 'submission.csv', _nn_root / 'submission_before_nn.csv')\ntry:\n    for _k, _v in NN_MODULES.items():\n        (_nn_code / _k).write_text(_v)\n    _nn_sys.path.insert(0, str(_nn_code))\n    from export_evidence import export as _nn_export\n    _nn_export(globals(), _nn_root)\n    from bundle import find_bundle as _nn_find\n    _nn_bundle = _nn_find('/kaggle/input', _nn_root, NN_PINS)\n    try:\n        import torch as _nn_t; _nn_gc.collect(); _nn_t.cuda.empty_cache()\n    except Exception:\n        pass\n    _nn_r = _nn_sp.run([_nn_sys.executable, str(_nn_code / 'neural_scores.py'), str(_nn_bundle), 'final_model.pt',\n                        str(_nn_root / 'frozen_candidate_evidence.parquet'), str(_nn_root / 'effective_query_spectra.parquet'),\n                        str(_nn_root / 'neural_candidate_features.parquet'), 'cuda'], cwd=str(_nn_root),\n                       capture_output=True, text=True, env={**_nn_os.environ, 'PYTHONPATH': str(_nn_code)})\n    print(_nn_r.stdout[-3000:], _nn_r.stderr[-3000:], flush=True)\n    assert _nn_r.returncode == 0, 'neural scoring failed'\n    from submission import apply as _nn_apply\n    NN_REPORT = _nn_apply(_nn_root / 'neural_candidate_features.parquet', _nn_root / 'submission_baseline0465.csv',\n                          _nn_root / 'submission.csv', weight=NN_WEIGHT, offset=NN_OFFSET)\n    print('NN_RERANK', _nn_json.dumps({k: NN_REPORT[k] for k in ('changed_rows', 'changed_top1', 'rows')}), flush=True)\nexcept Exception as _nn_e:\n    import traceback; traceback.print_exc()\n    _nn_sh.copy2(_nn_root / 'submission_before_nn.csv', _nn_root / 'submission.csv')\n    print('NN_RERANK failed, submission kept as before:', repr(_nn_e), flush=True)\n\n", "B": "# --- ours (fusion orchestrator): this script's own inputs only (all names kept inside a function) ---\ndef _ours_fu_install_filter(allowed, other_slugs):\n    import glob as gm\n    og, oig = gm.glob, gm.iglob\n    def keep(p):\n        p = str(p)\n        if not p.startswith('/kaggle/input/'):\n            return True\n        parts = p[len('/kaggle/input/'):].split('/')\n        if parts[0] == 'datasets' and len(parts) >= 3:\n            return parts[1] + '/' + parts[2] in allowed\n        return parts[0] not in other_slugs\n    def g(*a, **k):\n        return [p for p in og(*a, **k) if keep(p)]\n    def ig(*a, **k):\n        return (p for p in oig(*a, **k) if keep(p))\n    gm.glob, gm.iglob = g, ig\n    print('ours: input filter', sorted(allowed), flush=True)\n_ours_fu_install_filter(set(['ahmedberatozer/casmi26-fpnet-full1', 'ahmedberatozer/casmi26-glacier', 'ahmedberatozer/casmi26-iceberg', 'ahmedberatozer/casmi26-pubchem-tier', 'ahmedberatozer/casmi26-v2-pool', 'ahmedberatozer/casmi26-v3-models', 'ahmedberatozer/casmi26-v4b-models', 'dmitriigluzdov/casmi26-pubchem-popularity-prior', 'franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints', 'megayak/casmi26-simulated-ranker-rows', 'metric/rdkit-2026-3-3-wheel', 'prvsiyan/casmi26-fp-models-v2', 'prvsiyan/casmi26-ranker-features', 'prvsiyan/coconut-casmi26-candidates', 'sergiomalv/casmi-chebi-lmsd-structures', 'yasunorim/casmi26-own-coconut-candidates']), set(['casmi-spectrum-model-v1', 'casmi26-attributed-v4g-reproduction-assets', 'casmi26-fold-safe-fpnet', 'casmi26-natural-product-knowledge-table', 'casmi26-natural-product-panel-benchmark']))\ndel _ours_fu_install_filter\ntry:\n    from IPython.display import display\nexcept Exception:\n    display = print\n\n# ===== cell 0 =====\n'''CASMI26: Metric-Identity Fusion \u2014 Aggressive Experiment\nOne vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. Preserve the existing PubChem gate and same-formula GLACIER logic. This tests duplicate-vote bias, not a coefficient search. It can change top-1 and reduce the score.\nRun the complete repaired @obstacledeveloper adaptation first, then this independent overlay. submission_reference.csv is the completed baseline from the same run; experiment_audit.csv/json record changes and file hashes. This is not stacked with the other experiment. Same 17 source inputs and GPU environment; no additional dataset required. No new public score has been measured. A visible smoke run is not validation.'''\n\n\n# ===== cell 1 =====\n'''Credits & Attribution\n@obstacledeveloper \u2014 direct source: CASMI26 V1.1 Metric Audit and Safe Tail, version 1, scriptVersionId 356393001. Source: https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail?scriptVersionId=356393001\nThe source page displays public score 0.433; this adaptation has not reproduced that result.\n@bobthebot369 \u2014 inherited Sovereign Zenith ensemble and confidence-promotion ideas: https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith/notebook\n@ahmedberatozer \u2014 engine, ranker, pool, PubChem, full FPNet and simulator packages: https://www.kaggle.com/ahmedberatozer\n@prvsiyan \u2014 analog propagation, spectral matching and fingerprint models: https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline\n@dmitriigluzdov \u2014 PubChem popularity prior: https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior\n@hengck23 \u2014 inherited hengck23-dreams-enveda-casmi26 structural candidate resource: https://www.kaggle.com/hengck23\nOfficial metric implementation, version 14: https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank\nICEBERG/GLACIER research tooling: https://github.com/coleygroup/ms-pred\nCompetition: https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra\nRDKit official metric wheel: https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel\nChEBI/LIPID MAPS and COCONUT resources remain credited in inherited code and input metadata.\nThe public notebook declares Apache-2.0; upstream datasets/components retain their own licences. Substantial code, including embedded modules, is reused verbatim. Our adaptation changes presentation and adds provenance, not authorship of the engines. Upstream comments are retained as provenance, not independently established component-performance claims.'''\n\n\n# ===== cell 2 =====\n'''Experiment provenance and reasoning\nDirect starting point: @obstacledeveloper, linked in Credits & Attribution below. The upstream engine/fusion/simulator code is substantially reused, not claimed as original. Our contribution here is the separately audited overlay described above. Previous analog expansion, top-1 shielding, adduct-specific merging and single-pass variants did not improve our confirmed 0.420 result. These experiments test structural ranking changes instead of repeating those parameter changes. Metric identity uses the inherited official canonicalization and heavy-atom composition helpers. GLACIER tooling: https://github.com/coleygroup/ms-pred ; research: https://arxiv.org/abs/2606.29161 . Global cross-formula ranking is our unvalidated hypothesis, not a claim made by that paper. Public leaderboard feedback does not establish private-set generalization.'''\n\n\n# ===== cell 3 =====\n'''# CASMI26 V1.1 \u2014 metric audit and safe tail\n\nStarting point: this account's V1 submission, public MRR **0.425** on 2026-10-08.\nThis is a conservative candidate-coverage experiment, not a claimed leaderboard improvement.\n\nChanges: pin official RDKit 2026.03.3; preserve the V1 inference/ranking pipeline;\ndeduplicate only after ranking using the official metric identity; append ranked overflow\nand mass-compatible DreaMS candidates into vacant slots; assert every original matchable\nidentity has equal or better rank. The old DreaMS branch was bypassed by `NameError: Path`.\n\nNo hidden-test labels, manual labels, or network access are used. The visible test overlaps\ntraining data, so its score is an operational check and is **not** a validation estimate.\n\nSource attribution: copied V1 ensemble from BobtheBot369 / Ahmed Berat \u00d6zer / prvsiyan\nand the other authors credited in its code. Scoring reference: Kaggle Competition Metrics,\n`https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank`, version 14, Apache-2.0.\n'''\n\n\n# ===== cell 4 =====\n'''## 1. Setup\n\nLocates every dataset, installs the competition's RDKit wheel, sets the popularity-prior parameters for the PubChem channel (`POP_LAM = 0.25`, `POP_UNION = 200`) and checks the v4b model files against their manifest.\n'''\n\n\n# ===== cell 5 =====\n'''CASMI26 v4n = v4i + ICE_LAM = GL_LAM = 1.0 (v4l, LB 0.380) + engine FP bank fpnet_full1 (v4m, LB 0.378)'''\n'''CASMI26 v4i = v4h + PubChem-only channel pass-1 depth N1 1000 -> 5000 (recall; gate/slots unchanged)'''\n'''CASMI26 v4h = v4g + GLACIER third re-scoring term: z(ranker) + 0.5 z(ICEBERG) + 0.5 z(GLACIER) (casmi26-glacier)'''\n'''casmi26 v28 = v17 + pool popularity re-rank of the base list (mu 0.15, mode A); v17 = Ahmed v4g + popularity prior in the PubChem list (see make_v17.py)'''\n'''CASMI26 v4f = v4b (FPNet A engine bank, LB 0.354) + ICEBERG post-ranker isomer re-scoring = the v3.6 model (v1 engine + FPNet A + moderate generation + gated PubChem channel, LB 0.358)'''\n'''+ fe_v4 feature families computed on the SAME v1 engine run (derivation evidence / class-3 priors, fragmentation 2.0,'''\n'''analog-structure relations, DreamsFP views) + a ranker retrained on the v1-engine simulation sim1x with them.'''\n'''Built by work/analysis/fe_v4/build_v4v1.py -- edit there.'''\nimport os, sys, glob, re, subprocess, time, json, hashlib, pickle\nfrom pathlib import Path\nT0 = time.time()\nLIB_TAU, REL_TH = 0.9, 600.0\nTOPN, ICE_LAM, ICE_BUDGET, ICE_PC = 60, 1.0, 5400, True\nSLOTS_AGG, SLOTS_GENTLE = [2, 4, 6, 8, 10], [4, 8, 12, 16, 20]\nos.makedirs('/kaggle/working/numba_cache', exist_ok=True)\nos.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'\ndef find(pattern):\n    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)\n    if not hits:\n        raise FileNotFoundError(pattern)\n    return hits[0]\ndef sha256(path, n=1 << 22):\n    h = hashlib.sha256()\n    with open(path, 'rb') as f:\n        for b in iter(lambda: f.read(n), b''):\n            h.update(b)\n    return h.hexdigest()\nPOP_LAM, POP_UNION = 0.25, 200            # v17: popularity prior for the PubChem-only list\nos.environ['CASMI_POP_DIR'] = os.path.dirname(find('pc_lsid.npy')); os.environ['CASMI_POP_LAM'] = str(POP_LAM); os.environ['CASMI_POP_UNION'] = str(POP_UNION)\nprint('v17 popularity prior', POP_LAM, POP_UNION, os.environ['CASMI_POP_DIR'])\ntag = f'cp{sys.version_info.major}{sys.version_info.minor}'\nwhl = [w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w and Path(w).name.startswith('rdkit-2026.3.3-')]\nassert whl, 'Official RDKit 2026.3.3 wheel is missing'\nprint('rdkit wheel:', whl)\nr = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], capture_output=True, text=True)\nprint(r.stdout[-500:], r.stderr[-500:])\nr.check_returncode()\nimport rdkit; print('rdkit', rdkit.__version__)\nassert rdkit.__version__ == '2026.03.3'\nV4CODE = os.path.dirname(os.path.dirname(find('casmi26-v4b-models/**/casmi/engine.py')))\nV4MOD = os.path.dirname(find('casmi26-v4b-models/**/MANIFEST.json'))\nV3CODE = os.path.dirname(os.path.dirname(find('casmi26-v3-models/**/casmi/engine.py')))\nV3MOD = os.path.dirname(find('casmi26-v3-models/**/ranker_0.pkl'))\nPOOL_DIR = os.path.dirname(find('pool_meta.parquet'))\nPC_DIR = os.path.dirname(find('pc_smiles.npy'))\nCOMP = os.path.dirname(find('test.parquet'))\nMAN = json.load(open(os.path.join(V4MOD, 'MANIFEST.json')))\nbad = []\nfor rel, h in sorted(MAN['files'].items()):\n    p = os.path.join(V4CODE, rel[len('code/'):]) if rel.startswith('code/') else os.path.join(V4MOD, rel)\n    if not os.path.exists(p) or sha256(p) != h:\n        bad.append(rel)\nassert not bad, f'dataset files differ from MANIFEST.json: {bad}'\nprint('V4CODE', V4CODE, '\\nV4MOD', V4MOD, '\\nV3CODE', V3CODE, '\\nPOOL', POOL_DIR, '\\nPC', PC_DIR, '\\nCOMP', COMP)\nprint(f\"{len(MAN['files'])} files match MANIFEST.json; families {MAN['families']}\", f'{time.time()-T0:.0f}s')\n\n'''Compatibility repair: select the simulator RDKit 2025.3.6 wheel by interpreter and platform tags, including bundled .whl.ice files. Fail before inference when no compatible build exists; preserve the main RDKit 2026.03.3 environment.'''\n\ndef simulator_rdkit_wheel(pkg):\n    from pip._vendor.packaging.tags import sys_tags\n    from pip._vendor.packaging.utils import parse_wheel_filename\n    compatible = set(sys_tags())\n    bundled = sorted(glob.glob(os.path.join(pkg, 'wheels', 'rdkit-2025.3.6-*.whl*')))\n    external = sorted(glob.glob('/kaggle/input/**/rdkit-2025.3.6-*.whl', recursive=True))\n    for path in bundled + external:\n        name = os.path.basename(path).removesuffix('.ice')\n        _, version, _, tags = parse_wheel_filename(name)\n        if str(version) == '2025.3.6' and compatible.intersection(tags):\n            return path\n    raise FileNotFoundError(\n        f'RDKit 2025.3.6 wheel compatible with Python {sys.version_info.major}.{sys.version_info.minor} '\n        f'and this platform is required for ICEBERG/GLACIER; candidates: {bundled + external}')\n\n_simulator_pkg = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))\nprint('Simulator RDKit wheel:', simulator_rdkit_wheel(_simulator_pkg))\n\n\n# ===== cell 6 =====\n\"\"\"Read-only E004 observer, before the source smoke switch.\"\"\"\n\n\ndef _e004_preflight(scope):\n    import json\n    import os\n    import platform\n    import subprocess\n    import pandas as pd\n    import rdkit\n\n    slug = 'enveda-CASMI26-molecule-id-mass-spectra'\n    comp = os.path.realpath(scope['COMP'])\n    assert comp in {f'/kaggle/input/{slug}', f'/kaggle/input/competitions/{slug}'}\n    assert platform.python_version_tuple()[:2] == ('3', '12')\n    assert rdkit.__version__ == '2026.03.3'\n    # Do not initialize a parent CUDA context before the source's Engine2 subprocess.\n    gpu = subprocess.check_output(\n        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()\n    assert 'T4' in gpu, gpu\n    ids = sorted(set(pd.read_parquet(os.path.join(comp, 'test.parquet'),\n                                    columns=['molecule_id']).molecule_id.astype(str)))\n    sample = pd.read_csv(os.path.join(comp, 'sample_submission.csv'), dtype=str)\n    assert ids and set(sample.molecule_id) == set(ids)\n    result = dict(runtime_input_path=comp, input_ids=ids, python=platform.python_version(),\n                  rdkit=rdkit.__version__, gpu=gpu)\n    print('E004_PREFLIGHT', json.dumps({k: v for k, v in result.items() if k != 'input_ids'}))\n    return result\n\n\nE004_PREFLIGHT = _e004_preflight(globals())\ndel _e004_preflight\n\n\n# ===== cell 7 =====\n'''## 2. Commit switch\n\nWhen the visible placeholder test is detected, the notebook runs only 12 molecules with a short ICEBERG budget, so saving a version is quick. The scoring rerun on the hidden test runs everything. Nothing else depends on this switch.\n'''\n\n\n# ===== cell 8 =====\n'''SMOKE switch: commits (dummy test) run 12 molecules; the scoring rerun (hidden test) runs everything.'''\nimport pandas as pd, hashlib\n_te = pd.read_parquet(os.path.join(COMP, 'test.parquet'), columns=['molecule_id', 'spectrum_id'])\n_sig = hashlib.md5(','.join(sorted(_te.spectrum_id.astype(str))).encode()).hexdigest()\nIS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or _sig != '654e030e4173780e945252b4a5adf70b'\nSMOKE_N = 12\nif not IS_RERUN:\n    SM = '/kaggle/working/smoke_comp'; os.makedirs(SM, exist_ok=True)\n    _full = pd.read_parquet(os.path.join(COMP, 'test.parquet'))\n    _keep = sorted(_full.molecule_id.unique())[:SMOKE_N]\n    _full[_full.molecule_id.isin(_keep)].to_parquet(os.path.join(SM, 'test.parquet'))\n    for _f in ['train.parquet', 'sample_submission.csv']:\n        if not os.path.exists(os.path.join(SM, _f)):\n            os.symlink(os.path.join(COMP, _f), os.path.join(SM, _f))\n    COMP = SM\n    ICE_BUDGET = 300\nprint('RERUN', IS_RERUN, '| sig', _sig, '| COMP', COMP, '| ICE_BUDGET', ICE_BUDGET)\n\n\n# ===== cell 9 =====\n'''## 3. Engine 2: two-ranker engine (BIO + AFIX)\n\nThis engine runs first in its own process, while GPU memory is still empty. It writes a top-40 list per molecule with tautomer-canonical InChIKey-14 keys.\n\n- **Candidates** come from COCONUT, ChEBI/LIPID MAPS (BIO) and the training structures.\n- **Evidence** comes from library matching (including adduct-shifted matches), analog propagation, prvsiyan's fingerprint models and MetFrag-lite fragment matching.\n- **Ranking** blends two gradient-boosted rankers.\n'''\n\n\n# ===== cell 10 =====\n'''Our engine (two-ranker + BIO + AFIX) in a subprocess, then weighted reciprocal-rank fusion with the v4g lists.'''\nENG_DIR = '/kaggle/working/eng'; os.makedirs(ENG_DIR, exist_ok=True)\nENG_FILES = {'pv.py': '\"\"\"prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied\\nverbatim where possible so local simulations and the Kaggle notebook share one source of truth.\\nAdditions are marked `# ADDED`.\\n\"\"\"\\nimport math\\nimport numpy as np\\nfrom numba import njit, prange\\n\\n\\nclass CFG:\\n    PPM_WIN = 10.0\\n    PPM_FALLBACK = 30.0\\n    INT_FLOOR = 0.002\\n    MAX_PEAKS = 256\\n    MZ_TOL = 0.01\\n    INT_POWER = 1.0\\n    ENT_WEIGHT = True\\n    ANALOG_WIN = 200.0\\n    N_ANALOG = 80\\n    SIM_POWER = 3.0\\n    W1_PRIORS = (0.30, 0.60)\\n    SEEDS = (0, 1, 2, 3)\\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\\n    TOPN = 25\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# similarity kernels\\n# ----------------------------------------------------------------------------------------------\\n@njit(cache=True, fastmath=True)\\ndef _clean(mz, it, floor, topk, power, ent_weight):\\n    n = len(mz)\\n    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)\\n    mx = 0.0\\n    for i in range(n):\\n        if it[i] > mx: mx = it[i]\\n    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)\\n    thr = floor * mx; c = 0\\n    for i in range(n):\\n        if it[i] >= thr: c += 1\\n    idx = np.empty(c, np.int64); j = 0\\n    for i in range(n):\\n        if it[i] >= thr: idx[j] = i; j += 1\\n    if c > topk:\\n        v = np.empty(c, np.float32)\\n        for i in range(c): v[i] = it[idx[i]]\\n        o = np.argsort(v)[c - topk:]\\n        k2 = np.empty(topk, np.int64)\\n        for i in range(topk): k2[i] = idx[o[i]]\\n        k2.sort(); idx = k2; c = topk\\n    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)\\n    s = 0.0\\n    for i in range(c):\\n        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v\\n    if s > 0:\\n        for i in range(c): oi[i] /= s\\n    if ent_weight:\\n        S = 0.0\\n        for i in range(c):\\n            if oi[i] > 0: S -= oi[i] * np.log(oi[i])\\n        if S < 3.0:\\n            w = 0.25 + 0.25 * S; s2 = 0.0\\n            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]\\n            if s2 > 0:\\n                for i in range(c): oi[i] /= s2\\n    return om, oi\\n\\n\\n@njit(cache=True, fastmath=True)\\ndef entropy_sim(qmz, qp, cmz, cp, tol):\\n    i = 0; j = 0; n = len(qmz); m = len(cmz)\\n    SA = 0.0\\n    for x in range(n):\\n        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])\\n    SB = 0.0\\n    for x in range(m):\\n        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])\\n    SAB = 0.0; tot = 0.0\\n    buf = np.empty(n + m, np.float64); b = 0\\n    while i < n and j < m:\\n        d = qmz[i] - cmz[j]\\n        if d < -tol: buf[b] = qp[i]; i += 1; b += 1\\n        elif d > tol: buf[b] = cp[j]; j += 1; b += 1\\n        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1\\n    while i < n: buf[b] = qp[i]; i += 1; b += 1\\n    while j < m: buf[b] = cp[j]; j += 1; b += 1\\n    for x in range(b): tot += buf[x]\\n    if tot <= 0: return 0.0\\n    for x in range(b):\\n        v = buf[x] / tot\\n        if v > 0: SAB -= v * np.log(v)\\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\\n\\n\\n@njit(cache=True, fastmath=True, parallel=True)\\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\\n    out = np.zeros(len(cand), np.float32)\\n    for k in prange(len(cand)):\\n        c = cand[k]; a = off[c]; b = off[c + 1]\\n        if b <= a: continue\\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\\n        if len(cm) == 0: continue\\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\\n    return out\\n\\n\\n@njit(cache=True, fastmath=True)\\ndef entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):\\n    a = entropy_sim(qmz, qp, cmz, cp, tol)\\n    if shift > -0.001 and shift < 0.001: return a\\n    sm = np.empty(len(cmz), np.float32)\\n    for i in range(len(cmz)): sm[i] = cmz[i] + shift\\n    b = entropy_sim(qmz, qp, sm, cp, tol)\\n    return a if a > b else b\\n\\n\\n# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)\\n@njit(cache=True, fastmath=True, parallel=True)\\ndef search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):\\n    out = np.zeros(hi - lo, np.float32)\\n    for k in prange(hi - lo):\\n        a = roff[lo + k]; b = roff[lo + k + 1]\\n        if b <= a: continue\\n        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])\\n    return out\\n\\n\\n# ADDED: library search where each reference is shifted by (query precursor - reference precursor):\\n# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)\\n@njit(cache=True, fastmath=True, parallel=True)\\ndef search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):\\n    out = np.zeros(len(cand), np.float32)\\n    for k in prange(len(cand)):\\n        c = cand[k]; a = off[c]; b = off[c + 1]\\n        if b <= a: continue\\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\\n        if len(cm) == 0: continue\\n        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])\\n    return out\\n\\n\\n# ADDED: clean a list of spectra into a flat store\\n@njit(cache=True)\\ndef clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):\\n    n = len(rows)\\n    lens = np.zeros(n + 1, np.int64)\\n    tmp_m = []\\n    tmp_i = []\\n    for k in range(n):\\n        r = rows[k]\\n        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)\\n        tmp_m.append(cm); tmp_i.append(cp)\\n        lens[k + 1] = lens[k] + len(cm)\\n    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)\\n    for k in range(n):\\n        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]\\n    return lens, om, oi\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# adducts\\n# ----------------------------------------------------------------------------------------------\\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\\nE = 0.00054857990; PROTON = MASS[\\'H\\'] - E; H2O = 2 * MASS[\\'H\\'] + MASS[\\'O\\']\\nNH4 = MASS[\\'N\\'] + 4 * MASS[\\'H\\']; FORMATE = MASS[\\'C\\'] + 2 * MASS[\\'H\\'] + 2 * MASS[\\'O\\']\\nACETATE = 2 * MASS[\\'C\\'] + 4 * MASS[\\'H\\'] + 2 * MASS[\\'O\\']\\nADDUCTS = {\\n    \"[M+H]+\": (1, 1, PROTON), \"[M+NH4]+\": (1, 1, NH4 - E), \"[M+Na]+\": (1, 1, MASS[\\'Na\\'] - E),\\n    \"[M+K]+\": (1, 1, MASS[\\'K\\'] - E), \"[M-H2O+H]+\": (1, 1, PROTON - H2O), \"[M-2H2O+H]+\": (1, 1, PROTON - 2 * H2O),\\n    \"[M+2H]2+\": (1, 2, 2 * PROTON), \"[M]+\": (1, 1, -E), \"[M-H2O]+\": (1, 1, -E - H2O),\\n    \"[M+CH3OH+H]+\": (1, 1, PROTON + MASS[\\'C\\'] + 4 * MASS[\\'H\\'] + MASS[\\'O\\']),\\n    \"[M+CH3CN+H]+\": (1, 1, PROTON + 2 * MASS[\\'C\\'] + 3 * MASS[\\'H\\'] + MASS[\\'N\\']),\\n    \"[M-H]-\": (1, 1, -PROTON), \"[M-H2O-H]-\": (1, 1, -PROTON - H2O), \"[M+CH2O2-H]-\": (1, 1, FORMATE - PROTON),\\n    \"[M+C2H4O2-H]-\": (1, 1, ACETATE - PROTON), \"[M+Cl]-\": (1, 1, MASS[\\'Cl\\'] + E), \"[M]-\": (1, 1, E),\\n    \"[M-2H]-\": (1, 2, -2 * PROTON), \"[M+Na-2H]-\": (1, 1, MASS[\\'Na\\'] - 2 * PROTON),\\n    \"[2M+H]+\": (2, 1, PROTON), \"[2M+Na]+\": (2, 1, MASS[\\'Na\\'] - E), \"[2M+NH4]+\": (2, 1, NH4 - E),\\n    \"[2M+K]+\": (2, 1, MASS[\\'K\\'] - E), \"[2M-H]-\": (2, 1, -PROTON), \"[2M+CH2O2-H]-\": (2, 1, FORMATE - PROTON),\\n    \"[2M+C2H4O2-H]-\": (2, 1, ACETATE - PROTON), \"[2M+Na-2H]-\": (2, 1, MASS[\\'Na\\'] - 2 * PROTON),\\n    \"[3M+H]+\": (3, 1, PROTON), \"[3M-H]-\": (3, 1, -PROTON),\\n}\\n\\n\\ndef neutral_mass(mz, adduct):\\n    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)\\n    for a, (n, z, d) in ADDUCTS.items():\\n        m = (ad == a)\\n        if m.any(): out[m] = (mz[m] * z - d) / n\\n    return out\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# ranking features (31)\\n# ----------------------------------------------------------------------------------------------\\nN_ANALOG = 80\\nP_SIM = 3.0\\nN_FEAT = 31\\n\\n\\ndef _rank_norm(x):\\n    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)\\n\\n\\ndef _z(x):\\n    s = x.std()\\n    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)\\n\\n\\ndef rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):\\n    nc = cand_fp.shape[0]\\n    cf = cand_fp.astype(np.float32); cs = cf.sum(1)\\n    lv = np.asarray(cand_lib, np.float32)\\n    lvmax = float(lv.max()) if nc else 0.0\\n    if analog_fp is not None and len(analog_sim):\\n        af = analog_fp.astype(np.float32); asum = af.sum(1)\\n        inter = cf @ af.T\\n        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)\\n        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)\\n        ap = (tan * (w ** P_SIM)[None, :]).max(1)\\n        a1 = (tan * w[None, :]).max(1)\\n        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])\\n        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)\\n    else:\\n        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0\\n    apmax = float(ap.max()) if nc else 0.0\\n    if model_logits is not None:\\n        raw = cf @ np.asarray(model_logits, np.float32)\\n        nrm = raw / np.sqrt(np.maximum(cs, 1.0))\\n        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),\\n                 (raw == raw.max()).astype(np.float32)]\\n    else:\\n        mfeat = [np.zeros(nc, np.float32)] * 6\\n    if model_logits is not None and nc:\\n        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))\\n        lbest = int(np.argmax(lv)) if lvmax > 0 else -1\\n        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0\\n        abest = int(np.argmax(ap)) if apmax > 0 else -1\\n        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0\\n        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),\\n                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]\\n    else:\\n        xfeat = [np.zeros(nc, np.float32)] * 6\\n    if frag is not None:\\n        fr = np.asarray(frag, np.float32)\\n        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]\\n    else:\\n        ffeat = [np.zeros(nc, np.float32)] * 4\\n    return np.column_stack([\\n        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),\\n        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,\\n        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),\\n        np.full(nc, np.log(max(nc, 1))),\\n        *mfeat, *ffeat, *xfeat,\\n    ]).astype(np.float32)\\n\\n\\n# ----------------------------------------------------------------------------------------------\\n# MetFrag-lite\\n# ----------------------------------------------------------------------------------------------\\nAMU = {\\'C\\': 12.0, \\'H\\': 1.00782503207, \\'N\\': 14.0030740048, \\'O\\': 15.9949146196, \\'P\\': 30.97376163,\\n       \\'S\\': 31.97207100, \\'F\\': 18.99840322, \\'Cl\\': 34.96885268, \\'Br\\': 78.9183371, \\'I\\': 126.904473,\\n       \\'Na\\': 22.9897692809, \\'K\\': 38.96370668, \\'Si\\': 27.9769265325, \\'B\\': 11.0093054, \\'Se\\': 79.9165213}\\nH = AMU[\\'H\\']\\n\\n\\ndef mol_graph(smi):\\n    from rdkit import Chem, RDLogger\\n    RDLogger.DisableLog(\\'rdApp.*\\')\\n    m = Chem.MolFromSmiles(smi)\\n    if m is None: return None\\n    n = m.GetNumAtoms()\\n    w = np.zeros(n)\\n    for a in m.GetAtoms():\\n        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H\\n    if (w == 0).any(): return None\\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\\n    return w, bonds, n\\n\\n\\ndef _components(n, bonds, drop):\\n    adj = [[] for _ in range(n)]\\n    for i, (a, b) in enumerate(bonds):\\n        if i in drop: continue\\n        adj[a].append(b); adj[b].append(a)\\n    seen = np.zeros(n, bool); comps = []\\n    for s in range(n):\\n        if seen[s]: continue\\n        stack = [s]; seen[s] = True; cur = [s]\\n        while stack:\\n            u = stack.pop()\\n            for v in adj[u]:\\n                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)\\n        comps.append(cur)\\n    return comps\\n\\n\\ndef fragment_masses(smi, max_breaks=2, max_bonds=34):\\n    g = mol_graph(smi)\\n    if g is None: return np.zeros(0)\\n    w, bonds, n = g\\n    nb = len(bonds)\\n    if nb == 0 or nb > max_bonds: return np.array([w.sum()])\\n    out = {w.sum()}\\n    for i in range(nb):\\n        for c in _components(n, bonds, {i}):\\n            out.add(float(w[c].sum()))\\n    if max_breaks >= 2:\\n        for i in range(nb):\\n            for j in range(i + 1, nb):\\n                for c in _components(n, bonds, {i, j}):\\n                    out.add(float(w[c].sum()))\\n    return np.array(sorted(out))\\n\\n\\ndef frag_masses_safe(smi):\\n    try: return fragment_masses(smi)\\n    except Exception: return np.zeros(0)\\n\\n\\n# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)\\nCARRIERS = {\\n    \"[M+H]+\": (PROTON,), \"[M-H2O+H]+\": (PROTON,), \"[M-2H2O+H]+\": (PROTON,),\\n    \"[M+NH4]+\": (PROTON, NH4 - E), \"[M+Na]+\": (PROTON, MASS[\\'Na\\'] - E), \"[M+K]+\": (PROTON, MASS[\\'K\\'] - E),\\n    \"[M-H]-\": (-PROTON,), \"[M-H2O-H]-\": (-PROTON,), \"[M+CH2O2-H]-\": (-PROTON, FORMATE - PROTON),\\n    \"[M+Cl]-\": (-PROTON, MASS[\\'Cl\\'] + E),\\n}\\n\\n\\ndef explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\\n    car = CARRIERS.get(adduct, (PROTON,) if \"+\" in adduct[-2:] else (-PROTON,))\\n    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))\\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\\n    if tot <= 0: return 0.0\\n    idx = np.searchsorted(ion, peak_mz)\\n    ok = np.zeros(len(peak_mz), bool)\\n    for off in (-1, 0):\\n        k = np.clip(idx + off, 0, len(ion) - 1)\\n        ok |= np.abs(ion[k] - peak_mz) <= tol\\n    return float(w[ok].sum() / tot)\\n\\n\\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\\n    ion = []\\n    for dh in h_shifts:\\n        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))\\n    ion = np.sort(np.concatenate(ion))\\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\\n    if tot <= 0: return 0.0\\n    idx = np.searchsorted(ion, peak_mz)\\n    ok = np.zeros(len(peak_mz), bool)\\n    for off in (-1, 0):\\n        k = np.clip(idx + off, 0, len(ion) - 1)\\n        ok |= np.abs(ion[k] - peak_mz) <= tol\\n    return float(w[ok].sum() / tot)', 'pv_fp.py': '\"\"\"Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch.\"\"\"\\nimport math\\nimport numpy as np\\n# ----------------------------------------------------------------------------------------------\\n# spectrum -> fingerprint model\\n# ----------------------------------------------------------------------------------------------\\nimport torch, torch.nn as nn, torch.nn.functional as F\\n\\nFP_MAX_PEAKS = 128\\nADDUCT_LIST = [\"[M+H]+\", \"[M+NH4]+\", \"[M+Na]+\", \"[M+K]+\", \"[M-H2O+H]+\", \"[M-2H2O+H]+\", \"[M]+\",\\n               \"[M-H]-\", \"[M-H2O-H]-\", \"[M+CH2O2-H]-\", \"[M+C2H4O2-H]-\", \"[M+Cl]-\", \"[M]-\",\\n               \"[M+2H]2+\", \"[M-2H]-\", \"[2M+H]+\", \"[2M+Na]+\", \"[2M+NH4]+\", \"[2M-H]-\", \"[2M+K]+\",\\n               \"[2M+CH2O2-H]-\", \"[2M+C2H4O2-H]-\", \"[2M+Na-2H]-\", \"[M+Na-2H]-\", \"[M-H2O]+\", \"<unk>\"]\\nADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}\\nINSTR_LIST = [\"timsTOF\", \"Orbitrap\", \"QTOF\", \"IT\", \"other\"]\\n\\n\\ndef instr_family(s):\\n    if s is None: return 4\\n    t = str(s).lower()\\n    if \\'timstof\\' in t: return 0\\n    if \\'orbitrap\\' in t or \\'qft\\' in t or \\'ftms\\' in t or \\'hybrid ft\\' in t or \\'itft\\' in t or \\'exactive\\' in t: return 1\\n    if \\'tof\\' in t: return 2\\n    if \\'trap\\' in t or \\'qq\\' in t: return 3\\n    return 4\\n\\n\\ndef prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\\n    keep = (mz <= prec_mz + 1.5)\\n    mz, it = mz[keep], it[keep]\\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\\n    mx = it.max()\\n    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\\n    keep = it >= floor * mx\\n    mz, it = mz[keep], it[keep]\\n    if len(mz) > max_peaks:\\n        order = np.argsort(-it)\\n        bucket = (mz // win).astype(np.int64)\\n        cnt = {}; sel = []\\n        for i in order:\\n            b = bucket[i]; c = cnt.get(b, 0)\\n            if c < per_win: cnt[b] = c + 1; sel.append(i)\\n        sel = np.array(sel)\\n        if len(sel) > max_peaks:\\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\\n        elif len(sel) < max_peaks:\\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\\n            need = max_peaks - len(sel)\\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\\n        mz, it = mz[sel], it[sel]\\n    o = np.argsort(mz)\\n    mz, it = mz[o], it[o]\\n    v = np.sqrt(it / it.max())\\n    return mz.astype(np.float32), v.astype(np.float32)\\n\\n\\nclass SinEmb(nn.Module):\\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\\n        super().__init__()\\n        n = dim // 2\\n        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\\n        self.register_buffer(\\'inv\\', (2 * math.pi) / wav)\\n\\n    def forward(self, x):\\n        a = x.unsqueeze(-1) * self.inv\\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\\n\\n\\nclass Block(nn.Module):\\n    def __init__(self, d, h, drop):\\n        super().__init__(); self.h = h\\n        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)\\n        self.n2 = nn.LayerNorm(d)\\n        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\\n        self.drop = nn.Dropout(drop)\\n\\n    def forward(self, x, pad):\\n        B, N, D = x.shape; y = self.n1(x)\\n        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\\n        m = (~pad)[:, None, None, :]\\n        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)\\n        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\\n        return x + self.drop(self.ff(self.n2(x)))\\n\\n\\nclass FPNet(nn.Module):\\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\\n        super().__init__()\\n        self.d = d\\n        self.mz_emb = SinEmb(d)\\n        self.nl_emb = SinEmb(d)\\n        self.pk = nn.Linear(2 * d + 1, d)\\n        self.prec_emb = SinEmb(d)\\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\\n        self.gl = nn.Linear(d + 3, d)\\n        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\\n        self.norm = nn.LayerNorm(d)\\n        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\\n\\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\\n        B, N = mz.shape\\n        nl = (prec[:, None] - mz).clamp(min=0)\\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\\n        g = self.gl(torch.cat([self.prec_emb(prec),\\n                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\\n                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\\n        x = torch.cat([g.unsqueeze(1), p], 1)\\n        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\\n        for b in self.blocks: x = b(x, pad)\\n        x = self.norm(x)\\n        cls = x[:, 0]\\n        msk = (~pad[:, 1:]).float().unsqueeze(-1)\\n        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\\n        return self.head(torch.cat([cls, mean], -1))\\n\\n\\ndef load_fp_models(paths, dev):\\n    single, merged = [], []\\n    nbits = None\\n    for pth in paths:\\n        ck = torch.load(pth, map_location=\\'cpu\\', weights_only=False)\\n        net = FPNet(ck[\\'nbits\\'], d=ck[\\'d\\'], layers=ck[\\'layers\\']).to(dev).eval()\\n        net.load_state_dict(ck[\\'model\\'])\\n        (merged if \\'merged\\' in str(pth).replace(\\'\\\\\\\\\\', \\'/\\').split(\\'/\\')[-1] else single).append(net)\\n        nbits = ck[\\'nbits\\']\\n    return single, merged, nbits\\n\\n\\ndef merge_peaks(specs):\\n    \"\"\"specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum.\"\"\"\\n    specs = [(a, b) for a, b in specs if len(a)]\\n    if not specs: return np.zeros(0), np.zeros(0)\\n    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])\\n    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])\\n    o = np.argsort(mz); mz, it = mz[o], it[o]\\n    keep = np.ones(len(mz), bool)\\n    for j in range(1, len(mz)):\\n        if mz[j] - mz[j - 1] < 0.005:\\n            if it[j] >= it[j - 1]: keep[j - 1] = False\\n            else: keep[j] = False\\n    return mz[keep], it[keep]\\n\\n\\n@torch.no_grad()\\ndef logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):\\n    \"\"\"P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode.\"\"\"\\n    keep = [i for i, (a, _) in enumerate(P) if len(a)]\\n    if not keep: return None\\n    P = [P[i] for i in keep]\\n    B = len(P); N = max(len(a) for a, _ in P)\\n    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)\\n    for i, (a, b) in enumerate(P):\\n        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False\\n    T = lambda x: torch.as_tensor(x, device=dev)\\n    args = (T(mz), T(it), T(pad),\\n            T(np.asarray(prec, np.float32)[keep]),\\n            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX[\\'<unk>\\']) for a in np.asarray(adduct, object)[keep]])),\\n            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),\\n            T(np.asarray(ce, np.float32)[keep]),\\n            T(np.asarray(mode, np.float32)[keep]))\\n    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)\\n\\n\\ndef molecule_logits(models, specs, prec, adduct, instr, ce, mode):\\n    \"\"\"prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).\\n    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum.\"\"\"\\n    single, merged, dev = models\\n    out = []\\n    if single:\\n        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]\\n        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)\\n        if za is not None: out.append(za)\\n    if merged:\\n        mz, it = merge_peaks(specs)\\n        pm = float(np.median(prec))\\n        P = [prep_peaks(mz, it, pm)]\\n        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])\\n        if zb is not None: out.append(zb)\\n    return np.mean(out, axis=0) if out else None', 'casmi_engine.py': '\"\"\"End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,\\nbut reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).\\n\\nStages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission\\n\"\"\"\\nimport os, sys, glob, time, pickle, math\\nos.environ.setdefault(\"OPENBLAS_NUM_THREADS\", \"1\")\\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\\nimport pv\\nfrom pv import CFG\\nimport re as _re\\n_MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100, P=30.97376163,\\n             Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473, Si=27.9769265325, Se=79.9165213,\\n             Na=22.9897692809, K=38.96370668, B=11.0093054, As=74.9215965, Fe=55.9349375, D=2.0141017778)\\n_TOK = _re.compile(r\"([A-Z][a-z]?)(\\\\d*)\")\\n\\n\\ndef formula_mass(f):\\n    \"\"\"ADDED: neutral monoisotopic mass by arithmetic on molecular_formula (prvsiyan fix; immune to\\n    riken\\'s 0.005 Da precursor precision and gnps adduct mislabels). NaN if an element is untabulated.\"\"\"\\n    if not isinstance(f, str) or not f: return np.nan\\n    m = 0.0\\n    for el, n in _TOK.findall(f):\\n        if el not in _MONO: return np.nan\\n        m += _MONO[el] * (int(n) if n else 1)\\n    return m\\n\\nPOOL = None\\n\\n\\nclass RANK:\\n    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged\\n    SEEDS = (0, 1)\\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\\n\\nT0 = time.time()\\nLOCAL = os.environ.get(\"CASMI_LOCAL\") == \"1\"\\nROOTS = [r\"C:\\\\Users\\\\HW-LEE\\\\Desktop\\\\CASMI\"] if LOCAL else [\"/kaggle/input\"]\\n\\n\\ndef find(name):\\n    for root in ROOTS:\\n        hits = glob.glob(os.path.join(root, \"**\", name), recursive=True)\\n        if hits: return sorted(hits, key=len)[0]\\n    raise FileNotFoundError(name)\\n\\n\\ndef log(msg):\\n    print(f\"[{time.time()-T0:6.0f}s] {msg}\", flush=True)\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# library\\n# ---------------------------------------------------------------------------------------------\\ndef load_library(path):\\n    \"\"\"Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB.\"\"\"\\n    f = pq.ParquetFile(path)\\n    n_rows = f.metadata.num_rows\\n    npk = 0\\n    for i in range(f.num_row_groups):\\n        c = f.read_row_group(i, columns=[\"ms2_mzs\"]).column(0).combine_chunks()\\n        npk += len(c.values)\\n    off = np.zeros(n_rows + 1, np.int64)\\n    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)\\n    prec, add, pol, ik, smi = [], [], [], [], []\\n    fo, tims = [], []\\n    r0 = p0 = 0\\n    for i in range(f.num_row_groups):\\n        t = f.read_row_group(i, columns=[\"inchikey14\", \"normalized_smiles\", \"adduct\", \"precursor_mz\", \"ionization_mode\",\\n                                         \"ms2_mzs\", \"ms2_normalized_intensities\", \"molecular_formula\", \"instrument_type\"])\\n        mzc = t.column(\"ms2_mzs\").combine_chunks(); itc = t.column(\"ms2_normalized_intensities\").combine_chunks()\\n        o = mzc.offsets.to_numpy().astype(np.int64)\\n        n = len(o) - 1; k = int(o[-1] - o[0])\\n        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\\n        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\\n        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])\\n        prec.append(t.column(\"precursor_mz\").to_numpy(zero_copy_only=False).astype(np.float64))\\n        add += t.column(\"adduct\").cast(pa.string()).to_pylist()\\n        pol.append(np.array(t.column(\"ionization_mode\").cast(pa.string()).to_pylist(), dtype=object) == \"positive\")\\n        ik += t.column(\"inchikey14\").cast(pa.string()).to_pylist()\\n        smi += t.column(\"normalized_smiles\").cast(pa.string()).to_pylist()\\n        fo += t.column(\"molecular_formula\").cast(pa.string()).to_pylist()\\n        tims.append(np.array([isinstance(x, str) and x.lower().replace(\"-\", \"\").replace(\" \", \"\") == \"timstof\"\\n                              for x in t.column(\"instrument_type\").cast(pa.string()).to_pylist()]))\\n        r0 += n; p0 += k\\n        del t, mzc, itc\\n    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)\\n    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)\\n    nmp = pv.neutral_mass(prec, add)\\n    _fm = {x: formula_mass(x) for x in set(v for v in fo if isinstance(v, str))}\\n    nmf = np.array([_fm.get(x, np.nan) if isinstance(x, str) else np.nan for x in fo])\\n    nm = np.where(np.isfinite(nmf), nmf, nmp)\\n    tims = np.concatenate(tims)\\n    log(f\"library mass index: formula {np.isfinite(nmf).mean():.1%}, timsTOF rows {tims.mean():.1%}\")\\n    del fo\\n    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol, tims=tims,\\n                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)\\n# ---------------------------------------------------------------------------------------------\\n_g = {}\\n\\n\\ndef _fp_init():\\n    from rdkit import RDLogger\\n    from rdkit.Chem import rdFingerprintGenerator\\n    RDLogger.DisableLog(\"rdApp.*\")\\n    _g[\"bits\"] = np.load(find(\"fp_bits.npy\"))\\n    _g[\"m2\"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\\n    _g[\"m3\"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\\n    _g[\"rk\"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\\n\\n\\ndef fp_and_mass(smi):\\n    from rdkit import Chem\\n    from rdkit.Chem import MACCSkeys\\n    from rdkit.Chem.Descriptors import ExactMolWt\\n    if not _g: _fp_init()\\n    m = Chem.MolFromSmiles(smi)\\n    if m is None: return None\\n    try:\\n        fp = np.concatenate([_g[\"m2\"].GetFingerprintAsNumPy(m).astype(np.uint8),\\n                             _g[\"m3\"].GetFingerprintAsNumPy(m).astype(np.uint8),\\n                             _g[\"rk\"].GetFingerprintAsNumPy(m).astype(np.uint8),\\n                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g[\"bits\"]]\\n        return np.packbits(fp), float(ExactMolWt(m))\\n    except Exception:\\n        return None\\n\\n\\ndef canon_key(smi):\\n    from rdkit import Chem, RDLogger\\n    from rdkit.Chem.MolStandardize import rdMolStandardize\\n    RDLogger.DisableLog(\"rdApp.*\")\\n    if \"te\" not in _g: _g[\"te\"] = rdMolStandardize.TautomerEnumerator()\\n    try:\\n        m = Chem.MolFromSmiles(smi)\\n        return Chem.MolToInchiKey(_g[\"te\"].Canonicalize(m))[:14] if m is not None else None\\n    except Exception:\\n        return None\\n\\n\\ndef build_pool(L, workers):\\n    from multiprocessing import Pool as MPool\\n    d = os.path.dirname(find(\"coco_fp.npy\"))\\n    cm = pickle.load(open(os.path.join(d, \"coco_meta.pkl\"), \"rb\"))\\n    co_fp = np.load(os.path.join(d, \"coco_fp.npy\")); co_mass = np.load(os.path.join(d, \"coco_mass.npy\"))\\n    co_keys = np.asarray(cm[\"keys\"], dtype=object); co_smi = np.asarray(cm[\"smiles\"], dtype=object)\\n    # ADDED: ChEBI + LIPID MAPS (prvsiyan/chebi-lipidmaps-casmi26), same 6,930-bit packed fingerprints\\n    bd = os.path.dirname(find(\"bio_fp.npy\")); bm = pickle.load(open(os.path.join(bd, \"bio_meta.pkl\"), \"rb\"))\\n    bkeys = np.asarray(bm[\"keys\"], dtype=object); bnew = ~pd.Series(bkeys).isin(set(co_keys)).values\\n    co_fp = np.vstack([co_fp, np.load(os.path.join(bd, \"bio_fp.npy\"))[bnew]])\\n    co_mass = np.concatenate([co_mass, np.load(os.path.join(bd, \"bio_mass.npy\"))[bnew]])\\n    co_keys = np.concatenate([co_keys, bkeys[bnew]])\\n    co_smi = np.concatenate([co_smi, np.asarray(bm[\"smiles\"], dtype=object)[bnew]])\\n    log(f\"+ ChEBI/LIPID MAPS {int(bnew.sum()):,} new structures\")\\n    # v14 (from Hongbin Gu, guhongbin/casmi26-v447-pubchemlite): PubChemLite CC0 candidates\\n    # (franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints), same 6,930-bit packed fingerprints.\\n    # New arrays are built first and swapped in only if every step succeeds; any failure keeps the v13 pool.\\n    try:\\n        pcd = os.path.dirname(find(\"pubchemlite_fp.npy\"))\\n        pcm = pickle.load(open(os.path.join(pcd, \"pubchemlite_meta.pkl\"), \"rb\"))\\n        pckeys = np.asarray(pcm[\"keys\"], dtype=object)\\n        pcnew = ~pd.Series(pckeys).isin(set(co_keys)).values\\n        _fp = np.vstack([co_fp, np.load(os.path.join(pcd, \"pubchemlite_fp.npy\"))[pcnew]])\\n        _ms = np.concatenate([co_mass, np.load(os.path.join(pcd, \"pubchemlite_mass.npy\"))[pcnew]])\\n        _ks = np.concatenate([co_keys, pckeys[pcnew]])\\n        _sm = np.concatenate([co_smi, np.asarray(pcm[\"smiles\"], dtype=object)[pcnew]])\\n        assert len(_fp) == len(_ms) == len(_ks) == len(_sm), (len(_fp), len(_ms), len(_ks), len(_sm))\\n        co_fp, co_mass, co_keys, co_smi = _fp, _ms, _ks, _sm\\n        log(f\"+ PubChemLite CC0 {int(pcnew.sum()):,} new structures\")\\n    except Exception as _pe:\\n        log(f\"PubChemLite CC0 skipped: {_pe!r}\")\\n    tr = pd.DataFrame({\"ik\": L[\"ik\"], \"smi\": L[\"smi\"]}).dropna().drop_duplicates(\"ik\")\\n    try:  # ours (yasunorim, 10-21): own COCONUT table, structures not yet in the pool, fingerprinted like the training structures\\n        _od = [p for p in glob.glob(\"/kaggle/input/**/coco_meta.pkl\", recursive=True) if \"own-coconut\" in p]\\n        if _od:\\n            _om = pickle.load(open(_od[0], \"rb\"))\\n            _ot = pd.DataFrame({\"ik\": np.asarray(_om[\"keys\"], dtype=object), \"smi\": np.asarray(_om[\"smiles\"], dtype=object)}).dropna()\\n            _ot = _ot[~_ot.ik.isin(set(co_keys)) & ~_ot.ik.isin(set(tr.ik))].drop_duplicates(\"ik\")\\n            tr = pd.concat([tr, _ot], ignore_index=True)\\n            log(f\"+ ours: own COCONUT {len(_ot):,} new structures\")\\n        else:\\n            log(\"own COCONUT not mounted\")\\n    except Exception as _oe:\\n        log(f\"own COCONUT skipped: {_oe!r}\")\\n    tr = tr[~tr.ik.isin(set(co_keys))]\\n    log(f\"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}\")\\n    with MPool(workers) as mp:\\n        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)\\n    ok = np.array([r is not None for r in res])\\n    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])\\n    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])\\n    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])\\n    good = np.isfinite(mass)\\n    o = np.argsort(np.where(good, mass, 1e18), kind=\"mergesort\")[: good.sum()]\\n    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])\\n    P[\"k2i\"] = pd.Series(np.arange(len(o)), index=P[\"keys\"]); P[\"k2i\"] = P[\"k2i\"][~P[\"k2i\"].index.duplicated()]\\n    log(f\"pool {len(o):,} structures\")\\n    return P\\n\\n\\ndef pool_fps(idx):\\n    return np.unpackbits(np.asarray(POOL[\"fp\"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]\\n\\n\\ndef pool_window(t, ppm):\\n    a = np.searchsorted(POOL[\"mass\"], t * (1 - ppm / 1e6), \"left\")\\n    b = np.searchsorted(POOL[\"mass\"], t * (1 + ppm / 1e6), \"right\")\\n    return np.arange(a, b)\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)\\n# ---------------------------------------------------------------------------------------------\\ndef _reps(L, rows):\\n    npk = np.diff(L[\"off\"])[rows]\\n    df = pd.DataFrame({\"row\": rows, \"p\": L[\"row_p\"][rows], \"npk\": npk, \"tims\": L[\"tims\"][rows]})\\n    # ADDED: prefer the test instrument (timsTOF) representative, then the richest spectrum\\n    df = df.sort_values([\"p\", \"tims\", \"npk\", \"row\"], ascending=[True, False, False, True]).drop_duplicates(\"p\")\\n    rep = df.row.values; rep_nm = L[\"nm\"][rep]\\n    o = np.argsort(rep_nm, kind=\"mergesort\"); rep = rep[o]\\n    roff, rmz, rit = pv.clean_store(rep, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\\n    return dict(rep=rep, rep_p=L[\"row_p\"][rep], rep_nm=L[\"nm\"][rep], roff=roff, rmz=rmz, rit=rit)\\n\\n\\ndef build_index(L):\\n    ok = np.isfinite(L[\"nm\"]) & (L[\"row_p\"] >= 0)\\n    rows = np.where(ok)[0]\\n    o = np.argsort(L[\"nm\"][rows], kind=\"mergesort\")\\n    I = dict(lib_rows=rows[o], lib_nm=L[\"nm\"][rows[o]])\\n    I[\"all\"] = _reps(L, rows)\\n    I[\"pos\"] = _reps(L, rows[L[\"pol\"][rows] == 1])\\n    I[\"neg\"] = _reps(L, rows[L[\"pol\"][rows] == -1])\\n    log(f\"index: {len(rows):,} library rows; representatives {len(I[\\'all\\'][\\'rep\\']):,}\")\\n    return I\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# per-molecule channels (identical to e02_channels.molecule_channels)\\n# ---------------------------------------------------------------------------------------------\\ndef clean(mz, it):\\n    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\\n                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\\n\\n\\ndef analog_search(R, Q, target):\\n    lo = np.searchsorted(R[\"rep_nm\"], target - CFG.ANALOG_WIN, \"left\")\\n    hi = np.searchsorted(R[\"rep_nm\"], target + CFG.ANALOG_WIN, \"right\")\\n    asim = np.zeros(hi - lo, np.float32)\\n    if hi > lo and Q:\\n        shift = (target - R[\"rep_nm\"][lo:hi]).astype(np.float32)\\n        for qm, qp in Q:\\n            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R[\"roff\"], R[\"rmz\"], R[\"rit\"], CFG.MZ_TOL, shift))\\n    return R[\"rep_p\"][lo:hi], asim\\n\\n\\ndef top_analogs(p, s):\\n    if len(p) == 0: return []\\n    df = pd.DataFrame({\"p\": p, \"s\": s}).groupby(\"p\", sort=False).s.max().sort_values(ascending=False, kind=\"mergesort\")\\n    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))\\n\\n\\ndef molecule_channels(L, I, specs, precs, pols, target):\\n    Q = [clean(a, b) for a, b in specs]\\n    keep = [i for i, (a, _) in enumerate(Q) if len(a)]\\n    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]\\n    tol = target * CFG.PPM_WIN / 1e6\\n    lo = np.searchsorted(I[\"lib_nm\"], target - tol, \"left\"); hi = np.searchsorted(I[\"lib_nm\"], target + tol, \"right\")\\n    crow = I[\"lib_rows\"][lo:hi]\\n    u, inv = np.unique(L[\"row_p\"][crow], return_inverse=True)\\n    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)\\n    for j, (qm, qp) in enumerate(Q):\\n        if len(crow) == 0: break\\n        s = pv.search(qm, qp, crow, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\\n                      CFG.INT_POWER, CFG.ENT_WEIGHT)\\n        np.maximum.at(M[j], inv, s)\\n        sh = (precs[j] - L[\"prec\"][crow]).astype(np.float32)\\n        s2 = pv.search_shift_rows(qm, qp, crow, L[\"off\"], L[\"mz\"], L[\"it\"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\\n                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)\\n        np.maximum.at(MS[j], inv, s2)\\n    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)\\n    z = np.zeros(0, np.float32)\\n    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)\\n    p, s = analog_search(I[\"all\"], Q, target)\\n    ana = top_analogs(p, s)\\n    ps, ss = [], []\\n    for sign, name in ((1, \"pos\"), (-1, \"neg\")):\\n        QQ = [q for q, pl in zip(Q, pols) if pl == sign]\\n        if QQ:\\n            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)\\n    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []\\n    return lib, ana, anapol\\n\\n\\ndef parse_ce(v):\\n    try:\\n        a = np.abs(np.atleast_1d(np.asarray(v, float)))\\n        return float(a.mean()) if len(a) else 25.0\\n    except Exception:\\n        return 25.0\\n\\n\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# ranking features: prvsiyan\\'s 31 + our blocks (identical to e05_rank.py BLOCKS)\\n# ---------------------------------------------------------------------------------------------\\ndef _rel(v):\\n    v = np.asarray(v, np.float32)\\n    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]\\n\\n\\ndef _analog_feats(cfp, ana):\\n    nc = len(cfp)\\n    if not ana: return np.zeros((nc, 5), np.float32)\\n    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)\\n    inter = cf @ afp.T\\n    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)\\n    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)\\n    ap = (tan * (w ** 3)[None, :]).max(1)\\n    ap6 = (tan * (w ** 6)[None, :]).max(1)\\n    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)\\n\\n\\ndef blk_base(r, cfp):\\n    ana = r[\"ana\"]\\n    afp = pool_fps([p for p, _ in ana]) if ana else None\\n    sims = np.array([s for _, s in ana], np.float32)\\n    return pv.rank_features(cfp, r[\"lib\"], afp, sims, r[\"zlog\"], r[\"frag\"])\\n\\n\\ndef blk_mass(r, cfp):\\n    ppm = (POOL[\"mass\"][r[\"cand\"]] - r[\"target\"]) / r[\"target\"] * 1e6\\n    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)\\n\\n\\ndef blk_libx(r, cfp):\\n    cnt = np.log1p(r[\"lib_cnt\"])\\n    return np.column_stack(_rel(r[\"lib_mean\"]) + _rel(r[\"libsh\"]) + [cnt, cnt - cnt.max()]).astype(np.float32)\\n\\n\\ndef blk_anapol(r, cfp):\\n    return _analog_feats(cfp, r[\"anapol\"])\\n\\n\\ndef blk_fragx(r, cfp):\\n    return np.column_stack(_rel(r[\"frag_ad\"]) + _rel(r[\"frag_mean\"]) + [r[\"frag_pol\"]]).astype(np.float32)\\n\\n\\nBLOCKS = {\"base\": blk_base, \"mass\": blk_mass, \"libx\": blk_libx, \"anapol\": blk_anapol, \"fragx\": blk_fragx}\\n\\n\\n# ---------------------------------------------------------------------------------------------\\n# run\\n# ---------------------------------------------------------------------------------------------\\ndef compute_channels(L, I, models, te):\\n    mols = list(te.groupby(\"molecule_id\", sort=True))\\n    recs = []\\n    for gi, (mid, sub) in enumerate(mols):\\n        specs = []\\n        for r in sub.itertuples():\\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\\n            k_ = a_ <= float(r.precursor_mz) + 2.0\\n            a_, b_ = a_[k_], b_[k_]\\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\\n        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)\\n        nms = nms[np.isfinite(nms)]\\n        rec = dict(mid=mid, cand=np.zeros(0, np.int64))\\n        if len(nms):\\n            target = float(np.median(nms))\\n            modes = np.where(sub.ionization_mode.astype(str).values == \"positive\", 1.0, -1.0)\\n            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)\\n            cand = pool_window(target, CFG.PPM_WIN)\\n            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)\\n            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)\\n            import pv_fp\\n            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),\\n                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),\\n                                         ces, modes)\\n            u, lmax, lmean, lsh, lcnt = lib\\n            pos = {int(x): i for i, x in enumerate(u)}\\n            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0\\n            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,\\n                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),\\n                       ana=ana, anapol=anapol)\\n            for nm_, arr in ((\"lib\", lmax), (\"lib_mean\", lmean), (\"libsh\", lsh), (\"lib_cnt\", lcnt)):\\n                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v\\n        recs.append(rec)\\n        if gi % 50 == 0: log(f\"  channels {gi}/{len(mols)}\")\\n    return mols, recs\\n\\n\\ndef compute_frag(recs, workers):\\n    from multiprocessing import Pool as MPool\\n    uc = np.unique(np.concatenate([r[\"cand\"] for r in recs]))\\n    log(f\"MetFrag-lite on {len(uc):,} candidates\")\\n    with MPool(workers) as mp:\\n        fr = mp.map(pv.frag_masses_safe, list(POOL[\"smiles\"][uc]), chunksize=16)\\n    fmap = dict(zip(uc.tolist(), fr))\\n    for r in recs:\\n        if not len(r[\"cand\"]): continue\\n        peaks = []\\n        for a_, b_ in r.pop(\"specs\"):\\n            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\\n        F1 = np.zeros((len(r[\"cand\"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\\n        for j, c in enumerate(r[\"cand\"]):\\n            fm = fmap[int(c)]\\n            for k, (x, y) in enumerate(peaks):\\n                F1[j, k] = pv.explain_score(fm, x, y, mode=r[\"mode\"], tol=CFG.MZ_TOL)\\n                F2[j, k] = pv.explain_score_adduct(fm, x, y, r[\"adducts\"][k], tol=CFG.MZ_TOL)\\n                F3[j, k] = pv.explain_score(fm, x, y, mode=r[\"modes\"][k], tol=CFG.MZ_TOL)\\n        r[\"frag\"] = F1.max(1); r[\"frag_ad\"] = F2.max(1); r[\"frag_mean\"] = F3.mean(1); r[\"frag_pol\"] = F3.max(1)\\n\\n\\ndef _hgb(X, Y, W, seed, gbm):\\n    from sklearn.ensemble import HistGradientBoostingClassifier\\n    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)\\n    return m\\n\\n\\ndef fit_rankers(rank_train_path):\\n    \"\"\"Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim).\"\"\"\\n    z = np.load(rank_train_path, allow_pickle=True)\\n    X, Y, S = z[\"X\"], z[\"Y\"], z[\"S\"]\\n    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]\\n    log(f\"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features\")\\n    return rankers, [str(b) for b in z[\"blocks\"]]\\n\\n\\ndef fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):\\n    \"\"\"prvsiyan\\'s ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim).\"\"\"\\n    z = np.load(pv_rank_path)\\n    rankers = [_hgb(z[\"X\"], z[\"Y\"], np.where(z[\"M\"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]\\n    log(f\"prvsiyan ranker: {len(rankers)} GBMs on {z[\\'X\\'].shape[0]:,} rows x {z[\\'X\\'].shape[1]} features\")\\n    return rankers, z[\"X\"].shape[1]\\n\\n\\ndef score_molecules(recs, rankers, blocks, use_fp):\\n    out = {}\\n    for r in recs:\\n        if not len(r[\"cand\"]): continue\\n        rr = r if use_fp else dict(r, zlog=None)\\n        cfp = pool_fps(r[\"cand\"])\\n        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\\n        out[r[\"mid\"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)\\n    return out\\n\\n\\ndef blend_scores(a, b, wa):\\n    \"\"\"Within-molecule rank blend (both inputs are per-candidate probabilities).\"\"\"\\n    out = {}\\n    for mid in a:\\n        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])\\n        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]\\n    return out\\n\\n\\ndef make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):\\n    \"\"\"score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all.\"\"\"\\n    from multiprocessing import Pool as MPool\\n    cand = {r[\"mid\"]: r[\"cand\"] for r in recs if len(r[\"cand\"])}\\n    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind=\"mergesort\")][:topn + 15] for mid, sc in S.items()}\\n               for n, S in score_sets.items()}\\n    allc = [v for o in ordered.values() for v in o.values()]\\n    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)\\n    with MPool(workers) as mp:\\n        ck = mp.map(canon_key, list(POOL[\"smiles\"][short]), chunksize=64)\\n    ckey = dict(zip(short.tolist(), ck))\\n    samp = pd.read_csv(sample_path)\\n    subs = {}\\n    for n, o in ordered.items():\\n        rows = []\\n        for mid, _ in mols:\\n            out, seen = [], set()\\n            for c in o.get(mid, []):\\n                k = ckey.get(int(c)) or POOL[\"keys\"][c]\\n                if k in seen: continue\\n                seen.add(k); out.append(POOL[\"smiles\"][c])\\n                if len(out) == topn: break\\n            while len(out) < topn:\\n                out.append(\"CCO\")\\n            rows.append((mid, \";\".join(out[:topn])))\\n        sub = samp[[\"molecule_id\"]].merge(pd.DataFrame(rows, columns=[\"molecule_id\", \"smiles\"]), on=\"molecule_id\", how=\"left\")\\n        sub[\"smiles\"] = sub[\"smiles\"].fillna(\"CCO\")\\n        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0\\n        assert (sub.smiles.str.split(\";\").map(len) <= topn).all()\\n        subs[n] = sub\\n    return subs\\n\\n\\ndef main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):\\n    global POOL\\n    import torch\\n    import pv_fp\\n    L = load_library(train_path); log(f\"library {len(L[\\'prec\\']):,} spectra\")\\n    POOL = build_pool(L, workers)\\n    L[\"row_p\"] = POOL[\"k2i\"].reindex(L[\"ik\"]).fillna(-1).values.astype(np.int64)\\n    del L[\"ik\"], L[\"smi\"]\\n    I = build_index(L)\\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\\n    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)\\n    log(f\"fp models: {len(single)} single + {len(merged)} merged on {dev}\")\\n    te = pq.read_table(test_path).to_pandas()\\n    if limit:\\n        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]\\n    log(f\"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules\")\\n    mols, recs = compute_channels(L, I, (single, merged, dev), te)\\n    del L, I\\n    compute_frag(recs, workers)\\n    ours, blocks = fit_rankers(our_rank_path)\\n    s_ours = score_molecules(recs, ours, blocks, use_fp=False)\\n    del ours\\n    pvr, nfeat = fit_pv_rankers(pv_rank_path)\\n    s_pv = score_molecules(recs, pvr, [\"base\"], use_fp=True)\\n    del pvr\\n    subs = make_submissions(mols, recs, {\"blend\": blend_scores(s_pv, s_ours, w_pv), \"pv\": s_pv, \"ours\": s_ours},\\n                            sample_path, workers)\\n    log(\"submissions ready\")\\n    return subs, recs, {\"pv\": s_pv, \"ours\": s_ours}\\n'}\nfor _n, _src in ENG_FILES.items():\n    open(os.path.join(ENG_DIR, _n), 'w').write(_src)\nopen(os.path.join(ENG_DIR, 'eng_runner.py'), 'w').write('\"\"\"Our engine (two-ranker, BIO + AFIX) -> {molecule_id: {smiles: [...], keys: [...]}} (blend order, top 40).\"\"\"\\nimport os, sys, json, time\\nimport numpy as np\\nif __name__ == \\'__main__\\':\\n    here = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\\n    test, train, sample, out = sys.argv[1:5]\\n    import casmi_engine as E\\n    _orig_find = E.find\\n    def _find(n):                                  # v4g ships its own fp_bits.npy (10,226 bits): take prvsiyan\\'s 6,930\\n        if n == \\'fp_bits.npy\\':\\n            return os.path.join(os.path.dirname(_orig_find(\\'coco_fp.npy\\')), \\'fp_bits.npy\\')\\n        return _orig_find(n)\\n    E.find = _find\\n    E.RANK.W_A = (0.35, 0.55); E.RANK.SEEDS = (0, 1); E.CFG.N_ANALOG = 200\\n    import glob\\n    fp_models = sorted(p for p in glob.glob(\\'/kaggle/input/**/fp_*.pt\\', recursive=True) if \\'casmi26-fp-models-v2\\' in p)\\n    print(\\'engine fp models\\', fp_models, flush=True)\\n    T0 = time.time()\\n    subs, recs, S = E.main(test, train, sample, E.find(\\'sim_rank_rows_nofp.npz\\'), E.find(\\'rank_train.npz\\'), fp_models,\\n                           workers=os.cpu_count(), w_pv=0.88)\\n    bl = E.blend_scores(S[\\'pv\\'], S[\\'ours\\'], 0.88)\\n    res = {}\\n    for r in recs:\\n        mid = r[\\'mid\\']\\n        if mid not in bl:\\n            continue\\n        order = np.argsort(-bl[mid], kind=\\'mergesort\\')[:80]\\n        smis, keys, seen = [], [], set()\\n        for c in r[\\'cand\\'][order]:\\n            s = E.POOL[\\'smiles\\'][c]; k = E.canon_key(s)\\n            if k is None or k in seen:\\n                continue\\n            seen.add(k); smis.append(s); keys.append(k)\\n            if len(smis) >= 40:\\n                break\\n        res[str(mid)] = dict(smiles=smis, keys=keys)\\n    json.dump(res, open(out, \\'w\\'))\\n    print(\\'engine lists\\', len(res), f\\'{time.time()-T0:.0f}s\\', flush=True)\\n')\nENG = {}\ntry:\n    _t0 = time.time()\n    rr = subprocess.run([sys.executable, os.path.join(ENG_DIR, 'eng_runner.py'), os.path.join(COMP, 'test.parquet'),\n                         os.path.join(COMP, 'train.parquet'), os.path.join(COMP, 'sample_submission.csv'),\n                         '/kaggle/working/eng_lists.json'], capture_output=True, text=True, timeout=4 * 3600, cwd=ENG_DIR)\n    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])\n    ENG = json.load(open('/kaggle/working/eng_lists.json'))\n    print('engine lists', len(ENG), f'{time.time()-_t0:.0f}s')\nexcept Exception as e:\n    print('ENGINE FAILED -> v4g lists kept:', repr(e))\n\n\n# ===== cell 11 =====\n'''## 4. PubChem-only channel with a popularity prior\n\nThis channel searches about 105 million PubChem structures that are not in the main candidate pool. Its score combines the spectrum fit with how well documented a structure is.\n\n$$\\\\text{score} = z(f\\\\cdot z) + 0.25\\\\,\\\\big(\\\\log(1+\\\\#\\\\text{PubChem substances}) + \\\\log(1+\\\\#\\\\text{PubMed papers})\\\\big)$$\n\nThe 200 most documented structures in the mass window are always added to the shortlist. A gate later decides how many PubChem candidates enter the final list. This prior added +0.004 on the leaderboard.\n'''\n\n\n# ===== cell 12 =====\nV17_PATCH = \"\\n\\n# ---- v17 patch: popularity prior (log1p SID + log1p PMID from PubChem, mapped onto this store by stereo-stripped SMILES) ----\\n_POP_DIR = os.environ.get('CASMI_POP_DIR'); _POP_LAM = float(os.environ.get('CASMI_POP_LAM', '0')); _POP_UNION = int(os.environ.get('CASMI_POP_UNION', '0'))\\n_init_worker0 = init_worker\\n\\n\\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\\n    _init_worker0(pc_dir, bits_path, pool_meta_path, code_dir)\\n    if _POP_DIR:\\n        _W['ls'] = np.load(os.path.join(_POP_DIR, 'pc_lsid.npy'), mmap_mode='r')\\n        _W['lp'] = np.load(os.path.join(_POP_DIR, 'pc_lpmid.npy'), mmap_mode='r')\\n\\n\\ndef probe_one(task):\\n    mid, target, z = task\\n    t0 = time.time()\\n    chem = _W['chem']; Chem = _W['Chem']; gen = _W['ecfp4']\\n    tol = float(target) * PPM * 1e-6\\n    a0 = int(np.searchsorted(_W['mass'], float(target) - tol, 'left'))\\n    smis = _window(float(target))\\n    n = len(smis)\\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0, fz_top=None)\\n    if n == 0:\\n        diag['secs'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    raw_e = _W['raw_e']; z_e = z[_W['sel_e']].astype(np.float32)\\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\\n    for i, s in enumerate(smis):\\n        m = Chem.MolFromSmiles(s)\\n        if m is None:\\n            continue\\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\\n    sc1 = E @ z_e\\n    sc1[~ok] = -np.inf\\n    n1 = min(PC_N1, int(ok.sum()))\\n    diag['n_pass1'] = n1\\n    if n1 == 0:\\n        diag['secs'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\\n    use_pop = ('ls' in _W) and _POP_LAM > 0\\n    pop = (np.asarray(_W['ls'][a0:a0 + n], np.float32) + np.asarray(_W['lp'][a0:a0 + n], np.float32)) if use_pop else np.zeros(n, np.float32)\\n    if use_pop and _POP_UNION > 0:\\n        pp = np.where(ok, pop, -np.inf); ku = min(_POP_UNION, int(ok.sum()))\\n        top1 = np.union1d(top1, np.argpartition(-pp, ku - 1)[:ku])\\n    bits = _W['bits']\\n    fz, idx = [], []\\n    for i in top1:\\n        fp = chem.raw_fingerprint(smis[i])\\n        if fp is None:\\n            continue\\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\\n    diag['n_pass2'] = len(idx)\\n    if not idx:\\n        diag['secs'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    fz = np.asarray(fz, np.float64); idx = np.asarray(idx)\\n    keyc = {}\\n    def key(j):\\n        if j not in keyc:\\n            keyc[j] = chem.score_key(smis[idx[j]]); diag['n_keyed'] += 1\\n        return keyc[j]\\n    for j in np.argsort(-fz, kind='stable'):        # gate quantity = v16's pc_fz[0] (best non-pool structure by raw f.z)\\n        k = key(j)\\n        if k is not None and k not in _W['pool_keys']:\\n            diag['fz_top'] = float(fz[j]); break\\n    zf = (fz - fz.mean()) / (fz.std() + 1e-9)\\n    sc = zf + _POP_LAM * pop[idx]\\n    order = np.argsort(-sc, kind='stable')\\n    out, out_s, out_k, out_p, out_sc, seen = [], [], [], [], [], set()\\n    for j in order:\\n        k = key(j)\\n        if k is None or k in seen:\\n            continue\\n        seen.add(k)\\n        if k in _W['pool_keys']:\\n            diag['n_in_pool'] += 1\\n            continue\\n        out.append(smis[idx[j]]); out_s.append(float(sc[j])); out_k.append(k); out_p.append(float(pop[idx[j]])); out_sc.append(float(sc[j]))\\n        if len(out) >= K:\\n            break\\n    diag['n_listed'] = len(out)\\n    diag['S'] = float(out_sc[0]) if out_sc else 0.0\\n    diag['top_pop'] = float(out_p[0]) if out_p else 0.0\\n    diag['secs'] = time.time() - t0\\n    return mid, out, out_s, out_k, diag\\n\"\n'''PubChem-only channel in its own process (S1 code snapshot + FPNet A+B of casmi26-v3-models) -- identical to v3.6'''\nCORE = '\"\"\"P_pubchem probe core (lever 3). Embedded verbatim into the Kaggle notebook (written to /kaggle/working and imported,\\nso worker processes can import it) and imported directly by the local smoke test.\\n\\nPer molecule:\\n  1. f.z logits of the A+B FPNet ensemble, exactly like Engine.model_logits: z = 0.5 * (mean per-spectrum logits +\\n     mean per-polarity merged-spectrum logits); models averaged inside ModelBank.\\n  2. PubChem window: +-10 ppm around the target neutral mass (median over the molecule\\'s spectra), no subsampling.\\n  3. pass 1 (src/casmi/pubchem.py tier): partial f.z on the ECFP4 block of the selected bits, keep top PC_N1 = 1000.\\n  4. pass 2: full selected-bit fingerprints, full f.z, sort descending.\\n  5. walk down the f.z order, compute the metric score key (tautomer-canonical InChIKey14); keep a structure only if\\n     its key is NOT in the candidate pool (train U COCONUT) and not already listed; stop at 25.\\n  An empty list becomes the single \\'CCO\\' fallback (never padded).\\nReading: sum of reciprocal ranks of PubChem-only truths / ~130 public molecules.\\n\"\"\"\\nfrom __future__ import annotations\\n\\nimport os\\nimport time\\n\\nimport numpy as np\\n\\nPC_N1 = 5000\\nPPM = 10.0\\nK = 25\\n\\n_W = {}\\n\\n\\n# ------------------------------------------------------------------------------------------ model part (main process)\\ndef molecule_logits(bank, spectra):\\n    \"\"\"spectra: list of dict(mz, it, mode, adduct, prec, ce, ce_n). Returns z_avg (nbits,) float32 or None.\"\"\"\\n    from casmi import chem, fpnet\\n    from casmi.spectra import merge_spectra\\n    items_s = []\\n    for s in spectra:\\n        pm, pi = fpnet.prep_peaks(s[\\'mz\\'], s[\\'it\\'], s[\\'prec\\'])\\n        ce_ok = s[\\'ce\\'] is not None and not np.isnan(s[\\'ce\\'])\\n        items_s.append(dict(mz=pm, it=pi, prec=float(s[\\'prec\\']), adduct_ix=chem.adduct_index(s[\\'adduct\\']),\\n                            ce=float(s[\\'ce\\']) if ce_ok else 0.0, ce_known=1.0 if ce_ok else 0.0,\\n                            n_merged=min(int(s.get(\\'ce_n\\', 1)), 8), mode=float(s[\\'mode\\'])))\\n    items_m = []\\n    for md in (1, -1):\\n        grp = [s for s in spectra if s[\\'mode\\'] == md]\\n        if not grp:\\n            continue\\n        mm, ii = merge_spectra([(s[\\'mz\\'], s[\\'it\\']) for s in grp])\\n        prec = float(np.median([s[\\'prec\\'] for s in grp]))\\n        pm, pi = fpnet.prep_peaks(mm, ii, prec)\\n        ces = [s[\\'ce\\'] for s in grp if s[\\'ce\\'] is not None and not np.isnan(s[\\'ce\\'])]\\n        adducts = [s[\\'adduct\\'] for s in grp]\\n        ad = max(adducts, key=adducts.count)          # deterministic tie-break (first occurrence)\\n        items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad),\\n                            ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\\n                            n_merged=min(sum(max(1, int(s.get(\\'ce_n\\', 1))) for s in grp), 8), mode=float(md)))\\n    if not items_s:\\n        return None\\n    zs, _ = bank.logits_el(items_s)\\n    zm, _ = bank.logits_el(items_m)\\n    return (0.5 * (zs.mean(0) + zm.mean(0))).astype(np.float32)\\n\\n\\n# ------------------------------------------------------------------------------------------ PubChem part (workers)\\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\\n    import sys\\n    if code_dir and code_dir not in sys.path:\\n        sys.path.insert(0, code_dir)\\n    import pandas as pd\\n    from casmi import chem\\n    _W[\\'chem\\'] = chem\\n    _W[\\'mass\\'] = np.load(os.path.join(pc_dir, \\'pc_mass.npy\\'), mmap_mode=\\'r\\')\\n    _W[\\'off\\'] = np.load(os.path.join(pc_dir, \\'pc_off.npy\\'), mmap_mode=\\'r\\')\\n    _W[\\'buf\\'] = np.load(os.path.join(pc_dir, \\'pc_smiles.npy\\'), mmap_mode=\\'r\\')\\n    bits = np.load(bits_path)\\n    _W[\\'bits\\'] = bits\\n    _W[\\'sel_e\\'] = np.where(bits < 4096)[0]\\n    _W[\\'raw_e\\'] = bits[_W[\\'sel_e\\']]\\n    _W[\\'pool_keys\\'] = set(pd.read_parquet(pool_meta_path, columns=[\\'key\\']).key.values.tolist())\\n    r = chem._rdkit()\\n    _W[\\'ecfp4\\'] = r[\\'gen\\'].GetMorganGenerator(radius=2, fpSize=4096)\\n    _W[\\'Chem\\'] = r[\\'Chem\\']\\n\\n\\ndef _window(target):\\n    tol = target * PPM * 1e-6\\n    a = int(np.searchsorted(_W[\\'mass\\'], target - tol, \\'left\\')); b = int(np.searchsorted(_W[\\'mass\\'], target + tol, \\'right\\'))\\n    off = np.asarray(_W[\\'off\\'][a:b + 1]).astype(np.int64)\\n    if b <= a:\\n        return []\\n    raw = bytes(_W[\\'buf\\'][off[0]:off[-1]])\\n    base = off[0]\\n    return [raw[off[i] - base:off[i + 1] - base].decode(\\'ascii\\') for i in range(b - a)]\\n\\n\\ndef probe_one(task):\\n    \"\"\"task = (molecule_id, target_mass, z (nbits,) float32). Returns (molecule_id, [smiles], [f.z], [score keys], diag dict).\"\"\"\\n    mid, target, z = task\\n    t0 = time.time()\\n    chem = _W[\\'chem\\']; Chem = _W[\\'Chem\\']; gen = _W[\\'ecfp4\\']\\n    smis = _window(float(target))\\n    n = len(smis)\\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0)\\n    if n == 0:\\n        diag[\\'secs\\'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    raw_e = _W[\\'raw_e\\']; z_e = z[_W[\\'sel_e\\']].astype(np.float32)\\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\\n    for i, s in enumerate(smis):\\n        m = Chem.MolFromSmiles(s)\\n        if m is None:\\n            continue\\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\\n    sc1 = E @ z_e\\n    sc1[~ok] = -np.inf\\n    n1 = min(PC_N1, int(ok.sum()))\\n    diag[\\'n_pass1\\'] = n1\\n    if n1 == 0:\\n        diag[\\'secs\\'] = time.time() - t0\\n        return mid, [], [], [], diag\\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\\n    bits = _W[\\'bits\\']\\n    fz, idx = [], []\\n    for i in top1:\\n        fp = chem.raw_fingerprint(smis[i])\\n        if fp is None:\\n            continue\\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\\n    diag[\\'n_pass2\\'] = len(idx)\\n    order = np.argsort(-np.asarray(fz), kind=\\'stable\\')\\n    out, out_fz, out_k, seen = [], [], [], set()\\n    for o in order:\\n        s = smis[idx[o]]\\n        k = chem.score_key(s)\\n        diag[\\'n_keyed\\'] += 1\\n        if k is None or k in seen:\\n            continue\\n        seen.add(k)\\n        if k in _W[\\'pool_keys\\']:\\n            diag[\\'n_in_pool\\'] += 1\\n            continue\\n        out.append(s); out_fz.append(float(fz[o])); out_k.append(k)\\n        if len(out) >= K:\\n            break\\n    diag[\\'n_listed\\'] = len(out)\\n    diag[\\'secs\\'] = time.time() - t0\\n    return mid, out, out_fz, out_k, diag\\n'\nRUNNER = '\"\"\"PubChem-only channel for the final submission (runs in its own process so its casmi package\\n(S1 snapshot, FPNet A+B) never collides with the v1 engine\\'s casmi package).\\n\\nPer molecule: z = A+B logits (probe_core2.molecule_logits, identical to the probe that scored 0.037);\\nbest_pool_fz = max f.z over the pool window (+-10 ppm, fallback 30 ppm) with the same z (as P1 logits.py);\\nPubChem-only list = probe_core2.probe_one (top 25 by f.z, keys not in the pool) with f.z values and score keys.\\nWrites a json {molecule_id: {pc, pc_fz, pc_keys, best_pool_fz, target}}.\\nusage: python pc_runner.py <v3_code_dir> <v3_models_dir> <pool_dir> <pubchem_dir> <test_parquet> <out_json> <n_workers> [max_mols]\\n\"\"\"\\nimport os, sys, json, time\\nimport numpy as np, pandas as pd\\n\\nif __name__ == \\'__main__\\':\\n    V3CODE, V3MOD, POOL_DIR, PC_DIR, TEST, OUT, NW = sys.argv[1:8]\\n    MAXM = int(sys.argv[8]) if len(sys.argv) > 8 else 0\\n    NW = int(NW)\\n    sys.path.insert(0, V3CODE); sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\\n    import torch, glob\\n    import probe_core2 as pc\\n    from casmi import fpnet, chem\\n    T0 = time.time()\\n    models = sorted(glob.glob(os.path.join(V3MOD, \\'fpnet_*.pt\\')))\\n    assert len(models) == 2, models\\n    dev = \\'cuda\\' if torch.cuda.is_available() else \\'cpu\\'\\n    bank = fpnet.ModelBank(models, device=dev)\\n    bits = np.load(os.path.join(POOL_DIR, \\'fp_bits.npy\\')); nbits = len(bits)\\n    pmass = pd.read_parquet(os.path.join(POOL_DIR, \\'pool_meta.parquet\\'), columns=[\\'mass\\']).mass.values.astype(np.float64)\\n    assert np.all(np.diff(pmass) >= 0), \\'pool not mass-sorted\\'\\n    pfp = np.load(os.path.join(POOL_DIR, \\'pool_fp.npy\\'), mmap_mode=\\'r\\')\\n    te = pd.read_parquet(TEST)\\n    te[\\'nm\\'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\\n    tasks, extra = [], {}\\n    groups = list(te.groupby(\\'molecule_id\\', sort=False))\\n    if MAXM:\\n        groups = groups[:MAXM]\\n    for mid, sub in groups:\\n        nms = sub.nm.values[np.isfinite(sub.nm.values)]\\n        if not len(nms):\\n            continue\\n        target = float(np.median(nms))\\n        spectra = []\\n        for r in sub.itertuples():\\n            ce = r.collision_energy_ev\\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\\n                                mode=1 if r.ionization_mode == \\'positive\\' else -1, adduct=r.adduct,\\n                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,\\n                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\\n        spectra = spectra[:16]\\n        try:\\n            z = pc.molecule_logits(bank, spectra)\\n        except Exception as e:\\n            print(\\'ERROR logits\\', mid, repr(e), flush=True); z = None\\n        if z is None:\\n            continue\\n        tol = target * 10e-6\\n        a = np.searchsorted(pmass, target - tol, \\'left\\'); b = np.searchsorted(pmass, target + tol, \\'right\\')\\n        if b <= a:\\n            tol = target * 30e-6\\n            a = np.searchsorted(pmass, target - tol, \\'left\\'); b = np.searchsorted(pmass, target + tol, \\'right\\')\\n        if b > a:\\n            F = np.unpackbits(np.asarray(pfp[a:b]), axis=1)[:, :nbits].astype(np.float64)\\n            best = float((F @ z.astype(np.float64)).max())\\n        else:\\n            best = float(\\'nan\\')\\n        extra[mid] = dict(best_pool_fz=best, target=target)\\n        tasks.append((mid, target, z))\\n    del bank; torch.cuda.empty_cache()\\n    print(f\\'logits for {len(tasks)} molecules {time.time()-T0:.0f}s\\', flush=True)\\n    res = {}\\n    import multiprocessing as mp\\n    ctx = mp.get_context(\\'fork\\' if sys.platform != \\'win32\\' else \\'spawn\\')\\n    with ctx.Pool(NW, initializer=pc.init_worker, initargs=(PC_DIR, os.path.join(POOL_DIR, \\'fp_bits.npy\\'),\\n                                                            os.path.join(POOL_DIR, \\'pool_meta.parquet\\'), V3CODE)) as P:\\n        for k, (mid, smis, fzs, keys, d) in enumerate(P.imap_unordered(pc.probe_one, tasks, chunksize=1)):\\n            res[mid] = dict(pc=smis, pc_fz=fzs, pc_keys=keys, fz_top=d.get(\\'fz_top\\'), S=d.get(\\'S\\'), top_pop=d.get(\\'top_pop\\', 0.0), **extra[mid])\\n            if (k + 1) % 25 == 0 or k + 1 == len(tasks):\\n                print(f\\'  pubchem {k+1}/{len(tasks)} {time.time()-T0:.0f}s\\', flush=True)\\n    json.dump(res, open(OUT, \\'w\\'))\\n    print(\\'pc_runner done\\', len(res), f\\'{time.time()-T0:.0f}s\\', flush=True)\\n'\nopen('/kaggle/working/probe_core2.py', 'w').write(CORE + V17_PATCH)\nopen('/kaggle/working/pc_runner.py', 'w').write(RUNNER)\nprint('v17 patch appended to probe_core2:', 'def probe_one' in V17_PATCH)\nPC = {}\ntry:\n    rr = subprocess.run([sys.executable, '/kaggle/working/pc_runner.py', V3CODE, V3MOD, POOL_DIR, PC_DIR,\n                         os.path.join(COMP, 'test.parquet'), '/kaggle/working/pc_lists.json', '4'],\n                        capture_output=True, text=True, timeout=4 * 3600)\n    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])\n    PC = json.load(open('/kaggle/working/pc_lists.json'))\nexcept Exception as e:\n    print('PUBCHEM CHANNEL FAILED -> base lists only:', repr(e))\nprint('pubchem lists', len(PC), f'{time.time()-T0:.0f}s')\n\n\n# ===== cell 13 =====\n'''## 5. Engine 1: Ahmed Berat \u00d6zer's v4n\n\nThis step builds the spectral library from `train.parquet` and links the precomputed candidate pool: training structures plus COCONUT, about 711k in total. The pool comes with FPNet fingerprints and fragment tables.\n'''\n\n\n# ===== cell 14 =====\n'''v1 engine code (the 0.358 model's engine) + fe_v4; pool symlinks; spectrum cache from train.parquet'''\nsys.path.insert(0, V4CODE)\nos.makedirs('work', exist_ok=True)\nfor f in ['pool_meta.parquet', 'pool_fp.npy', 'pool_frag_off.npy', 'pool_frag_mass.npy', 'fp_bits.npy', 'train_structs.parquet', 'train_fp_sel.npy']:\n    src = os.path.join(POOL_DIR, f)\n    dst = f'work/{f}'\n    if not os.path.exists(dst):\n        os.symlink(src, dst)\nfrom casmi.build import build_spec_cache\nbuild_spec_cache(os.path.join(COMP, 'train.parquet'), 'work')\nimport casmi; print('casmi from', casmi.__file__)\nsys.path.insert(1, os.path.join(V4CODE, 'fe_v4'))\nprint(f'cache built {time.time()-T0:.0f}s')\n\n\n# ===== cell 15 =====\n'''v1 engine exactly as in v3.6 (FPNet A, EngineCfg(generate=True)) wrapped by V1FE (recording hooks + families)'''\nimport numpy as np, pandas as pd, torch, lightgbm as lgb\nfrom casmi.library import Library\nfrom casmi.pool import Pool\nfrom casmi.engine import Engine, EngineCfg, FEATURES\nfrom casmi import fpnet, chem\nimport v1engine\ndev = 'cuda' if torch.cuda.is_available() else 'cpu'\nL = Library('work'); P = Pool('work')\ntfp = np.load('work/train_fp_sel.npy')\n_FULL = glob.glob('/kaggle/input/**/casmi26-fpnet-full1/**/fpnet_full1.pt', recursive=True)   # v4m\nbank = fpnet.ModelBank([_FULL[0] if _FULL else os.path.join(V4MOD, 'fpnet_0.pt')], device=dev)\nprint('engine FP bank:', _FULL[0] if _FULL else 'fpnet_0 (A) FALLBACK', flush=True)\nE = Engine(L, P, tfp, EngineCfg(generate=True), bank)\nV = v1engine.V1FE(E, [os.path.join(V4MOD, m['file']) for m in MAN['fe_models']], MAN['families'], device=dev)\nif 'derivation' in MAN['families']:\n    n_bad = V.mods['derivation'].self_test()\n    print('derivation prior self-test mismatches:', n_bad)\n    assert n_bad == 0, 'RDKit build differs from the one the derivation priors were built with'\nRK = pickle.load(open(os.path.join(V4MOD, MAN['ranker']['file']), 'rb'))\nRK_FEATS = list(RK['features']); RK_B = [lgb.Booster(model_str=s) for s in RK['boosters']]\ndef rank_score(X, names):\n    pos = {n: i for i, n in enumerate(names)}\n    miss = [f for f in RK_FEATS if f not in pos]\n    assert not miss, f'ranker features not produced (would be zero-filled): {miss}'\n    Xs = np.ascontiguousarray(X[:, [pos[f] for f in RK_FEATS]], dtype=np.float32)\n    return np.mean([b.predict(Xs, num_threads=4) for b in RK_B], axis=0)\nprint('engine ready; ranker features', len(RK_FEATS), 'boosters', len(RK_B), f'{time.time()-T0:.0f}s')\n\n\n# ===== cell 16 =====\n'''## 6. Main candidate lists + pool popularity re-rank\n\nFor each molecule, v4n scores candidates with FPNet fingerprints, its feature families and a 4-booster LightGBM ranker, and keeps the top 60. The list is then re-ranked by\n\n$$z(\\\\text{ranker score}) + 0.15\\\\,\\\\big(\\\\log(1+\\\\#\\\\text{substances}) + \\\\log(1+\\\\#\\\\text{papers})\\\\big)$$\n\nOnly pool structures are re-ranked. Generated (novel) candidates keep their positions and form their own simulator groups. This step added +0.010 on the leaderboard.\n'''\n\n\n# ===== cell 17 =====\n'''all molecules -> base lists (v3.6 spectra construction and ordering)'''\nte = pd.read_parquet(os.path.join(COMP, 'test.parquet'))\nte['nm'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\nmols = list(te.groupby('molecule_id', sort=False))\nBASE, n_err = {}, 0\nPOOLPOP_MU = 0.15\nPOOL_POP = np.load(find('pool_lsid.npy')).astype(np.float64) + np.load(find('pool_lpmid.npy')).astype(np.float64)\nassert len(POOL_POP) == len(P.key), (len(POOL_POP), len(P.key))\nn_pp = 0\nprint('v19 pool popularity re-rank, mu', POOLPOP_MU, 'pool', len(POOL_POP))\nfor gi, (mid, sub) in enumerate(mols):\n    nms = sub.nm.values[np.isfinite(sub.nm.values)]\n    smis, keys, lib_max, scs, forms = [], [], 0.0, [], []\n    pids_ = []\n    if len(nms):\n        target = float(np.median(nms))\n        spectra = []\n        for r in sub.itertuples():\n            ce = r.collision_energy_ev\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                                mode=1 if r.ionization_mode == 'positive' else -1, adduct=r.adduct,\n                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,\n                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\n        try:\n            C, X, F, names, info = V.run(spectra, target)\n            lib_max = float(info.get('lib_max', 0.0))\n            if len(C.pid):\n                score = rank_score(np.concatenate([X, F], 1), list(FEATURES) + list(names))\n                order = np.argsort(-score, kind='stable')\n                seen = set()\n                for i in order:\n                    k = C.key[i]\n                    if k in seen: continue\n                    seen.add(k); smis.append(C.smiles[i]); keys.append(k); scs.append(float(score[i])); forms.append(C.formula[i]); pids_.append(int(C.pid[i]))\n                    if len(smis) >= TOPN: break\n        except Exception as e:\n            print('ERROR', mid, repr(e)); n_err += 1\n    if POOLPOP_MU > 0 and len(scs) > 1:            # v19: popularity re-rank of the top-60 base list\n        try:\n            s_ = np.asarray(scs, np.float64); z_ = (s_ - s_.mean()) / (s_.std() + 1e-9)\n            pid_ = np.asarray(pids_); isg_ = pid_ < 0\n            f_ = z_ + POOLPOP_MU * np.where(isg_, 0.0, POOL_POP[np.maximum(pid_, 0)])\n            pool_i = np.where(~isg_)[0]; o_ = np.empty(len(pid_), np.int64)\n            o_[isg_] = np.where(isg_)[0]; o_[~isg_] = pool_i[np.argsort(-f_[pool_i], kind='stable')]   # generated keep their slots\n            smis = [smis[i] for i in o_]; keys = [keys[i] for i in o_]; scs = [float(f_[i]) for i in o_]\n            forms = [forms[i] + ('|gen' if isg_[i] else '') for i in o_]      # generated form their own ICEBERG groups\n            n_pp += 1\n        except Exception as e:\n            print('poolpop failed', mid, repr(e))\n    BASE[mid] = (smis, keys, lib_max, scs, forms)\n    if (gi + 1) % 25 == 0 or gi == len(mols) - 1:\n        print(f'  {gi+1}/{len(mols)} molecules {time.time()-T0:.0f}s', flush=True)\nprint('errors', n_err, '| popularity re-ranked lists', n_pp)\n'''Keep ranked overflow for append-only final coverage.'''\nBASE_TAIL_CANDIDATES = {str(k): list(v[0]) for k, v in BASE.items()}\n\n\n# ===== cell 18 =====\n'''## 7. ICEBERG + GLACIER spectrum simulators\n\nBoth models predict each candidate's spectrum and compare it with the measured one. Candidates with the same molecular formula are then re-ranked by\n\n$$z(\\\\text{score}) + 1.0\\\\,z(\\\\text{ICEBERG}) + 1.0\\\\,z(\\\\text{GLACIER})$$\n\nMost wrong answers share the right answer's formula, so this is where isomers get separated.\n\n- Second-engine candidates join the simulator input, so they are scored too.\n- **GLACIER uses [M+H]+ spectra only.** Restricting it this way added +0.006 on the leaderboard.\n'''\n\n\n# ===== cell 19 =====\n'''--- CLAW FIX (b418/b419): stage health is now RECORDED, never silently swallowed ---'''\n'''`glob('/kaggle/input/**/x')` WITHOUT recursive=True matches one directory level only, so it'''\n'''misses /kaggle/input/datasets/<owner>/<slug>/. That silently disabled ICEBERG+GLACIER:'''\n'''ICE FAILED -> ranker order kept: FileNotFoundError('cp313 rdkit wheel not found')  [b417 log]'''\n'''rdkit-2025.3.6-cp312-...whl is not a supported wheel on this platform               [b417 log]'''\n'''recursive=True finds the wheel under BOTH mount layouts (same 2025.3.6 cp313 file either way).'''\nCLAW_ICE_HEALTH = 'ICE NOT RUN'\nCLAW_GL_HEALTH = 'GL NOT RUN'\nBASE_ICE = False         # b421: base-list ICEBERG fusion OFF (b420 showed it costs 0.009); see header\n'''ICEBERG post-ranker isomer re-scoring (casmi26-iceberg). Optional: any failure keeps the ranker order.'''\nimport gc\nICE_SCORES, ice_stats = {}, dict(molecules=0, changed_top25=0, changed_top1=0)\n\n\n'''--- CLAW FIX (b415c): make ICEBERG/GLACIER runnable on a cp313 Kaggle image ---'''\n'''The ICEBERG package bundles rdkit-2025.3.6 built for **cp312**; this image is cp313,'''\n'''so pip rejects that wheel and install_site() raises before anything is scored'''\n'''(n_mols_scored == 0 for ICEBERG *and* GLACIER). The MANIFEST pins 2025.3.6 because'''\n'''\"FragmentEngine Kekulize differs in 2026.03\", so swapping in the runtime's 2026.03'''\n'''rdkit would change the scoring. Instead we install the SAME 2025.3.6 release built'''\n'''for cp313, so the pinned version is honoured exactly and RDKIT_REQ passes natively.'''\n''''''\n'''install_site() is idempotent via <site>/.ice_site_ok, which it compares against the'''\n'''name list of `pkg/wheels`. We pre-build the site and write that exact list, so both'''\n'''runners short-circuit and never invoke the failing pip.'''\ndef _claw_preseed_site(pkg, site):\n    import glob as _g, os as _o, shutil as _s, subprocess as _sp, sys as _sys\n    srcs = sorted(_g.glob(_o.path.join(pkg, 'wheels', '*.whl.ice')) +\n                  _g.glob(_o.path.join(pkg, 'wheels', '*.whl')))\n    # install everything EXCEPT the cp-tag-mismatched rdkit\n    keep = [w for w in srcs if not _o.path.basename(w).lower().startswith('rdkit')]\n    dropped = [_o.path.basename(w) for w in srcs if w not in keep]\n    new = [simulator_rdkit_wheel(pkg)]\n    keep = keep + new[:1]\n    _o.makedirs(site, exist_ok=True)\n    wdir = _o.path.join(site, '.wheels')\n    _o.makedirs(wdir, exist_ok=True)\n    whl = []\n    for w in keep:\n        dst = _o.path.join(wdir, _o.path.basename(w)[:-4] if w.endswith('.ice') else _o.path.basename(w))\n        _s.copyfile(w, dst)\n        whl.append(dst)\n    r = _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',\n                 '--no-compile', '--disable-pip-version-check', '--upgrade', '--target', site] + whl,\n                stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True)\n    print('[claw] site preseed rc', r.returncode, (r.stdout or '')[-400:], flush=True)\n    if r.returncode != 0:\n        raise RuntimeError('claw preseed pip failed: ' + (r.stdout or '')[-600:])\n    _s.rmtree(wdir, ignore_errors=True)\n    # the marker MUST equal exactly what install_site() derives from the PACKAGE list\n    want = '\\n'.join(_o.path.basename(w) for w in srcs)\n    with open(_o.path.join(site, '.ice_site_ok'), 'w') as f:\n        f.write(want)\n    print('[claw] site ready; substituted', dropped, '->', [_o.path.basename(w) for w in new[:1]], flush=True)\n    return want\n\n\ntry:\n    ICE_PKG = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))\n    sys.path.append(ICE_PKG)\n    import fuse as ice_fuse\n    del V, E, bank; gc.collect(); torch.cuda.empty_cache()\n    # priority: molecules without a library-level match first (class 1 rarely changes), then the rest\n    mids = sorted(BASE, key=lambda m: BASE[m][2])\n    cands = {m: (ice_fuse.ice_candidates(BASE[m][0], BASE[m][4], TOPN) if BASE_ICE else []) for m in mids}\n    if ENG:                              # ADDED (ICE_UNION): also score our engine's candidates (same-formula groups\n        from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF   # formed over the v4g top-N + engine top-40)\n        from rdkit import Chem as _Ch\n        def _f2(s):\n            try:\n                return _CMF(_Ch.MolFromSmiles(s))\n            except Exception:\n                return s\n        n_add = 0\n        for m in mids:\n            e = ENG.get(str(m))\n            if not e or not e.get('smiles'):\n                continue\n            bs, bk, bf = list(BASE[m][0][:TOPN]), set(BASE[m][1][:TOPN]), list(BASE[m][4][:TOPN])\n            es = [s for s, k in zip(e['smiles'][:40], e['keys'][:40]) if k not in bk]\n            ef = [_f2(s) for s in es]\n            extra = ice_fuse.ice_candidates(bs + es, bf + ef, len(bs) + len(es))\n            before = len(cands.get(m, []))\n            cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))\n            n_add += len(cands[m]) - before\n        print('ICE_UNION: engine candidates added to ICE input:', n_add, flush=True)\n    if ICE_PC:                          # also the PubChem-only list of molecules the v3.6 gate may use (lib_max < LIB_TAU)\n        from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n        from rdkit import Chem as _Chem\n        def _form(s):\n            try:\n                return CalcMolFormula(_Chem.MolFromSmiles(s))\n            except Exception:\n                return s\n        for m in mids:\n            p = PC.get(m) if isinstance(PC, dict) else None\n            if p and p.get('pc') and BASE[m][2] < LIB_TAU:\n                p['pc_form'] = [_form(x) for x in p['pc']]\n                extra = ice_fuse.ice_candidates(p['pc'], p['pc_form'], 25)\n                cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))\n    cands = {m: c for m, c in cands.items() if c}\n    items = ice_fuse.build_ice_input(te, cands)\n    print('ICE input molecules', len(items), 'candidates', sum(len(it['cands']) for it in items), f'{time.time()-T0:.0f}s', flush=True)\n    _claw_preseed_site(ICE_PKG, '/kaggle/working/ice_site')\n    ICE_SCORES = ice_fuse.run_ice(ICE_PKG, items, workdir='/kaggle/working/ice_work', device='cuda', budget_s=ICE_BUDGET,\n                                  site='/kaggle/working/ice_site', extra_args=['--any-rdkit'])\n    meta_f = '/kaggle/working/ice_work/ice_out.json.meta.json'\n    if os.path.exists(meta_f):\n        print('ICE meta', open(meta_f).read()[:1500])\nexcept Exception as e:\n    CLAW_ICE_HEALTH = f'ICE FAILED: {e!r}'\n    print('ICE FAILED -> ranker order kept:', repr(e), flush=True)\n'''v4h: GLACIER third re-scoring term (casmi26-glacier) on the SAME runner input; any GLACIER failure -> v4g (ICE only)'''\nGL_SCORES, gl_stats, GL_LAM, GL_BUDGET = {}, dict(molecules=0, changed_vs_ice_top25=0, changed_vs_ice_top1=0), 1.0, 4000\ntry:\n    GL_PKG = os.path.dirname(find('casmi26-glacier/**/gl_runner.py'))\n    sys.path.append(GL_PKG)\n    import gl_fuse\n    gc.collect(); torch.cuda.empty_cache()   # ICE ran in its own (finished) process; free the notebook's cached GPU memory\n    gl_items = [dict(it, spectra=[s for s in it['spectra'] if s.get('adduct') == '[M+H]+']) for it in items]   # v29: GLACIER on [M+H]+ only (v27: +0.006)\n    gl_items = [it for it in gl_items if it['spectra']]\n    print('v29 GLACIER [M+H]+ only: molecules', len(gl_items), 'of', len(items))\n    GL_SCORES = gl_fuse.run_gl(GL_PKG, gl_items, workdir='/kaggle/working/gl_work', device='cuda', budget_s=GL_BUDGET,\n                               site='/kaggle/working/ice_site', wheels=os.path.join(ICE_PKG, 'wheels'), extra_args=['--any-rdkit'])\n    meta_f = '/kaggle/working/gl_work/gl_out.json.meta.json'\n    print('GL meta', open(meta_f).read()[:1500] if os.path.exists(meta_f) else 'missing')\nexcept Exception as e:\n    GL_SCORES = {}\n    CLAW_GL_HEALTH = f'GL FAILED: {e!r}'\n    print('GL FAILED -> v4g behaviour (ICE only):', repr(e), flush=True)\n# N1A complement: Seyit Kaan Gunes; integration as in Ahmed Berat Ozer v15 / guhongbin v465.\n# ---- N1A_MCE (gold roadmap 2026-10-09 section 3.2: N1 arm A + I14b merged CE on the proxies only) ---------------\n# Molecules WITHOUT a real [M+H]+ spectrum are GLACIER-blind in H1. N1 arm A scores their [M+NH4]+ / [M-H2O+H]+\n# spectra as [M+H]+ proxies (NH4: M = prec - 18.033823, peaks >= M + 1.007276 - 1.5 dropped; water loss:\n# M = prec + 18.010565 - 1.007276, peaks < prec - 1.5 kept; precursor -> M + 1.007276, relabelled [M+H]+;\n# [M-2H2O+H]+ / Na / K never), and a merged-CE proxy (>= 2 distinct CE buckets) is predicted at EVERY bucket and the\n# predictions merge()d like the SUM_CE path (I14b). = bench kdev_glbench_n1 arm N1A_D on the blind molecules (521\n# bench molecules: 21 blind molecules with a proxy, lift over random +0.270 [0.117, 0.443]; vs H1 on all np molecules\n# +0.027 [0.010, 0.047]; merged vs mean CE on those 21 +0.0026). Arm B (proxies also where [M+H]+ exists, -0.047) and\n# merged CE for real [M+H]+ spectra (no gain) are NOT done: molecules WITH a real [M+H]+ keep the H1 run above\n# untouched (stock gl_runner, mean CE, byte-identical input). The bench's own code runs here, embedded verbatim\n# (sha256-checked): glb_core.py (n1_proxy / build_variants / mol_flags / arm_tags / arm_scores) in this process,\n# glb_driver.py (driver around the attached gl_runner.py: same env, model, Predictor, canonicalisation, CE bucketing,\n# merge, entropy similarity) in its own process like gl_runner. N1 scores are added to GL_SCORES (-> the PubChem-list\n# re-rank below and the fused re-rank of cell 19, whose `if ice:` gate is opened ONLY for N1 molecules with scores).\n# Any failure -> no N1 scores = H1 behaviour (the commit raises instead). The 12 dummy commit molecules never\n# trigger N1, so the commit adds a forced exercise: synthetic [M+H]+-less copies 'n1x:<mid>' ([M+H]+ -> [M+NH4]+ with\n# an injected NH4 precursor peak, the first one also -> [M-H2O+H]+) run through the same code + asserts.\nN1_ARM = 'N1A_D'                 # bench arm: blind molecule -> spectrum tags (NH4p, WLp), CE mode 'merged'\nN1_BUDGET = 1200.0 if IS_RERUN else 600.0\nN1_WORK, N1_SITE, N1_DEVICE, N1_DRIVER_EXTRA = '/kaggle/working/n1_work', '/kaggle/working/ice_site', 'cuda', []\nN1_MIDS, N1_GL_MIDS, N1_STATS, N1_DRV, N1_HEALTH = set(), set(), {}, {}, 'N1 NOT RUN'\nN1X_MIDS, N1X_GL, N1X_STATS = set(), {}, {}\n# k2 (E006 port), variant A = N1-complement. E006 (cell 31) re-ranks a GLACIER-blind molecule with ICEBERG when its ICE\n# dict has finite scores (in practice [M+Na]+ without [M+H]+: `elif ice_fallback and ice and not gl`). N1 GL scores\n# would move such a molecule to the GLACIER branch, so they are kept ONLY for N1 molecules WITHOUT any finite ICE\n# score (E006 has no forward-model evidence there: its ICE fallback on an all-None dict is a no-op). False = the plain\n# port (N1 GL scores for every N1 molecule, E2 behaviour). Molecules with a real [M+H]+ are never touched either way.\n# Challenger route: flexonafft, casmi26-closing-the-evidence-gap v47 (v22).\nN1_COMPLEMENT = False\nprint('[N1 challenger] configuration: N1_COMPLEMENT=False; real [M+H]+ stays on H1', flush=True)\nN1_ICE_DROPPED = set()           # N1 molecules left to E006's ICE fallback by the complement filter\nN1_CORE_SHA256 = 'f7a3b8cd8802a7ba3b69a297603241784a63c069044cf7bd6efda7d68cc8fdfa'      # = subs/out/kdev_glbench_n1_full/glb_core.py\nN1_DRIVER_SHA256 = 'e919a154b57bda5a52573481348e38d8b4e7c97883532ea5b385bc3cb6f98502'    # = subs/out/kdev_glbench_n1_full/glb_driver.py\nN1_GL_RUNNER_SHA256 = '5c4f74313ac54263d05d987c39d012d81ec57c5f35e1095ecd49a5bcb6440fca'   # casmi26-glacier gl_runner.py (B1 MANIFEST, bench)\n# kdev_glbench_n1 glb_core.py, verbatim (proxy construction, variants, arm resolution, aggregation)\nN1_CORE_SRC = ''.join([\n    '\"\"\"casmi26 kdev_glbench_n1 -- pure-python core of the roadmap 3.2 GLACIER bench (no torch / rdkit import).\\n',\n    '\\n',\n    'Query handling (N1, roadmap 3.2), on fuse.build_ice_input(..., covered_only=False) spectrum dicts:\\n',\n    '  [M+NH4]+    M = prec - 18.033823; prec -> M + 1.007276; drop peaks mz >= M + 1.007276 - 1.5; relabel [M+H]+\\n',\n    '  [M-H2O+H]+  M = prec + 18.010565 - 1.007276; prec -> M + 1.007276; keep peaks mz < original prec - 1.5; relabel\\n',\n    '  (a proxy left without peaks is dropped; [M-2H2O+H]+, [M+Na]+, [M+K]+ are never proxied)\\n',\n    '  Hcut (diagnostic): a real [M+H]+ spectrum with the same precursor-region cut (peaks mz >= prec - 1.5 dropped), the\\n',\n    '  like-for-like reference for the proxies (separates \"different adduct\" from \"precursor region removed\").\\n',\n    '\\n',\n    'Scoring: the driver returns per-spectrum GLACIER similarities for the frozen bench candidates in two CE modes\\n',\n    \"('mean' = production gl_runner: one prediction at the bucket of mean(ce_ev); 'merged' = I14b: one prediction per\\n\",\n    \"distinct CE bucket of a merged-CE spectrum, merged like the SUM_CE path). A molecule's arm score per candidate =\\n\",\n    \"mean over the arm's spectra with a similarity = gl_ex_mean restricted to those spectra.\\n\",\n    '\\n',\n    'Ranking metric: expected reciprocal rank of the truth (cands[mid][0]) under a UNIFORM RANDOM tie-break (no truth-first\\n',\n    'bias: a truth tied with k decoys gets mean(1/(r..r+k))); unscored candidates rank below every scored one (tied among\\n',\n    'themselves). Baseline = a random order of all frozen candidates, mean(1/1..1/n). An arm \"covers\" a molecule when the\\n',\n    'truth and >= 2 candidates are scored; elsewhere the arm has no evidence (= the random baseline in fill-in totals).\\n',\n    '\"\"\"\\n',\n    'import math\\n',\n    'from collections import Counter, OrderedDict\\n',\n    '\\n',\n    'import numpy as np\\n',\n    '\\n',\n    'PROTON = 1.007276\\n',\n    'NH4_ADD = 18.033823           # [M+NH4]+ precursor m/z - M\\n',\n    'H2O = 18.010565\\n',\n    'CUT_DA = 1.5\\n',\n    'MIN_PEAKS = 1\\n',\n    \"PROXY_TAG = OrderedDict([('[M+NH4]+', 'NH4p'), ('[M-H2O+H]+', 'WLp')])\\n\",\n    'EPS_TIE = 1e-12\\n',\n    'N_BOOT = 2000\\n',\n    'BOOT_SEED = 12345\\n',\n    '\\n',\n    \"# arm -> (spectrum tags, CE mode, description); 'N1A' is resolved per molecule (see arm_tags)\\n\",\n    'ARMS = OrderedDict([\\n',\n    \"    ('A_H', (('H',), 'mean', '(a) [M+H]+ only, mean CE = B1 production GL')),\\n\",\n    \"    ('B_NH4', (('NH4p',), 'mean', '(b) [M+NH4]+ proxies relabelled [M+H]+')),\\n\",\n    \"    ('C_WL', (('WLp',), 'mean', '(c) [M-H2O+H]+ proxies relabelled [M+H]+')),\\n\",\n    \"    ('D_Hmerged', (('H',), 'merged', '(d) [M+H]+ only, merged-CE per-bucket prediction (I14b)')),\\n\",\n    \"    ('A_Hcut', (('Hcut',), 'mean', 'diag: [M+H]+ with the proxy-style precursor cut')),\\n\",\n    \"    ('B_NH4m', (('NH4p',), 'merged', 'diag: NH4 proxies, merged CE')),\\n\",\n    \"    ('C_WLm', (('WLp',), 'merged', 'diag: water-loss proxies, merged CE')),\\n\",\n    \"    ('HNa', (('H', 'Na'), 'mean', 'diag: [M+H]+ and [M+Na]+ (old fwdbench COV; P4 analogue)')),\\n\",\n    \"    ('AB', (('H', 'NH4p', 'WLp'), 'mean', 'diag: N1 arm B = [M+H]+ plus all proxies')),\\n\",\n    \"    ('N1A', (None, 'mean', 'N1 arm A: [M+H]+ if the molecule has one, else its proxies')),\\n\",\n    \"    ('N1A_D', (None, 'merged', 'diag: N1 arm A with merged CE (N1A + I14b)')),\\n\",\n    '])\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------ query handling --\\n',\n    'def _keep_below(sp, cut, min_peaks):\\n',\n    \"    mz, it = sp.get('mz') or [], sp.get('it') or []\\n\",\n    '    keep = [(float(m), float(i)) for m, i in zip(mz, it) if float(m) < cut]\\n',\n    '    if len(keep) < min_peaks or not any(i > 0 for _, i in keep):\\n',\n    '        return None\\n',\n    '    return [m for m, _ in keep], [i for _, i in keep]\\n',\n    '\\n',\n    '\\n',\n    'def n1_proxy(sp, min_peaks=MIN_PEAKS):\\n',\n    '    \"\"\"[M+NH4]+ / [M-H2O+H]+ positive spectrum dict -> relabelled [M+H]+ proxy dict, or None.\"\"\"\\n',\n    \"    if not isinstance(sp, dict) or sp.get('mode') != 'positive':\\n\",\n    '        return None\\n',\n    \"    ad = sp.get('adduct')\\n\",\n    '    try:\\n',\n    \"        prec = float(sp['prec'])\\n\",\n    '    except Exception:\\n',\n    '        return None\\n',\n    '    if not math.isfinite(prec):\\n',\n    '        return None\\n',\n    \"    if ad == '[M+NH4]+':\\n\",\n    '        M = prec - NH4_ADD\\n',\n    '        cut = M + PROTON - CUT_DA\\n',\n    \"    elif ad == '[M-H2O+H]+':\\n\",\n    '        M = prec + H2O - PROTON\\n',\n    '        cut = prec - CUT_DA\\n',\n    '    else:\\n',\n    '        return None\\n',\n    '    k = _keep_below(sp, cut, min_peaks)\\n',\n    '    if k is None:\\n',\n    '        return None\\n',\n    '    out = dict(sp)\\n',\n    \"    out.update(mz=k[0], it=k[1], prec=M + PROTON, adduct='[M+H]+', mode='positive', n1_src=ad)\\n\",\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    'def h_cut(sp, min_peaks=MIN_PEAKS):\\n',\n    '    \"\"\"Real [M+H]+ spectrum with the proxy-style cut (peaks mz >= prec - 1.5 dropped); None if nothing is left.\"\"\"\\n',\n    \"    if not isinstance(sp, dict) or sp.get('mode') != 'positive' or sp.get('adduct') != '[M+H]+':\\n\",\n    '        return None\\n',\n    \"    k = _keep_below(sp, float(sp['prec']) - CUT_DA, min_peaks)\\n\",\n    '    if k is None:\\n',\n    '        return None\\n',\n    '    return dict(sp, mz=k[0], it=k[1])\\n',\n    '\\n',\n    '\\n',\n    'def build_variants(items_all):\\n',\n    '    \"\"\"fuse.build_ice_input(Q, cands, covered_only=False) items -> driver items whose spectra carry sid / tag\\n',\n    '    (tags H, Hcut, Na, NH4p, WLp), plus counts. sid = \\'<index of the spectrum in the molecule>:<tag>\\'.\"\"\"\\n',\n    '    out, st = [], Counter()\\n',\n    '    for it in items_all:\\n',\n    '        sps = []\\n',\n    \"        for j, sp in enumerate(it.get('spectra') or []):\\n\",\n    \"            if sp.get('mode') != 'positive':\\n\",\n    '                continue\\n',\n    \"            ad = sp.get('adduct')\\n\",\n    \"            if ad == '[M+H]+':\\n\",\n    \"                sps.append(dict(sp, sid=f'{j}:H', tag='H')); st['H'] += 1\\n\",\n    '                c = h_cut(sp)\\n',\n    '                if c is None:\\n',\n    \"                    st['Hcut_empty'] += 1\\n\",\n    '                else:\\n',\n    \"                    sps.append(dict(c, sid=f'{j}:Hcut', tag='Hcut')); st['Hcut'] += 1\\n\",\n    \"            elif ad == '[M+Na]+':\\n\",\n    \"                sps.append(dict(sp, sid=f'{j}:Na', tag='Na')); st['Na'] += 1\\n\",\n    '            elif ad in PROXY_TAG:\\n',\n    '                tag = PROXY_TAG[ad]\\n',\n    '                p = n1_proxy(sp)\\n',\n    '                if p is None:\\n',\n    \"                    st[tag + '_empty'] += 1\\n\",\n    '                else:\\n',\n    \"                    sps.append(dict(p, sid=f'{j}:{tag}', tag=tag)); st[tag] += 1\\n\",\n    \"        out.append(dict(mid=str(it['mid']), cands=list(it['cands']), spectra=sps))\\n\",\n    '    return out, dict(st)\\n',\n    '\\n',\n    '\\n',\n    'def proxy_prec_check(items):\\n',\n    '    \"\"\"Real-data check of the adduct arithmetic: proxy precursor (M + 1.007276) vs the molecule\\'s real [M+H]+\\n',\n    '    precursor m/z (median over its [M+H]+ spectra), in mDa, per proxy tag.\"\"\"\\n',\n    '    out = {}\\n',\n    '    for tag in PROXY_TAG.values():\\n',\n    '        d = []\\n',\n    '        for it in items:\\n',\n    \"            H = [float(sp['prec']) for sp in it['spectra'] if sp.get('tag') == 'H']\\n\",\n    '            if H:\\n',\n    \"                d += [1000.0 * (float(sp['prec']) - float(np.median(H))) for sp in it['spectra'] if sp.get('tag') == tag]\\n\",\n    '        out[tag] = dict(n=len(d), median_mda=float(np.median(d)) if d else None,\\n',\n    '                        max_abs_mda=float(np.max(np.abs(d))) if d else None)\\n',\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    '# ---------------------------------------------------------------------------------------------------- ranking ----\\n',\n    'def rr_random_tb(vals, ti=0, eps=EPS_TIE):\\n',\n    '    \"\"\"Expected reciprocal rank of vals[ti] under a uniform random tie-break; None entries rank below all scored\\n',\n    '    ones. None if vals[ti] is None.\"\"\"\\n',\n    '    t = vals[ti]\\n',\n    '    if t is None:\\n',\n    '        return None\\n',\n    '    gt = eq = 0\\n',\n    '    for j, v in enumerate(vals):\\n',\n    '        if j == ti or v is None:\\n',\n    '            continue\\n',\n    '        if v > t + eps:\\n',\n    '            gt += 1\\n',\n    '        elif abs(v - t) <= eps:\\n',\n    '            eq += 1\\n',\n    '    return sum(1.0 / (gt + 1 + k) for k in range(eq + 1)) / (eq + 1)\\n',\n    '\\n',\n    '\\n',\n    'def rr_truth_first(vals, ti=0, eps=EPS_TIE):\\n',\n    '    \"\"\"Old-bench convention with the truth winning every tie (the bias this bench removes).\"\"\"\\n',\n    '    t = vals[ti]\\n',\n    '    if t is None:\\n',\n    '        return None\\n',\n    '    return 1.0 / (1 + sum(1 for j, v in enumerate(vals) if j != ti and v is not None and v > t + eps))\\n',\n    '\\n',\n    '\\n',\n    'def rr_split(vals, ti=0, eps=EPS_TIE):\\n',\n    '    \"\"\"fwdbench bench_core.mrr_table convention (ties count half).\"\"\"\\n',\n    '    t = vals[ti]\\n',\n    '    if t is None:\\n',\n    '        return None\\n',\n    '    gt = sum(1 for j, v in enumerate(vals) if j != ti and v is not None and v > t + eps)\\n',\n    '    eq = sum(1 for j, v in enumerate(vals) if j != ti and v is not None and abs(v - t) <= eps)\\n',\n    '    return 1.0 / (1 + gt + 0.5 * eq)\\n',\n    '\\n',\n    '\\n',\n    'def rr_rand(n):\\n',\n    \"    return sum(1.0 / k for k in range(1, n + 1)) / n if n > 0 else float('nan')\\n\",\n    '\\n',\n    '\\n',\n    'def covered(vals, ti=0):\\n',\n    '    return vals[ti] is not None and sum(v is not None for v in vals) >= 2\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------ aggregation ----\\n',\n    'def mol_flags(item):\\n',\n    \"    tags = Counter(sp['tag'] for sp in item['spectra'])\\n\",\n    \"    return dict(has_H=tags['H'] > 0, has_NH4p=tags['NH4p'] > 0, has_WLp=tags['WLp'] > 0, has_Na=tags['Na'] > 0)\\n\",\n    '\\n',\n    '\\n',\n    'def arm_tags(arm, flags):\\n',\n    '    tags, mode, _ = ARMS[arm]\\n',\n    '    if tags is None:                                        # N1 arm A: proxies only where no real [M+H]+ exists\\n',\n    \"        tags = ('H',) if flags['has_H'] else ('NH4p', 'WLp')\\n\",\n    '    return tags, mode\\n',\n    '\\n',\n    '\\n',\n    'def arm_scores(mol_out, tags, mode):\\n',\n    '    \"\"\"mol_out = driver output of one molecule {\\'cands\\': [...], \\'spec\\': {sid: {\\'tag\\',\\'nb\\',\\'mean\\',\\'merged\\'}}} ->\\n',\n    '    per-candidate list (aligned with mol_out[\\'cands\\']) of the mean similarity over the arm\\'s spectra (None if none).\"\"\"\\n',\n    \"    cands = mol_out.get('cands') or []\\n\",\n    \"    cols = [s[mode] for sid, s in sorted((mol_out.get('spec') or {}).items(), key=lambda kv: _sid_key(kv[0]))\\n\",\n    \"            if s.get('tag') in tags and s.get(mode) is not None]\\n\",\n    '    out = []\\n',\n    '    for i in range(len(cands)):\\n',\n    '        xs = [c[i] for c in cols if i < len(c) and c[i] is not None]\\n',\n    '        out.append(float(np.mean(xs)) if xs else None)\\n',\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    'def _sid_key(sid):\\n',\n    \"    a, _, b = str(sid).partition(':')\\n\",\n    '    try:\\n',\n    '        return (int(a), b)\\n',\n    '    except ValueError:\\n',\n    '        return (10 ** 9, str(sid))\\n',\n    '\\n',\n    '\\n',\n    'def molecule_table(items, out, done):\\n',\n    '    \"\"\"-> list of row dicts, one per molecule the driver finished: rr per arm (None = arm does not cover it).\"\"\"\\n',\n    '    rows = []\\n',\n    '    for it in items:\\n',\n    \"        mid = it['mid']\\n\",\n    '        if mid not in done:\\n',\n    '            continue\\n',\n    \"        mo = out.get(mid) or {'cands': it['cands'], 'spec': {}}\\n\",\n    \"        assert list(mo.get('cands') or it['cands']) == list(it['cands']), ('candidate order changed', mid)\\n\",\n    '        fl = mol_flags(it)\\n',\n    \"        n = len(it['cands'])\\n\",\n    \"        nb_h = [s.get('nb', 1) for s in (mo.get('spec') or {}).values() if s.get('tag') == 'H']\\n\",\n    \"        r = dict(mid=mid, set=mid.split(':')[0], n=n, rand=rr_rand(n), has_merged_H=any(b >= 2 for b in nb_h), **fl)\\n\",\n    '        for arm in ARMS:\\n',\n    '            tags, mode = arm_tags(arm, fl)\\n',\n    '            v = arm_scores(mo, tags, mode)\\n',\n    \"            r['rr_' + arm] = rr_random_tb(v) if covered(v) else None\\n\",\n    \"            if arm == 'A_H':\\n\",\n    \"                r['rrtf_A_H'] = rr_truth_first(v) if covered(v) else None\\n\",\n    \"                r['rrsp_A_H'] = rr_split(v) if covered(v) else None\\n\",\n    \"                r['top1_A_H'] = (None if not covered(v) else float(rr_random_tb(v) == 1.0))\\n\",\n    '        rows.append(r)\\n',\n    '    return rows\\n',\n    '\\n',\n    '\\n',\n    '# ----------------------------------------------------------------------------------------------------- stats -----\\n',\n    'def _boot_idx(n, seed=BOOT_SEED, nb=N_BOOT):\\n',\n    '    return np.random.default_rng(seed).integers(0, n, (nb, n))\\n',\n    '\\n',\n    '\\n',\n    \"def paired(rows, a, b, sel=None, label=''):\\n\",\n    '    \"\"\"Paired comparison of arms a and b over the rows both cover (and sel(row) if given).\"\"\"\\n',\n    \"    R = [r for r in rows if r['rr_' + a] is not None and r['rr_' + b] is not None and (sel is None or sel(r))]\\n\",\n    '    n = len(R)\\n',\n    '    res = dict(label=label, a=a, b=b, n=n)\\n',\n    '    if n == 0:\\n',\n    '        return res\\n',\n    \"    ra = np.array([r['rr_' + a] for r in R]); rb = np.array([r['rr_' + b] for r in R])\\n\",\n    \"    rn = np.array([r['rand'] for r in R])\\n\",\n    '    d = rb - ra\\n',\n    '    idx = _boot_idx(n)\\n',\n    '    la, lb = ra - rn, rb - rn\\n',\n    \"    with np.errstate(divide='ignore', invalid='ignore'):\\n\",\n    '        bo_ratio = lb[idx].mean(1) / la[idx].mean(1)\\n',\n    \"    ratio = float(lb.mean() / la.mean()) if la.mean() > 0 else float('nan')\\n\",\n    '    res.update(mrr_a=float(ra.mean()), mrr_b=float(rb.mean()), rand=float(rn.mean()), lift_a=float(la.mean()),\\n',\n    '               lift_b=float(lb.mean()), delta=float(d.mean()),\\n',\n    '               delta_ci=[float(x) for x in np.percentile(d[idx].mean(1), [2.5, 97.5])],\\n',\n    '               ratio=ratio, ratio_ci=[float(x) for x in np.nanpercentile(bo_ratio, [2.5, 97.5])] if n > 1 else [ratio, ratio],\\n',\n    '               wins=int((d > 1e-9).sum()), ties=int((np.abs(d) <= 1e-9).sum()), losses=int((d < -1e-9).sum()),\\n',\n    \"               by_set={s: int(sum(r['set'] == s for r in R)) for s in sorted({r['set'] for r in R})})\\n\",\n    '    return res\\n',\n    '\\n',\n    '\\n',\n    \"def arm_table(rows, arm, sets=('np', 'e180', 'all')):\\n\",\n    '    out = {}\\n',\n    '    for s in sets:\\n',\n    \"        R = [r for r in rows if r['rr_' + arm] is not None and (s == 'all' or r['set'] == s)]\\n\",\n    '        if not R:\\n',\n    '            out[s] = dict(n=0)\\n',\n    '            continue\\n',\n    \"        rr = np.array([r['rr_' + arm] for r in R]); rn = np.array([r['rand'] for r in R])\\n\",\n    '        out[s] = dict(n=len(R), mrr=float(rr.mean()), rand=float(rn.mean()), lift=float((rr - rn).mean()),\\n',\n    '                      top1=float(np.mean(rr >= 1.0 - 1e-12)))\\n',\n    '    return out\\n',\n    '\\n',\n    '\\n',\n    \"def filled_delta(rows, a, b, sel=None, label=''):\\n\",\n    '    \"\"\"Mean over ALL selected molecules with a missing arm filled by the random baseline (no evidence).\"\"\"\\n',\n    '    R = [r for r in rows if sel is None or sel(r)]\\n',\n    '    n = len(R)\\n',\n    '    res = dict(label=label, a=a, b=b, n=n)\\n',\n    '    if n == 0:\\n',\n    '        return res\\n',\n    \"    fa = np.array([r['rr_' + a] if r['rr_' + a] is not None else r['rand'] for r in R])\\n\",\n    \"    fb = np.array([r['rr_' + b] if r['rr_' + b] is not None else r['rand'] for r in R])\\n\",\n    '    d = fb - fa\\n',\n    '    idx = _boot_idx(n)\\n',\n    '    res.update(mrr_a=float(fa.mean()), mrr_b=float(fb.mean()), delta=float(d.mean()),\\n',\n    '               delta_ci=[float(x) for x in np.percentile(d[idx].mean(1), [2.5, 97.5])],\\n',\n    '               n_changed=int((np.abs(d) > 1e-9).sum()))\\n',\n    '    return res\\n',\n    '\\n',\n    '\\n',\n    'def fmt_p(p):\\n',\n    \"    if not p.get('n'):\\n\",\n    '        return f\"{p.get(\\'label\\', \\'\\')}: {p[\\'a\\']} vs {p[\\'b\\']} n=0\"\\n',\n    '    s = (f\"{p.get(\\'label\\', \\'\\')}: n={p[\\'n\\']} {p[\\'a\\']} {p[\\'mrr_a\\']:.4f} vs {p[\\'b\\']} {p[\\'mrr_b\\']:.4f} (rand {p[\\'rand\\']:.4f}) \"\\n',\n    '         f\"delta {p[\\'delta\\']:+.4f} CI [{p[\\'delta_ci\\'][0]:+.4f}, {p[\\'delta_ci\\'][1]:+.4f}] \"\\n',\n    '         f\"lift {p[\\'lift_a\\']:.4f} -> {p[\\'lift_b\\']:.4f} ratio {p[\\'ratio\\']:.3f} CI [{p[\\'ratio_ci\\'][0]:.3f}, {p[\\'ratio_ci\\'][1]:.3f}] \"\\n',\n    '         f\"W/T/L {p[\\'wins\\']}/{p[\\'ties\\']}/{p[\\'losses\\']} sets {p[\\'by_set\\']}\")\\n',\n    '    return s\\n',\n    '\\n',\n    '\\n',\n    'def gates(rows):\\n',\n    '    \"\"\"Roadmap 3.2 gates. N1-NH4 / N1-WL: kill if the proxy lift < 50% of the paired [M+H]+ lift (same molecules).\\n',\n    '    I14b: adopt only if delta >= +0.01 on merged-CE e180 molecules and >= 0 on np merged-CE molecules.\"\"\"\\n',\n    '    g = OrderedDict()\\n',\n    \"    for key, arm in (('N1_NH4', 'B_NH4'), ('N1_WL', 'C_WL')):\\n\",\n    \"        p = paired(rows, 'A_H', arm, label=f'{key} paired [M+H]+ vs proxy')\\n\",\n    \"        pc = paired(rows, 'A_Hcut', arm, label=f'{key} paired [M+H]+cut vs proxy (diag)')\\n\",\n    \"        if not p.get('n'):\\n\",\n    \"            v = 'NO DATA'\\n\",\n    \"        elif not math.isfinite(p['ratio']):\\n\",\n    \"            v = 'UNDEFINED ([M+H]+ lift <= 0)'\\n\",\n    '        else:\\n',\n    \"            v = 'PASS' if p['ratio'] >= 0.5 else 'KILL'\\n\",\n    \"        g[key] = dict(verdict=v, rule='proxy lift >= 0.5 x paired [M+H]+ lift', paired=p, paired_cut=pc)\\n\",\n    \"    pe = paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'e180' and r['has_merged_H'], label='I14b e180 merged-CE')\\n\",\n    \"    pn = paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'np' and r['has_merged_H'], label='I14b np merged-CE')\\n\",\n    \"    if not pe.get('n') or not pn.get('n'):\\n\",\n    \"        v = 'NO DATA'\\n\",\n    '    else:\\n',\n    \"        v = 'ADOPT' if (pe['delta'] >= 0.01 and pn['delta'] >= 0.0) else 'REJECT'\\n\",\n    \"    g['I14b'] = dict(verdict=v, rule='delta >= +0.01 on e180 merged-CE and >= 0 on np merged-CE', e180=pe, np=pn,\\n\",\n    \"                     np_all=paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'np', label='I14b np all'),\\n\",\n    \"                     e180_all=paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'e180', label='I14b e180 all'))\\n\",\n    '    return g\\n',\n    '\\n',\n    '\\n',\n    'def blind_report(rows):\\n',\n    '    \"\"\"Molecules with NO real [M+H]+ spectrum (GLACIER-blind in B1): what N1 arm A recovers there.\"\"\"\\n',\n    \"    B = [r for r in rows if not r['has_H']]\\n\",\n    \"    res = OrderedDict(n_blind=len(B), n_blind_np=sum(r['set'] == 'np' for r in B),\\n\",\n    \"                      n_blind_with_proxy=sum(r['has_NH4p'] or r['has_WLp'] for r in B),\\n\",\n    \"                      n_blind_proxy_covered=sum(r['rr_N1A'] is not None for r in B))\\n\",\n    \"    C = [r for r in B if r['rr_N1A'] is not None]\\n\",\n    '    if C:\\n',\n    \"        rr = np.array([r['rr_N1A'] for r in C]); rn = np.array([r['rand'] for r in C])\\n\",\n    '        idx = _boot_idx(len(C))\\n',\n    '        res.update(mrr_proxy=float(rr.mean()), rand=float(rn.mean()), lift=float((rr - rn).mean()),\\n',\n    '                   lift_ci=[float(x) for x in np.percentile((rr - rn)[idx].mean(1), [2.5, 97.5])],\\n',\n    '                   top1=float(np.mean(rr >= 1.0 - 1e-12)),\\n',\n    \"                   by_src={k: int(sum(1 for r in C if r[k])) for k in ('has_NH4p', 'has_WLp')},\\n\",\n    \"                   mids=[r['mid'] for r in C])\\n\",\n    \"    res['overall_np'] = filled_delta(rows, 'A_H', 'N1A', sel=lambda r: r['set'] == 'np', label='N1A vs A, all np (blind filled with rand)')\\n\",\n    \"    res['overall_all'] = filled_delta(rows, 'A_H', 'N1A', label='N1A vs A, all molecules')\\n\",\n    \"    res['overall_np_D'] = filled_delta(rows, 'A_H', 'N1A_D', sel=lambda r: r['set'] == 'np', label='N1A+I14b vs A, all np')\\n\",\n    '    return res\\n',\n    '\\n',\n    '\\n',\n    'def score_corr(new, old):\\n',\n    '    \"\"\"new / old: {mid: {smiles: score|None}} -> pooled Pearson + mean per-molecule Spearman over common scored.\"\"\"\\n',\n    '    xs, ys, sp = [], [], []\\n',\n    '    for mid, d in new.items():\\n',\n    '        o = old.get(mid) or {}\\n',\n    '        pairs = [(v, o.get(s)) for s, v in d.items() if v is not None and o.get(s) is not None]\\n',\n    '        if len(pairs) < 3:\\n',\n    '            continue\\n',\n    '        a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])\\n',\n    '        xs.extend(a); ys.extend(b)\\n',\n    '        ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))\\n',\n    '        if ra.std() > 0 and rb.std() > 0:\\n',\n    '            sp.append(float(np.corrcoef(ra, rb)[0, 1]))\\n',\n    '    if len(xs) < 3:\\n',\n    '        return dict(n_mols=0)\\n',\n    '    return dict(n_mols=len(sp), n_pairs=len(xs), pearson=float(np.corrcoef(xs, ys)[0, 1]),\\n',\n    \"                spearman_mean=float(np.mean(sp)) if sp else float('nan'))\\n\",\n    '\\n',\n    '\\n',\n    '# -------------------------------------------------------------------------------------------------- self-test ----\\n',\n    'def selftest():\\n',\n    '    \"\"\"Forced unit tests of the pure functions (run on Kaggle before any GPU work and locally by the build).\"\"\"\\n',\n    '    M = 300.0\\n',\n    '    nh4 = dict(mz=[100.0, 250.0, 299.5, 300.0, 301.0, 318.03], it=[0.1, 1.0, 0.2, 0.3, 0.4, 0.5], prec=M + NH4_ADD,\\n',\n    \"               adduct='[M+NH4]+', mode='positive', instrument='timsTOF', ce_ev=[20.0], ce_orig='20', ce_units='eV')\\n\",\n    '    p = n1_proxy(nh4)\\n',\n    \"    assert p is not None and p['mz'] == [100.0, 250.0, 299.5] and p['it'] == [0.1, 1.0, 0.2], p\\n\",\n    \"    assert abs(p['prec'] - (M + PROTON)) < 1e-9 and p['adduct'] == '[M+H]+' and p['n1_src'] == '[M+NH4]+'\\n\",\n    \"    assert p['ce_ev'] == [20.0] and p['instrument'] == 'timsTOF' and nh4['adduct'] == '[M+NH4]+'   # input untouched\\n\",\n    \"    wl = dict(nh4, mz=[100.0, 281.4, 281.5, 283.0], it=[1.0, 0.5, 0.5, 0.9], prec=M + PROTON - H2O, adduct='[M-H2O+H]+')\\n\",\n    '    p = n1_proxy(wl)\\n',\n    \"    assert p is not None and p['mz'] == [100.0, 281.4] and abs(p['prec'] - (M + PROTON)) < 1e-9, p\\n\",\n    \"    for ad in ('[M+Na]+', '[M+K]+', '[M-2H2O+H]+', '[M+H]+', '[M-H]-'):\\n\",\n    '        assert n1_proxy(dict(nh4, adduct=ad)) is None, ad\\n',\n    \"    assert n1_proxy(dict(nh4, mode='negative')) is None\\n\",\n    '    assert n1_proxy(dict(nh4, mz=[310.0, 315.0], it=[1.0, 1.0])) is None             # nothing below the cut\\n',\n    '    assert n1_proxy(dict(nh4, mz=[100.0], it=[0.0])) is None                           # no positive intensity\\n',\n    \"    h = dict(nh4, adduct='[M+H]+', prec=M + PROTON, mz=[100.0, 299.5, 299.6, 301.0], it=[1.0, 1.0, 1.0, 1.0])\\n\",\n    '    c = h_cut(h)\\n',\n    \"    assert c is not None and c['mz'] == [100.0, 299.5] and c['prec'] == h['prec'] and c['adduct'] == '[M+H]+', c\\n\",\n    '    assert h_cut(nh4) is None\\n',\n    \"    items = [dict(mid='np:X', cands=['A', 'B'], spectra=[h, nh4, wl, dict(h, mode='negative', adduct='[M-H]-'),\\n\",\n    \"                                                        dict(h, adduct='[M+Na]+'), dict(h, adduct='[M+K]+')])]\\n\",\n    '    v, st = build_variants(items)\\n',\n    \"    assert [s['sid'] for s in v[0]['spectra']] == ['0:H', '0:Hcut', '1:NH4p', '2:WLp', '4:Na'], v\\n\",\n    '    assert st == dict(H=1, Hcut=1, NH4p=1, WLp=1, Na=1), st\\n',\n    '    # random tie-break expected RR\\n',\n    '    assert abs(rr_random_tb([0.5, 0.9, 0.5, 0.1]) - (1 / 2 + 1 / 3) / 2) < 1e-12\\n',\n    '    assert rr_random_tb([None, 0.2, 0.1]) is None and rr_random_tb([0.2, None, 0.1]) == 1.0\\n',\n    '    assert abs(rr_random_tb([0.0, 0.0, 0.0, 0.0]) - rr_rand(4)) < 1e-12\\n',\n    '    assert rr_truth_first([0.0, 0.0, 0.0, 0.0]) == 1.0 and abs(rr_split([0.0, 0.0, 0.0]) - 0.5) < 1e-12\\n',\n    '    assert abs(rr_rand(3) - (1 + 0.5 + 1 / 3) / 3) < 1e-12\\n',\n    '    assert covered([0.1, None, 0.2]) and not covered([0.1, None]) and not covered([None, 0.1, 0.2])\\n',\n    \"    # aggregation: mean over the arm's spectra, None-aware; N1A tag resolution\\n\",\n    \"    mo = {'cands': ['A', 'B', 'C'], 'spec': {\\n\",\n    \"        '0:H': dict(tag='H', nb=3, mean=[0.2, 0.4, None], merged=[0.3, 0.1, None]),\\n\",\n    \"        '1:H': dict(tag='H', nb=1, mean=[0.4, None, None], merged=[0.4, None, None]),\\n\",\n    \"        '2:NH4p': dict(tag='NH4p', nb=1, mean=[0.9, 0.1, 0.5], merged=[0.9, 0.1, 0.5])}}\\n\",\n    \"    am = arm_scores(mo, ('H',), 'mean')\\n\",\n    '    assert len(am) == 3 and np.allclose(am[:2], [0.3, 0.4]) and am[2] is None, am\\n',\n    \"    assert np.allclose(arm_scores(mo, ('H',), 'merged')[:2], [0.35, 0.1])\\n\",\n    \"    assert arm_tags('N1A', dict(has_H=True)) == (('H',), 'mean')\\n\",\n    \"    assert arm_tags('N1A', dict(has_H=False)) == (('NH4p', 'WLp'), 'mean')\\n\",\n    \"    assert arm_tags('N1A_D', dict(has_H=False)) == (('NH4p', 'WLp'), 'merged')\\n\",\n    '    # end-to-end on two synthetic molecules\\n',\n    \"    its = [dict(mid='np:X', cands=['A', 'B', 'C'], spectra=[dict(tag='H'), dict(tag='NH4p')]),\\n\",\n    \"           dict(mid='np:Y', cands=['A', 'B', 'C'], spectra=[dict(tag='NH4p')]),\\n\",\n    \"           dict(mid='e180:Z', cands=['A', 'B'], spectra=[dict(tag='H')])]\\n\",\n    \"    out = {'np:X': mo, 'np:Y': {'cands': ['A', 'B', 'C'], 'spec': {'0:NH4p': dict(tag='NH4p', nb=1, mean=[0.1, 0.5, 0.0], merged=[0.1, 0.5, 0.0])}}}\\n\",\n    \"    rows = molecule_table(its, out, done={'np:X', 'np:Y'})\\n\",\n    \"    assert [r['mid'] for r in rows] == ['np:X', 'np:Y']\\n\",\n    '    x, y = rows\\n',\n    \"    assert abs(x['rr_A_H'] - 0.5) < 1e-12 and x['rr_B_NH4'] == 1.0 and x['rr_N1A'] == x['rr_A_H'] and x['has_merged_H']\\n\",\n    \"    assert y['rr_A_H'] is None and abs(y['rr_N1A'] - 0.5) < 1e-12 and not y['has_H']\\n\",\n    \"    p = paired(rows, 'A_H', 'B_NH4')\\n\",\n    \"    assert p['n'] == 1 and abs(p['delta'] - 0.5) < 1e-12\\n\",\n    \"    f = filled_delta(rows, 'A_H', 'N1A')\\n\",\n    \"    assert f['n'] == 2 and abs(f['delta'] - (0.5 - rr_rand(3)) / 2) < 1e-12\\n\",\n    '    g = gates(rows)                                          # A lift < 0 here (0.5 vs rand 0.611)\\n',\n    \"    assert g['N1_NH4']['verdict'].startswith('UNDEFINED') and g['N1_WL']['verdict'] == 'NO DATA', g['N1_NH4']['verdict']\\n\",\n    \"    assert g['I14b']['verdict'] == 'NO DATA'\\n\",\n    '\\n',\n    '    def row(s, a, b, d, mh=True):\\n',\n    \"        return dict(mid=f'{s}:{a}{b}{d}', set=s, rand=0.2, rr_A_H=a, rr_B_NH4=b, rr_A_Hcut=None, rr_C_WL=None,\\n\",\n    '                    rr_D_Hmerged=d, has_merged_H=mh)\\n',\n    \"    gk = gates([row('np', 1.0, 0.5, 1.0), row('np', 0.6, 0.4, 0.6), row('e180', 1.0, None, 1.0)])   # ratio 0.25/0.6\\n\",\n    \"    assert gk['N1_NH4']['verdict'] == 'KILL' and abs(gk['N1_NH4']['paired']['ratio'] - 0.25 / 0.6) < 1e-12\\n\",\n    \"    assert gk['I14b']['verdict'] == 'REJECT' and gk['I14b']['e180']['delta'] == 0.0      # +0.01 needed on e180\\n\",\n    \"    gp = gates([row('np', 1.0, 0.9, 1.0), row('np', 0.6, 0.6, 0.5), row('e180', 0.5, None, 1.0)])   # ratio 1.1/1.2\\n\",\n    \"    assert gp['N1_NH4']['verdict'] == 'PASS' and gp['I14b']['verdict'] == 'REJECT'          # np merged delta < 0\\n\",\n    \"    ga = gates([row('np', 0.5, 0.5, 0.5), row('e180', 0.5, None, 0.52)])\\n\",\n    \"    assert ga['I14b']['verdict'] == 'ADOPT' and abs(ga['I14b']['e180']['delta'] - 0.02) < 1e-12\\n\",\n    '    b = blind_report(rows)\\n',\n    \"    assert b['n_blind'] == 1 and b['n_blind_proxy_covered'] == 1\\n\",\n    \"    sc = score_corr({'m': {'a': 1.0, 'b': 2.0, 'c': 3.0}}, {'m': {'a': 2.0, 'b': 4.0, 'c': 6.5}})\\n\",\n    \"    assert sc['n_mols'] == 1 and sc['pearson'] > 0.99 and abs(sc['spearman_mean'] - 1.0) < 1e-12\\n\",\n    \"    return 'glb_core selftest ok (proxy NH4/WL/Hcut, variants, random tie-break RR, aggregation, paired, gates, blind)'\\n\",\n])\n# kdev_glbench_n1 glb_driver.py, verbatim (GLACIER driver: mean + merged-CE per spectrum, own process)\nN1_DRIVER_SRC = ''.join([\n    '#!/usr/bin/env python\\n',\n    '\"\"\"casmi26 kdev_glbench_n1 -- GLACIER bench driver (runs in its OWN process, like gl_runner.py).\\n',\n    '\\n',\n    'Imports the attached casmi26-glacier gl_runner.py as a module and reuses ITS environment setup (offline wheel site,\\n',\n    'shims, RDKit 2025.03 check), model loading, batched Predictor (canonical-SMILES dedupe, OOM split, fail-fast), query\\n',\n    \"cleaning, CE bucketing, summed-CE merge() and entropy similarity. The 'mean' mode calls gl_runner.score_molecule on\\n\",\n    \"one spectrum at a time, so a molecule's mean over its [M+H]+ spectra is exactly the production gl_ex_mean.\\n\",\n    \"Added: per-spectrum output, and the I14b 'merged' mode (a merged-CE eV spectrum = >= 2 distinct CE buckets is\\n\",\n    'predicted at each bucket and the predictions are merge()d like the SUM_CE path; single-CE / NCE / no-CE spectra:\\n',\n    'merged = mean).\\n',\n    '\\n',\n    'usage: python glb_driver.py <gl_pkg> <in.json> <out.json> [--device cuda] [--budget S] [--site DIR] [--wheels DIR]\\n',\n    '                            [--bs 32] [--chunk-jobs 3000] [--fake]\\n',\n    'in.json : [{\"mid\", \"cands\": [smiles], \"spectra\": [{\"sid\", \"tag\", \"mz\", \"it\", \"prec\", \"adduct\", \"mode\", \"instrument\",\\n',\n    '            \"ce_ev\", \"ce_orig\", \"ce_units\"}]}]\\n',\n    'out.json: {mid: {\"cands\": [smiles], \"spec\": {sid: {\"tag\", \"nb\", \"mean\": [sim|null], \"merged\": [sim|null]}}}}\\n',\n    '          (lists aligned with \"cands\"; null = rejected SMILES / failed prediction)\\n',\n    'out.json.meta.json: status, counts, timings, versions, errors, \"done\" = molecules fully scored (input order).\\n',\n    '--fake: no torch / model / wheel site (local dry run): deterministic pseudo-random predictions, SMILES accepted as is.\\n',\n    '\"\"\"\\n',\n    'import os, sys, json, time, math, argparse, traceback, hashlib, importlib.util\\n',\n    '\\n',\n    'sys.dont_write_bytecode = True\\n',\n    \"NEED = ('setup_env', 'load_model', 'Predictor', 'smiles_ok', '_CANON', 'molecule_jobs', 'score_molecule',\\n\",\n    \"        'prep_spectrum', 'prep_query', 'entropy_sim', 'merge', 'post', 'ce_bucket', 'spectrum_ce', 'is_covered',\\n\",\n    \"        'instr_token', 'SUM_CE', 'FLOOR', 'TOPK', 'POWER', 'ENTW', 'TOL', 'PPM', 'write_json', 'Log')\\n\",\n    '\\n',\n    '\\n',\n    'def sha256(path):\\n',\n    '    h = hashlib.sha256()\\n',\n    \"    with open(path, 'rb') as f:\\n\",\n    \"        for b in iter(lambda: f.read(1 << 20), b''):\\n\",\n    '            h.update(b)\\n',\n    '    return h.hexdigest()\\n',\n    '\\n',\n    '\\n',\n    'def load_gl(pkg):\\n',\n    '    \"\"\"gl_runner.py of the package as a module (no side effects beyond its imports: numpy, numba).\"\"\"\\n',\n    \"    spec = importlib.util.spec_from_file_location('gl_runner', os.path.join(pkg, 'gl_runner.py'))\\n\",\n    '    G = importlib.util.module_from_spec(spec)\\n',\n    \"    sys.modules['gl_runner'] = G\\n\",\n    '    spec.loader.exec_module(G)\\n',\n    '    miss = [n for n in NEED if not hasattr(G, n)]\\n',\n    '    if miss:\\n',\n    \"        raise RuntimeError(f'gl_runner API differs from the one this driver was written for: missing {miss}')\\n\",\n    '    return G\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------- per-spectrum plan ----\\n',\n    'def merged_buckets(G, sp):\\n',\n    '    \"\"\"I14b: sorted distinct CE buckets of a merged-CE eV spectrum when there are >= 2, else None.\"\"\"\\n',\n    \"    if sp.get('ce_units') == 'NCE':\\n\",\n    '        return None\\n',\n    '    b = set()\\n',\n    \"    for x in sp.get('ce_ev') or []:\\n\",\n    '        try:\\n',\n    '            x = float(x)\\n',\n    '        except (TypeError, ValueError):\\n',\n    '            continue\\n',\n    '        if math.isfinite(x):\\n',\n    '            cb = G.ce_bucket(x)\\n',\n    '            if math.isfinite(cb):\\n',\n    '                b.add(float(cb))\\n',\n    '    return sorted(b) if len(b) >= 2 else None\\n',\n    '\\n',\n    '\\n',\n    'def spec_plan(G, sp):\\n',\n    '    \"\"\"None if gl_runner would not score this spectrum, else {\\'entry\\': molecule_plan entry, \\'mb\\': buckets|None}.\"\"\"\\n',\n    '    if not G.is_covered(sp):\\n',\n    '        return None\\n',\n    '    qm, qp = G.prep_spectrum(sp)\\n',\n    \"    return dict(entry=(qm, qp, sp['adduct'], G.instr_token(sp), G.ce_bucket(G.spectrum_ce(sp))), mb=merged_buckets(G, sp))\\n\",\n    '\\n',\n    '\\n',\n    'def spec_jobs(G, plan, cands):\\n',\n    \"    J = set(G.molecule_jobs([plan['entry']], cands))\\n\",\n    \"    if plan['mb']:\\n\",\n    \"        _, _, ad, ins, _ = plan['entry']\\n\",\n    '        for c in cands:\\n',\n    \"            for b in plan['mb']:\\n\",\n    '                J.add((c, ad, ins, b))\\n',\n    '    return J\\n',\n    '\\n',\n    '\\n',\n    'def spec_scores(G, plan, cands, cache):\\n',\n    '    \"\"\"-> (mean {c: sim|None}, merged {c: sim|None}, n_buckets).\"\"\"\\n',\n    \"    mean = G.score_molecule([plan['entry']], cands, cache)\\n\",\n    \"    if not plan['mb']:\\n\",\n    '        return mean, dict(mean), 1\\n',\n    \"    qm, qp, ad, ins, _ = plan['entry']\\n\",\n    '    merged = {}\\n',\n    '    for c in cands:\\n',\n    \"        parts = [cache.get((c, ad, ins, b)) for b in plan['mb']]\\n\",\n    '        if any(p is None for p in parts):            # = the SUM_CE path: every component must be predicted\\n',\n    '            merged[c] = None\\n',\n    '            continue\\n',\n    '        pr = G.merge(parts)\\n',\n    '        pm, pp = G.prep_query(pr[0], pr[1], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\\n',\n    '        s = float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))\\n',\n    '        merged[c] = s if math.isfinite(s) else None\\n',\n    \"    return mean, merged, len(plan['mb'])\\n\",\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------ fake mode ------\\n',\n    'class FakePredictor:\\n',\n    '    \"\"\"Same interface as gl_runner.Predictor (predict(jobs, deadline) -> bool, .cache); deterministic random peaks.\"\"\"\\n',\n    '\\n',\n    '    def __init__(self, G):\\n',\n    '        self.G, self.cache = G, {}\\n',\n    '        self.n_pred = self.n_fail = self.n_feat_fail = 0\\n',\n    '        self.t_pred, self.bs = 0.0, 0\\n',\n    '\\n',\n    '    def predict(self, jobs, deadline):\\n',\n    '        import numpy as np\\n',\n    '        t0 = time.time()\\n',\n    '        for j in sorted(jobs):\\n',\n    '            if j in self.cache:\\n',\n    '                continue\\n',\n    '            if time.time() > deadline:\\n',\n    '                self.t_pred += time.time() - t0\\n',\n    '                return False\\n',\n    '            c = self.G._CANON.get(j[0], (None,))[0]\\n',\n    '            if c is None:\\n',\n    '                self.cache[j] = None\\n',\n    '                continue\\n',\n    '            r = np.random.default_rng(int(hashlib.md5(repr((c, j[1], j[2], float(j[3]))).encode()).hexdigest()[:8], 16))\\n',\n    '            mz = np.round(r.uniform(40.0, 400.0, 25), 2)\\n',\n    '            o = self.G.post(np.stack([mz, r.uniform(0.05, 1.0, 25)], 1).astype(np.float32))\\n',\n    '            self.cache[j] = (o[0].astype(np.float64), o[1].astype(np.float64))\\n',\n    '            self.n_pred += 1\\n',\n    '        self.t_pred += time.time() - t0\\n',\n    '        return True\\n',\n    '\\n',\n    '\\n',\n    'def fake_smiles_ok(G):\\n',\n    '    def ok(smi):\\n',\n    '        good = isinstance(smi, str) and bool(smi)\\n',\n    \"        G._CANON[smi] = (smi if good else None, max(1, len(smi) // 2) if good else 0, '' if good else 'empty')\\n\",\n    \"        return good, ('' if good else 'empty')\\n\",\n    '    return ok\\n',\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------- self-test -----\\n',\n    'def selftest(G):\\n',\n    '    \"\"\"Forced unit test of the merged-CE logic on a synthetic cache (no model); run by the notebook before the GPU\\n',\n    '    job and by the local build.\"\"\"\\n',\n    '    import numpy as np\\n',\n    \"    sp = dict(mz=[50.0, 80.0, 120.0, 150.5], it=[0.2, 1.0, 0.5, 0.3], prec=151.0, adduct='[M+H]+', mode='positive',\\n\",\n    \"              instrument='timsTOF', ce_ev=[20.0, 40.0, 60.0], ce_orig='20,40,60', ce_units='eV')\\n\",\n    '    p = spec_plan(G, sp)\\n',\n    \"    assert p is not None and p['mb'] == [20.0, 40.0, 60.0], p\\n\",\n    \"    assert p['entry'][2:] == ('[M+H]+', 'QTOF', 40.0), p['entry'][2:]\\n\",\n    \"    cands = ['CCO', 'CCC']\\n\",\n    '    J = spec_jobs(G, p, cands)\\n',\n    \"    assert J == {(c, '[M+H]+', 'QTOF', b) for c in cands for b in (20.0, 40.0, 60.0)}, J\\n\",\n    '    rng = np.random.default_rng(0)\\n',\n    '    cache = {}\\n',\n    '    for j in sorted(J):\\n',\n    '        mz = np.sort(rng.choice([50.0, 80.0, 120.0, 150.5, 60.0, 99.0], 4, replace=False))\\n',\n    '        cache[j] = (mz.astype(np.float64), rng.uniform(0.1, 1.0, 4).astype(np.float64))\\n',\n    \"    cache[('CCC', '[M+H]+', 'QTOF', 60.0)] = None\\n\",\n    '    mean, merged, nb = spec_scores(G, p, cands, cache)\\n',\n    \"    assert nb == 3 and merged['CCC'] is None and mean['CCC'] is not None and mean['CCO'] is not None, (mean, merged)\\n\",\n    \"    pr = G.merge([cache[('CCO', '[M+H]+', 'QTOF', b)] for b in (20.0, 40.0, 60.0)])\\n\",\n    '    qm, qp = G.prep_spectrum(sp)\\n',\n    '    pm, pp = G.prep_query(pr[0], pr[1], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\\n',\n    \"    assert abs(merged['CCO'] - float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))) < 1e-12\\n\",\n    \"    pm, pp = G.prep_query(*cache[('CCO', '[M+H]+', 'QTOF', 40.0)], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\\n\",\n    \"    assert abs(mean['CCO'] - float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))) < 1e-12\\n\",\n    '    p1 = spec_plan(G, dict(sp, ce_ev=[40.0]))                         # single CE: merged == mean\\n',\n    '    m1, g1, n1 = spec_scores(G, p1, cands, cache)\\n',\n    \"    assert p1['mb'] is None and n1 == 1 and m1 == g1 and abs(m1['CCO'] - mean['CCO']) < 1e-12\\n\",\n    '    assert merged_buckets(G, dict(sp, ce_ev=[20.0, 21.0])) is None    # same bucket twice\\n',\n    '    assert merged_buckets(G, dict(sp, ce_ev=[0.0, 20.0])) == [5.0, 20.0]\\n',\n    \"    assert merged_buckets(G, dict(sp, ce_units='NCE', ce_orig='20,40')) is None\\n\",\n    '    assert merged_buckets(G, dict(sp, ce_ev=[])) is None\\n',\n    \"    assert spec_plan(G, dict(sp, adduct='[M+NH4]+')) is None and spec_plan(G, dict(sp, mode='negative')) is None\\n\",\n    '    pn = spec_plan(G, dict(sp, ce_ev=[]))                              # no CE: SUM_CE jobs, merged == mean\\n',\n    \"    assert pn['mb'] is None and spec_jobs(G, pn, ['CCO']) == {('CCO', '[M+H]+', 'QTOF', c) for c in G.SUM_CE}\\n\",\n    \"    return 'glb_driver selftest ok (merged-CE buckets, jobs, merge == SUM_CE path, single-CE identity, coverage)'\\n\",\n    '\\n',\n    '\\n',\n    '# ------------------------------------------------------------------------------------------------------ main -----\\n',\n    'def parse(argv=None):\\n',\n    '    ap = argparse.ArgumentParser()\\n',\n    \"    ap.add_argument('pkg'); ap.add_argument('inp'); ap.add_argument('out')\\n\",\n    \"    ap.add_argument('--device', default='cuda')\\n\",\n    \"    ap.add_argument('--budget', type=float, default=0.0)\\n\",\n    \"    ap.add_argument('--site', default='/tmp/glb_site')\\n\",\n    \"    ap.add_argument('--wheels', default=None)\\n\",\n    \"    ap.add_argument('--bs', type=int, default=32)\\n\",\n    \"    ap.add_argument('--chunk-jobs', type=int, default=3000)\\n\",\n    \"    ap.add_argument('--fake', action='store_true')\\n\",\n    '    return ap.parse_args(argv)\\n',\n    '\\n',\n    '\\n',\n    'def main(argv=None):\\n',\n    '    a = parse(argv)\\n',\n    '    if not a.fake:\\n',\n    \"        os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # = gl_runner.main (deterministic torch)\\n\",\n    '    t_start = time.time()\\n',\n    \"    deadline = t_start + a.budget if a.budget > 0 else float('inf')\\n\",\n    \"    meta = dict(model='glacier-bench', status='init', fake=a.fake, n_mols=0, n_mols_done=0, n_spec=0, n_spec_covered=0,\\n\",\n    '                n_spec_merged=0, n_jobs=0, errors=[], done=[])\\n',\n    '    G = None\\n',\n    '    result = {}\\n',\n    '\\n',\n    '    def write(status=None):\\n',\n    '        if status:\\n',\n    \"            meta['status'] = status\\n\",\n    \"        meta['seconds'] = round(time.time() - t_start, 2)\\n\",\n    \"        tmp = a.out + '.tmp'\\n\",\n    \"        with open(tmp, 'w') as f:\\n\",\n    '            json.dump(result, f, allow_nan=False)\\n',\n    '        os.replace(tmp, a.out)\\n',\n    \"        with open(a.out + '.meta.json.tmp', 'w') as f:          # atomic too: a kill mid-write never leaves bad JSON\\n\",\n    '            json.dump(meta, f, allow_nan=False)\\n',\n    \"        os.replace(a.out + '.meta.json.tmp', a.out + '.meta.json')\\n\",\n    '\\n',\n    '    def log(*x):\\n',\n    \"        print(f'[glb {time.time() - t_start:7.1f}s]', *x, file=sys.stderr, flush=True)\\n\",\n    '\\n',\n    '    try:\\n',\n    '        mols = json.load(open(a.inp))\\n',\n    \"        meta['n_mols'] = len(mols)\\n\",\n    '        write()\\n',\n    '        G = load_gl(a.pkg)\\n',\n    \"        meta['gl_runner_sha256'] = sha256(os.path.join(a.pkg, 'gl_runner.py'))\\n\",\n    '        if a.fake:\\n',\n    \"            meta['rdkit'] = 'fake'\\n\",\n    '            G.smiles_ok = fake_smiles_ok(G)\\n',\n    '            pred = FakePredictor(G)\\n',\n    \"            log('FAKE mode: pseudo-random predictions, no model')\\n\",\n    '        else:\\n',\n    '            ns = argparse.Namespace(pkg=a.pkg, site=a.site, wheels=a.wheels, any_rdkit=False)\\n',\n    \"            meta['rdkit'] = G.setup_env(ns, log)\\n\",\n    '            import torch\\n',\n    \"            meta['torch'] = torch.__version__\\n\",\n    '            dev = a.device\\n',\n    \"            if dev.startswith('cuda') and not torch.cuda.is_available():\\n\",\n    \"                log('cuda unavailable -> cpu'); dev = 'cpu'\\n\",\n    \"            meta['device'] = dev\\n\",\n    '            torch.use_deterministic_algorithms(True, warn_only=True)\\n',\n    '            torch.backends.cudnn.benchmark = False\\n',\n    '            model = G.load_model(os.path.abspath(a.pkg), dev, log)\\n',\n    '            pred = G.Predictor(model, dev, a.bs, log)\\n',\n    '            log(f\\'model loaded on {dev} (rdkit {meta[\"rdkit\"]}, torch {torch.__version__}, numba {G.HAVE_NUMBA})\\')\\n',\n    \"        meta['t_load'] = round(time.time() - t_start, 2)\\n\",\n    '        okc = {}\\n',\n    '        chunk, chunk_jobs = [], set()\\n',\n    '\\n',\n    '        def flush():\\n',\n    '            nonlocal chunk, chunk_jobs\\n',\n    '            if not chunk:\\n',\n    '                return True\\n',\n    '            t0 = time.time()\\n',\n    '            if not pred.predict(chunk_jobs, deadline):\\n',\n    '                return False\\n',\n    '            for mid, cands, plans in chunk:\\n',\n    '                spec = {}\\n',\n    '                for sid, tag, pl in plans:\\n',\n    '                    try:\\n',\n    '                        mean, merged, nb = spec_scores(G, pl, cands, pred.cache)\\n',\n    '                    except Exception as e:\\n',\n    \"                        meta['errors'].append(f'{mid} {sid}: score {repr(e)[:200]}')\\n\",\n    '                        continue\\n',\n    \"                    spec[sid] = dict(tag=tag, nb=nb, mean=[mean.get(c) for c in result[mid]['cands']],\\n\",\n    \"                                     merged=[merged.get(c) for c in result[mid]['cands']])\\n\",\n    \"                result[mid]['spec'] = spec\\n\",\n    \"                meta['done'].append(mid)\\n\",\n    \"            meta['n_mols_done'] = len(meta['done'])\\n\",\n    '            write()\\n',\n    \"            log(f'chunk: {len(chunk)} mols {len(chunk_jobs)} jobs {time.time() - t0:.1f}s '\\n\",\n    '                f\\'({meta[\"n_mols_done\"]}/{meta[\"n_mols\"]} mols done, {pred.n_pred} predictions)\\')\\n',\n    '            chunk, chunk_jobs = [], set()\\n',\n    '            return True\\n',\n    '\\n',\n    '        stopped = False\\n',\n    '        for mol in mols:\\n',\n    \"            mid = str(mol['mid'])\\n\",\n    \"            allc = list(dict.fromkeys(mol.get('cands') or []))\\n\",\n    '            result[mid] = dict(cands=allc, spec={})\\n',\n    '            for c in allc:\\n',\n    '                if c not in okc:\\n',\n    '                    okc[c], why = G.smiles_ok(c)\\n',\n    '                    if not okc[c] and sum(1 for v in okc.values() if not v) <= 5:\\n',\n    \"                        meta['errors'].append(f'smiles rejected ({why}): {str(c)[:100]}')\\n\",\n    '            cands = [c for c in allc if okc[c]]\\n',\n    '            plans = []\\n',\n    \"            for sp in mol.get('spectra') or []:\\n\",\n    \"                meta['n_spec'] += 1\\n\",\n    '                try:\\n',\n    '                    pl = spec_plan(G, sp)\\n',\n    '                except Exception as e:\\n',\n    '                    meta[\\'errors\\'].append(f\\'{mid} {sp.get(\"sid\")}: plan {repr(e)[:200]}\\')\\n',\n    '                    continue\\n',\n    '                if pl is None:\\n',\n    '                    continue\\n',\n    \"                meta['n_spec_covered'] += 1\\n\",\n    \"                meta['n_spec_merged'] += int(bool(pl['mb']))\\n\",\n    \"                plans.append((sp['sid'], sp.get('tag'), pl))\\n\",\n    '            if not cands or not plans:\\n',\n    \"                meta['done'].append(mid)                         # nothing GLACIER could score: done, empty\\n\",\n    '                continue\\n',\n    '            J = set()\\n',\n    '            for _, _, pl in plans:\\n',\n    '                J |= spec_jobs(G, pl, cands)\\n',\n    \"            meta['n_jobs'] += len(J)\\n\",\n    '            chunk.append((mid, cands, plans))\\n',\n    '            chunk_jobs |= J\\n',\n    '            if len(chunk_jobs) >= a.chunk_jobs and not flush():\\n',\n    '                stopped = True\\n',\n    '                break\\n',\n    '        if not stopped and not flush():\\n',\n    '            stopped = True\\n',\n    '        meta.update(n_pred=pred.n_pred, n_pred_fail=pred.n_fail, n_feat_fail=pred.n_feat_fail, t_pred=round(pred.t_pred, 2),\\n',\n    '                    pred_per_s=round(pred.n_pred / max(pred.t_pred, 1e-9), 2), bs_final=pred.bs,\\n',\n    \"                    n_bad_smiles=sum(1 for v in okc.values() if not v), n_mols_done=len(meta['done']))\\n\",\n    \"        if not a.fake and meta.get('device', '').startswith('cuda'):\\n\",\n    '            import torch\\n',\n    \"            meta['max_mem_gb'] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)\\n\",\n    \"        write('budget' if stopped else 'ok')\\n\",\n    \"        log('status', meta['status'], {k: meta[k] for k in ('n_mols', 'n_mols_done', 'n_spec_covered', 'n_spec_merged',\\n\",\n    \"                                                            'n_pred', 'n_pred_fail', 'pred_per_s')})\\n\",\n    '        return 0\\n',\n    '    except BaseException as e:                                   # never crash the notebook: status error + partial out\\n',\n    \"        meta['errors'].append(traceback.format_exc()[-3000:])\\n\",\n    \"        log('ERROR', repr(e)[:300])\\n\",\n    '        try:\\n',\n    \"            write('error')\\n\",\n    '        except Exception:\\n',\n    '            return 2\\n',\n    '        return 0\\n',\n    '\\n',\n    '\\n',\n    \"if __name__ == '__main__':\\n\",\n    '    sys.exit(main())\\n',\n])\n\n\ndef _n1_mod(name, path):\n    import importlib.util as _iu\n    spec = _iu.spec_from_file_location(name, path)\n    mod = _iu.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef n1_select(items_all, K):\n    \"\"\"N1 arm A (bench glb_core) on fuse.build_ice_input(..., covered_only=False) items -> (driver items, counts):\n    the molecules WITHOUT a real positive [M+H]+ spectrum that have >= 1 [M+NH4]+ / [M-H2O+H]+ proxy, carrying only\n    those proxies (bench tags NH4p / WLp, sid '<spectrum index>:<tag>'), in input (= GLACIER priority) order.\"\"\"\n    var, _ = K.build_variants(items_all)\n    raw = {str(it['mid']): sum(1 for s in it.get('spectra') or [] if s.get('mode') == 'positive'\n                               and s.get('adduct') in K.PROXY_TAG) for it in items_all}\n    out = []\n    st = dict(molecules=0, with_H=0, blind=0, blind_nh4_wl=0, n1=0, proxies=0, NH4p=0, WLp=0, proxy_empty=0)\n    for it in var:\n        fl = K.mol_flags(it)\n        st['molecules'] += 1\n        if fl['has_H']:                  # arm A: a real [M+H]+ exists -> the H1 run above scores this molecule\n            st['with_H'] += 1\n            continue\n        st['blind'] += 1\n        tags, _mode = K.arm_tags(N1_ARM, fl)\n        sps = [s for s in it['spectra'] if s.get('tag') in tags]\n        st['blind_nh4_wl'] += int(raw.get(it['mid'], 0) > 0)\n        st['proxy_empty'] += raw.get(it['mid'], 0) - len(sps)\n        if not sps or not it['cands']:\n            continue\n        st['n1'] += 1\n        st['proxies'] += len(sps)\n        for s in sps:\n            st[s['tag']] += 1\n        out.append(dict(mid=it['mid'], cands=list(it['cands']), spectra=sps))\n    return out, st\n\n\ndef n1_run(items, budget):\n    \"\"\"The bench glb_driver.py in its own process (= gl_fuse.run_gl pattern) -> (out, meta, done). Never raises for a\n    driver failure: the driver checkpoints after every chunk, so a budget stop / crash / kill keeps the molecules it\n    finished; unreadable output -> nothing done.\"\"\"\n    fin, fout = os.path.join(N1_WORK, 'n1_in.json'), os.path.join(N1_WORK, 'n1_out.json')\n    for f in (fout, fout + '.meta.json'):\n        if os.path.exists(f):\n            os.remove(f)\n    with open(fin, 'w') as f:\n        json.dump(items, f)\n    cmd = [sys.executable, os.path.join(N1_WORK, 'glb_driver.py'), GL_PKG, fin, fout, '--device', N1_DEVICE,\n           '--budget', str(float(budget)), '--site', N1_SITE, '--wheels', os.path.join(ICE_PKG, 'wheels')] + list(N1_DRIVER_EXTRA)\n    t0 = time.time()\n    try:\n        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace',\n                           timeout=float(budget) + 600)\n        tail = (r.stdout or '')[-3000:]\n    except subprocess.TimeoutExpired as e:\n        tail = f'TIMEOUT after {time.time() - t0:.0f}s ' + (e.stdout.decode(errors='replace') if isinstance(e.stdout, bytes)\n                                                            else str(e.stdout or ''))[-2000:]\n    print(tail, flush=True)\n    meta, out = {}, {}\n    try:\n        with open(fout + '.meta.json') as f:\n            meta = json.load(f)\n    except Exception as e:\n        print('[N1] driver meta unreadable:', repr(e), flush=True)\n    try:\n        with open(fout) as f:\n            out = json.load(f)\n    except Exception as e:\n        print('[N1] driver output unreadable:', repr(e), flush=True)\n    done = set(meta.get('done') or []) if meta else {m for m, v in out.items() if isinstance(v, dict) and v.get('spec')}\n    return out, meta, done\n\n\ndef n1_scores(out, done, items, K, mode=None):\n    \"\"\"Driver output -> {mid: {smiles: score | None}} = bench arm_scores of N1_ARM (mean over the molecule's proxy\n    spectra of the per-spectrum similarity in the arm's CE mode, or `mode`); finished molecules only, candidate order\n    as sent; non-finite / outside [0, 1] -> None; molecules without any score are left out.\"\"\"\n    tags, amode = K.arm_tags(N1_ARM, dict(has_H=False))\n    res = {}\n    for it in items:\n        mid = str(it['mid'])\n        mo = out.get(mid)\n        if mid not in done or not isinstance(mo, dict):\n            continue\n        cands = [str(c) for c in mo.get('cands') or []]\n        if cands != [str(c) for c in dict.fromkeys(it['cands'])]:\n            print('[N1] candidate order changed -> molecule dropped', mid, flush=True)\n            continue\n        d = {}\n        for c, x in zip(cands, K.arm_scores(mo, tags, mode or amode)):\n            d[c] = float(x) if x is not None and math.isfinite(x) and -1e-9 <= x <= 1.0 + 1e-9 else None\n        if any(x is not None for x in d.values()):\n            res[mid] = d\n    return res\n\n\ndef n1_synthetic_te(te_df, mids, K):\n    \"\"\"Commit-only forced exercise input: for every molecule in `mids` with a positive [M+H]+ spectrum a copy\n    'n1x:<mid>' WITHOUT any [M+H]+: each positive [M+H]+ row -> an [M+NH4]+ row (precursor + 18.033823 - 1.007276,\n    plus an injected peak AT the NH4 precursor that the proxy cut must drop); the first one also -> an [M-H2O+H]+ row\n    (precursor - 18.010565, peaks unchanged); original NH4 / water-loss rows are left out; every other row (negative,\n    Na, K, 2M, ...) is copied unchanged and must never be proxied.\n    -> (DataFrame, {xmid: [(kind, original [M+H]+ precursor, mz list, intensity list) per row]})\"\"\"\n    rows, ref = [], {}\n    want = {str(m) for m in mids}\n    for mid, sub in te_df.groupby(te_df.molecule_id.astype(str), sort=False):\n        if mid not in want:\n            continue\n        recs = sub.to_dict('records')\n        if not any(r.get('ionization_mode') == 'positive' and r.get('adduct') == '[M+H]+' for r in recs):\n            continue\n        xm, first, rf = f'n1x:{mid}', True, []\n        for r in recs:\n            r = dict(r, molecule_id=xm)\n            if r.get('ionization_mode') == 'positive' and r.get('adduct') == '[M+H]+':\n                prec = float(r['precursor_mz'])\n                mz = [float(x) for x in r['ms2_mzs']]\n                it = [float(x) for x in r['ms2_normalized_intensities']]\n                if first:\n                    rows.append(dict(r, adduct='[M-H2O+H]+', precursor_mz=prec - K.H2O, ms2_mzs=np.asarray(mz),\n                                     ms2_normalized_intensities=np.asarray(it)))\n                    rf.append(('WL', prec, mz, it))\n                    first = False\n                p2 = prec - K.PROTON + K.NH4_ADD\n                rows.append(dict(r, adduct='[M+NH4]+', precursor_mz=p2, ms2_mzs=np.asarray(mz + [p2]),\n                                 ms2_normalized_intensities=np.asarray(it + [1.0])))\n                rf.append(('NH4', prec, mz, it))\n            elif r.get('ionization_mode') == 'positive' and r.get('adduct') in K.PROXY_TAG:\n                continue                 # an original NH4 / water-loss row: left out, so every proxy traces to a converted row\n            else:\n                rows.append(r)\n                rf.append(('keep', None, None, None))\n        ref[xm] = rf\n    return pd.DataFrame(rows), ref\n\n\ndef n1_check_proxies(x_items, ref, K):\n    \"\"\"Commit asserts on the synthetic proxies: relabel, precursor back at the original [M+H]+ m/z (< 1e-6 Da), cut =\n    exactly the original peaks below the bench cut (injected NH4 precursor peak gone), one proxy per converted row.\"\"\"\n    n = 0\n    for it in x_items:\n        rf = ref[it['mid']]\n        for s in it['spectra']:\n            j, tag = s['sid'].split(':')\n            kind, prec0, mz0, it0 = rf[int(j)]\n            assert (tag, kind) in (('NH4p', 'NH4'), ('WLp', 'WL')), (it['mid'], s['sid'], kind)\n            assert s['adduct'] == '[M+H]+' and s['mode'] == 'positive' and s['n1_src'] == (\n                '[M+NH4]+' if kind == 'NH4' else '[M-H2O+H]+'), (it['mid'], s['sid'])\n            assert abs(float(s['prec']) - prec0) < 1e-6, (it['mid'], s['sid'], s['prec'], prec0)\n            if kind == 'NH4':\n                p2 = prec0 - K.PROTON + K.NH4_ADD\n                cut = (p2 - K.NH4_ADD) + K.PROTON - K.CUT_DA\n            else:\n                cut = (prec0 - K.H2O) - K.CUT_DA\n            keep = [(m, i) for m, i in zip(mz0, it0) if m < cut]\n            assert s['mz'] == [m for m, _ in keep] and s['it'] == [i for _, i in keep], (it['mid'], s['sid'])\n            assert max(s['mz']) < float(s['prec']) - K.CUT_DA + 1e-6, (it['mid'], s['sid'])\n            n += 1\n        assert {s['tag'] for s in it['spectra']} <= {'NH4p', 'WLp'}, it['mid']\n    return n\n\n\ndef _n1_spearman(a, b):\n    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)\n    ra, rb = np.argsort(np.argsort(a)), np.argsort(np.argsort(b))\n    return float(np.corrcoef(ra, rb)[0, 1]) if ra.std() > 0 and rb.std() > 0 else float('nan')\n\n\ndef n1_ice_finite(ice):\n    \"\"\"Number of finite ICEBERG scores in one molecule's ICE dict (None / NaN / inf / non-numeric = no evidence; the\n    per-value test of fuse.rerank and of E006's _finite_coverage).\"\"\"\n    import math as _m\n    n = 0\n    for v in (ice.values() if isinstance(ice, dict) else ()):\n        try:\n            n += v is not None and _m.isfinite(float(v))\n        except (TypeError, ValueError):\n            pass\n    return n\n\n\ndef n1_complement(scores, ice_scores):\n    \"\"\"k2 variant A: drop the N1 GL scores of the molecules whose ICE dict (ice_scores[str(mid)]) holds a finite\n    score -> (kept {mid: same score dict}, dropped {mid}). Kept dicts are passed through unchanged.\"\"\"\n    ice_scores = ice_scores if isinstance(ice_scores, dict) else {}\n    drop = {m for m in scores if n1_ice_finite(ice_scores.get(str(m))) > 0}\n    return {m: v for m, v in scores.items() if m not in drop}, drop\n\n\ntry:\n    import math\n    _n1_h1_gl = isinstance(GL_SCORES, dict) and any(isinstance(g, dict) and any(v is not None for v in g.values())\n                                                  for g in GL_SCORES.values())\n    if not _n1_h1_gl:\n        raise RuntimeError('H1 GLACIER produced no scores (GL failed / not run) -> N1 not attempted')\n    os.makedirs(N1_WORK, exist_ok=True)\n    for _n1_fn, _n1_s, _n1_h in (('glb_core.py', N1_CORE_SRC, N1_CORE_SHA256), ('glb_driver.py', N1_DRIVER_SRC, N1_DRIVER_SHA256)):\n        assert hashlib.sha256(_n1_s.encode('utf-8')).hexdigest() == _n1_h, ('embedded bench source changed', _n1_fn)\n        with open(os.path.join(N1_WORK, _n1_fn), 'w', encoding='utf-8') as _n1_f:\n            _n1_f.write(_n1_s)\n    N1K = _n1_mod('glb_core', os.path.join(N1_WORK, 'glb_core.py'))\n    N1DR = _n1_mod('glb_driver', os.path.join(N1_WORK, 'glb_driver.py'))\n    print('[N1]', N1K.selftest(), flush=True)\n    _n1_G = N1DR.load_gl(GL_PKG)            # attached gl_runner.py as a module (numpy / numba only; no torch here)\n    print('[N1]', N1DR.selftest(_n1_G), flush=True)\n    _n1_sg = sha256(os.path.join(GL_PKG, 'gl_runner.py'))\n    print(f'[N1] code: glb_core {N1_CORE_SHA256[:12]} glb_driver {N1_DRIVER_SHA256[:12]} (= kdev_glbench_n1) | '\n          f'gl_runner.py {_n1_sg[:12]} matches the B1/bench MANIFEST: {_n1_sg == N1_GL_RUNNER_SHA256} | arm {N1_ARM} '\n          f'(tags, mode) {N1K.arm_tags(N1_ARM, dict(has_H=False))} | budget {N1_BUDGET:.0f}s', flush=True)\n    _n1_items_all = ice_fuse.build_ice_input(te, cands, covered_only=False)\n    _n1_items, N1_STATS = n1_select(_n1_items_all, N1K)\n    print(f\"[N1] molecules: GL/ICE input {N1_STATS['molecules']} | with real [M+H]+ {N1_STATS['with_H']} (H1 run, \"\n          f\"unchanged) | blind {N1_STATS['blind']} (with NH4/WL spectra {N1_STATS['blind_nh4_wl']}) | N1 molecules \"\n          f\"{N1_STATS['n1']} | proxies built {N1_STATS['proxies']} (NH4p {N1_STATS['NH4p']}, WLp {N1_STATS['WLp']}; \"\n          f\"empty after the cut {N1_STATS['proxy_empty']})\", flush=True)\n    _n1_x_items, _n1_x_ref = [], {}\n    if not IS_RERUN:                     # forced commit exercise: synthetic blind copies through the same code\n        _n1_tx, _n1_x_ref = n1_synthetic_te(te, list(cands), N1K)\n        _n1_x_all = ice_fuse.build_ice_input(_n1_tx, {f'n1x:{m}': c for m, c in cands.items() if f'n1x:{m}' in _n1_x_ref},\n                                          covered_only=False)\n        _n1_x_items, N1X_STATS = n1_select(_n1_x_all, N1K)\n        N1X_STATS['proxy_checked'] = n1_check_proxies(_n1_x_items, _n1_x_ref, N1K)\n        assert N1X_STATS['blind'] == N1X_STATS['molecules'] == len(_n1_x_ref) >= 1 and N1X_STATS['with_H'] == 0, N1X_STATS\n        assert N1X_STATS['n1'] >= 1 and N1X_STATS['proxy_checked'] == N1X_STATS['proxies'] >= N1X_STATS['n1'], N1X_STATS\n        assert N1X_STATS['proxies'] + N1X_STATS['proxy_empty'] == sum(k != 'keep' for v in _n1_x_ref.values() for k, *_ in v)\n        # arm-B guard: a molecule WITH a real [M+H]+ plus NH4 / water-loss spectra must NOT become an N1 molecule\n        _n1_src = sorted(_n1_x_ref)[0].split(':', 1)[1]\n        _n1_gdf = pd.concat([te[te.molecule_id.astype(str) == _n1_src], _n1_tx[_n1_tx.molecule_id == f'n1x:{_n1_src}']],\n                            ignore_index=True).assign(molecule_id=f'n1b:{_n1_src}')\n        _n1_gi, _n1_gs = n1_select(ice_fuse.build_ice_input(_n1_gdf, {f'n1b:{_n1_src}': {str(m): c for m, c in cands.items()}[_n1_src]},\n                                                           covered_only=False), N1K)\n        assert _n1_gi == [] and _n1_gs['with_H'] == 1 and _n1_gs['n1'] == 0, ('arm-B guard: [M+H]+ molecule selected', _n1_gs)\n        N1X_STATS['armB_guard'] = 'ok'\n        N1X_MIDS = {it['mid'] for it in _n1_x_items}\n    _n1_run = _n1_items + _n1_x_items\n    _n1_exp_merged = sum(bool(N1DR.merged_buckets(_n1_G, s)) for it in _n1_run for s in it['spectra'])\n    _n1_out, N1_DRV, _n1_done = n1_run(_n1_run, N1_BUDGET) if _n1_run else ({}, {'status': 'nothing to score'}, set())\n    print(f\"[N1] driver: status {N1_DRV.get('status')} | molecules sent {len(_n1_run)} (real {len(_n1_items)}, synthetic \"\n          f\"{len(_n1_x_items)}) done {len(_n1_done)} | spectra covered {N1_DRV.get('n_spec_covered')} (merged-CE \"\n          f\"{N1_DRV.get('n_spec_merged')}, expected {_n1_exp_merged}) | predictions {N1_DRV.get('n_pred')} (fail \"\n          f\"{N1_DRV.get('n_pred_fail')}) | {N1_DRV.get('seconds')}s | errors {(N1_DRV.get('errors') or [])[:2]}\", flush=True)\n    _n1_sc = n1_scores(_n1_out, _n1_done, _n1_run, N1K)\n    _n1_real = {m: v for m, v in _n1_sc.items() if not m.startswith('n1x:')}\n    _n1_clash = sorted(m for m in _n1_real if any(x is not None for x in (GL_SCORES.get(m) or {}).values()))\n    if not IS_RERUN:\n        assert not _n1_clash, ('N1 molecule already scored by the H1 run', _n1_clash[:5])\n        N1X_GL = {m: v for m, v in _n1_sc.items() if m.startswith('n1x:')}\n        assert N1_DRV.get('status') == 'ok' and N1X_MIDS <= _n1_done, (N1_DRV.get('status'), sorted(N1X_MIDS - _n1_done)[:5])\n        assert len(N1X_GL) >= 1 and N1_DRV.get('n_spec_merged') == _n1_exp_merged, (len(N1X_GL), N1_DRV.get('n_spec_merged'))\n        _n1_nb1 = _n1_nbm = 0\n        for _n1_m in N1X_MIDS:              # single-CE spectra: merged-CE mode == mean (production gl_runner) exactly;\n            for _n1_sp in (_n1_out[_n1_m].get('spec') or {}).values():     # merged-CE ones: the I14b path acts\n                if _n1_sp.get('nb') == 1:\n                    assert _n1_sp['mean'] == _n1_sp['merged'], _n1_m\n                    _n1_nb1 += 1\n                else:\n                    _n1_nbm += int(_n1_sp['mean'] != _n1_sp['merged'])\n        # the dummy commit molecules carry 20,40,60 eV [M+H]+ spectra -> merged-CE proxies (8 in the local dry run): the\n        # per-bucket prediction + merge() must run on them with the real model and change their scores vs mean CE\n        assert _n1_exp_merged >= 1 and _n1_nbm >= 1, ('merged-CE path not exercised on the commit', _n1_exp_merged, _n1_nbm)\n        _n1_rho = []                        # proxies (= the [M+H]+ peaks, cut) vs the real H1 GL scores of the source\n        _n1_mean = n1_scores(_n1_out, _n1_done, _n1_x_items, N1K, mode='mean')\n        for _n1_m, _n1_d in _n1_mean.items():\n            _n1_h = GL_SCORES.get(_n1_m.split(':', 1)[1]) or {}\n            _n1_p = [(v, _n1_h.get(c)) for c, v in _n1_d.items() if v is not None and _n1_h.get(c) is not None]\n            if len(_n1_p) >= 4:\n                _n1_r = _n1_spearman([a for a, _ in _n1_p], [b for _, b in _n1_p])\n                if math.isfinite(_n1_r):\n                    _n1_rho.append(_n1_r)\n        N1X_STATS.update(gl_scored=len(N1X_GL), single_ce_identical=_n1_nb1, merged_changed=_n1_nbm, rho_n=len(_n1_rho),\n                         rho_median=float(np.median(_n1_rho)) if _n1_rho else float('nan'))\n        print(f\"[N1X] synthetic exercise: molecules {N1X_STATS['molecules']} | N1 molecules {N1X_STATS['n1']} | proxies \"\n              f\"built {N1X_STATS['proxies']} (NH4p {N1X_STATS['NH4p']}, WLp {N1X_STATS['WLp']}; empty \"\n              f\"{N1X_STATS['proxy_empty']}) | proxy checks OK {N1X_STATS['proxy_checked']} | arm-B guard \"\n              f\"{N1X_STATS['armB_guard']} | merged-CE spectra \"\n              f\"{N1_DRV.get('n_spec_merged')} (expected {_n1_exp_merged}; merged != mean CE {_n1_nbm}) | single-CE \"\n              f\"merged==mean {_n1_nb1} | GL-scored \"\n              f\"synthetic molecules {len(N1X_GL)} | proxy (mean CE) vs H1 GL Spearman median \"\n              f\"{N1X_STATS['rho_median']:.3f} over {len(_n1_rho)}\", flush=True)\n        assert not _n1_rho or N1X_STATS['rho_median'] >= 0.3, ('proxy scores do not track the H1 GL scores', _n1_rho)\n    else:\n        _n1_real = {m: v for m, v in _n1_real.items() if m not in _n1_clash}\n    if N1_COMPLEMENT:                    # k2 variant A: E006's ICE fallback keeps the blind molecules ICEBERG covers\n        _n1_n0 = len(_n1_real)\n        _n1_real, N1_ICE_DROPPED = n1_complement(_n1_real, ICE_SCORES)\n        if not IS_RERUN:                 # forced commit exercise of the filter on THIS commit's real ICE output: a\n            #                              synthetic copy 'n1x:<mid>' stands for a blind molecule with <mid>'s ICE dict\n            _n1_xi = {m: (ICE_SCORES.get(m.split(':', 1)[1]) if isinstance(ICE_SCORES, dict) else None) for m in N1X_GL}\n            _n1_xk, _n1_xd = n1_complement(N1X_GL, _n1_xi)\n            assert _n1_xd == {m for m, v in _n1_xi.items() if n1_ice_finite(v) > 0}, _n1_xd\n            assert set(_n1_xk) == set(N1X_GL) - _n1_xd and all(_n1_xk[m] is N1X_GL[m] for m in _n1_xk)\n            assert _n1_xd, ('complement filter not exercised: no synthetic N1 molecule whose source has a finite ICE '\n                            'score', len(N1X_GL), sum(n1_ice_finite(v) for v in _n1_xi.values()))\n            _n1_t = n1_complement({'f': {}, 'n': {}, 'x': {}, 'b': {}, 'm': {}, 'e': {}},\n                                  {'f': {'p': 0.5, 'q': None}, 'n': {'p': None}, 'x': {'p': float('nan'), 'q': float('inf')},\n                                   'b': {'p': 'bad', 'q': [1]}, 'e': {}})\n            assert set(_n1_t[0]) == {'n', 'x', 'b', 'm', 'e'} and _n1_t[1] == {'f'}, _n1_t\n            assert n1_complement({}, ICE_SCORES) == ({}, set()) and n1_complement({'f': {}}, None) == ({'f': {}}, set())\n            N1X_STATS['complement'] = dict(synthetic=len(N1X_GL), dropped=len(_n1_xd), kept=len(_n1_xk))\n            print(f\"[N1X] complement filter exercise: synthetic N1 molecules with GL {len(N1X_GL)} | source ICE dict has a \"\n                  f\"finite score -> dropped {len(_n1_xd)} | kept {len(_n1_xk)} | unit cases OK\", flush=True)\n        print(f\"[N1] complement (k2 variant A): N1 molecules with GL scores {_n1_n0} | dropped (finite ICE score, E006 ICE \"\n              f\"fallback kept) {len(N1_ICE_DROPPED)} | kept {len(_n1_real)}\", flush=True)\n    _n1_health = (f\"N1 OK: {len(_n1_real)} of {len(_n1_items)} N1 molecules GL-scored \"\n                  f\"({sum(sum(v is not None for v in d.values()) for d in _n1_real.values())} candidates), driver \"\n                  f\"{N1_DRV.get('status')}, clashes dropped {len(_n1_clash)}\")\n    N1_MIDS, N1_GL_MIDS = {str(it['mid']) for it in _n1_items}, set(_n1_real)\n    GL_SCORES.update(_n1_real)           # last statement: N1 scores exist only if everything above succeeded\n    N1_HEALTH = _n1_health\nexcept Exception as e:\n    for _n1_m in list(N1_GL_MIDS):\n        GL_SCORES.pop(_n1_m, None)\n    N1_MIDS, N1_GL_MIDS = set(), set()\n    N1_HEALTH = f'N1 FAILED: {e!r}'\n    print('[N1] FAILED -> H1 behaviour (no N1 GL scores):', repr(e)[:600], flush=True)\n    if not IS_RERUN:\n        raise\nprint(f'[N1] GL-scored N1 molecules: {len(N1_GL_MIDS)} of {len(N1_MIDS)} | health: {N1_HEALTH} | '\n      f'{time.time()-T0:.0f}s', flush=True)\ntry:                                     # counts distinguish selected proxies from retained forward scores (log only)\n    N1_CHALLENGER_COVERAGE = dict(\n        complement=bool(N1_COMPLEMENT), eligible_molecules=len(N1_MIDS),\n        proxy_covered_molecules=len(N1_GL_MIDS),\n        proxy_covered_with_finite_ice=sum(n1_ice_finite(ICE_SCORES.get(str(m), {})) > 0\n                                           for m in N1_GL_MIDS),\n        real_h_molecules_skipped=int(N1_STATS.get('with_H', 0)), health=N1_HEALTH)\n    print('N1_CHALLENGER_COVERAGE', json.dumps(N1_CHALLENGER_COVERAGE), flush=True)\nexcept Exception as e:\n    print('[N1 challenger] coverage log failed (log only):', repr(e), flush=True)\n# ---- /N1A_MCE ---------------------------------------------------------------------------------------------------\ndef gl_of(m):\n    g = GL_SCORES.get(str(m), {}) if isinstance(GL_SCORES, dict) else {}\n    return g if isinstance(g, dict) and any(v is not None for v in g.values()) else {}\ndef gl_rerank(order0, smis, keys, scs, forms, ice, gl, top_n, tag=''):\n    # z(scs) + ICE_LAM z(ice) + GL_LAM z(gl) in the same slots; no GL scores / any failure -> order0 (= v4g order)\n    if not gl:\n        return order0\n    try:\n        o = gl_fuse.rerank_multi(smis, keys, scs, forms, [ice, gl], [ICE_LAM, GL_LAM], top_n=top_n)\n    except Exception as e:\n        print('gl rerank failed', tag, repr(e)); return order0\n    for k, v in (('molecules', 1), ('changed_vs_ice_top25', int(o[:25] != order0[:25])), ('changed_vs_ice_top1', int(o[:1] != order0[:1]))):\n        gl_stats[tag + k] = gl_stats.get(tag + k, 0) + v\n    return o\nfor m in list(BASE):\n    smis, keys, lib_max, scs, forms = BASE[m]\n    ice = ICE_SCORES.get(str(m), {}) if ICE_SCORES else {}\n    gl = gl_of(m)\n    if BASE_ICE and ice and any(v is not None for v in ice.values()):\n        try:\n            order = ice_fuse.rerank(smis, keys, scs, forms, ice, lam=ICE_LAM, top_n=TOPN)\n            order = gl_rerank(order, smis, keys, scs, forms, ice, gl, TOPN)\n            s2 = [smis[i] for i in order]; k2 = [keys[i] for i in order]\n            ice_stats['molecules'] += 1\n            ice_stats['changed_top25'] += int(s2[:25] != smis[:25]); ice_stats['changed_top1'] += int(bool(s2) and s2[0] != smis[0])\n            smis, keys = s2, k2\n        except Exception as e:\n            print('rerank failed', m, repr(e))\n    BASE[m] = (smis[:25], keys[:25], lib_max)\n    p = PC.get(m) if (ICE_PC and isinstance(PC, dict)) else None\n    if p and p.get('pc_form') and ice:\n        try:                             # re-order the PubChem list by z(f.z) + lam z(ice) inside same-formula groups;\n            o = ice_fuse.rerank(p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, lam=ICE_LAM, top_n=25)\n            o = gl_rerank(o, p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, gl, 25, 'pc_')\n            fz0 = p['pc_fz'][0]          # the gate keeps using the ORIGINAL top f.z (rel), so gate decisions are unchanged\n            p['pc'] = [p['pc'][i] for i in o]; p['pc_keys'] = [p['pc_keys'][i] for i in o]\n            p['pc_fz'] = [fz0] + [p['pc_fz'][i] for i in o][1:]\n            ice_stats['pc_changed'] = ice_stats.get('pc_changed', 0) + int(o[:1] != [0])\n        except Exception as e:\n            print('pc rerank failed', m, repr(e))\nprint('ICE rerank stats', ice_stats, f'{time.time()-T0:.0f}s')\ntry:                                     # N1A_MCE: the N1 GL scores also reach the PubChem-list re-rank above (log only)\n    _n1_base = {str(m): m for m in BASE}\n    _n1_pc = sum(1 for s in N1_GL_MIDS if ICE_PC and isinstance(PC, dict) and (PC.get(_n1_base.get(s, s)) or {}).get('pc_form')\n                 and isinstance(ICE_SCORES, dict) and ICE_SCORES.get(s))\n    print(f'[N1] PubChem-list GL re-rank: N1 molecules with GL scores {len(N1_GL_MIDS)} | with a gated PubChem list '\n          f'(lib_max < LIB_TAU) and an ICE dict = re-ranked with their N1 GL scores {_n1_pc}', flush=True)\nexcept Exception as e:\n    print('[N1] PubChem-list count failed (log only):', repr(e), flush=True)\nprint('GL rerank stats', gl_stats, f'{time.time()-T0:.0f}s')\n\n\n# ===== cell 20 =====\n'''## 8. Gated PubChem merge\n\nPubChem-only candidates enter the main list only when there is no strong library match (`lib_max < 0.9`). When the PubChem fingerprint fit beats the best pool candidate by a wide margin, they take slots 2, 4, 6, 8 and 10. Otherwise they take slots 4, 8, 12, 16 and 20.\n'''\n\n\n# ===== cell 21 =====\n'''gated PubChem merge (v3.6, unchanged) -> submission.csv'''\ndef merge(base, base_keys, pc, pc_keys, slots, n=25):\n    bk = set(base_keys)\n    pcs = [s for s, k in zip(pc, pc_keys) if k not in bk]\n    out, bi, pj = [], 0, 0\n    for pos in range(1, n + 1):\n        if pos in slots and pj < len(pcs):\n            out.append(pcs[pj]); pj += 1\n        elif bi < len(base):\n            out.append(base[bi]); bi += 1\n        elif pj < len(pcs):\n            out.append(pcs[pj]); pj += 1\n    return out\n'''---- CLAW PROMOTION RULE (b417) --------------------------------------------------'''\n'''The shipped gate only INTERLEAVES PubChem-only structures into fixed slots, so the'''\n'''main list always keeps rank 1. When the best PubChem proposal is a strong outlier --'''\n'''S = z(f.z over the pass-2 set) + 0.25*pop, carried out of the worker -- AND is well'''\n'''documented (pop >= 5), let it lead; everything else keeps its usual slot.'''\n'''NOTE: S is the Z-SCORED quantity (median ~6). The raw p['pc_fz'][0] (median -1437)'''\n'''must never be compared to S_TAU; that would fire on ~71% of molecules.'''\n'''EVIDENCE: b416 (promotion, ICEBERG broken) = 0.417 vs base415 (ICE broken, no'''\n'''promotion) 0.374 and b415c (ICE working, no promotion) 0.393. The promotion overcame'''\n'''losing ICE, so its marginal value is >= +0.024 (est. ~+0.037 using the notebook's own'''\n'''ICE +0.013 / GL +0.006 measurements). b417 isolates it on the WORKING-ICE base.'''\nS_TAU, POP_TAU = 6.0, 5.0\n\n\ndef promote(base, base_keys, pc, pc_keys, slots, n=25):\n    \"\"\"Confident PubChem proposal leads the usual list; every other entry keeps its slot.\"\"\"\n    usual = merge(base, base_keys, pc, pc_keys, slots, n=n)\n    if not pc:\n        return usual\n    top = pc[0]\n    return ([top] + [x for x in usual if x != top])[:n]\n\n\n'''self-test of both pure functions (this also runs in the cheap smoke pass)'''\n_b, _bk = ['A', 'B', 'C'], ['ka', 'kb', 'kc']\n_p, _pk = ['X', 'Y'], ['kx', 'ky']\nassert merge(_b, _bk, _p, _pk, [2, 4]) == ['A', 'X', 'B', 'Y', 'C'], merge(_b, _bk, _p, _pk, [2, 4])\nassert promote(_b, _bk, _p, _pk, [2, 4]) == ['X', 'A', 'B', 'Y', 'C'], promote(_b, _bk, _p, _pk, [2, 4])\nassert merge(_b, _bk, _p, _pk, [2, 4])[0] == 'A', 'shipped gate must keep rank 1'\nassert merge(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']\nassert promote(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']\nassert promote(_b, _bk, _p, _pk, [2, 4])[:1] == ['X'] and len(promote(_b, _bk, _p, _pk, [2, 4])) == 5\nprint('merge/promote self-test OK')\n\nrows, stats = [], dict(untouched=0, gentle=0, aggressive=0, no_pc=0, promoted=0)\nfor mid in te.molecule_id.unique():\n    smis, keys, lib_max = BASE.get(mid, ([], [], 0.0))\n    p = PC.get(mid)\n    if p is None or not p['pc']:\n        stats['no_pc'] += 1; final = smis\n    elif lib_max >= LIB_TAU:\n        stats['untouched'] += 1; final = smis\n    else:\n        rel = (p['fz_top'] if p.get('fz_top') is not None else p['pc_fz'][0]) - p['best_pool_fz'] if np.isfinite(p['best_pool_fz']) else 1e9\n        slots = SLOTS_AGG if rel > REL_TH else SLOTS_GENTLE\n        stats['aggressive' if rel > REL_TH else 'gentle'] += 1\n        S = p.get('S'); pop_top = float(p.get('top_pop') or 0.0)\n        # Skomuro LB 0.403 Rule: Promote on Popularity OR Structure-Evidence Margin!\n        # rel > 250 captures confident PubChem outliers that popularity alone misses\n        # v11: rel promotion threshold 150.0 (public evidence: seyitkaangunes E006 sweep; antriksharya e2rank2 base)\n        if (S is not None and float(S) > S_TAU and pop_top >= POP_TAU) or (rel > 150.0):\n            stats['promoted'] += 1\n            final = promote(smis, keys, p['pc'], p['pc_keys'], slots)\n        else:\n            final = merge(smis, keys, p['pc'], p['pc_keys'], slots)\n    if not final: final = ['CCO']\n    rows.append((mid, ';'.join(final[:25])))\nprint('merge stats', stats)\nsub = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])\nsamp = pd.read_csv(os.path.join(COMP, 'sample_submission.csv'))\nsub = samp[['molecule_id']].merge(sub, on='molecule_id', how='left')\nsub['smiles'] = sub['smiles'].fillna('CCO')\nassert len(sub) == len(samp) and sub.molecule_id.is_unique and sub.smiles.notna().all()\nassert sub.smiles.map(lambda s: len(s.split(';'))).max() <= 25\nsub.to_csv('submission.csv', index=False)\nprint('CLAW STAGE HEALTH |', CLAW_ICE_HEALTH, '|', CLAW_GL_HEALTH, flush=True)\ntry:\n    _ice_n = json.load(open('/kaggle/working/ice_work/ice_out.json.meta.json')).get('n_mols_scored', 0)\n    _gl_n = json.load(open('/kaggle/working/gl_work/gl_out.json.meta.json')).get('n_mols_scored', 0)\nexcept Exception:\n    _ice_n = _gl_n = -1\nprint(f'CLAW STAGE SCORED | ice={_ice_n} gl={_gl_n}', flush=True)\njson.dump(dict(secs=round(time.time() - T0), merge=stats, n_errors=n_err,\n               ice_health=CLAW_ICE_HEALTH, gl_health=CLAW_GL_HEALTH,\n               ice_n_scored=_ice_n, gl_n_scored=_gl_n,\n               families=MAN['families'],\n               gpu=torch.cuda.get_device_name(0) if dev == 'cuda' else None), open('run_manifest.json', 'w'), indent=1)\nprint('done', sub.shape, f'{time.time()-T0:.0f}s')\nsub.head()\n\n\n# ===== cell 22 =====\n'''## 9. Two-engine fusion + final simulator re-rank\n\nThe two lists are combined by weighted reciprocal-rank fusion. A candidate's score is $\\\\frac{1}{3 + r_{\\\\text{main}}} + \\\\frac{0.6}{3 + r_{\\\\text{engine 2}}}$, and the top 40 are kept. ICEBERG and GLACIER then re-rank same-formula groups of the fused list one last time, and the top 25 become the submission.\n'''\n\n\n# ===== cell 23 =====\nALPHA, KRR = 0.6, 3.0\nCELL19_ICE = False       # b421: cell-19 ICE contribution EXPLICITLY off.\n'''Reproduces BY DESIGN what b416 got by accident: when ICE errored it still wrote a'''\n'''shell dict of all-None values, cell-19's `if ice:` gate passed, and ICE's column then'''\n'''contributed nothing (covered count 0) -- leaving GLACIER alone. b420 (ICE real, both'''\n'''applied) scored 0.402; b416 (GLACIER only) scored 0.417. This isolates that one'''\n'''arithmetic difference on the healthy base.'''\nfrom rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF2\nfrom rdkit import Chem as _Ch2\ndef _form2(s):\n    try:\n        return _CMF2(_Ch2.MolFromSmiles(s))\n    except Exception:\n        return s\ndef fuse2(v_smis, e_smis, e_keys, n=40):\n    sc, smi_of = {}, {}\n    for r, s in enumerate(v_smis, 1):\n        k = chem.score_key(s) or s\n        sc[k] = sc.get(k, 0.0) + 1.0 / (KRR + r); smi_of.setdefault(k, s)\n    for r, (s, k) in enumerate(zip(e_smis, e_keys), 1):\n        if not k:\n            continue\n        sc[k] = sc.get(k, 0.0) + ALPHA / (KRR + r); smi_of.setdefault(k, s)\n    order = sorted(sc, key=lambda k: -sc[k])[:n]\n    return [smi_of[k] for k in order], order, [sc[k] for k in order]\n\nPREFUSION_LISTS = {str(r.molecule_id): str(r.smiles).split(';') for r in pd.read_csv('submission.csv').itertuples(index=False)}\nFUSION_OVERFLOW = {}\nif ENG:\n    v4 = pd.read_csv('submission.csv')\n    changed = n_ice = 0; out = []\n    for mid, s in zip(v4.molecule_id, v4.smiles):\n        vs = [x for x in s.split(';') if x and x != 'CCO']\n        e = ENG.get(str(mid))\n        if e and e['smiles']:\n            fsm, fk, fsc = fuse2(vs, e['smiles'], e['keys'])\n            ice = ICE_SCORES.get(str(mid), {}) if isinstance(ICE_SCORES, dict) else {}\n            if ice:\n                try:\n                    o = ice_fuse.rerank(fsm, fk, fsc, [_form2(x) for x in fsm], (ice if CELL19_ICE else {}), lam=ICE_LAM, top_n=len(fsm))\n                    _g = gl_of(mid)\n                    if _g:\n                        o = gl_fuse.rerank_multi(fsm, fk, fsc, [_form2(x) for x in fsm], [(ice if CELL19_ICE else {}), _g], [ICE_LAM, GL_LAM], top_n=len(fsm))\n                    n_ice += int(o[:25] != list(range(min(25, len(o)))))\n                    fsm = [fsm[i] for i in o]\n                except Exception as ex:\n                    print('fused ICE rerank failed', mid, repr(ex))\n            changed += int(bool(vs) and fsm[:1] != vs[:1])\n            FUSION_OVERFLOW[str(mid)] = list(fsm[25:])\n            vs = fsm[:25]\n        out.append((mid, ';'.join(vs[:25]) if vs else 'CCO'))\n    fs = pd.DataFrame(out, columns=['molecule_id', 'smiles'])\n    assert len(fs) == len(v4) and fs.molecule_id.is_unique and fs.smiles.notna().all()\n    assert fs.smiles.map(lambda s: len(s.split(';'))).max() <= 25\n    fs.to_csv('submission.csv', index=False)\n\n\nprint('V1 inference complete; applying separately audited final pass next.')\n\n\n# ===== cell 24 =====\n'''## Final metric audit\n\nThe final pass cannot demote any existing metric-equivalent answer: it retains the first\noccurrence of every valid identity in order, then appends new identities up to 25 candidates.\nRuntime assertions check this for every active molecule. This guarantee applies to the\nsame predictions and current metric; it does not predict the score of a fresh inference run.\n\nOutputs: `submission.csv`, `baseline_submission.csv`, `candidate_audit.csv`,\n`candidate_audit.json`, and stage health in `run_manifest.json`.\n'''\n\n\n# ===== cell 25 =====\nfrom functools import lru_cache\nfrom rdkit import Chem, RDLogger\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\nRDLogger.DisableLog('rdApp.*')\n_tautomer_canonicalizer = rdMolStandardize.TautomerEnumerator()\ndef canonicalize_smiles_to_ik14(smiles: str) -> str:\n    \"\"\"Canonicalize SMILES to InChIKey14, or None if parsing fails.\n\n    Stereochemistry is removed *before* the tautomer canonicalization, not left to fall out of\n    the InChIKey14 truncation afterwards. Truncating to InChIKey14 discards the stereo block, so\n    it looks like it should already make stereochemistry free -- but the canonicalization runs\n    first, and `TautomerEnumerator` will not move a double bond whose position a neighbouring\n    stereocentre constrains. The same skeleton drawn with and without that stereocentre can\n    therefore land on two different canonical tautomers, and different tautomers are different\n    connectivity, which is the part of the InChIKey14 that survives truncation. Stripping first\n    makes the canonical form depend only on the skeleton, which is what the competition promises.\n\n    Both the answer key and the submission go through this function, so the two sides can never\n    be treated asymmetrically.\n    \"\"\"\n    if not isinstance(smiles, str):\n        return None\n    try:\n        mol = Chem.MolFromSmiles(smiles)\n        if mol is None:\n            return None\n        Chem.RemoveStereochemistry(mol)\n        canonical_mol = _tautomer_canonicalizer.Canonicalize(mol)\n        inchikey = Chem.MolToInchiKey(canonical_mol)\n        return inchikey[:14]\n    # RDKit's MolSanitizeException family subclasses ValueError; RuntimeError covers the C++\n    # layer and TypeError covers Boost.Python argument errors. One participant's malformed\n    # SMILES must never take the whole scoring run down, so a failure here is just a non-match.\n    except (ValueError, RuntimeError, TypeError):\n        return None\n\ndef heavy_atom_composition(smiles: str):\n    \"\"\"Heavy-atom element counts (excluding H, ignoring charge) as a hashable key, or None\n    if parsing fails.\n\n    Cheap (no tautomer enumeration) -- used to gate the expensive canonicalization. Anything\n    that shares an InChIKey14 with an accepted structure has the same heavy-atom skeleton, so\n    a heavy-atom-composition mismatch lets us skip canonicalization with no change to any\n    score. We deliberately do NOT gate on full molecular formula: protomers / charge variants\n    (e.g. a zwitterion C(=O)[O-]...[N+] vs its neutral-acid protomer C(=O)O...[N+]) share an\n    InChIKey14 but differ in H count and charge, so a formula gate would wrongly reject them.\n    \"\"\"\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\n    if mol is None:\n        return None\n    counts = {}\n    for atom in mol.GetAtoms():\n        sym = atom.GetSymbol()\n        if sym != 'H':\n            counts[sym] = counts.get(sym, 0) + 1\n    return tuple(sorted(counts.items()))\n\n@lru_cache(maxsize=150000)\ndef metric_identity(smiles):\n    key = canonicalize_smiles_to_ik14(smiles)\n    return (key, heavy_atom_composition(smiles)) if key else None\n\ndef safe_extend(original, extra, limit=25):\n    \"\"\"Preserve first valid metric identities; fill only unoccupied ranks.\"\"\"\n    out, seen = [], set()\n    for smi in list(original) + list(extra):\n        smi = str(smi).strip()\n        identity = metric_identity(smi)\n        if identity is not None and identity not in seen:\n            seen.add(identity)\n            out.append(smi)\n            if len(out) == limit:\n                break\n    return out\n\ndef first_ranks(smiles):\n    ranks = {}\n    for rank, smi in enumerate(smiles, 1):\n        identity = metric_identity(smi)\n        if identity is not None:\n            ranks.setdefault(identity, rank)\n    return ranks\n\ndef assert_monotone(original, revised):\n    old, new = first_ranks(original), first_ranks(revised)\n    assert all(identity in new and new[identity] <= rank for identity, rank in old.items())\n\n\n# ===== cell 26 =====\nMETRIC_SOURCE_SHA256 = 'c61d163f1bb768313fd0863822952f035f343924f37e3f5e4b5883ccef7e6201'\n'''Final pass: no change to any V1 model, fusion weight, promotion gate or simulator.'''\nimport collections\nimport hashlib\nfrom pathlib import Path\nassert rdkit.__version__ == '2026.03.3'\nassert n_err == 0, f'Base inference errors: {n_err}'\n_before = pd.read_csv('submission.csv')\n_before.to_csv('baseline_submission.csv', index=False)\n_active = set(te.molecule_id.astype(str))\nassert _active <= set(map(str, ENG)), 'Engine 2 did not cover all active molecules'\n_stage_meta = {}\nfor _stage, _path in [('ice', 'ice_work/ice_out.json.meta.json'), ('glacier', 'gl_work/gl_out.json.meta.json')]:\n    if Path(_path).is_file():\n        _stage_meta[_stage] = json.loads(Path(_path).read_text())\n    else:\n        _stage_meta[_stage] = {'status': 'missing'}\nassert _stage_meta['ice'].get('status') == 'ok', _stage_meta['ice']\nif (te.adduct == '[M+H]+').any():\n    assert _stage_meta['glacier'].get('status') == 'ok', _stage_meta['glacier']\n\n_dm_smiles = _dm_mass = _dm_order = None\n_dm_paths = sorted(glob.glob('/kaggle/input/**/dreams_np_mass.npy', recursive=True))\n_dreams_used = 0\n_audit_rows, _out = [], []\n_targets = te.groupby('molecule_id').nm.median().to_dict()\nfor _row in _before.itertuples(index=False):\n    _mid = str(_row.molecule_id)\n    _original = [s.strip() for s in str(_row.smiles).split(';') if s.strip()]\n    if _mid not in _active:  # Visible smoke placeholders; hidden rerun processes every molecule.\n        _out.append((_row.molecule_id, _row.smiles))\n        continue\n    _ranks = first_ranks(_original)\n    _clean = safe_extend(_original, [])\n    _extras = list(FUSION_OVERFLOW.get(_mid, [])) + list(BASE_TAIL_CANDIDATES.get(_mid, []))\n    _engine = ENG.get(_mid, {})\n    _extras.extend(_engine.get('smiles', []))\n    _pc = PC.get(_row.molecule_id, PC.get(_mid, {}))\n    _extras.extend(_pc.get('pc', []))\n    _revised = safe_extend(_clean, _extras)\n    _after_ranked = len(_revised)\n    _target = float(_targets.get(_row.molecule_id, _targets.get(_mid, np.nan)))\n    if len(_revised) < 25 and np.isfinite(_target) and _target > 0 and _dm_paths:\n        if _dm_mass is None:\n            _dd = Path(_dm_paths[0]).parent\n            _dm_mass = np.load(_dd / 'dreams_np_mass.npy').astype(np.float64)\n            _dm_smiles = np.load(_dd / 'dreams_np_smiles.npy', allow_pickle=True)\n            assert len(_dm_mass) == len(_dm_smiles)\n            _dm_order = np.argsort(_dm_mass, kind='stable')\n            _dm_sorted = _dm_mass[_dm_order]\n        _lo, _hi = np.searchsorted(_dm_sorted, [_target * (1 - 10e-6), _target * (1 + 10e-6)])\n        _ix = _dm_order[_lo:_hi]\n        _ix = _ix[np.argsort(np.abs(_dm_mass[_ix] - _target), kind='stable')]\n        _mass_candidates = [s.decode() if isinstance(s, bytes) else str(s) for s in _dm_smiles[_ix]]\n        _revised = safe_extend(_revised, _mass_candidates)\n        _dreams_used += len(_revised) - _after_ranked\n    assert _revised, f'No valid candidates for {_mid}'\n    assert len(_revised) <= 25 and len(first_ranks(_revised)) == len(_revised)\n    assert_monotone(_original, _revised)\n    _new_ranks = first_ranks(_revised)\n    _audit_rows.append(dict(molecule_id=_mid, original_count=len(_original),\n        original_unique=len(_ranks), invalid_count=sum(metric_identity(s) is None for s in _original),\n        duplicate_count=sum(metric_identity(s) is not None for s in _original) - len(_ranks),\n        revised_count=len(_revised), ranked_added=_after_ranked - len(_clean),\n        dreams_added=len(_revised) - _after_ranked,\n        promoted_original=sum(_new_ranks[k] < v for k, v in _ranks.items()),\n        changed=_original != _revised))\n    _out.append((_row.molecule_id, ';'.join(_revised)))\n_final = pd.DataFrame(_out, columns=['molecule_id', 'smiles'])\n_expected = pd.read_csv(os.path.join(COMP, 'sample_submission.csv')).molecule_id\nassert _final.molecule_id.is_unique and set(_final.molecule_id) == set(_expected)\nassert _final.smiles.notna().all() and _final.smiles.str.split(';').map(len).between(1, 25).all()\nif IS_RERUN:\n    assert len(_active) == len(_final), 'A hidden molecule was left unprocessed'\n_final.to_csv('submission.csv', index=False)\n_audit_df = pd.DataFrame(_audit_rows)\n_audit_df.to_csv('candidate_audit.csv', index=False)\n_summary = dict(experiment='V1.1 metric-safe tail', rdkit=rdkit.__version__,\n    hidden_rerun=IS_RERUN, active_molecules=len(_active), output_rows=len(_final),\n    changed_molecules=int(_audit_df.changed.sum()),\n    invalid_removed=int(_audit_df.invalid_count.sum()),\n    equivalent_duplicates_removed=int(_audit_df.duplicate_count.sum()),\n    existing_candidates_promoted=int(_audit_df.promoted_original.sum()),\n    ranked_candidates_added=int(_audit_df.ranked_added.sum()),\n    dreams_candidates_added=int(_audit_df.dreams_added.sum()),\n    monotone_rank_assertions_passed=True,\n    metric_source_sha256=METRIC_SOURCE_SHA256,\n    baseline_sha256=hashlib.sha256(Path('baseline_submission.csv').read_bytes()).hexdigest(),\n    submission_sha256=hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest(),\n    stages=_stage_meta, runtime_seconds=round(time.time()-T0, 1),\n    validation_note='Visible test is a smoke check only and overlaps training; no generalization score claimed.')\nPath('candidate_audit.json').write_text(json.dumps(_summary, indent=2, default=str))\nprint(json.dumps({k:v for k,v in _summary.items() if k != 'stages'}, indent=2))\ndisplay(_audit_df)\n\n\n# ===== cell 27 =====\n'''Independent experimental overlay\nOne vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. Preserve the existing PubChem gate and same-formula GLACIER logic. This tests duplicate-vote bias, not a coefficient search. It can change top-1 and reduce the score.'''\n\ndef metric_fusion(first, second, identity, alpha=0.6, offset=3.0, limit=40):\n    scores, representatives = {}, {}\n    collapsed = 0\n    for candidates, weight in ((first, 1.0), (second, alpha)):\n        seen = set()\n        for candidate in candidates:\n            key = identity(candidate)\n            if key is None:\n                continue\n            if key in seen:\n                collapsed += 1\n                continue\n            seen.add(key)\n            scores[key] = scores.get(key, 0.0) + weight / (offset + len(seen))\n            representatives.setdefault(key, candidate)\n    ordered = sorted(scores, key=lambda key: -scores[key])[:limit]\n    return [representatives[key] for key in ordered], [scores[key] for key in ordered], collapsed\n\nEXPERIMENT = 'metric_identity_fusion'\n_reference = pd.read_csv('submission.csv')\n_reference.to_csv('submission_reference.csv', index=False)\n_rows, _audit = [], []\nfor _row in _reference.itertuples(index=False):\n    _mid = str(_row.molecule_id)\n    _original = str(_row.smiles).split(';')\n    _revised, _collapsed = list(_original), 0\n    _engine = ENG.get(_mid)\n    if _mid in PREFUSION_LISTS and _engine and _engine.get('smiles'):\n        _first = [s for s in PREFUSION_LISTS[_mid] if s and s != 'CCO']\n        _candidates, _scores, _collapsed = metric_fusion(_first, _engine['smiles'], metric_identity,\n                                                       alpha=ALPHA, offset=KRR)\n        _keys = [chem.score_key(s) or s for s in _candidates]\n        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}\n        _gl = gl_of(_mid)\n        if _ice and _gl:\n            _order = gl_fuse.rerank_multi(_candidates, _keys, _scores,\n                [_form2(s) for s in _candidates], [{}, _gl], [ICE_LAM, GL_LAM], top_n=len(_candidates))\n            _candidates = [_candidates[i] for i in _order]\n        if _candidates:\n            _revised = safe_extend(_candidates[:25], _candidates[25:] + _original)\n    _rows.append((_row.molecule_id, ';'.join(_revised)))\n    _audit.append(dict(molecule_id=_mid, collapsed_source_votes=_collapsed,\n        changed=_revised != _original, changed_top1=_revised[:1] != _original[:1]))\n\n_result = pd.DataFrame(_rows, columns=['molecule_id', 'smiles'])\nassert list(_result.molecule_id) == list(_reference.molecule_id)\nassert _result.molecule_id.is_unique and _result.smiles.notna().all()\nassert _result.smiles.map(lambda value: 1 <= len(value.split(';')) <= 25).all()\n_result.to_csv('submission.csv', index=False)\npd.DataFrame(_audit).to_csv('experiment_audit.csv', index=False)\n_changed = sum(row['changed'] for row in _audit)\n_report = dict(experiment=EXPERIMENT, changed_molecules=_changed,\n               changed_top1=sum(row['changed_top1'] for row in _audit),\n               reference_sha256=sha256('submission_reference.csv'),\n               submission_sha256=sha256('submission.csv'), measured_score=None)\njson.dump(_report, open('experiment_audit.json', 'w'), indent=2)\nprint(_report)\nif not _changed:\n    print('NO-OP: predictions match the reference; do not spend a second submission on this file.')\n\n\n# ===== cell 28 =====\n\"\"\"Observe the frozen source output and preserve same-run fusion inputs for later work.\"\"\"\n\n\ndef _e004_postflight(scope):\n    import hashlib\n    import json\n    import time\n    from pathlib import Path\n    import pandas as pd\n    import rdkit\n\n    pre = scope['E004_PREFLIGHT']\n    ids = set(pre['input_ids'])\n    processed = set(scope['te'].molecule_id.astype(str))\n    full = bool(scope['IS_RERUN'])\n    assert processed == (ids if full else set(sorted(ids)[:scope['SMOKE_N']]))\n    frame = pd.read_csv('submission.csv', dtype=str, keep_default_na=False)\n    assert list(frame.columns) == ['molecule_id', 'smiles']\n    assert frame.molecule_id.is_unique and set(frame.molecule_id) == ids\n    assert frame.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25).all()\n    base, eng, pc = scope['BASE'], scope['ENG'], scope['PC']\n    assert set(map(str, base)) == processed\n    assert processed <= set(map(str, eng))\n    assert scope['n_err'] == 0\n    if not full:\n        assert all(v[0] for v in base.values())\n        assert all(eng[mid]['smiles'] and pc.get(mid, {}).get('pc') for mid in processed)\n    stages = {}\n    for stage in ('ice', 'gl'):\n        path = Path(f'{stage}_work/{stage}_out.json.meta.json')\n        meta = json.loads(path.read_text()) if path.is_file() else {'status': 'missing'}\n        # The frozen source owns full-run failure policy. Extra acceptance checks\n        # only gate the visible smoke, where all expected stages are exercised.\n        if not full:\n            assert meta['status'] == 'ok' and meta['device'].startswith('cuda'), (stage, meta)\n            assert meta['rdkit'] == '2025.03.6'\n            assert not meta.get('errors') and meta.get('n_pred_fail', 0) == 0\n            assert meta.get('n_feat_fail', 0) == 0\n            if meta.get('n_mols_covered', 0):\n                assert meta['n_mols_scored'] > 0 and meta['n_scored'] > 0\n        stages[stage] = meta\n    payload = {\n        'prefusion': {mid: scope['PREFUSION_LISTS'][mid] for mid in sorted(processed)},\n        'engine2': {mid: eng[mid] for mid in sorted(processed)},\n        'pubchem': {mid: pc.get(mid, {}) for mid in sorted(processed)},\n        'base_tail': {mid: scope['BASE_TAIL_CANDIDATES'][mid] for mid in sorted(processed)},\n        'fusion_overflow': {mid: scope['FUSION_OVERFLOW'].get(mid, []) for mid in sorted(processed)},\n    }\n    Path('e004_candidate_inputs.json').write_text(json.dumps(payload, indent=2, default=str) + '\\n')\n    hashes = {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in (\n        'submission.csv', 'baseline_submission.csv', 'submission_reference.csv',\n        'candidate_audit.json', 'experiment_audit.json', 'e004_candidate_inputs.json')}\n    report = dict(experiment='E004', source_version_id=356740494, source_version=80,\n                  source_code_sha256='eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2',\n                  **pre, is_full_run=full, processed_ids=sorted(processed),\n                  processed_molecules=len(processed), input_molecules=len(ids),\n                  output_rows=len(frame), placeholder_fill_rows=len(ids - processed),\n                  base_errors=int(scope['n_err']), base_nonempty=sum(bool(v[0]) for v in base.values()),\n                  engine2_nonempty=sum(bool(eng[mid]['smiles']) for mid in processed),\n                  pubchem_nonempty=sum(bool(pc.get(mid, {}).get('pc')) for mid in processed),\n                  main_rdkit=rdkit.__version__, stages=stages, hashes=hashes,\n                  elapsed_seconds=round(time.time() - scope['T0'], 2),\n                  limitations=['Visible smoke is not independent validation.',\n                               'Author 0.436 is not this reproduction score.',\n                               'Whole v80 recipe differs from E003 in multiple factors.'])\n    Path('e004_verification.json').write_text(json.dumps(report, indent=2) + '\\n')\n    print('E004_VERIFICATION', json.dumps(report))\n    return report\n\n\nE004_REPORT = _e004_postflight(globals())\ndel _e004_postflight\n\n\n# ===== cell 29 =====\n# v7c: MetFrag-lite fragment evidence for molecules with NO forward-model coverage (no GLACIER, no finite ICEBERG)\nFRAG_GAP = {}\ntry:\n    FRAG_SRC = '\"\"\"Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\\'s <=2-bond-cleavage fragments explain.\\n\\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\\nremember the fewest cleavages that give each mass, then match every peak against\\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\\nA peak\\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\\nand a spectrum\\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\\n\\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\\n\\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\\n\"\"\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\\'s periodic table\\nAMU = {\\'C\\': 12.0, \\'H\\': 1.00782503207, \\'N\\': 14.0030740048, \\'O\\': 15.9949146196, \\'P\\': 30.97376163,\\n       \\'S\\': 31.97207100, \\'F\\': 18.99840322, \\'Cl\\': 34.96885268, \\'Br\\': 78.9183371, \\'I\\': 126.904473,\\n       \\'Na\\': 22.9897692809, \\'K\\': 38.96370668, \\'Si\\': 27.9769265325, \\'B\\': 11.0093054, \\'Se\\': 79.9165213}\\nH = AMU[\\'H\\']\\nE = 0.00054857990\\nPROTON = H - E\\nH2O = 2 * H + AMU[\\'O\\']\\nNH4 = AMU[\\'N\\'] + 4 * H\\nFORMATE = AMU[\\'C\\'] + 2 * H + 2 * AMU[\\'O\\']\\n\\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\\nCARRIERS = {\\n    \\'[M+H]+\\': (PROTON,), \\'[M-H2O+H]+\\': (PROTON,), \\'[M-2H2O+H]+\\': (PROTON,),\\n    \\'[M+NH4]+\\': (PROTON, NH4 - E), \\'[M+Na]+\\': (PROTON, AMU[\\'Na\\'] - E), \\'[M+K]+\\': (PROTON, AMU[\\'K\\'] - E),\\n    \\'[M-H]-\\': (-PROTON,), \\'[M-H2O-H]-\\': (-PROTON,), \\'[M+CH2O2-H]-\\': (-PROTON, FORMATE - PROTON),\\n    \\'[M+Cl]-\\': (-PROTON, AMU[\\'Cl\\'] + E),\\n}\\n\\nDEFAULT_CFG = dict(\\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\\n    intensity=\\'linear\\',          # \\'linear\\' | \\'sqrt\\' | a float power applied to `it`\\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\\n    tol=0.005,                   # absolute m/z tolerance, Da\\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\\n    max_breaks=2,                # bond cleavages per fragment (1..3)\\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\\n)\\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\\'sqrt\\', tol=0.01, h_shifts=tuple(range(-2, 3)))\\n\\n\\ndef resolve_cfg(cfg=None):\\n    \"\"\"DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently).\"\"\"\\n    out = dict(DEFAULT_CFG)\\n    for k, v in (cfg or {}).items():\\n        if k not in out:\\n            raise KeyError(f\\'frag_rescore: unknown cfg key {k!r}\\')\\n        out[k] = v\\n    out[\\'h_shifts\\'] = tuple(int(d) for d in out[\\'h_shifts\\'])\\n    if not 1 <= int(out[\\'max_breaks\\']) <= 3:\\n        raise ValueError(\\'frag_rescore: max_breaks must be 1, 2 or 3\\')\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------------------------------------------\\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\\n# ---------------------------------------------------------------------------------------------------------------\\ndef _find(parent, a):\\n    while parent[a] != a:\\n        parent[a] = parent[parent[a]]\\n        a = parent[a]\\n    return a\\n\\n\\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\\n    \"\"\"Union the kept bonds; if the graph splits, append each piece\\'s mass with cleavage count k.\"\"\"\\n    for a in range(n):\\n        parent[a] = a\\n        acc[a] = 0.0\\n    ncomp = n\\n    for e in range(ea.shape[0]):\\n        if removed[e]:\\n            continue\\n        ra = _find(parent, ea[e])\\n        rb = _find(parent, eb[e])\\n        if ra != rb:\\n            parent[ra] = rb\\n            ncomp -= 1\\n    if ncomp < 2:\\n        return cnt\\n    for a in range(n):\\n        r = _find(parent, a)\\n        acc[r] += w[a]\\n    for a in range(n):\\n        if parent[a] == a:\\n            out_m[cnt] = acc[a]\\n            out_k[cnt] = k\\n            cnt += 1\\n    return cnt\\n\\n\\ndef _enumerate(n, w, ea, eb, max_breaks, base_components):\\n    \"\"\"All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces.\"\"\"\\n    nb = ea.shape[0]\\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\\n    out_m = np.empty(cap, np.float64)\\n    out_k = np.empty(cap, np.int8)\\n    parent = np.empty(n, np.int64)\\n    acc = np.empty(n, np.float64)\\n    removed = np.zeros(nb, np.bool_)\\n    cnt = 0\\n    for i in range(nb):\\n        removed[i] = True\\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\\n        if max_breaks >= 2:\\n            for j in range(i + 1, nb):\\n                removed[j] = True\\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 2)\\n                if max_breaks >= 3:\\n                    for l in range(j + 1, nb):\\n                        removed[l] = True\\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\\n                        removed[l] = False\\n                removed[j] = False\\n        removed[i] = False\\n    return out_m[:cnt], out_k[:cnt]\\n\\n\\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\\n    from numba import njit\\n    _find = njit(cache=True, nogil=True)(_find)\\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\\n    HAVE_NUMBA = True\\nexcept Exception:  # pragma: no cover - fallback path\\n    HAVE_NUMBA = False\\n\\n\\ndef mol_graph(smi):\\n    \"\"\"(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable.\"\"\"\\n    from rdkit import Chem, RDLogger\\n    RDLogger.DisableLog(\\'rdApp.*\\')\\n    try:\\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\\n    except Exception:\\n        m = None\\n    if m is None or m.GetNumAtoms() == 0:\\n        return None\\n    pt = Chem.GetPeriodicTable()\\n    w = np.empty(m.GetNumAtoms(), np.float64)\\n    for a in m.GetAtoms():\\n        s = a.GetSymbol()\\n        am = AMU.get(s)\\n        if am is None:\\n            am = pt.GetMostCommonIsotopeMass(s)\\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\\n    return w, ea, eb, len(Chem.GetMolFrags(m))\\n\\n\\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40):\\n    \"\"\"Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\\n\\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\\n    SMILES does not parse.\\n    \"\"\"\\n    g = mol_graph(smi)\\n    if g is None:\\n        return None\\n    w, ea, eb, npieces = g\\n    nb = len(ea)\\n    mb = int(max_breaks)\\n    if mb >= 3 and nb > max_bonds_3:\\n        mb = 2\\n    if mb >= 2 and nb > max_bonds_2:\\n        mb = 1\\n    levels = [np.array([w.sum()])]\\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\\n        from rdkit import Chem\\n        m = Chem.MolFromSmiles(smi)\\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\\n    if nb == 0:\\n        return levels + [np.zeros(0) for _ in range(int(max_breaks))]\\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces)\\n    fm = np.concatenate([levels[0], fm])\\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\\n    key = np.round(fm, 5)\\n    order = np.lexsort((fk, key))                  # by mass, then fewest cleavages first\\n    key, fm, fk = key[order], fm[order], fk[order]\\n    first = np.ones(len(key), bool)\\n    first[1:] = key[1:] != key[:-1]\\n    fm, fk = fm[first], fk[first]\\n    return [np.sort(fm[fk == k]) for k in range(int(max_breaks) + 1)]\\n\\n\\n# ---------------------------------------------------------------------------------------------------------------\\n# scoring\\n# ---------------------------------------------------------------------------------------------------------------\\ndef _carriers(adduct, mode):\\n    c = CARRIERS.get(adduct)\\n    if c is not None:\\n        return c\\n    if isinstance(adduct, str) and adduct.endswith(\\'-\\'):\\n        return (-PROTON,)\\n    if isinstance(adduct, str) and adduct.endswith(\\'+\\'):\\n        return (PROTON,)\\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\\n\\n\\ndef _prep_spectrum(s, c):\\n    \"\"\"Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair.\"\"\"\\n    mz = np.asarray(s.get(\\'mz\\', ()), np.float64).ravel()\\n    it = np.asarray(s.get(\\'it\\', ()), np.float64).ravel()\\n    n = min(len(mz), len(it))\\n    mz, it = mz[:n], it[:n]\\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\\n    mz, it = mz[ok], it[ok]\\n    if len(mz) == 0:\\n        return None\\n    p = c[\\'intensity\\']\\n    iw = it if p == \\'linear\\' else np.sqrt(it) if p == \\'sqrt\\' else it ** float(p)\\n    if c[\\'mz_power\\']:\\n        prec = float(s.get(\\'prec\\') or mz.max())\\n        iw = iw * (mz / prec) ** float(c[\\'mz_power\\'])\\n    tot = iw.sum()\\n    if not tot > 0:\\n        return None\\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\\'adduct\\'), s.get(\\'mode\\')) for dh in c[\\'h_shifts\\']])\\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\\n    return iw / tot, q, len(shifts), len(mz)\\n\\n\\ndef _score_spectrum(levels, weights, prep, tol):\\n    iw, q, ns, npk = prep\\n    best = np.zeros(npk)\\n    for arr, wt in zip(levels, weights):\\n        if len(arr) == 0 or wt <= 0:\\n            continue\\n        a = np.searchsorted(arr, q - tol, \\'left\\')\\n        b = np.searchsorted(arr, q + tol, \\'right\\')\\n        hit = (b > a).reshape(ns, npk).any(axis=0)\\n        best = np.where(hit & (wt > best), wt, best)\\n    return float(iw @ best)\\n\\n\\ndef candidate_levels(smi, cfg=None, cache=None):\\n    \"\"\"Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter).\"\"\"\\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\\n\\n\\ndef _levels_cached(smi, c, cache):\\n    key = (smi, int(c[\\'max_breaks\\']), int(c[\\'max_bonds_2\\']), int(c[\\'max_bonds_3\\']))\\n    if cache is not None and key in cache:\\n        return cache[key]\\n    try:\\n        lv = fragment_levels(smi, c[\\'max_breaks\\'], c[\\'max_bonds_2\\'], c[\\'max_bonds_3\\'])\\n    except Exception:\\n        lv = None\\n    if cache is not None:\\n        cache[key] = lv\\n    return lv\\n\\n\\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\\n    \"\"\"One score per candidate: mean explained-intensity share over the molecule\\'s spectra (None: unparsable).\"\"\"\\n    c = resolve_cfg(cfg)\\n    cache = {} if cache is None else cache\\n    weights = [1.0, float(c[\\'w1\\']), float(c[\\'w2\\']), float(c[\\'w3\\'])][: int(c[\\'max_breaks\\']) + 1]\\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\\n    tol = float(c[\\'tol\\'])\\n    out = []\\n    for smi in cand_smiles:\\n        lv = _levels_cached(smi, c, cache)\\n        if lv is None:\\n            out.append(None)\\n            continue\\n        if not preps:\\n            out.append(0.0)\\n            continue\\n        out.append(float(np.mean([_score_spectrum(lv, weights, p, tol) for p in preps])))\\n    return out\\n\\n\\ndef warmup():\\n    \"\"\"Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it.\"\"\"\\n    fragment_levels(\\'CCOC(=O)c1ccccc1O\\')\\n    return HAVE_NUMBA\\n'\n    open('/kaggle/working/frag_rescore.py', 'w').write(FRAG_SRC)\n    if '/kaggle/working' not in sys.path:\n        sys.path.insert(0, '/kaggle/working')\n    import frag_rescore\n    frag_rescore.warmup()\n    _fcache = {}\n    _by = {str(m): g for m, g in te.groupby('molecule_id', sort=False)}\n    _t0 = time.time(); _nf = 0\n    for _mid, _lst in PREFUSION_LISTS.items():\n        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}\n        if gl_of(_mid) or any(v is not None for v in (_ice or {}).values()):\n            continue\n        _e = ENG.get(_mid) or {}\n        _c = [s for s in dict.fromkeys(list(_lst) + list(_e.get('smiles', []))) if s and s != 'CCO']\n        _g = _by.get(_mid)\n        if _g is None or len(_c) < 2:\n            continue\n        _spec = [dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                      adduct=r.adduct, prec=float(r.precursor_mz), mode=1 if r.ionization_mode == 'positive' else -1)\n                 for r in _g.itertuples()][:16]\n        try:\n            _sc = frag_rescore.score_molecule(_spec, _c, cache=_fcache)\n            FRAG_GAP[_mid] = {s: float(v) for s, v in zip(_c, _sc) if v is not None}\n            _nf += 1\n        except Exception as _ex:\n            print('frag failed', _mid, repr(_ex))\n    print('v7c FRAG gap molecules', _nf, 'of', len(PREFUSION_LISTS), f'{time.time()-_t0:.0f}s', flush=True)\nexcept Exception as _e:\n    FRAG_GAP = {}\n    print('v7c FRAG FAILED -> v7a behaviour:', repr(_e), flush=True)\n\n\n# ===== cell 30 =====\ndef _factorial_entry(scope):\n    \"\"\"Append-only E005\u2013E007 overlay; original v80 functions remain authoritative.\n\n    Embedded function-locally after the complete frozen E004 notebook. This module\n    never installs dependencies, changes model scores, or recanonicalizes SMILES.\n    \"\"\"\n\n    import hashlib\n    import json\n    import math\n    from pathlib import Path\n\n\n    def _finite_coverage(scores, candidates, keys):\n        count = 0\n        for smi, key in zip(candidates, keys):\n            value = scores.get(smi)\n            if value is None:\n                value = scores.get(key)\n            try:\n                count += value is not None and math.isfinite(float(value))\n            except (TypeError, ValueError):\n                pass\n        return count\n\n\n    def rerank_row(scope, mid, original, *, window, ice_fallback, processed):\n        \"\"\"Rebuild final metric fusion from same-run prefusion and engine lists.\n\n        ``original`` is the metric-safe reference BEFORE the frozen final overlay,\n        not the E004 final output. The original identity, representative selection,\n        same-formula slot rerank and tail fill are reused without modification.\n        \"\"\"\n        if window not in (40, 80):\n            raise ValueError(\"Only the prespecified 40/80 candidate windows are supported\")\n        original = list(original)\n        audit = {\n            \"molecule_id\": mid,\n            \"processed\": bool(processed),\n            \"branch\": \"placeholder\",\n            \"candidate_count\": 0,\n            \"collapsed_source_votes\": 0,\n            \"ice_finite_count\": 0,\n            \"gl_finite_count\": 0,\n            \"fallback_changed\": False,\n            \"fallback_changed_top1\": False,\n            \"n1_proxy\": mid in scope.get(\"N1_GL_MIDS\", set()),\n            \"joint_ice_proxy_gl\": False,\n        }\n        if not processed:\n            return original, audit\n        audit[\"branch\"] = \"no_engine\"\n        engine = scope[\"ENG\"].get(mid)\n        if mid not in scope[\"PREFUSION_LISTS\"] or not engine or not engine.get(\"smiles\"):\n            return original, audit\n        first = [s for s in scope[\"PREFUSION_LISTS\"][mid] if s and s != \"CCO\"]\n        candidates, scores, collapsed = scope[\"metric_fusion\"](\n            first,\n            engine[\"smiles\"],\n            scope[\"metric_identity\"],\n            alpha=scope[\"ALPHA\"],\n            offset=scope[\"KRR\"],\n            limit=window,\n        )\n        keys = [scope[\"chem\"].score_key(s) or s for s in candidates]\n        ice_scores = scope[\"ICE_SCORES\"]\n        ice = ice_scores.get(mid, {}) if isinstance(ice_scores, dict) else {}\n        gl = scope[\"gl_of\"](mid)\n        audit.update(\n            branch=\"unrescored\",\n            candidate_count=len(candidates),\n            collapsed_source_votes=collapsed,\n            ice_finite_count=_finite_coverage(ice, candidates, keys),\n            gl_finite_count=_finite_coverage(gl, candidates, keys),\n        )\n        before = list(candidates)\n        if ice and gl:\n            # flexonafft v47: joint ICE + proxy GL only on N1 treatment rows.\n            # ice_fallback=False still rebuilds the exact E004 GL-only control.\n            order = scope[\"gl_fuse\"].rerank_multi(\n                candidates,\n                keys,\n                scores,\n                [scope[\"_form2\"](s) for s in candidates],\n                [ice if ice_fallback and mid in scope.get(\"N1_GL_MIDS\", set()) else {}, gl],\n                [scope[\"ICE_LAM\"], scope[\"GL_LAM\"]],\n                top_n=len(candidates),\n            )\n            candidates = [candidates[i] for i in order]\n            audit[\"branch\"] = \"glacier\"\n            audit[\"joint_ice_proxy_gl\"] = bool(ice_fallback and audit[\"n1_proxy\"])\n        elif (ice_fallback and not gl and scope.get(\"FRAG_GAP\", {}).get(mid)\n              and not any(v is not None for v in (ice or {}).values())):\n            # v7c: no forward-model evidence at all -> MetFrag-lite gap re-rank (lambda 0.5), once\n            order = scope[\"gl_fuse\"].rerank_multi(\n                candidates, keys, scores, [scope[\"_form2\"](s) for s in candidates],\n                [scope[\"FRAG_GAP\"][mid]], [0.5], top_n=len(candidates))\n            candidates = [candidates[i] for i in order]\n            audit[\"branch\"] = \"frag_gap\"\n        elif ice_fallback and ice and not gl:\n            # Public v1 / script 356785434: ICE is used only where gl_of(mid) is empty.\n            # Keep its truthy-dict gate: all-None ICE is an observable harmless no-op.\n            order = scope[\"ice_fuse\"].rerank(\n                candidates,\n                keys,\n                scores,\n                [scope[\"_form2\"](s) for s in candidates],\n                ice,\n                lam=scope[\"ICE_LAM\"],\n                top_n=len(candidates),\n            )\n            candidates = [candidates[i] for i in order]\n            audit[\"branch\"] = \"ice_fallback\"\n        result = (\n            scope[\"safe_extend\"](candidates[:25], candidates[25:] + original)\n            if candidates\n            else original\n        )\n        if audit[\"branch\"] == \"ice_fallback\":\n            without_ice = (\n                scope[\"safe_extend\"](before[:25], before[25:] + original) if before else original\n            )\n            audit[\"fallback_changed\"] = result != without_ice\n            audit[\"fallback_changed_top1\"] = result[:1] != without_ice[:1]\n        return result, audit\n\n\n    def run_overlay(scope, *, experiment, window, ice_fallback, provenance, workdir=\".\"):\n        \"\"\"Save the E004 bytes, validate same-run control, then emit one treatment.\"\"\"\n        import pandas as pd\n\n        root = Path(workdir)\n        path = root / \"submission.csv\"\n        control_bytes = path.read_bytes()\n        control_sha = hashlib.sha256(control_bytes).hexdigest()\n        base_report = json.loads((root / \"e004_verification.json\").read_text())\n        assert base_report[\"experiment\"] == \"E004\"\n        assert base_report[\"hashes\"][\"submission.csv\"] == control_sha\n        # Copy bytes directly; never reserialize or overwrite original E004 audit files.\n        (root / \"submission_e004_control.csv\").write_bytes(control_bytes)\n        control = pd.read_csv(path, dtype=str, keep_default_na=False)\n        reference = pd.read_csv(root / \"submission_reference.csv\", dtype=str, keep_default_na=False)\n        assert list(reference.columns) == list(control.columns) == [\"molecule_id\", \"smiles\"]\n        assert list(reference.molecule_id) == list(control.molecule_id)\n        assert control.molecule_id.is_unique\n        processed = set(base_report[\"processed_ids\"])\n        assert processed == set(scope[\"te\"].molecule_id.astype(str))\n        controls = dict(zip(control.molecule_id, control.smiles))\n        output, audits = [], []\n        for row in reference.itertuples(index=False):\n            mid, original = str(row.molecule_id), str(row.smiles).split(\";\")\n            active = mid in processed\n            # The exact original pre-overlay tail is essential for paired equivalence.\n            rebuilt, _ = rerank_row(\n                scope, mid, original, window=40, ice_fallback=False, processed=active\n            )\n            assert \";\".join(rebuilt) == controls[mid], f\"E004 control reconstruction mismatch: {mid}\"\n            result, audit = rerank_row(\n                scope, mid, original, window=window, ice_fallback=ice_fallback, processed=active\n            )\n            assert 1 <= len(result) <= 25\n            if active:\n                identities = [scope[\"metric_identity\"](s) for s in result]\n                assert None not in identities and len(identities) == len(set(identities)), mid\n            else:\n                assert \";\".join(result) == controls[mid], f\"Placeholder changed: {mid}\"\n            old = controls[mid].split(\";\")\n            audit.update(changed=result != old, changed_top1=result[:1] != old[:1])\n            audits.append(audit)\n            output.append((mid, \";\".join(result)))\n        frame = pd.DataFrame(output, columns=[\"molecule_id\", \"smiles\"])\n        assert list(frame.molecule_id) == list(control.molecule_id)\n        frame.to_csv(path, index=False)\n        report = {\n            \"experiment\": experiment,\n            \"window\": window,\n            \"ice_fallback\": bool(ice_fallback),\n            \"source_experiment\": \"E004\",\n            \"base_report_file\": \"e004_verification.json\",\n            \"control_file\": \"submission_e004_control.csv\",\n            \"control_sha256\": control_sha,\n            \"submission_sha256\": hashlib.sha256(path.read_bytes()).hexdigest(),\n            \"processed_ids\": sorted(processed),\n            \"output_rows\": len(frame),\n            \"changed_rows\": sum(a[\"changed\"] for a in audits),\n            \"changed_top1\": sum(a[\"changed_top1\"] for a in audits),\n            \"fallback_rows\": sum(a[\"branch\"] == \"ice_fallback\" for a in audits),\n            \"fallback_finite_rows\": sum(\n                a[\"branch\"] == \"ice_fallback\" and a[\"ice_finite_count\"] > 0 for a in audits\n            ),\n            \"fallback_changed_rows\": sum(a[\"fallback_changed\"] for a in audits),\n            \"fallback_changed_top1\": sum(a[\"fallback_changed_top1\"] for a in audits),\n            \"same_run_control_reconstructed\": True,\n            \"n1_challenger\": {\n                \"complement\": bool(scope.get(\"N1_COMPLEMENT\", True)),\n                \"proxy_covered_molecules\": len(scope.get(\"N1_GL_MIDS\", set())),\n                \"joint_rerank_rows\": sum(a[\"joint_ice_proxy_gl\"] for a in audits),\n                \"joint_finite_ice_rows\": sum(a[\"joint_ice_proxy_gl\"] and a[\"ice_finite_count\"] > 0 for a in audits),\n                \"joint_finite_both_rows\": sum(a[\"joint_ice_proxy_gl\"] and a[\"ice_finite_count\"] > 0\n                                              and a[\"gl_finite_count\"] > 0 for a in audits),\n            },\n            \"provenance\": provenance,\n            \"limitations\": [\n                \"Visible smoke is not independent validation.\",\n                \"An inactive fallback in visible smoke says nothing about hidden benefit.\",\n            ],\n        }\n        pd.DataFrame(audits).to_csv(root / \"experiment_audit_factorial.csv\", index=False)\n        (root / \"experiment_audit_factorial.json\").write_text(\n            json.dumps(dict(report, rows=audits), indent=2, allow_nan=False) + \"\\n\"\n        )\n        (root / \"experiment_verification.json\").write_text(\n            json.dumps(report, indent=2, allow_nan=False) + \"\\n\"\n        )\n        print(\"FACTORIAL_VERIFICATION\", json.dumps(report))\n        print(\"N1_CHALLENGER_FINAL\", json.dumps(report[\"n1_challenger\"]), flush=True)\n        return report\n\n    return run_overlay(scope, **{'experiment': 'E006', 'window': 40, 'ice_fallback': True, 'provenance': {'base_notebook_sha256': '1f0f9b3026a93ddb5ad6fa6993c8ac0a4bfaddc54594eb1dd6dda7a20d8ef0d6', 'base_source_code_sha256': 'eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2', 'helper_source_sha256': 'a027a69ddda2c2e14639e3ec5196d5f1eb2ab57cba10ef929723148f31fc8f34', 'source_script_version_id': 356740494, 'ice_policy_source_script_version_id': 356785434, 'ice_policy_source': 'samanyu1808/casmi26-iceberg-where-glacier-is-blind-lb-0-438/1'}})\n\nFACTORIAL_REPORT = _factorial_entry(globals())\ndel _factorial_entry\n\n\n# ===== cell 31 =====\n# ---- E2R2: engine-2 rank-2 insert after the final E006 overlay (MODE 'rank2' only) ----\n# Idea: antriksharya, public notebook \"CASMI26 | LB 0.443 | v7c + engine-2 rank-2 insert\" (Apache 2.0); reimplemented.\n# Change: candidates are keyed with this notebook's metric_identity (stereo stripped, tautomer-canonical InChIKey-14\n# + heavy-atom composition) instead of chem.score_key, which keeps stereo.\n\n\ndef e2r2_make_key(identity, placeholder='CCO'):\n    \"\"\"SMILES -> metric identity, or None for a non-string, a blank string, the placeholder (or any SMILES with the\n    placeholder's identity) and an unparseable SMILES. A token whose key is None is never inserted and never matched.\"\"\"\n    ph = identity(placeholder)\n\n    def key(smiles):\n        if not isinstance(smiles, str) or not smiles.strip() or smiles.strip() == placeholder:\n            return None\n        k = identity(smiles)\n        return None if k is None or k == ph else k\n\n    return key\n\n\ndef e2r2_row(toks, eng_list, key, limit=25, placeholder='CCO'):\n    \"\"\"Put engine 2's top candidate x2 at rank 2 of one final list (best first). Pure function.\n\n    x2 is the first entry of `eng_list` that is a non-blank string other than the placeholder. It is used only when\n    key(x2) is not None; if that entry is unusable the row is left alone (later entries are not tried).\n      empty row, or rank 1 is the placeholder  -> unchanged, 'skipped'\n      no engine list                           -> unchanged, 'no_eng'\n      x2 missing, unparseable or holds ';'     -> unchanged, 'invalid_x2'\n      key(x2) == key(rank 1)                   -> unchanged, 'same_as_rank1'\n      x2's identity already at rank 2          -> that token stays, deeper twins of it are dropped, 'already_rank2'\n      otherwise                                -> every token with key(x2) is dropped, x2 goes to rank 2, the other\n                                                  tokens keep their order, the list is cut to `limit`, 'inserted'\n    Returns (new_toks, tag, moved_from). moved_from is 'absent' or the old 1-based rank (str) of x2's identity when\n    x2 ends at rank 2, else None.\n    \"\"\"\n    toks = list(toks)\n    if not toks or not isinstance(toks[0], str) or toks[0].strip() in ('', placeholder):\n        return toks, 'skipped', None\n    if isinstance(eng_list, str):\n        eng_list = [eng_list]\n    if not eng_list:\n        return toks, 'no_eng', None\n    x2 = next((s.strip() for s in eng_list if isinstance(s, str) and s.strip() not in ('', placeholder)), None)\n    kx = key(x2) if x2 is not None and not any(c in x2 for c in ';\\r\\n') else None\n    if kx is None:\n        return toks, 'invalid_x2', None\n    keys = [key(t) for t in toks]\n    if kx == keys[0]:\n        return toks, 'same_as_rank1', None\n    hits = [i for i, k in enumerate(keys) if k == kx]\n    if hits and hits[0] == 1:\n        return [t for i, t in enumerate(toks) if i == 1 or keys[i] != kx][:limit], 'already_rank2', '2'\n    rest = [t for t, k in zip(toks, keys) if k != kx]\n    return (rest[:1] + [x2] + rest[1:])[:limit], 'inserted', (str(hits[0] + 1) if hits else 'absent')\n\n\ndef _e2r2_apply(scope, base_bytes, workdir='.'):\n    \"\"\"Apply e2r2_row to every row of the final submission.csv, check the result, then replace the file.\"\"\"\n    import hashlib\n    import json\n    import os\n    from collections import Counter\n    from pathlib import Path\n    import pandas as pd\n\n    root = Path(workdir)\n    sub_path, tmp_path = root / 'submission.csv', root / 'submission_e2r2.tmp.csv'\n    (root / 'submission_base_e2r2.csv').write_bytes(base_bytes)\n    base = pd.read_csv(root / 'submission_base_e2r2.csv', dtype=str, keep_default_na=False)\n    sample = pd.read_csv(os.path.join(scope['COMP'], 'sample_submission.csv'), dtype=str, keep_default_na=False)\n    key = e2r2_make_key(scope['metric_identity'])\n    eng = scope['ENG']\n    assert isinstance(eng, dict), type(eng)\n    active = set(scope['te'].molecule_id.astype(str))\n    strict = not bool(scope.get('IS_RERUN', True))\n\n    def engine_list(mid):\n        e = eng.get(mid)\n        if e is None and mid.strip().lstrip('-').isdigit():\n            e = eng.get(int(mid))\n        if isinstance(e, dict):\n            return e.get('smiles') or []\n        return list(e) if isinstance(e, (list, tuple)) else []\n\n    def n_dup(ts):\n        ks = [k for k in map(key, ts) if k is not None]\n        return len(ks) - len(set(ks))\n\n    def check(frame):\n        assert list(frame.columns) == ['molecule_id', 'smiles'], list(frame.columns)\n        assert list(frame.molecule_id) == list(base.molecule_id)\n        assert len(frame) == len(sample) and frame.molecule_id.is_unique\n        assert set(frame.molecule_id) == set(sample.molecule_id)\n        assert frame.smiles.notna().all() and (frame.smiles.str.strip() != '').all()\n        assert frame.smiles.map(lambda v: 1 <= len(v.split(';')) <= 25).all()\n\n    check(base)\n    tags, moved, rows, examples = Counter(), Counter(), [], []\n    eligible_active = tail_dropped = twins_dropped = 0\n    for mid, s in zip(base.molecule_id, base.smiles):\n        old = s.split(';')\n        new, tag, moved_from = e2r2_row(old, engine_list(mid), key)\n        tags[tag] += 1\n        eligible_active += int(tag != 'skipped' and mid in active)\n        if moved_from is not None:\n            moved[moved_from] += 1\n        if new == old:\n            rows.append((mid, s))\n            continue\n        # changed row: rank 1 kept, x2's identity exactly once (rank 2), other tokens in order, no new duplicates\n        kx = key(new[1])\n        n_tw = sum(key(t) == kx for t in old)\n        assert tag in ('inserted', 'already_rank2') and mid in active, (mid, tag)\n        assert new[0] == old[0] and kx is not None and 2 <= len(new) <= 25 and all(t.strip() for t in new), mid\n        assert sum(key(t) == kx for t in new) == 1, mid\n        assert [t for t in new if key(t) != kx] == [t for t in old if key(t) != kx][:len(new) - 1], mid\n        assert n_dup(new) <= n_dup(old), mid\n        twins_dropped += max(0, n_tw - 1)\n        tail_dropped += len(old) - n_tw + 1 - len(new)\n        if len(examples) < 3:\n            examples.append(dict(molecule_id=mid, tag=tag, moved_from=moved_from, before=old[:3], after=new[:3]))\n        rows.append((mid, ';'.join(new)))\n    result = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])\n    check(result)\n    if strict:  # commit run: every processed molecule must reach the overlay with an engine-2 list\n        assert eligible_active == len(active) and tags['no_eng'] == 0, (eligible_active, len(active), dict(tags))\n    changed = int(sum(a != b for a, b in zip(result.smiles, base.smiles)))\n    stats = dict(mode='rank2', key='metric_identity', is_rerun=not strict, rows=len(base),\n                 eligible=len(base) - tags['skipped'], inserted=tags['inserted'],\n                 moved_from={k: moved[k] for k in sorted(moved, key=lambda k: -1 if k == 'absent' else int(k))},\n                 same_as_rank1=tags['same_as_rank1'], no_eng=tags['no_eng'], invalid_x2=tags['invalid_x2'],\n                 already_rank2=tags['already_rank2'], skipped=tags['skipped'], changed_rows=changed,\n                 tail_dropped=tail_dropped, extra_twins_dropped=twins_dropped,\n                 base_sha256=hashlib.sha256(base_bytes).hexdigest())\n    new_bytes = base_bytes\n    if changed:\n        result.to_csv(tmp_path, index=False)\n        back = pd.read_csv(tmp_path, dtype=str, keep_default_na=False)\n        check(back)\n        assert list(back.smiles) == list(result.smiles)\n        new_bytes = tmp_path.read_bytes()\n    stats['submission_sha256'] = hashlib.sha256(new_bytes).hexdigest()\n    (root / 'e2r2_stats.json').write_text(json.dumps(dict(stats, status='applied', examples=examples), indent=1) + '\\n')\n    if changed:\n        os.replace(tmp_path, sub_path)\n    print('E2R2_APPLIED', json.dumps(stats), flush=True)\n    for ex in examples:\n        print('E2R2_EXAMPLE', json.dumps(ex), flush=True)\n    return stats\n\n\nimport json as _e2r2_json\nimport os as _e2r2_os\nimport sys as _e2r2_sys\nimport traceback as _e2r2_tb\n_E2R2_STRICT = not bool(globals().get('IS_RERUN', True))  # commit run: fail loudly; scoring rerun: keep the E006 file\n_e2r2_base_bytes = None\nE2R2_STATS = None\ntry:\n    with open('submission.csv', 'rb') as _e2r2_f:  # the final E006 file (cwd is /kaggle/working)\n        _e2r2_base_bytes = _e2r2_f.read()\n    print('E2R2 reads', _e2r2_os.path.abspath('submission.csv'), len(_e2r2_base_bytes), 'bytes', flush=True)\n    E2R2_STATS = _e2r2_apply(globals(), _e2r2_base_bytes)\nexcept Exception as _e2r2_exc:\n    if _E2R2_STRICT:\n        raise\n    _e2r2_state = 'not read'\n    if _e2r2_base_bytes is not None:\n        try:\n            _e2r2_now = None\n            if _e2r2_os.path.isfile('submission.csv'):\n                with open('submission.csv', 'rb') as _e2r2_f:\n                    _e2r2_now = _e2r2_f.read()\n            if _e2r2_now != _e2r2_base_bytes:\n                with open('submission.csv', 'wb') as _e2r2_f:\n                    _e2r2_f.write(_e2r2_base_bytes)\n            _e2r2_state = 'untouched' if _e2r2_now == _e2r2_base_bytes else 'restored from the E006 bytes'\n        except Exception as _e2r2_exc2:\n            _e2r2_state = f'RESTORE FAILED {_e2r2_exc2!r}'\n    try:\n        if _e2r2_os.path.exists('submission_e2r2.tmp.csv'):\n            _e2r2_os.remove('submission_e2r2.tmp.csv')\n        with open('e2r2_stats.json', 'w') as _e2r2_f:\n            _e2r2_f.write(_e2r2_json.dumps(dict(status='failed', submission=_e2r2_state,\n                                                error=f'{type(_e2r2_exc).__name__}: {_e2r2_exc}'[:500])) + '\\n')\n    except Exception:\n        pass\n    try:\n        print('!' * 100, flush=True)\n        print(f'E2R2_FAILED {type(_e2r2_exc).__name__}: {_e2r2_exc} | submission.csv {_e2r2_state} '\n              '| the E006 list is the submission', flush=True)\n        _e2r2_tb.print_exc(file=_e2r2_sys.stdout)\n        print('!' * 100, flush=True)\n    except Exception:\n        print('E2R2_FAILED (error report failed) | submission.csv', _e2r2_state, flush=True)\n\n\n# ===== cell 32 =====\n# ==== ours (yasunorim): drop scratch trees from /kaggle/working (the output listing is paged at 500 files) ====\nimport os as _os, shutil as _sh\nfor _d in ('ice_site', 'gl_site', '__pycache__', 'ours_mods'):\n    _p = _os.path.join('/kaggle/working', _d)\n    if _os.path.isdir(_p):\n        _sh.rmtree(_p, ignore_errors=True)\n_n = sum(len(f) for _, _, f in _os.walk('/kaggle/working'))\nprint('files left in /kaggle/working', _n, '| submission.csv', _os.path.isfile('/kaggle/working/submission.csv'), flush=True)\n\n"}
K, WA, WB = 10.0, 1.0, 1.0
LIMIT = 9 * 3600 - 25 * 60          # leave 25 min for the fusion and the kernel's own exit
os.makedirs('/tmp/ours_fu', exist_ok=True)
for nm, src in SCRIPTS.items():
    open(f'/tmp/ours_fu/{nm}.py', 'w').write(src)
KEEP = set(os.listdir(W))


def run(nm, timeout):
    print(f'=== ours: run {nm} (timeout {timeout:.0f}s, elapsed {time.time() - T0:.0f}s)', flush=True)
    t = time.time()
    p = subprocess.Popen([sys.executable, '-u', f'/tmp/ours_fu/{nm}.py'], cwd=W, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    tail = []
    try:
        import threading
        def pump():
            for line in p.stdout:
                print(f'[{nm}] ' + line, end='', flush=True)
        th = threading.Thread(target=pump, daemon=True); th.start()
        rc = p.wait(timeout=max(60, timeout))
        th.join(30)
    except subprocess.TimeoutExpired:
        print(f'=== ours: {nm} timed out, killing', flush=True)
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except Exception as e:
            print('kill failed', e)
        rc = -9
    sub = os.path.join(W, 'submission.csv')
    ok = rc == 0 and os.path.exists(sub)
    dst = f'/tmp/ours_fu/sub_{nm}.csv'
    if ok:
        shutil.copy(sub, dst)
    print(f'=== ours: {nm} rc={rc} ok={ok} in {time.time() - t:.0f}s', flush=True)
    for f in os.listdir(W):            # clean the working dir for the next pipeline
        if f in KEEP:
            continue
        q = os.path.join(W, f)
        try:
            shutil.rmtree(q) if os.path.isdir(q) and not os.path.islink(q) else os.remove(q)
        except Exception as e:
            print('cleanup', f, e)
    return dst if ok else None


def read(path):
    rows = list(csv.reader(open(path)))
    return rows[0], {r[0]: [s for s in r[1].split(';') if s] for r in rows[1:]}


sub_a = run('A', LIMIT - (time.time() - T0))
sub_b = run('B', LIMIT - (time.time() - T0)) if sub_a else None
if sub_a is None:
    # A failed: B gets the whole remaining time; its file goes out alone
    sub_b = run('B', LIMIT - (time.time() - T0))
    assert sub_b, 'both pipelines failed'
    shutil.copy(sub_b, os.path.join(W, 'submission.csv'))
    print('=== ours: A failed, submission = B', flush=True)
elif sub_b is None:
    shutil.copy(sub_a, os.path.join(W, 'submission.csv'))
    print('=== ours: B failed, submission = A', flush=True)
else:
    head, LA = read(sub_a)
    _, LB = read(sub_b)
    try:
        from rdkit import Chem, RDLogger
        RDLogger.DisableLog('rdApp.*')
        _cache = {}
        def key(s):
            if s not in _cache:
                m = Chem.MolFromSmiles(s)
                _cache[s] = Chem.MolToInchiKey(m)[:14] if m is not None else s
            return _cache[s]
        print('ours: fusion keys = InChIKey first block', flush=True)
    except Exception as e:
        key = lambda s: s
        print('ours: fusion keys = SMILES (no RDKit:', e, ')', flush=True)
    out, n_top1, n_b_only, n_mol = {}, 0, 0, 0
    for mid, la in LA.items():
        lb = LB.get(mid, [])
        score, first, smi = {}, {}, {}
        for w, lst in ((WA, la), (WB, lb)):
            seen = set()
            for r, s in enumerate(lst[:25]):
                kk = key(s)
                if kk in seen:
                    continue
                seen.add(kk)
                score[kk] = score.get(kk, 0.0) + w / (K + r + 1)
                smi.setdefault(kk, s)
        order_a = {key(s): i for i, s in enumerate(la)}
        fused = sorted(score, key=lambda kk: (-score[kk], order_a.get(kk, 1e9)))[:25]
        out[mid] = [smi[kk] for kk in fused]
        n_mol += 1
        if la and out[mid] and key(la[0]) != fused[0]:
            n_top1 += 1
        n_b_only += sum(1 for kk in fused if kk not in order_a)
    with open(os.path.join(W, 'submission.csv'), 'w', newline='') as f:
        wr = csv.writer(f)
        wr.writerow(head)
        for mid, lst in out.items():
            wr.writerow([mid, ';'.join(lst)])
    print(f'=== ours: fused {n_mol} molecules; top-1 changed vs A in {n_top1}; B-only rows in the top 25: {n_b_only}; '
          f'molecules only in B: {len(set(LB) - set(LA))}', flush=True)
print(f'=== ours: done in {time.time() - T0:.0f}s', flush=True)
